<a href="https://colab.research.google.com/github/chayansaha2/PlanProof-AI/blob/main/PlanProof_AI_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# PlanProof AI — Research Environment Check

import sys
import platform
import os

print("Project: PlanProof AI")
print("Python version:", sys.version.split()[0])
print("Platform:", platform.system())
print("Working directory:", os.getcwd())
print("Environment ready!")


Project: PlanProof AI
Python version: 3.13.15
Platform: Linux
Working directory: /content
Environment ready!


In [ ]:

# Our first planning task

plan = {
    "task_order": [
        "literature_review",
        "run_experiments",
        "write_results",
        "submit_report",
    ],
    "required_tasks": [
        "literature_review",
        "run_experiments",
        "write_results",
        "submit_report",
    ],
    "precedence_constraints": [
        ("literature_review", "write_results"),
        ("run_experiments", "write_results"),
        ("write_results", "submit_report"),
    ],
}

print("Tasks:", len(plan["task_order"]))
print("Constraints:", len(plan["precedence_constraints"]))
print("Plan:", plan["task_order"])


Tasks: 4
Constraints: 3
Plan: ['literature_review', 'run_experiments', 'write_results', 'submit_report']


In [ ]:

# ==========================================
# PlanProof AI — Version 0.1
# Deterministic Constraint Verification
# ==========================================

def validate_plan(plan):
    """Verify required tasks and task-order constraints."""

    order = plan["task_order"]
    required = plan["required_tasks"]
    constraints = plan["precedence_constraints"]

    violations = []

    # 1. Detect duplicate tasks
    if len(order) != len(set(order)):
        violations.append({
            "type": "duplicate_task",
            "message": "Duplicate task IDs detected."
        })

    # Record the position of each task
    positions = {
        task: index for index, task in enumerate(order)
    }

    # 2. Check required tasks
    for task in required:
        if task not in positions:
            violations.append({
                "type": "missing_task",
                "message": f"Missing required task: {task}"
            })

    # 3. Check precedence constraints
    for before, after in constraints:
        if before not in positions or after not in positions:
            continue

        if positions[before] >= positions[after]:
            violations.append({
                "type": "precedence_violation",
                "message": f"{before} must occur before {after}."
            })

    return {
        "valid": len(violations) == 0,
        "violations": violations
    }


# Test 1: Check the original plan
result = validate_plan(plan)

print("VALID PLAN TEST")
print("Accepted:", result["valid"])
print("Violations:", result["violations"])

# Test 2: Deliberately break the task order
invalid_plan = {
    **plan,
    "task_order": [
        "literature_review",
        "write_results",
        "run_experiments",
        "submit_report"
    ]
}

result = validate_plan(invalid_plan)

print("\nINVALID PLAN TEST")
print("Accepted:", result["valid"])
print("Violations detected:", len(result["violations"]))

for violation in result["violations"]:
    print("-", violation["message"])


VALID PLAN TEST
Accepted: True
Violations: []

INVALID PLAN TEST
Accepted: False
Violations detected: 1
- run_experiments must occur before write_results.


In [ ]:

# ==========================================
# PlanProof AI — Automatic Plan Repair v0.1
# ==========================================

from heapq import heappush, heappop


def repair_plan(plan):
    """Repair task ordering and add missing required tasks."""

    original_order = plan["task_order"]
    required = plan["required_tasks"]
    constraints = plan["precedence_constraints"]

    # Preserve first occurrence of each task.
    # Add required tasks that are missing.
    tasks = list(dict.fromkeys(original_order))

    for task in required:
        if task not in tasks:
            tasks.append(task)

    # Every constraint must refer to a known task.
    for before, after in constraints:
        if before not in tasks or after not in tasks:
            return {
                "success": False,
                "repaired_plan": None,
                "message": (
                    f"Constraint references an unknown task: "
                    f"{before} -> {after}"
                ),
            }

    # Build a directed graph.
    graph = {task: [] for task in tasks}
    indegree = {task: 0 for task in tasks}

    for before, after in constraints:
        if after not in graph[before]:
            graph[before].append(after)
            indegree[after] += 1

    # Original position is our tie-breaking priority.
    # Newly added required tasks come after original tasks.
    original_positions = {}
    for index, task in enumerate(original_order):
        original_positions.setdefault(task, index)

    priority = {
        task: original_positions.get(task, len(original_order) + i)
        for i, task in enumerate(tasks)
    }

    # Kahn's algorithm with a priority queue.
    available = []

    for task in tasks:
        if indegree[task] == 0:
            heappush(available, (priority[task], task))

    repaired_order = []

    while available:
        _, task = heappop(available)
        repaired_order.append(task)

        for next_task in graph[task]:
            indegree[next_task] -= 1

            if indegree[next_task] == 0:
                heappush(
                    available,
                    (priority[next_task], next_task)
                )

    # If not every task was ordered, constraints contain a cycle.
    if len(repaired_order) != len(tasks):
        return {
            "success": False,
            "repaired_plan": None,
            "message": "Repair impossible: cyclic constraints detected.",
        }

    repaired = {
        **plan,
        "task_order": repaired_order,
    }

    # Never report successful repair without verification.
    verification = validate_plan(repaired)

    if not verification["valid"]:
        return {
            "success": False,
            "repaired_plan": repaired,
            "message": "Repair failed verification.",
            "violations": verification["violations"],
        }

    return {
        "success": True,
        "repaired_plan": repaired,
        "message": "Repaired plan passed verification.",
    }


# Test the repair engine using our invalid plan.
repair_result = repair_plan(invalid_plan)

print("PLAN REPAIR TEST")
print("Success:", repair_result["success"])
print("Message:", repair_result["message"])

if repair_result["success"]:
    print("Original order:", invalid_plan["task_order"])
    print("Repaired order:",
          repair_result["repaired_plan"]["task_order"])
    print("Final verification:",
          validate_plan(repair_result["repaired_plan"]))


PLAN REPAIR TEST
Success: True
Message: Repaired plan passed verification.
Original order: ['literature_review', 'write_results', 'run_experiments', 'submit_report']
Repaired order: ['literature_review', 'run_experiments', 'write_results', 'submit_report']
Final verification: {'valid': True, 'violations': []}


In [ ]:

# ==========================================
# PlanProof AI — Repair Robustness Tests
# ==========================================

def run_test(name, condition):
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {name}")
    assert condition, f"Test failed: {name}"


# Test 1: An ordinary valid plan remains valid.
valid_result = repair_plan(plan)

run_test(
    "Valid plan is accepted",
    valid_result["success"]
    and validate_plan(valid_result["repaired_plan"])["valid"]
)


# Test 2: A missing required task is restored.
missing_plan = {
    **plan,
    "task_order": [
        "literature_review",
        "write_results",
        "submit_report",
    ],
}

missing_result = repair_plan(missing_plan)

run_test(
    "Missing required task is restored",
    missing_result["success"]
    and "run_experiments"
    in missing_result["repaired_plan"]["task_order"]
    and validate_plan(missing_result["repaired_plan"])["valid"]
)


# Test 3: Duplicate tasks are removed.
duplicate_plan = {
    **plan,
    "task_order": [
        "literature_review",
        "run_experiments",
        "run_experiments",
        "write_results",
        "submit_report",
    ],
}

duplicate_result = repair_plan(duplicate_plan)

run_test(
    "Duplicate tasks are removed",
    duplicate_result["success"]
    and len(duplicate_result["repaired_plan"]["task_order"])
    == len(set(duplicate_result["repaired_plan"]["task_order"]))
    and validate_plan(duplicate_result["repaired_plan"])["valid"]
)


# Test 4: An impossible dependency cycle is rejected.
cyclic_plan = {
    "task_order": ["A", "B"],
    "required_tasks": ["A", "B"],
    "precedence_constraints": [("A", "B"), ("B", "A")],
}

cycle_result = repair_plan(cyclic_plan)

run_test(
    "Impossible dependency cycle is rejected",
    not cycle_result["success"]
    and cycle_result["repaired_plan"] is None
)


print("\nAll four robustness tests passed.")


[PASS] Valid plan is accepted
[PASS] Missing required task is restored
[PASS] Duplicate tasks are removed
[PASS] Impossible dependency cycle is rejected

All four robustness tests passed.


In [ ]:

# =====================================================
# PlanProof AI — Synthetic Research Benchmark v0.1
# =====================================================

import random
import statistics
from collections import Counter


def generate_benchmark_case(rng, task_count, edge_probability):
    """Generate a DAG of task dependencies and a shuffled plan."""

    tasks = [f"T{i}" for i in range(task_count)]
    constraints = []

    # Edges go from lower to higher IDs, guaranteeing
    # that the generated dependency graph has no cycles.
    for i in range(task_count):
        for j in range(i + 1, task_count):
            if rng.random() < edge_probability:
                constraints.append((tasks[i], tasks[j]))

    shuffled_order = tasks.copy()
    rng.shuffle(shuffled_order)

    return {
        "task_order": shuffled_order,
        "required_tasks": tasks.copy(),
        "precedence_constraints": constraints,
    }


def position_change_rate(original, repaired):
    """Fraction of task IDs whose positions changed."""

    original_positions = {
        task: i for i, task in enumerate(original)
    }
    repaired_positions = {
        task: i for i, task in enumerate(repaired)
    }

    if not original:
        return 0.0

    changed = sum(
        original_positions[task] != repaired_positions[task]
        for task in original_positions
    )

    return changed / len(original)


def run_benchmark(number_of_cases=500, seed=42):
    rng = random.Random(seed)

    initially_valid = 0
    initially_invalid = 0
    repaired_valid = 0
    repair_failures = 0
    changed_rates = []
    violation_counts = Counter()
    task_counts = []

    for _ in range(number_of_cases):
        task_count = rng.randint(5, 30)
        edge_probability = rng.uniform(0.05, 0.25)

        case = generate_benchmark_case(
            rng, task_count, edge_probability
        )

        initial_result = validate_plan(case)
        task_counts.append(task_count)

        if initial_result["valid"]:
            initially_valid += 1
        else:
            initially_invalid += 1

        for violation in initial_result["violations"]:
            violation_counts[violation["type"]] += 1

        repaired_result = repair_plan(case)

        if repaired_result["success"]:
            final_plan = repaired_result["repaired_plan"]
            final_result = validate_plan(final_plan)

            if final_result["valid"]:
                repaired_valid += 1
                changed_rates.append(
                    position_change_rate(
                        case["task_order"],
                        final_plan["task_order"],
                    )
                )
            else:
                repair_failures += 1
        else:
            repair_failures += 1

    def percentage(numerator, denominator):
        return 100 * numerator / denominator if denominator else 0.0

    print("=" * 48)
    print("PLANPROOF AI — BENCHMARK RESULTS")
    print("=" * 48)
    print("Random seed:", seed)
    print("Cases:", number_of_cases)
    print("Average tasks per case:",
          round(statistics.mean(task_counts), 2))

    print("\nINITIAL PLANS")
    print("Valid:", initially_valid)
    print("Invalid:", initially_invalid)
    print("Initial validity rate:",
          f"{percentage(initially_valid, number_of_cases):.2f}%")

    print("\nAFTER REPAIR")
    print("Successfully repaired and verified:", repaired_valid)
    print("Repair failures:", repair_failures)
    print("Final validity rate:",
          f"{percentage(repaired_valid, number_of_cases):.2f}%")

    print("\nREPAIR IMPACT")
    print("Average position-change rate among successful repairs:",
          f"{100 * statistics.mean(changed_rates):.2f}%"
          if changed_rates else "N/A")
    print("Initial violation counts:", dict(violation_counts))

    return {
        "seed": seed,
        "cases": number_of_cases,
        "initial_valid": initially_valid,
        "initial_invalid": initially_invalid,
        "repaired_valid": repaired_valid,
        "repair_failures": repair_failures,
        "violation_counts": dict(violation_counts),
    }


benchmark_results = run_benchmark()


PLANPROOF AI — BENCHMARK RESULTS
Random seed: 42
Cases: 500
Average tasks per case: 17.18

INITIAL PLANS
Valid: 25
Invalid: 475
Initial validity rate: 5.00%

AFTER REPAIR
Successfully repaired and verified: 500
Repair failures: 0
Final validity rate: 100.00%

REPAIR IMPACT
Average position-change rate among successful repairs: 79.93%
Initial violation counts: {'precedence_violation': 6175}


In [ ]:

# ======================================================
# PlanProof AI — Experiment 2: Preserve Valid Plans
# ======================================================

def run_preservation_experiment(number_of_cases=500, seed=42):
    rng = random.Random(seed)

    initially_valid = 0
    invalid_cases = 0
    repaired_successfully = 0

    # Original repair strategy: repair every plan.
    all_plan_change_rates = []

    # Improved policy: leave valid plans untouched.
    conditional_change_rates = []
    valid_plans_preserved = 0

    for _ in range(number_of_cases):
        n = rng.randint(5, 30)
        p = rng.uniform(0.05, 0.25)

        case = generate_benchmark_case(rng, n, p)
        original = case["task_order"]
        initial = validate_plan(case)

        # Evaluate the current repair algorithm on every plan.
        old_result = repair_plan(case)

        if old_result["success"]:
            old_order = old_result["repaired_plan"]["task_order"]
            all_plan_change_rates.append(
                position_change_rate(original, old_order)
            )

        # Improved policy: do not modify already-valid plans.
        if initial["valid"]:
            initially_valid += 1
            valid_plans_preserved += 1

            conditional_change_rates.append(0.0)
            continue

        invalid_cases += 1
        new_result = repair_plan(case)

        if new_result["success"]:
            repaired_successfully += 1
            new_order = new_result["repaired_plan"]["task_order"]

            conditional_change_rates.append(
                position_change_rate(original, new_order)
            )

    print("=" * 52)
    print("PLAN PRESERVATION EXPERIMENT")
    print("=" * 52)
    print("Cases:", number_of_cases)
    print("Initially valid:", initially_valid)
    print("Initially invalid:", invalid_cases)

    print("\nCURRENT STRATEGY")
    print(
        "Average position-change rate:",
        f"{100 * statistics.mean(all_plan_change_rates):.2f}%"
    )

    print("\nCONDITIONAL REPAIR STRATEGY")
    print("Valid plans left untouched:", valid_plans_preserved)
    print("Invalid plans repaired:", repaired_successfully)
    print(
        "Average position-change rate across all plans:",
        f"{100 * statistics.mean(conditional_change_rates):.2f}%"
    )

    return {
        "cases": number_of_cases,
        "initially_valid": initially_valid,
        "invalid_cases": invalid_cases,
        "repaired_successfully": repaired_successfully,
        "valid_plans_preserved": valid_plans_preserved,
        "current_mean_change_rate": statistics.mean(
            all_plan_change_rates
        ),
        "conditional_mean_change_rate": statistics.mean(
            conditional_change_rates
        ),
    }


preservation_results = run_preservation_experiment()


PLAN PRESERVATION EXPERIMENT
Cases: 500
Initially valid: 25
Initially invalid: 475

CURRENT STRATEGY
Average position-change rate: 79.93%

CONDITIONAL REPAIR STRATEGY
Valid plans left untouched: 25
Invalid plans repaired: 475
Average position-change rate across all plans: 79.93%


In [ ]:

# ======================================================
# PlanProof AI — Experiment 3: Pairwise Order Preservation
# ======================================================

from collections import deque


def pairwise_order_change_rate(original, repaired):
    """Fraction of task pairs whose relative order changed."""

    original_pos = {task: i for i, task in enumerate(original)}
    repaired_pos = {task: i for i, task in enumerate(repaired)}

    common = list(original_pos)
    if len(common) < 2:
        return 0.0

    changed_pairs = 0
    total_pairs = 0

    for i in range(len(common)):
        for j in range(i + 1, len(common)):
            a, b = common[i], common[j]

            before_relation = original_pos[a] < original_pos[b]
            after_relation = repaired_pos[a] < repaired_pos[b]

            total_pairs += 1
            if before_relation != after_relation:
                changed_pairs += 1

    return changed_pairs / total_pairs


def fifo_repair(plan):
    """Topologically sort tasks using FIFO tie-breaking."""

    tasks = list(dict.fromkeys(plan["task_order"]))

    for task in plan["required_tasks"]:
        if task not in tasks:
            tasks.append(task)

    graph = {task: [] for task in tasks}
    indegree = {task: 0 for task in tasks}

    for before, after in plan["precedence_constraints"]:
        if before not in graph or after not in graph:
            return None
        if after not in graph[before]:
            graph[before].append(after)
            indegree[after] += 1

    queue = deque(task for task in tasks if indegree[task] == 0)
    result = []

    while queue:
        task = queue.popleft()
        result.append(task)

        for nxt in graph[task]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                queue.append(nxt)

    if len(result) != len(tasks):
        return None

    repaired = {**plan, "task_order": result}

    if not validate_plan(repaired)["valid"]:
        return None

    return repaired


def compare_repair_strategies(number_of_cases=500, seed=42):
    rng = random.Random(seed)
    stable_changes = []
    fifo_changes = []
    stable_failures = 0
    fifo_failures = 0

    for _ in range(number_of_cases):
        n = rng.randint(5, 30)
        p = rng.uniform(0.05, 0.25)
        case = generate_benchmark_case(rng, n, p)

        stable = repair_plan(case)
        fifo = fifo_repair(case)

        if stable["success"]:
            stable_changes.append(
                pairwise_order_change_rate(
                    case["task_order"],
                    stable["repaired_plan"]["task_order"],
                )
            )
        else:
            stable_failures += 1

        if fifo is not None:
            fifo_changes.append(
                pairwise_order_change_rate(
                    case["task_order"],
                    fifo["task_order"],
                )
            )
        else:
            fifo_failures += 1

    print("PAIRWISE ORDER PRESERVATION")
    print("Cases:", number_of_cases)
    print("Stable repair failures:", stable_failures)
    print("FIFO repair failures:", fifo_failures)

    print(
        "Stable repair mean pairwise change:",
        f"{100 * statistics.mean(stable_changes):.2f}%"
    )
    print(
        "FIFO repair mean pairwise change:",
        f"{100 * statistics.mean(fifo_changes):.2f}%"
    )


compare_repair_strategies()


PAIRWISE ORDER PRESERVATION
Cases: 500
Stable repair failures: 0
FIFO repair failures: 0
Stable repair mean pairwise change: 26.63%
FIFO repair mean pairwise change: 39.59%


In [ ]:

# =====================================================
# PlanProof AI — Experiment 4: Multi-Seed Evaluation
# =====================================================

def run_multiseed_experiment(
    seeds=range(10),
    cases_per_seed=500,
):
    results = []

    for seed in seeds:
        rng = random.Random(seed)

        stable_rates = []
        fifo_rates = []
        stable_failures = 0
        fifo_failures = 0

        for _ in range(cases_per_seed):
            n = rng.randint(5, 30)
            p = rng.uniform(0.05, 0.25)

            case = generate_benchmark_case(rng, n, p)

            stable = repair_plan(case)
            fifo = fifo_repair(case)

            if stable["success"]:
                stable_rates.append(
                    pairwise_order_change_rate(
                        case["task_order"],
                        stable["repaired_plan"]["task_order"],
                    )
                )
            else:
                stable_failures += 1

            if fifo is not None:
                fifo_rates.append(
                    pairwise_order_change_rate(
                        case["task_order"],
                        fifo["task_order"],
                    )
                )
            else:
                fifo_failures += 1

        stable_mean = statistics.mean(stable_rates)
        fifo_mean = statistics.mean(fifo_rates)

        results.append({
            "seed": seed,
            "stable": stable_mean,
            "fifo": fifo_mean,
            "difference": fifo_mean - stable_mean,
            "stable_failures": stable_failures,
            "fifo_failures": fifo_failures,
        })

    print("MULTI-SEED RESULTS")
    print("Seed | Stable % | FIFO % | FIFO - Stable (pp)")

    for r in results:
        print(
            f"{r['seed']:>4} | "
            f"{100*r['stable']:>8.2f} | "
            f"{100*r['fifo']:>6.2f} | "
            f"{100*r['difference']:>17.2f}"
        )

    mean_stable = statistics.mean(r["stable"] for r in results)
    mean_fifo = statistics.mean(r["fifo"] for r in results)
    mean_difference = statistics.mean(
        r["difference"] for r in results
    )

    print("\nAGGREGATE RESULTS")
    print("Total cases:", len(results) * cases_per_seed)
    print(f"Mean stable change: {100*mean_stable:.2f}%")
    print(f"Mean FIFO change: {100*mean_fifo:.2f}%")
    print(
        "Mean difference (FIFO - stable): "
        f"{100*mean_difference:.2f} percentage points"
    )
    print(
        "Seeds where stable has lower change:",
        sum(r["difference"] > 0 for r in results),
        "of", len(results)
    )
    print(
        "Total stable failures:",
        sum(r["stable_failures"] for r in results)
    )
    print(
        "Total FIFO failures:",
        sum(r["fifo_failures"] for r in results)
    )

    return results


multiseed_results = run_multiseed_experiment()


MULTI-SEED RESULTS
Seed | Stable % | FIFO % | FIFO - Stable (pp)
   0 |    28.01 |  39.54 |             11.54
   1 |    27.59 |  40.16 |             12.57
   2 |    27.82 |  40.03 |             12.21
   3 |    27.11 |  39.23 |             12.11
   4 |    26.40 |  38.78 |             12.37
   5 |    27.86 |  40.09 |             12.24
   6 |    26.47 |  38.89 |             12.42
   7 |    27.33 |  40.20 |             12.87
   8 |    26.75 |  38.87 |             12.12
   9 |    28.35 |  40.33 |             11.99

AGGREGATE RESULTS
Total cases: 5000
Mean stable change: 27.37%
Mean FIFO change: 39.61%
Mean difference (FIFO - stable): 12.24 percentage points
Seeds where stable has lower change: 10 of 10
Total stable failures: 0
Total FIFO failures: 0


In [ ]:

# =====================================================
# PlanProof AI — Experiment 5: Optimality Gap
# Exhaustive reference for small planning problems
# =====================================================

from itertools import permutations


def is_valid_order(order, constraints):
    positions = {task: i for i, task in enumerate(order)}

    return all(
        positions[a] < positions[b]
        for a, b in constraints
    )


def exhaustive_optimal_repair(plan):
    """Find the valid ordering with minimum pairwise changes."""

    original = list(dict.fromkeys(plan["task_order"]))

    for task in plan["required_tasks"]:
        if task not in original:
            original.append(task)

    # This experiment assumes every task is required
    # and every constraint refers to a known task.
    if set(original) != set(plan["required_tasks"]):
        return None

    constraints = plan["precedence_constraints"]

    if any(
        a not in original or b not in original
        for a, b in constraints
    ):
        return None

    best_order = None
    best_cost = float("inf")

    for candidate in permutations(original):
        if not is_valid_order(candidate, constraints):
            continue

        cost = pairwise_order_change_rate(original, candidate)

        if cost < best_cost:
            best_cost = cost
            best_order = list(candidate)

    if best_order is None:
        return None

    return {
        "task_order": best_order,
        "pairwise_change_rate": best_cost,
    }


def run_optimality_experiment(cases=30, seed=2026):
    rng = random.Random(seed)

    stable_costs = []
    optimal_costs = []
    optimal_matches = 0

    for case_id in range(cases):
        n = rng.randint(5, 8)
        p = rng.uniform(0.10, 0.35)

        case = generate_benchmark_case(rng, n, p)

        stable = repair_plan(case)
        optimal = exhaustive_optimal_repair(case)

        if not stable["success"] or optimal is None:
            print("Unexpected failure in case", case_id)
            continue

        stable_order = stable["repaired_plan"]["task_order"]

        stable_cost = pairwise_order_change_rate(
            case["task_order"], stable_order
        )

        optimal_cost = optimal["pairwise_change_rate"]

        stable_costs.append(stable_cost)
        optimal_costs.append(optimal_cost)

        if abs(stable_cost - optimal_cost) < 1e-12:
            optimal_matches += 1

    if not stable_costs:
        print("No cases evaluated.")
        return None

    mean_stable = statistics.mean(stable_costs)
    mean_optimal = statistics.mean(optimal_costs)

    print("EXHAUSTIVE OPTIMALITY EXPERIMENT")
    print("Cases evaluated:", len(stable_costs))
    print(f"Stable mean change: {100*mean_stable:.2f}%")
    print(f"Optimal mean change: {100*mean_optimal:.2f}%")
    print(
        "Mean optimality gap:",
        f"{100*(mean_stable-mean_optimal):.2f} percentage points"
    )
    print(
        "Stable matched optimal cost:",
        f"{optimal_matches}/{len(stable_costs)}"
    )

    return {
        "cases": len(stable_costs),
        "stable_mean": mean_stable,
        "optimal_mean": mean_optimal,
        "mean_gap": mean_stable - mean_optimal,
        "optimal_matches": optimal_matches,
    }


optimality_results = run_optimality_experiment()


EXHAUSTIVE OPTIMALITY EXPERIMENT
Cases evaluated: 30
Stable mean change: 16.13%
Optimal mean change: 15.69%
Mean optimality gap: 0.44 percentage points
Stable matched optimal cost: 29/30


In [ ]:

# =====================================================
# PlanProof AI — Experiment 6: Find a Counterexample
# =====================================================

def find_counterexample(seed=2026, max_cases=100):
    rng = random.Random(seed)

    for case_id in range(max_cases):
        n = rng.randint(5, 8)
        p = rng.uniform(0.10, 0.35)

        case = generate_benchmark_case(rng, n, p)

        stable = repair_plan(case)
        optimal = exhaustive_optimal_repair(case)

        if not stable["success"] or optimal is None:
            continue

        stable_order = stable["repaired_plan"]["task_order"]

        stable_cost = pairwise_order_change_rate(
            case["task_order"], stable_order
        )
        optimal_cost = optimal["pairwise_change_rate"]

        if stable_cost > optimal_cost + 1e-12:
            print("COUNTEREXAMPLE FOUND")
            print("Case:", case_id)
            print("Original order:", case["task_order"])
            print("Constraints:", case["precedence_constraints"])
            print("Stable repair:", stable_order)
            print("Optimal repair:", optimal["task_order"])
            print(f"Stable cost: {100*stable_cost:.2f}%")
            print(f"Optimal cost: {100*optimal_cost:.2f}%")
            print(
                "Gap:",
                f"{100*(stable_cost-optimal_cost):.2f} percentage points"
            )

            return case, stable_order, optimal["task_order"]

    print("No counterexample found in this search.")
    return None


counterexample = find_counterexample()


COUNTEREXAMPLE FOUND
Case: 6
Original order: ['T5', 'T2', 'T1', 'T4', 'T0', 'T3']
Constraints: [('T0', 'T1'), ('T1', 'T2'), ('T1', 'T3'), ('T2', 'T5')]
Stable repair: ['T4', 'T0', 'T1', 'T2', 'T5', 'T3']
Optimal repair: ['T0', 'T1', 'T2', 'T5', 'T4', 'T3']
Stable cost: 60.00%
Optimal cost: 46.67%
Gap: 13.33 percentage points


In [ ]:

# =====================================================
# PlanProof AI — Local Search Repair v0.2
# =====================================================

def local_search_repair(plan):
    """Improve stable repair by testing valid task relocations."""

    initial = repair_plan(plan)

    if not initial["success"]:
        return initial

    current = initial["repaired_plan"]
    original = list(dict.fromkeys(plan["task_order"]))

    for task in plan["required_tasks"]:
        if task not in original:
            original.append(task)

    def cost(order):
        return pairwise_order_change_rate(original, order)

    current_order = current["task_order"][:]
    current_cost = cost(current_order)

    improved = True

    while improved:
        improved = False
        best_order = current_order
        best_cost = current_cost

        for source in range(len(current_order)):
            task = current_order[source]
            remaining = current_order[:source] + current_order[source + 1:]

            for destination in range(len(remaining) + 1):
                candidate = remaining[:destination] + [task] + remaining[destination:]

                if candidate == current_order:
                    continue

                candidate_plan = {
                    **current,
                    "task_order": candidate,
                }

                if not validate_plan(candidate_plan)["valid"]:
                    continue

                candidate_cost = cost(candidate)

                if candidate_cost < best_cost - 1e-12:
                    best_cost = candidate_cost
                    best_order = candidate

        if best_cost < current_cost - 1e-12:
            current_order = best_order
            current_cost = best_cost
            improved = True

    result_plan = {
        **current,
        "task_order": current_order,
    }

    return {
        "success": validate_plan(result_plan)["valid"],
        "repaired_plan": result_plan,
        "message": "Local search completed.",
        "pairwise_change_rate": current_cost,
    }


# Test on our discovered counterexample.
case, stable_order, optimal_order = counterexample

improved_result = local_search_repair(case)

print("COUNTEREXAMPLE COMPARISON")
print("Original:", case["task_order"])
print("Stable:", stable_order)
print("Local search:", improved_result["repaired_plan"]["task_order"])
print("Exhaustive optimum:", optimal_order)

print(
    "Stable cost:",
    f"{100*pairwise_order_change_rate(case['task_order'], stable_order):.2f}%"
)

print(
    "Local search cost:",
    f"{100*pairwise_order_change_rate(case['task_order'], improved_result['repaired_plan']['task_order']):.2f}%"
)

print(
    "Optimal cost:",
    f"{100*pairwise_order_change_rate(case['task_order'], optimal_order):.2f}%"
)


COUNTEREXAMPLE COMPARISON
Original: ['T5', 'T2', 'T1', 'T4', 'T0', 'T3']
Stable: ['T4', 'T0', 'T1', 'T2', 'T5', 'T3']
Local search: ['T0', 'T1', 'T2', 'T5', 'T4', 'T3']
Exhaustive optimum: ['T0', 'T1', 'T2', 'T5', 'T4', 'T3']
Stable cost: 60.00%
Local search cost: 46.67%
Optimal cost: 46.67%


In [ ]:

# =====================================================
# PlanProof AI — Experiment 7: Repair Quality Comparison
# =====================================================

import time


def compare_three_methods(cases=30, seed=2026):
    rng = random.Random(seed)

    stable_costs = []
    local_costs = []
    optimal_costs = []

    stable_times = []
    local_times = []
    optimal_times = []

    local_optimal_matches = 0
    stable_optimal_matches = 0

    for case_id in range(cases):
        n = rng.randint(5, 8)
        p = rng.uniform(0.10, 0.35)
        case = generate_benchmark_case(rng, n, p)

        start = time.perf_counter()
        stable = repair_plan(case)
        stable_times.append(time.perf_counter() - start)

        start = time.perf_counter()
        local = local_search_repair(case)
        local_times.append(time.perf_counter() - start)

        start = time.perf_counter()
        optimal = exhaustive_optimal_repair(case)
        optimal_times.append(time.perf_counter() - start)

        if not stable["success"] or not local["success"] or optimal is None:
            print("Unexpected failure in case:", case_id)
            continue

        original = case["task_order"]

        stable_cost = pairwise_order_change_rate(
            original, stable["repaired_plan"]["task_order"]
        )
        local_cost = pairwise_order_change_rate(
            original, local["repaired_plan"]["task_order"]
        )
        optimal_cost = optimal["pairwise_change_rate"]

        stable_costs.append(stable_cost)
        local_costs.append(local_cost)
        optimal_costs.append(optimal_cost)

        if abs(stable_cost - optimal_cost) < 1e-12:
            stable_optimal_matches += 1

        if abs(local_cost - optimal_cost) < 1e-12:
            local_optimal_matches += 1

    count = len(optimal_costs)

    if count == 0:
        print("No cases evaluated.")
        return None

    print("THREE-METHOD COMPARISON")
    print("Cases evaluated:", count)
    print()

    print(
        "Stable repair mean change:",
        f"{100*statistics.mean(stable_costs):.2f}%"
    )
    print(
        "Local-search mean change:",
        f"{100*statistics.mean(local_costs):.2f}%"
    )
    print(
        "Exhaustive optimum mean change:",
        f"{100*statistics.mean(optimal_costs):.2f}%"
    )

    print()
    print("Stable optimal-cost matches:",
          f"{stable_optimal_matches}/{count}")
    print("Local-search optimal-cost matches:",
          f"{local_optimal_matches}/{count}")

    print()
    print(
        "Mean local-search optimality gap:",
        f"{100*statistics.mean(
            [a-b for a, b in zip(local_costs, optimal_costs)]
        ):.2f} percentage points"
    )

    print()
    print("Mean runtime per case (milliseconds)")
    print(f"Stable: {1000*statistics.mean(stable_times):.3f}")
    print(f"Local search: {1000*statistics.mean(local_times):.3f}")
    print(f"Exhaustive: {1000*statistics.mean(optimal_times):.3f}")

    return {
        "cases": count,
        "stable_mean": statistics.mean(stable_costs),
        "local_mean": statistics.mean(local_costs),
        "optimal_mean": statistics.mean(optimal_costs),
        "stable_matches": stable_optimal_matches,
        "local_matches": local_optimal_matches,
    }


three_method_results = compare_three_methods()


THREE-METHOD COMPARISON
Cases evaluated: 30

Stable repair mean change: 16.13%
Local-search mean change: 15.69%
Exhaustive optimum mean change: 15.69%

Stable optimal-cost matches: 29/30
Local-search optimal-cost matches: 30/30

Mean local-search optimality gap: 0.00 percentage points

Mean runtime per case (milliseconds)
Stable: 0.021
Local search: 0.293
Exhaustive: 21.233


In [ ]:

# =====================================================
# PlanProof AI — Experiment 8: Expanded Optimality Test
# =====================================================

def expanded_optimality_test(
    seeds=range(10),
    cases_per_seed=10,
):
    all_results = []

    for seed in seeds:
        rng = random.Random(seed)
        seed_gaps = []
        seed_matches = 0

        for _ in range(cases_per_seed):
            n = rng.randint(5, 8)
            p = rng.uniform(0.10, 0.35)

            case = generate_benchmark_case(rng, n, p)

            local = local_search_repair(case)
            optimal = exhaustive_optimal_repair(case)

            if not local["success"] or optimal is None:
                continue

            local_cost = pairwise_order_change_rate(
                case["task_order"],
                local["repaired_plan"]["task_order"],
            )
            optimal_cost = optimal["pairwise_change_rate"]

            gap = local_cost - optimal_cost
            seed_gaps.append(gap)

            if abs(gap) < 1e-12:
                seed_matches += 1

        all_results.append({
            "seed": seed,
            "cases": len(seed_gaps),
            "matches": seed_matches,
            "mean_gap": (
                statistics.mean(seed_gaps)
                if seed_gaps else None
            ),
        })

    evaluated = sum(r["cases"] for r in all_results)
    matches = sum(r["matches"] for r in all_results)
    gaps = [
        r["mean_gap"]
        for r in all_results
        if r["mean_gap"] is not None
    ]

    print("EXPANDED OPTIMALITY RESULTS")
    print("Cases evaluated:", evaluated)
    print("Optimal-cost matches:", matches)
    print(
        "Optimal-cost match rate:",
        f"{100*matches/evaluated:.2f}%"
        if evaluated else "N/A"
    )

    if gaps:
        print(
            "Mean per-seed optimality gap:",
            f"{100*statistics.mean(gaps):.4f} percentage points"
        )

    print("\nPer-seed results")
    for r in all_results:
        print(
            f"Seed {r['seed']}: "
            f"{r['matches']}/{r['cases']} optimal-cost matches"
        )

    return all_results


expanded_results = expanded_optimality_test()


EXPANDED OPTIMALITY RESULTS
Cases evaluated: 100
Optimal-cost matches: 100
Optimal-cost match rate: 100.00%
Mean per-seed optimality gap: 0.0000 percentage points

Per-seed results
Seed 0: 10/10 optimal-cost matches
Seed 1: 10/10 optimal-cost matches
Seed 2: 10/10 optimal-cost matches
Seed 3: 10/10 optimal-cost matches
Seed 4: 10/10 optimal-cost matches
Seed 5: 10/10 optimal-cost matches
Seed 6: 10/10 optimal-cost matches
Seed 7: 10/10 optimal-cost matches
Seed 8: 10/10 optimal-cost matches
Seed 9: 10/10 optimal-cost matches


In [ ]:

# =====================================================
# PlanProof AI — Experiment 9: Scalability Evaluation
# =====================================================

import time


def scalability_experiment(
    cases_per_size=100,
    task_sizes=(10, 15, 20, 30),
    seed=2027,
):
    rng = random.Random(seed)
    results = []

    for n in task_sizes:
        stable_costs = []
        local_costs = []
        stable_times = []
        local_times = []
        stable_failures = 0
        local_failures = 0

        for _ in range(cases_per_size):
            p = rng.uniform(0.05, 0.25)
            case = generate_benchmark_case(rng, n, p)

            start = time.perf_counter()
            stable = repair_plan(case)
            stable_times.append(time.perf_counter() - start)

            start = time.perf_counter()
            local = local_search_repair(case)
            local_times.append(time.perf_counter() - start)

            if stable["success"]:
                stable_costs.append(
                    pairwise_order_change_rate(
                        case["task_order"],
                        stable["repaired_plan"]["task_order"],
                    )
                )
            else:
                stable_failures += 1

            if local["success"]:
                local_costs.append(
                    pairwise_order_change_rate(
                        case["task_order"],
                        local["repaired_plan"]["task_order"],
                    )
                )
            else:
                local_failures += 1

        row = {
            "tasks": n,
            "cases": cases_per_size,
            "stable_valid": len(stable_costs),
            "local_valid": len(local_costs),
            "stable_failures": stable_failures,
            "local_failures": local_failures,
            "stable_change": (
                statistics.mean(stable_costs)
                if stable_costs else None
            ),
            "local_change": (
                statistics.mean(local_costs)
                if local_costs else None
            ),
            "stable_ms": 1000 * statistics.mean(stable_times),
            "local_ms": 1000 * statistics.mean(local_times),
        }

        results.append(row)

    print("SCALABILITY EXPERIMENT")
    print("Seed:", seed)
    print("Cases per task size:", cases_per_size)
    print()

    print(
        "Tasks | Stable change | Local change | "
        "Stable ms | Local ms | Failures (S/L)"
    )

    for r in results:
        stable_change = (
            f"{100*r['stable_change']:.2f}%"
            if r["stable_change"] is not None else "N/A"
        )
        local_change = (
            f"{100*r['local_change']:.2f}%"
            if r["local_change"] is not None else "N/A"
        )

        print(
            f"{r['tasks']:>5} | "
            f"{stable_change:>13} | "
            f"{local_change:>11} | "
            f"{r['stable_ms']:>9.3f} | "
            f"{r['local_ms']:>8.3f} | "
            f"{r['stable_failures']}/{r['local_failures']}"
        )

    return results


scalability_results = scalability_experiment()


SCALABILITY EXPERIMENT
Seed: 2027
Cases per task size: 100

Tasks | Stable change | Local change | Stable ms | Local ms | Failures (S/L)
   10 |        19.67% |      17.91% |     0.030 |    1.654 | 0/0
   15 |        26.23% |      23.77% |     0.044 |    9.697 | 0/0
   20 |        32.04% |      28.62% |     0.066 |   39.821 | 0/0
   30 |        38.59% |      34.86% |     0.146 |  315.229 | 0/0


In [ ]:

# =====================================================
# PlanProof AI — Experiment 10: Quality–Runtime Trade-off
# =====================================================

def summarize_tradeoff(results):
    print("QUALITY–RUNTIME TRADE-OFF")
    print("-" * 68)
    print(
        "Tasks | Quality gain (pp) | Extra time (ms) | "
        "Local / Stable time"
    )

    for r in results:
        quality_gain = (
            100 * (r["stable_change"] - r["local_change"])
        )
        extra_time = r["local_ms"] - r["stable_ms"]
        ratio = r["local_ms"] / r["stable_ms"]

        print(
            f"{r['tasks']:>5} | "
            f"{quality_gain:>17.2f} | "
            f"{extra_time:>15.3f} | "
            f"{ratio:>18.1f}x"
        )


summarize_tradeoff(scalability_results)


QUALITY–RUNTIME TRADE-OFF
--------------------------------------------------------------------
Tasks | Quality gain (pp) | Extra time (ms) | Local / Stable time
   10 |              1.76 |           1.624 |               54.9x
   15 |              2.46 |           9.653 |              220.1x
   20 |              3.42 |          39.755 |              601.1x
   30 |              3.74 |         315.083 |             2156.1x


In [ ]:

# =====================================================
# PlanProof AI — Fast Local Search v0.3
# =====================================================

def fast_local_search_repair(plan):
    """First-improvement relocation search with verification."""

    initial = repair_plan(plan)

    if not initial["success"]:
        return initial

    current_plan = initial["repaired_plan"]
    current_order = current_plan["task_order"][:]

    original = list(dict.fromkeys(plan["task_order"]))
    for task in plan["required_tasks"]:
        if task not in original:
            original.append(task)

    def cost(order):
        return pairwise_order_change_rate(original, order)

    current_cost = cost(current_order)
    moves_accepted = 0

    while True:
        improved = False

        for source in range(len(current_order)):
            task = current_order[source]
            remaining = (
                current_order[:source] + current_order[source + 1:]
            )

            for destination in range(len(remaining) + 1):
                candidate = (
                    remaining[:destination]
                    + [task]
                    + remaining[destination:]
                )

                if candidate == current_order:
                    continue

                # Moving one task changes only its relative
                # order with tasks it crosses.
                # Check constraints involving that task only.
                positions = {
                    item: index
                    for index, item in enumerate(candidate)
                }

                valid_move = all(
                    positions[a] < positions[b]
                    for a, b in plan["precedence_constraints"]
                    if a == task or b == task
                )

                if not valid_move:
                    continue

                candidate_cost = cost(candidate)

                if candidate_cost < current_cost - 1e-12:
                    current_order = candidate
                    current_cost = candidate_cost
                    moves_accepted += 1
                    improved = True
                    break

            if improved:
                break

        if not improved:
            break

    repaired = {
        **current_plan,
        "task_order": current_order,
    }

    verification = validate_plan(repaired)

    if not verification["valid"]:
        return {
            "success": False,
            "repaired_plan": None,
            "message": "Final verification failed.",
            "violations": verification["violations"],
        }

    return {
        "success": True,
        "repaired_plan": repaired,
        "message": "Fast local search completed.",
        "pairwise_change_rate": current_cost,
        "moves_accepted": moves_accepted,
    }


print("Fast local search function defined.")


Fast local search function defined.


In [ ]:

# =====================================================
# PlanProof AI — Experiment 11: Speed Comparison
# =====================================================

def compare_search_speeds(
    cases_per_size=100,
    task_sizes=(10, 15, 20, 30),
    seed=2027,
):
    rng = random.Random(seed)
    rows = []

    for n in task_sizes:
        stable_costs = []
        best_costs = []
        fast_costs = []

        stable_times = []
        best_times = []
        fast_times = []

        for _ in range(cases_per_size):
            case = generate_benchmark_case(
                rng, n, rng.uniform(0.05, 0.25)
            )

            start = time.perf_counter()
            stable = repair_plan(case)
            stable_times.append(time.perf_counter() - start)

            start = time.perf_counter()
            best = local_search_repair(case)
            best_times.append(time.perf_counter() - start)

            start = time.perf_counter()
            fast = fast_local_search_repair(case)
            fast_times.append(time.perf_counter() - start)

            if not (stable["success"] and best["success"] and fast["success"]):
                raise RuntimeError("A repair method failed.")

            stable_costs.append(
                pairwise_order_change_rate(
                    case["task_order"],
                    stable["repaired_plan"]["task_order"],
                )
            )
            best_costs.append(
                pairwise_order_change_rate(
                    case["task_order"],
                    best["repaired_plan"]["task_order"],
                )
            )
            fast_costs.append(
                pairwise_order_change_rate(
                    case["task_order"],
                    fast["repaired_plan"]["task_order"],
                )
            )

        rows.append({
            "tasks": n,
            "stable_cost": statistics.mean(stable_costs),
            "best_cost": statistics.mean(best_costs),
            "fast_cost": statistics.mean(fast_costs),
            "stable_ms": 1000 * statistics.mean(stable_times),
            "best_ms": 1000 * statistics.mean(best_times),
            "fast_ms": 1000 * statistics.mean(fast_times),
        })

    print("SPEED AND QUALITY COMPARISON")
    print(
        "Tasks | Stable cost | Best-search cost | Fast-search cost"
        " | Stable ms | Best ms | Fast ms"
    )

    for r in rows:
        print(
            f"{r['tasks']:>5} | "
            f"{100*r['stable_cost']:>11.2f}% | "
            f"{100*r['best_cost']:>16.2f}% | "
            f"{100*r['fast_cost']:>15.2f}% | "
            f"{r['stable_ms']:>9.3f} | "
            f"{r['best_ms']:>7.3f} | "
            f"{r['fast_ms']:>7.3f}"
        )

    return rows


speed_results = compare_search_speeds()


SPEED AND QUALITY COMPARISON
Tasks | Stable cost | Best-search cost | Fast-search cost | Stable ms | Best ms | Fast ms
   10 |       19.67% |            17.91% |           17.91% |     0.028 |   1.559 |   1.182
   15 |       26.23% |            23.77% |           23.77% |     0.048 |  11.900 |   9.018
   20 |       32.04% |            28.62% |           28.62% |     0.072 |  41.364 |  37.507
   30 |       38.59% |            34.86% |           34.87% |     0.149 | 301.918 | 294.872


In [ ]:

def compare_search_quality(
    seeds=range(10),
    cases_per_seed=100,
    task_sizes=(10, 15, 20, 30),
):
    import random
    from collections import defaultdict

    results = defaultdict(
        lambda: {
            "cases": 0,
            "stable_failures": 0,
            "best_failures": 0,
            "fast_failures": 0,
            "fast_matches_best": 0,
            "fast_better": 0,
            "best_better": 0,
            "stable_cost": 0.0,
            "best_cost": 0.0,
            "fast_cost": 0.0,
        }
    )

    for seed in seeds:
        for n in task_sizes:
            rng = random.Random(seed * 1000 + n)

            for _ in range(cases_per_seed):
                plan = generate_benchmark_case(
                    rng, task_count=n, edge_probability=0.15
                )

                stable = repair_plan(plan)
                best = local_search_repair(plan)
                fast = fast_local_search_repair(plan)

                stable_valid = (
                    stable.get("success", False)
                    and validate_plan({
                        **plan, "task_order": stable["task_order"]
                    })["valid"]
                )
                best_valid = (
                    best.get("success", False)
                    and validate_plan({
                        **plan, "task_order": best["task_order"]
                    })["valid"]
                )
                fast_valid = (
                    fast.get("success", False)
                    and validate_plan({
                        **plan, "task_order": fast["task_order"]
                    })["valid"]
                )

                row = results[n]
                row["cases"] += 1
                row["stable_failures"] += not stable_valid
                row["best_failures"] += not best_valid
                row["fast_failures"] += not fast_valid

                if not (stable_valid and best_valid and fast_valid):
                    continue

                sc = pairwise_order_change_rate(
                    plan["task_order"], stable["task_order"]
                )
                bc = pairwise_order_change_rate(
                    plan["task_order"], best["task_order"]
                )
                fc = pairwise_order_change_rate(
                    plan["task_order"], fast["task_order"]
                )

                row["stable_cost"] += sc
                row["best_cost"] += bc
                row["fast_cost"] += fc

                if abs(fc - bc) < 1e-9:
                    row["fast_matches_best"] += 1
                elif fc < bc:
                    row["fast_better"] += 1
                else:
                    row["best_better"] += 1

    print("MULTI-SEED ROBUSTNESS RESULTS")
    print(
        "Tasks | Cases | Stable cost | Best cost | Fast cost | "
        "Fast matches best | Validity failures (S/B/F)"
    )

    for n in task_sizes:
        r = results[n]
        count = r["cases"]
        valid_count = count - max(
            r["stable_failures"],
            r["best_failures"],
            r["fast_failures"],
        )

        # Use all cases for the denominator so failures remain visible.
        denom = count
        print(
            f"{n:5d} | {count:5d} | "
            f"{100*r['stable_cost']/denom:10.2f}% | "
            f"{100*r['best_cost']/denom:8.2f}% | "
            f"{100*r['fast_cost']/denom:8.2f}% | "
            f"{r['fast_matches_best']:17d} | "
            f"{r['stable_failures']}/"
            f"{r['best_failures']}/"
            f"{r['fast_failures']}"
        )

    return dict(results)


robustness_results = compare_search_quality()


KeyError: 'task_order'

In [ ]:

def compare_search_quality(
    seeds=range(10),
    cases_per_seed=100,
    task_sizes=(10, 15, 20, 30),
):
    import random
    from collections import defaultdict

    results = defaultdict(lambda: {
        "cases": 0,
        "stable_failures": 0,
        "best_failures": 0,
        "fast_failures": 0,
        "fast_matches_best": 0,
        "fast_better": 0,
        "best_better": 0,
        "stable_cost": 0.0,
        "best_cost": 0.0,
        "fast_cost": 0.0,
        "valid_cases": 0,
    })

    def check_result(plan, result):
        # Handle unsuccessful repairs safely.
        if not isinstance(result, dict):
            return False, None

        order = result.get("task_order")
        if not result.get("success", False) or not isinstance(order, list):
            return False, None

        candidate = {**plan, "task_order": order}
        validation = validate_plan(candidate)

        if not validation.get("valid", False):
            return False, None

        cost = pairwise_order_change_rate(
            plan["task_order"], order
        )
        return True, cost

    for seed in seeds:
        for n in task_sizes:
            rng = random.Random(seed * 1000 + n)

            for _ in range(cases_per_seed):
                plan = generate_benchmark_case(
                    rng,
                    task_count=n,
                    edge_probability=0.15,
                )

                stable = repair_plan(plan)
                best = local_search_repair(plan)
                fast = fast_local_search_repair(plan)

                s_ok, sc = check_result(plan, stable)
                b_ok, bc = check_result(plan, best)
                f_ok, fc = check_result(plan, fast)

                row = results[n]
                row["cases"] += 1
                row["stable_failures"] += int(not s_ok)
                row["best_failures"] += int(not b_ok)
                row["fast_failures"] += int(not f_ok)

                # Compare costs only when all three methods succeed.
                if not (s_ok and b_ok and f_ok):
                    continue

                row["valid_cases"] += 1
                row["stable_cost"] += sc
                row["best_cost"] += bc
                row["fast_cost"] += fc

                if abs(fc - bc) < 1e-9:
                    row["fast_matches_best"] += 1
                elif fc < bc:
                    row["fast_better"] += 1
                else:
                    row["best_better"] += 1

    print("MULTI-SEED ROBUSTNESS RESULTS")
    print(
        "Tasks | Cases | Valid all 3 | Stable cost | Best cost | "
        "Fast cost | Fast=Best | Failures S/B/F"
    )

    for n in task_sizes:
        r = results[n]
        count = r["cases"]
        valid = r["valid_cases"]

        def mean_cost(key):
            return (
                f"{100 * r[key] / valid:.2f}%"
                if valid else "N/A"
            )

        print(
            f"{n:5d} | {count:5d} | {valid:11d} | "
            f"{mean_cost('stable_cost'):11s} | "
            f"{mean_cost('best_cost'):9s} | "
            f"{mean_cost('fast_cost'):9s} | "
            f"{r['fast_matches_best']:9d} | "
            f"{r['stable_failures']}/"
            f"{r['best_failures']}/"
            f"{r['fast_failures']}"
        )

    return {n: dict(row) for n, row in results.items()}


robustness_results = compare_search_quality()


MULTI-SEED ROBUSTNESS RESULTS
Tasks | Cases | Valid all 3 | Stable cost | Best cost | Fast cost | Fast=Best | Failures S/B/F
   10 |  1000 |           0 | N/A         | N/A       | N/A       |         0 | 1000/1000/1000
   15 |  1000 |           0 | N/A         | N/A       | N/A       |         0 | 1000/1000/1000
   20 |  1000 |           0 | N/A         | N/A       | N/A       |         0 | 1000/1000/1000
   30 |  1000 |           0 | N/A         | N/A       | N/A       |         0 | 1000/1000/1000


In [ ]:

def compare_search_quality(
    seeds=range(10),
    cases_per_seed=100,
    task_sizes=(10, 15, 20, 30),
):
    import random
    from collections import defaultdict

    results = defaultdict(lambda: {
        "cases": 0,
        "stable_failures": 0,
        "best_failures": 0,
        "fast_failures": 0,
        "fast_matches_best": 0,
        "fast_better": 0,
        "best_better": 0,
        "stable_cost": 0.0,
        "best_cost": 0.0,
        "fast_cost": 0.0,
        "valid_cases": 0,
    })

    def check_result(plan, result):
        if not isinstance(result, dict) or not result.get("success", False):
            return False, None

        # Correctly read the nested repaired plan.
        repaired_plan = result.get("repaired_plan")
        if not isinstance(repaired_plan, dict):
            return False, None

        order = repaired_plan.get("task_order")
        if not isinstance(order, list):
            return False, None

        validation = validate_plan(repaired_plan)
        if not validation.get("valid", False):
            return False, None

        cost = pairwise_order_change_rate(
            plan["task_order"], order
        )
        return True, cost

    for seed in seeds:
        for n in task_sizes:
            rng = random.Random(seed * 1000 + n)

            for _ in range(cases_per_seed):
                plan = generate_benchmark_case(
                    rng,
                    task_count=n,
                    edge_probability=0.15,
                )

                stable = repair_plan(plan)
                best = local_search_repair(plan)
                fast = fast_local_search_repair(plan)

                s_ok, sc = check_result(plan, stable)
                b_ok, bc = check_result(plan, best)
                f_ok, fc = check_result(plan, fast)

                row = results[n]
                row["cases"] += 1
                row["stable_failures"] += int(not s_ok)
                row["best_failures"] += int(not b_ok)
                row["fast_failures"] += int(not f_ok)

                if not (s_ok and b_ok and f_ok):
                    continue

                row["valid_cases"] += 1
                row["stable_cost"] += sc
                row["best_cost"] += bc
                row["fast_cost"] += fc

                if abs(fc - bc) < 1e-9:
                    row["fast_matches_best"] += 1
                elif fc < bc:
                    row["fast_better"] += 1
                else:
                    row["best_better"] += 1

    print("MULTI-SEED ROBUSTNESS RESULTS")
    print(
        "Tasks | Cases | Valid all 3 | Stable cost | Best cost | "
        "Fast cost | Fast=Best | Failures S/B/F"
    )

    for n in task_sizes:
        r = results[n]
        count = r["cases"]
        valid = r["valid_cases"]

        def mean_cost(key):
            return (
                f"{100 * r[key] / valid:.2f}%"
                if valid else "N/A"
            )

        print(
            f"{n:5d} | {count:5d} | {valid:11d} | "
            f"{mean_cost('stable_cost'):11s} | "
            f"{mean_cost('best_cost'):9s} | "
            f"{mean_cost('fast_cost'):9s} | "
            f"{r['fast_matches_best']:9d} | "
            f"{r['stable_failures']}/"
            f"{r['best_failures']}/"
            f"{r['fast_failures']}"
        )

    return {n: dict(row) for n, row in results.items()}


robustness_results = compare_search_quality()


MULTI-SEED ROBUSTNESS RESULTS
Tasks | Cases | Valid all 3 | Stable cost | Best cost | Fast cost | Fast=Best | Failures S/B/F
   10 |  1000 |        1000 | 21.53%      | 19.64%    | 19.65%    |       999 | 0/0/0
   15 |  1000 |        1000 | 27.37%      | 24.27%    | 24.27%    |       998 | 0/0/0
   20 |  1000 |        1000 | 32.02%      | 28.37%    | 28.38%    |       991 | 0/0/0
   30 |  1000 |        1000 | 37.57%      | 33.52%    | 33.52%    |       968 | 0/0/0


In [ ]:

import random
import time

def benchmark_search_runtime(
    seeds=range(10),
    cases_per_seed=100,
    task_sizes=(10, 15, 20, 30),
):
    results = {}

    for n in task_sizes:
        timings = {
            "stable": [],
            "best": [],
            "fast": [],
        }

        for seed in seeds:
            rng = random.Random(seed * 1000 + n)

            for _ in range(cases_per_seed):
                plan = generate_benchmark_case(
                    rng,
                    task_count=n,
                    edge_probability=0.15,
                )

                for name, fn in [
                    ("stable", repair_plan),
                    ("best", local_search_repair),
                    ("fast", fast_local_search_repair),
                ]:
                    start = time.perf_counter()
                    result = fn(plan)
                    elapsed_ms = (
                        time.perf_counter() - start
                    ) * 1000

                    if not result.get("success", False):
                        raise RuntimeError(
                            f"{name} failed for {n} tasks, seed {seed}"
                        )

                    timings[name].append(elapsed_ms)

        results[n] = {
            name: sum(values) / len(values)
            for name, values in timings.items()
        }

    print("RUNTIME ROBUSTNESS RESULTS")
    print("Tasks | Stable ms | Best-search ms | Fast-search ms")

    for n, row in results.items():
        print(
            f"{n:5d} | "
            f"{row['stable']:9.4f} | "
            f"{row['best']:14.4f} | "
            f"{row['fast']:14.4f}"
        )

    return results


runtime_results = benchmark_search_runtime()


RUNTIME ROBUSTNESS RESULTS
Tasks | Stable ms | Best-search ms | Fast-search ms
   10 |    0.0338 |         2.1050 |         1.5690
   15 |    0.0471 |        11.6686 |         9.1157
   20 |    0.0778 |        43.0731 |        40.0552
   30 |    0.1441 |       283.8734 |       325.3801


In [ ]:

# PlanProof AI — Reproducible Benchmark Setup
# Experiment 14: Runtime and reproducibility

import os
import sys
import json
import time
import random
import platform
import statistics
import pandas as pd

from datetime import datetime, timezone
from pathlib import Path

# Fixed experiment configuration
MASTER_SEED = 20261009
TASK_SIZES = (10, 15, 20, 30)
CASES_PER_SIZE = 100
WARMUP_CASES = 5
TIMING_REPEATS = 5
EDGE_PROBABILITY = 0.15

# Save outputs in a dedicated directory
RESULTS_DIR = Path("/content/planproof_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Record the experiment environment
environment = {
    "project": "PlanProof AI",
    "experiment": "runtime_reproducibility",
    "master_seed": MASTER_SEED,
    "task_sizes": list(TASK_SIZES),
    "cases_per_size": CASES_PER_SIZE,
    "warmup_cases": WARMUP_CASES,
    "timing_repeats": TIMING_REPEATS,
    "edge_probability": EDGE_PROBABILITY,
    "python_version": sys.version,
    "platform": platform.platform(),
    "started_utc": datetime.now(timezone.utc).isoformat(),
}

with open(RESULTS_DIR / "experiment_config.json", "w") as f:
    json.dump(environment, f, indent=2)

print("PlanProof AI experiment setup complete.")
print("Results directory:", RESULTS_DIR)
print("Python version:", platform.python_version())
print("Task sizes:", TASK_SIZES)
print("Cases per size:", CASES_PER_SIZE)
print("Timing repeats:", TIMING_REPEATS)
print("Configuration saved.")


PlanProof AI experiment setup complete.
Results directory: /content/planproof_results
Python version: 3.13.15
Task sizes: (10, 15, 20, 30)
Cases per size: 100
Timing repeats: 5
Configuration saved.


In [ ]:

required_functions = [
    "generate_benchmark_case",
    "validate_plan",
    "repair_plan",
    "local_search_repair",
    "fast_local_search_repair",
    "pairwise_order_change_rate",
]

missing = [
    name for name in required_functions
    if name not in globals() or not callable(globals()[name])
]

if missing:
    print("Missing functions:", missing)
    print("Run the earlier cells that define these functions.")
else:
    print("All required functions are available.")

    test_rng = random.Random(MASTER_SEED)
    test_plan = generate_benchmark_case(
        test_rng,
        task_count=10,
        edge_probability=EDGE_PROBABILITY,
    )

    for name, fn in [
        ("Stable", repair_plan),
        ("Best-search", local_search_repair),
        ("Fast-search", fast_local_search_repair),
    ]:
        result = fn(test_plan)
        repaired = result.get("repaired_plan", {})
        order = repaired.get("task_order", [])

        valid = (
            result.get("success", False)
            and validate_plan(repaired).get("valid", False)
        )

        print(
            f"{name:12s} | success={result.get('success', False)}"
            f" | valid={valid} | tasks={len(order)}"
        )


All required functions are available.
Stable       | success=True | valid=True | tasks=10
Best-search  | success=True | valid=True | tasks=10
Fast-search  | success=True | valid=True | tasks=10


In [ ]:

# PlanProof AI — Experiment 14, Step 3
# Generate a fixed benchmark dataset and save it to JSON.

benchmark_cases = []

for n in TASK_SIZES:
    for case_index in range(CASES_PER_SIZE):
        # Each case gets its own deterministic seed.
        case_seed = MASTER_SEED + n * 100_000 + case_index
        rng = random.Random(case_seed)

        plan = generate_benchmark_case(
            rng,
            task_count=n,
            edge_probability=EDGE_PROBABILITY,
        )

        benchmark_cases.append({
            "case_id": f"n{n}_case{case_index:03d}",
            "task_count": n,
            "seed": case_seed,
            "plan": plan,
        })

# Save the exact input plans, not just the random seeds.
dataset_path = RESULTS_DIR / "benchmark_dataset.json"

with open(dataset_path, "w") as f:
    json.dump(benchmark_cases, f, indent=2)

# Basic dataset checks
case_ids = [case["case_id"] for case in benchmark_cases]
assert len(case_ids) == len(set(case_ids))
assert len(benchmark_cases) == len(TASK_SIZES) * CASES_PER_SIZE

print("Benchmark dataset created successfully.")
print("Total plans:", len(benchmark_cases))
print("Plans per task size:")

for n in TASK_SIZES:
    count = sum(c["task_count"] == n for c in benchmark_cases)
    print(f"  {n} tasks: {count}")

print("Saved to:", dataset_path)


Benchmark dataset created successfully.
Total plans: 400
Plans per task size:
  10 tasks: 100
  15 tasks: 100
  20 tasks: 100
  30 tasks: 100
Saved to: /content/planproof_results/benchmark_dataset.json


In [ ]:

# PlanProof AI — Step 4: Verify the saved dataset

with open(dataset_path, "r") as f:
    reloaded_cases = json.load(f)

# Compare JSON-normalized versions of both datasets.
# This accounts for tuples becoming lists during JSON serialization.
expected = json.loads(json.dumps(benchmark_cases))

assert reloaded_cases == expected, "Dataset changed during save/load!"

# Verify plan structure and required-task coverage.
for case in reloaded_cases:
    plan = case["plan"]

    assert len(plan["task_order"]) == case["task_count"]
    assert len(set(plan["task_order"])) == case["task_count"]
    assert set(plan["task_order"]) == set(plan["required_tasks"])

    task_ids = set(plan["task_order"])

    # Every precedence constraint must reference known tasks.
    for before, after in plan["precedence_constraints"]:
        assert before in task_ids
        assert after in task_ids

print("Reproducibility check passed.")
print("Saved dataset matches the in-memory dataset after JSON normalization.")
print("Total plans:", len(reloaded_cases))
print("All plans pass structural checks.")


Reproducibility check passed.
Saved dataset matches the in-memory dataset after JSON normalization.
Total plans: 400
All plans pass structural checks.


In [ ]:

# PlanProof AI — Experiment 14, Step 5
# Repeated timing on the frozen dataset

import time
import statistics
import pandas as pd

algorithms = {
    "stable": repair_plan,
    "best_search": local_search_repair,
    "fast_search": fast_local_search_repair,
}

# Warm up each algorithm on a few existing benchmark plans.
for case in reloaded_cases[:WARMUP_CASES]:
    for fn in algorithms.values():
        fn(case["plan"])

records = []

for case_number, case in enumerate(reloaded_cases, start=1):
    plan = case["plan"]

    for algorithm_name, fn in algorithms.items():
        timings_ms = []
        final_result = None

        for repeat in range(TIMING_REPEATS):
            start = time.perf_counter()
            result = fn(plan)
            elapsed_ms = (time.perf_counter() - start) * 1000

            timings_ms.append(elapsed_ms)
            final_result = result

        # Read the result structure used by our algorithms.
        repaired_plan = (
            final_result.get("repaired_plan", {})
            if isinstance(final_result, dict)
            else {}
        )

        success = (
            isinstance(final_result, dict)
            and final_result.get("success", False)
        )

        valid = (
            success
            and isinstance(repaired_plan, dict)
            and validate_plan(repaired_plan).get("valid", False)
        )

        if valid:
            cost = pairwise_order_change_rate(
                plan["task_order"],
                repaired_plan["task_order"],
            )
            repaired_task_count = len(repaired_plan["task_order"])
        else:
            cost = None
            repaired_task_count = None

        records.append({
            "case_id": case["case_id"],
            "task_count": case["task_count"],
            "seed": case["seed"],
            "algorithm": algorithm_name,
            "success": success,
            "valid": bool(valid),
            "pairwise_change_rate": cost,
            "median_runtime_ms": statistics.median(timings_ms),
            "min_runtime_ms": min(timings_ms),
            "max_runtime_ms": max(timings_ms),
            "repeats": TIMING_REPEATS,
            "repaired_task_count": repaired_task_count,
        })

    if case_number % 50 == 0:
        print(f"Processed {case_number}/{len(reloaded_cases)} plans")

benchmark_df = pd.DataFrame(records)

csv_path = RESULTS_DIR / "per_case_benchmark_results.csv"
benchmark_df.to_csv(csv_path, index=False)

print("\nBenchmark completed.")
print("Rows recorded:", len(benchmark_df))
print("Expected rows:", len(reloaded_cases) * len(algorithms))
print("Saved to:", csv_path)

print("\nValidity summary:")
print(
    benchmark_df.groupby("algorithm")["valid"]
    .agg(["sum", "count"])
    .to_string()
)


Processed 50/400 plans
Processed 100/400 plans
Processed 150/400 plans
Processed 200/400 plans
Processed 250/400 plans
Processed 300/400 plans
Processed 350/400 plans
Processed 400/400 plans

Benchmark completed.
Rows recorded: 1200
Expected rows: 1200
Saved to: /content/planproof_results/per_case_benchmark_results.csv

Validity summary:
             sum  count
algorithm              
best_search  400    400
fast_search  400    400
stable       400    400


In [ ]:

# PlanProof AI — Experiment 14, Step 6
# Aggregate the reproducible benchmark results

summary_df = (
    benchmark_df
    .groupby(["task_count", "algorithm"], as_index=False)
    .agg(
        cases=("case_id", "count"),
        valid_cases=("valid", "sum"),
        median_runtime_ms=("median_runtime_ms", "median"),
        p95_runtime_ms=("median_runtime_ms", lambda x: x.quantile(0.95)),
        mean_order_change=("pairwise_change_rate", "mean"),
    )
)

summary_df["validity_rate_pct"] = (
    100 * summary_df["valid_cases"] / summary_df["cases"]
)

summary_df["mean_order_change_pct"] = (
    100 * summary_df["mean_order_change"]
)

summary_path = RESULTS_DIR / "benchmark_summary.csv"
summary_df.to_csv(summary_path, index=False)

display_columns = [
    "task_count",
    "algorithm",
    "cases",
    "validity_rate_pct",
    "mean_order_change_pct",
    "median_runtime_ms",
    "p95_runtime_ms",
]

print("SUMMARY BY TASK SIZE")
print(
    summary_df[display_columns]
    .round(4)
    .to_string(index=False)
)

print("\nSummary saved to:", summary_path)


SUMMARY BY TASK SIZE
 task_count   algorithm  cases  validity_rate_pct  mean_order_change_pct  median_runtime_ms  p95_runtime_ms
         10 best_search    100              100.0                21.1778             1.0460          3.6995
         10 fast_search    100              100.0                21.1778             0.8751          2.2967
         10      stable    100              100.0                23.0000             0.0179          0.0220
         15 best_search    100              100.0                24.5143            10.0384         33.0886
         15 fast_search    100              100.0                24.5143             7.2686         30.2640
         15      stable    100              100.0                28.0000             0.0322          0.0599
         20 best_search    100              100.0                28.3947            37.8859         97.2520
         20 fast_search    100              100.0                28.3947            30.6798         84.9087
       

In [ ]:

# PlanProof AI — Experiment 15
# Paired comparisons with bootstrap confidence intervals

import numpy as np
import pandas as pd

def bootstrap_mean_ci(values, seed=20261009, repeats=5000):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)

    samples = rng.choice(
        values,
        size=(repeats, len(values)),
        replace=True,
    ).mean(axis=1)

    low, high = np.percentile(samples, [2.5, 97.5])
    return float(values.mean()), float(low), float(high)


def paired_comparison(df, task_count, metric, method_a, method_b):
    subset = df[df["task_count"] == task_count]

    paired = subset.pivot(
        index="case_id",
        columns="algorithm",
        values=metric,
    ).dropna(subset=[method_a, method_b])

    # Positive values mean method B has a lower value than method A.
    # For order-change cost, positive means B preserves more ordering.
    improvement = paired[method_a] - paired[method_b]

    mean, low, high = bootstrap_mean_ci(
        improvement.to_numpy(),
        seed=MASTER_SEED + task_count,
    )

    return {
        "task_count": task_count,
        "metric": metric,
        "comparison": f"{method_a} minus {method_b}",
        "paired_cases": len(improvement),
        "mean_difference": mean,
        "ci95_low": low,
        "ci95_high": high,
        "b_lower_cases": int((improvement > 0).sum()),
        "ties": int((improvement == 0).sum()),
        "a_lower_cases": int((improvement < 0).sum()),
    }


comparison_rows = []

for n in TASK_SIZES:
    # Lower order-change cost is better.
    comparison_rows.append(
        paired_comparison(
            benchmark_df, n, "pairwise_change_rate",
            "stable", "best_search"
        )
    )
    comparison_rows.append(
        paired_comparison(
            benchmark_df, n, "pairwise_change_rate",
            "best_search", "fast_search"
        )
    )

    # Lower runtime is better.
    comparison_rows.append(
        paired_comparison(
            benchmark_df, n, "median_runtime_ms",
            "best_search", "fast_search"
        )
    )

comparison_df = pd.DataFrame(comparison_rows)

print("PAIRED COMPARISONS")
print(comparison_df.round(6).to_string(index=False))

comparison_path = RESULTS_DIR / "paired_comparisons.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nSaved to:", comparison_path)


PAIRED COMPARISONS
 task_count               metric                    comparison  paired_cases  mean_difference   ci95_low  ci95_high  b_lower_cases  ties  a_lower_cases
         10 pairwise_change_rate      stable minus best_search           100         0.018222   0.012222   0.025111             36    64              0
         10 pairwise_change_rate best_search minus fast_search           100         0.000000   0.000000   0.000000              0   100              0
         10    median_runtime_ms best_search minus fast_search           100         0.457119   0.341520   0.591185             96     0              4
         15 pairwise_change_rate      stable minus best_search           100         0.034857   0.027617   0.042667             69    31              0
         15 pairwise_change_rate best_search minus fast_search           100         0.000000   0.000000   0.000000              0   100              0
         15    median_runtime_ms best_search minus fast_search       

In [ ]:

# PlanProof AI — Experiment 16: Artifact manifest

import hashlib
import json
from datetime import datetime, timezone

artifact_names = [
    "experiment_config.json",
    "benchmark_dataset.json",
    "per_case_benchmark_results.csv",
    "benchmark_summary.csv",
    "paired_comparisons.csv",
]

manifest = {
    "project": "PlanProof AI",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "artifacts": [],
}

for filename in artifact_names:
    path = RESULTS_DIR / filename

    if path.exists():
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        manifest["artifacts"].append({
            "filename": filename,
            "bytes": path.stat().st_size,
            "sha256": digest,
        })
    else:
        print("Missing artifact:", filename)

manifest_path = RESULTS_DIR / "manifest.json"

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("\nARTIFACT MANIFEST")
print(json.dumps(manifest, indent=2))
print("\nManifest saved to:", manifest_path)



ARTIFACT MANIFEST
{
  "project": "PlanProof AI",
  "created_utc": "2026-10-09T15:28:56.641161+00:00",
  "artifacts": [
    {
      "filename": "experiment_config.json",
      "bytes": 433,
      "sha256": "b2c36d1bf023eec19e7b24afc30a9ef787e4fdd432fd536ce2c555930a74d918"
    },
    {
      "filename": "benchmark_dataset.json",
      "bytes": 920788,
      "sha256": "28b9dc8ed54f54f1d80701e4b215ec4212810d4a26749230127bb3fa96c04021"
    },
    {
      "filename": "per_case_benchmark_results.csv",
      "bytes": 150385,
      "sha256": "a14cc625c5ced5f7c8fe7d0904fba3f57b33ec4e4378fbf1259602dc97d5066e"
    },
    {
      "filename": "benchmark_summary.csv",
      "bytes": 1326,
      "sha256": "da61e02630818a7a36b390a7c41b6e676a33e8f5877b6560c1ba588ad3b1d4b9"
    },
    {
      "filename": "paired_comparisons.csv",
      "bytes": 1425,
      "sha256": "204fc1c99d94c25254cd9eb0b31d7f67fe5fcb46eca21b4a137e76b1fd1b12d3"
    }
  ]
}

Manifest saved to: /content/planproof_results/manifest.json

In [ ]:

# PlanProof AI — Research report and artifact bundle

import json
import zipfile
from pathlib import Path
from datetime import datetime, timezone

report_path = RESULTS_DIR / "research_report.md"

report = r"""
# PlanProof AI: Reproducible Planning and Repair Benchmark

## 1. Research objective

This experiment studies whether constraint-aware local-search repair
can preserve more of an original task ordering than stable topological
repair, while maintaining plan validity.

## 2. Research question

On a synthetic benchmark of task-ordering plans with precedence
constraints, how do stable repair, best-improvement local search,
and first-improvement local search compare in validity, order-change
cost, and runtime?

## 3. Methods

Three methods were evaluated:

- **Stable repair:** stable topological repair.
- **Best-search:** local search using best-improvement moves.
- **Fast-search:** local search using first-improvement moves.

The benchmark contains 400 frozen synthetic plans:
100 each with 10, 15, 20, and 30 tasks. The configured edge
probability is 0.15. The saved dataset is the authoritative set
of inputs for this experiment.

Each algorithm was run on every plan. Runtime was measured five
times per plan, with warm-up calls before the timed benchmark.
The per-case runtime statistic is the median of those five runs.

Plan validity was checked after each algorithm returned. Repair
quality was measured using pairwise order-change rate: the proportion
of task pairs whose relative order differs from the original plan.
Lower values indicate greater preservation of the original ordering.

Paired bootstrap confidence intervals were calculated using 5,000
resamples for the reported comparisons.

## 4. Results

All three algorithms returned valid plans for all 400 benchmark
cases, producing 1,200 valid algorithm-case results.

### Mean order-change cost

| Tasks | Stable repair | Best-search | Fast-search |
|---:|---:|---:|---:|
| 10 | 23.00% | 21.18% | 21.18% |
| 15 | 28.00% | 24.51% | 24.51% |
| 20 | 31.98% | 28.39% | 28.39% |
| 30 | 37.99% | 34.07% | 34.08% |

### Median runtime per plan

| Tasks | Stable repair (ms) | Best-search (ms) | Fast-search (ms) |
|---:|---:|---:|---:|
| 10 | 0.0179 | 1.0460 | 0.8751 |
| 15 | 0.0322 | 10.0384 | 7.2686 |
| 20 | 0.0504 | 37.8859 | 30.6798 |
| 30 | 0.1106 | 261.5291 | 254.1243 |

Runtime measurements are specific to the recorded Colab environment
and can vary with system load.

### Paired comparison: stable repair versus best-search

The mean reduction in order-change cost was:

- 10 tasks: 1.82 percentage points (95% bootstrap CI: 1.22 to 2.51).
- 15 tasks: 3.49 percentage points (95% bootstrap CI: 2.76 to 4.27).
- 20 tasks: 3.59 percentage points (95% bootstrap CI: 3.08 to 4.12).
- 30 tasks: 3.92 percentage points (95% bootstrap CI: 3.42 to 4.48).

These intervals are conditional on the sampled synthetic benchmark
distribution. They do not establish generalization to real-world plans.

## 5. Interpretation

In this benchmark, both search-based methods reduced mean pairwise
order-change cost compared with stable repair. Stable repair was
substantially faster. Fast-search had similar mean quality to
best-search, while its runtime advantage was not consistent across
all individual cases.

The experiments support a limited conclusion: for the tested synthetic
plans, local-search repair can preserve more of the original ordering
while maintaining the precedence constraints checked by the verifier.

## 6. Limitations

1. The plans are synthetic, not generated by real AI agents.
2. Only one benchmark-generation configuration is summarized here.
3. The tested plan sizes range from 10 to 30 tasks.
4. Pairwise order preservation is a proxy for repair quality; it does
   not measure semantic correctness, task usefulness, or execution success.
5. Runtime measurements are environment-dependent.
6. The reported bootstrap intervals do not establish universal
   performance or research novelty.
7. Additional testing is required for cycles, missing tasks, duplicate
   tasks, unknown task identifiers, and diverse constraint structures.
8. A literature review is required before claiming novelty.

## 7. Reproducibility artifacts

- `experiment_config.json`: configuration and environment metadata.
- `benchmark_dataset.json`: frozen input plans.
- `per_case_benchmark_results.csv`: per-plan, per-algorithm measurements.
- `benchmark_summary.csv`: aggregated metrics by task size and algorithm.
- `paired_comparisons.csv`: paired differences and bootstrap intervals.
- `manifest.json`: SHA-256 checksums for the five artifacts above.

The exact source notebook and its dependency versions should also be
preserved with the project before claiming full independent reproduction.

## 8. Next research steps

1. Save and version the exact notebook/source code and dependencies.
2. Test additional random seeds and constraint-generation regimes.
3. Add adversarial cases and explicit verifier/repair unit tests.
4. Compare no verification, verification only, and verification plus repair.
5. Evaluate AI-generated plans and inspect failures.
6. Conduct a literature review before asserting novelty.
"""

report_path.write_text(report.strip() + "\n", encoding="utf-8")

# Include the report and all primary experiment artifacts.
files_to_bundle = [
    "experiment_config.json",
    "benchmark_dataset.json",
    "per_case_benchmark_results.csv",
    "benchmark_summary.csv",
    "paired_comparisons.csv",
    "manifest.json",
    "research_report.md",
]

zip_path = Path("/content/PlanProof_AI_research_artifacts.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in files_to_bundle:
        path = RESULTS_DIR / filename
        if not path.exists():
            raise FileNotFoundError(f"Missing artifact: {path}")
        archive.write(path, arcname=filename)

# Verify the archive can be read and contains every expected artifact.
with zipfile.ZipFile(zip_path, "r") as archive:
    bad_file = archive.testzip()
    assert bad_file is None, f"Corrupt archive member: {bad_file}"
    assert set(archive.namelist()) == set(files_to_bundle)

print("Research report saved:", report_path)
print("ZIP bundle created:", zip_path)
print("Files included:", len(files_to_bundle))
print("ZIP integrity check passed.")
print("Bundle size:", round(zip_path.stat().st_size / 1024, 1), "KB")


Research report saved: /content/planproof_results/research_report.md
ZIP bundle created: /content/PlanProof_AI_research_artifacts.zip
Files included: 7
ZIP integrity check passed.
Bundle size: 90.5 KB


In [ ]:

from google.colab import files
files.download("/content/PlanProof_AI_research_artifacts.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# PlanProof AI — Experiment 17, Step 1
# Inspect verifier behavior on one deliberately invalid plan.

experiment_plan = {
    "task_order": [
        "write_results",
        "run_experiments",
        "submit_report",
    ],
    "required_tasks": [
        "run_experiments",
        "write_results",
        "submit_report",
    ],
    "precedence_constraints": [
        ("run_experiments", "write_results"),
        ("write_results", "submit_report"),
    ],
}

print("INPUT PLAN")
print(experiment_plan)

print("\nVERIFICATION RESULT")
verification_result = validate_plan(experiment_plan)
print(verification_result)

print("\nREPAIR RESULT")
repair_result = repair_plan(experiment_plan)
print(repair_result)


INPUT PLAN
{'task_order': ['write_results', 'run_experiments', 'submit_report'], 'required_tasks': ['run_experiments', 'write_results', 'submit_report'], 'precedence_constraints': [('run_experiments', 'write_results'), ('write_results', 'submit_report')]}

VERIFICATION RESULT
{'valid': False, 'violations': [{'type': 'precedence_violation', 'message': 'run_experiments must occur before write_results.'}]}

REPAIR RESULT
{'success': True, 'repaired_plan': {'task_order': ['run_experiments', 'write_results', 'submit_report'], 'required_tasks': ['run_experiments', 'write_results', 'submit_report'], 'precedence_constraints': [('run_experiments', 'write_results'), ('write_results', 'submit_report')]}, 'message': 'Repaired plan passed verification.'}


In [ ]:

# PlanProof AI — Experiment 17, Step 2
# Compare no verification, verification only, and verification + repair.

import random
import pandas as pd

EXPERIMENT_SEED = 20261017
CASES_PER_SIZE = 100
EXPERIMENT_TASK_SIZES = (10, 15, 20, 30)
EDGE_PROBABILITY = 0.15

experiment_cases = []

for n in EXPERIMENT_TASK_SIZES:
    for case_index in range(CASES_PER_SIZE):
        seed = EXPERIMENT_SEED + n * 100_000 + case_index
        rng = random.Random(seed)

        # Generate a valid dependency structure.
        case_plan = generate_benchmark_case(
            rng,
            task_count=n,
            edge_probability=EDGE_PROBABILITY,
        )

        # Deliberately reverse the generated order.
        # This creates an identical, reproducible challenge for all conditions.
        original_order = list(reversed(case_plan["task_order"]))

        # Keep the same tasks and constraints in every condition.
        challenge_plan = {
            **case_plan,
            "task_order": original_order,
        }

        experiment_cases.append({
            "case_id": f"exp17_n{n}_{case_index:03d}",
            "task_count": n,
            "seed": seed,
            "plan": challenge_plan,
        })

print("Generated cases:", len(experiment_cases))
print("Task sizes:", EXPERIMENT_TASK_SIZES)

# Check the baseline validity rate.
initial_valid = sum(
    validate_plan(case["plan"])["valid"]
    for case in experiment_cases
)

print("Initially valid:", initial_valid)
print(
    "Initial validity rate:",
    round(100 * initial_valid / len(experiment_cases), 2),
    "%"
)

# Save the exact experiment inputs.
exp17_dataset_path = RESULTS_DIR / "experiment17_dataset.json"

with open(exp17_dataset_path, "w") as f:
    json.dump(experiment_cases, f, indent=2)

print("Dataset saved to:", exp17_dataset_path)


Generated cases: 400
Task sizes: (10, 15, 20, 30)
Initially valid: 6
Initial validity rate: 1.5 %
Dataset saved to: /content/planproof_results/experiment17_dataset.json


In [ ]:

# PlanProof AI — Experiment 17, Step 3
# Conditions:
# A: No verification/action; keep original plan
# B: Verify and report; do not modify plan
# C: Verify, repair if needed, and verify again

import pandas as pd
import json

experiment17_rows = []

for case in experiment_cases:
    plan = case["plan"]
    initial_check = validate_plan(plan)
    initially_valid = initial_check["valid"]

    # A — No verification: accept the original plan unchanged.
    condition_a_plan = plan
    condition_a_valid = validate_plan(condition_a_plan)["valid"]

    experiment17_rows.append({
        "case_id": case["case_id"],
        "task_count": case["task_count"],
        "condition": "A_no_verification",
        "initial_valid": initially_valid,
        "final_valid": condition_a_valid,
        "violations_before": len(initial_check["violations"]),
        "violations_after": len(
            validate_plan(condition_a_plan)["violations"]
        ),
        "repair_success": None,
        "order_change_rate": 0.0,
    })

    # B — Verification only: report violations, do not modify.
    condition_b_check = validate_plan(plan)

    experiment17_rows.append({
        "case_id": case["case_id"],
        "task_count": case["task_count"],
        "condition": "B_verification_only",
        "initial_valid": initially_valid,
        "final_valid": condition_b_check["valid"],
        "violations_before": len(initial_check["violations"]),
        "violations_after": len(condition_b_check["violations"]),
        "repair_success": None,
        "order_change_rate": 0.0,
    })

    # C — Verification + repair.
    if initially_valid:
        # Preserve already-valid plans unchanged.
        repaired_plan = plan
        repair_success = True
    else:
        repair_result = repair_plan(plan)
        repair_success = repair_result.get("success", False)
        repaired_plan = repair_result.get("repaired_plan", {})

    if isinstance(repaired_plan, dict) and repaired_plan.get("task_order") is not None:
        final_check = validate_plan(repaired_plan)
        final_valid = final_check["valid"]

        change_rate = pairwise_order_change_rate(
            plan["task_order"],
            repaired_plan["task_order"],
        ) if len(plan["task_order"]) > 1 else 0.0

        violations_after = len(final_check["violations"])
    else:
        final_valid = False
        change_rate = None
        violations_after = None

    experiment17_rows.append({
        "case_id": case["case_id"],
        "task_count": case["task_count"],
        "condition": "C_verification_plus_repair",
        "initial_valid": initially_valid,
        "final_valid": final_valid,
        "violations_before": len(initial_check["violations"]),
        "violations_after": violations_after,
        "repair_success": repair_success,
        "order_change_rate": change_rate,
    })

experiment17_df = pd.DataFrame(experiment17_rows)

# Save per-case outcomes.
exp17_results_path = RESULTS_DIR / "experiment17_per_case.csv"
experiment17_df.to_csv(exp17_results_path, index=False)

print("Experiment 17 completed.")
print("Cases:", len(experiment_cases))
print("Condition-level rows:", len(experiment17_df))
print("Saved to:", exp17_results_path)

print("\nOVERALL CONDITION SUMMARY")

summary17 = (
    experiment17_df.groupby("condition", as_index=False)
    .agg(
        cases=("case_id", "count"),
        initially_valid_pct=("initial_valid", lambda x: 100 * x.mean()),
        final_valid_pct=("final_valid", lambda x: 100 * x.mean()),
        mean_violations_before=("violations_before", "mean"),
        mean_violations_after=("violations_after", "mean"),
    )
)

print(summary17.round(3).to_string(index=False))

print("\nREPAIR SUCCESS RATE")
repair_rows = experiment17_df[
    experiment17_df["condition"] == "C_verification_plus_repair"
]
print(
    "Successful repairs:",
    int(repair_rows["repair_success"].fillna(False).sum()),
    "/",
    len(repair_rows),
)


Experiment 17 completed.
Cases: 400
Condition-level rows: 1200
Saved to: /content/planproof_results/experiment17_per_case.csv

OVERALL CONDITION SUMMARY
                 condition  cases  initially_valid_pct  final_valid_pct  mean_violations_before  mean_violations_after
         A_no_verification    400                  1.5              1.5                  14.358                 14.358
       B_verification_only    400                  1.5              1.5                  14.358                 14.358
C_verification_plus_repair    400                  1.5            100.0                  14.358                  0.000

REPAIR SUCCESS RATE
Successful repairs: 400 / 400


/tmp/ipykernel_2292/1091934089.py:119: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  int(repair_rows["repair_success"].fillna(False).sum()),


In [ ]:

# PlanProof AI — Experiment 17, Step 4
# Analyze how much repair changes the original order.

repair_quality = repair_rows.copy()

repair_quality["order_change_pct"] = (
    repair_quality["order_change_rate"] * 100
)

print("REPAIR QUALITY SUMMARY")
print(
    repair_quality.groupby("task_count")["order_change_pct"]
    .agg(
        cases="count",
        mean_change_pct="mean",
        median_change_pct="median",
        p95_change_pct=lambda x: x.quantile(0.95),
    )
    .round(2)
    .to_string()
)

print("\nOVERALL REPAIR QUALITY")
print(
    repair_quality["order_change_pct"]
    .agg(["mean", "median", "min", "max"])
    .round(2)
    .to_string()
)

quality_path = RESULTS_DIR / "experiment17_repair_quality.csv"
repair_quality.to_csv(quality_path, index=False)

print("\nSaved to:", quality_path)


REPAIR QUALITY SUMMARY
            cases  mean_change_pct  median_change_pct  p95_change_pct
task_count                                                           
10            100            20.42              17.78           44.56
15            100            28.34              28.57           46.71
20            100            31.51              30.53           44.79
30            100            36.76              37.13           49.48

OVERALL REPAIR QUALITY
mean      29.26
median    30.53
min        0.00
max       57.47

Saved to: /content/planproof_results/experiment17_repair_quality.csv


In [ ]:

# PlanProof AI — Experiment 17, Step 5
# Compare all three repair methods on the same challenge plans.

import time
import pandas as pd

method_rows = []

methods = {
    "stable": repair_plan,
    "best_search": local_search_repair,
    "fast_search": fast_local_search_repair,
}

for case in experiment_cases:
    plan = case["plan"]

    for method_name, method in methods.items():
        start = time.perf_counter()
        result = method(plan)
        elapsed_ms = (time.perf_counter() - start) * 1000

        repaired = result.get("repaired_plan", {})
        success = result.get("success", False)

        valid = (
            success
            and isinstance(repaired, dict)
            and validate_plan(repaired).get("valid", False)
        )

        if valid:
            change_rate = pairwise_order_change_rate(
                plan["task_order"],
                repaired["task_order"],
            )
        else:
            change_rate = None

        method_rows.append({
            "case_id": case["case_id"],
            "task_count": case["task_count"],
            "method": method_name,
            "success": success,
            "valid": bool(valid),
            "order_change_pct": (
                100 * change_rate if change_rate is not None else None
            ),
            "runtime_ms": elapsed_ms,
        })

method_df = pd.DataFrame(method_rows)

method_summary = (
    method_df.groupby(["task_count", "method"], as_index=False)
    .agg(
        cases=("case_id", "count"),
        valid_cases=("valid", "sum"),
        mean_order_change_pct=("order_change_pct", "mean"),
        median_order_change_pct=("order_change_pct", "median"),
        median_runtime_ms=("runtime_ms", "median"),
    )
)

method_summary["validity_pct"] = (
    100 * method_summary["valid_cases"] / method_summary["cases"]
)

display(
    method_summary[
        [
            "task_count", "method", "cases", "validity_pct",
            "mean_order_change_pct", "median_runtime_ms"
        ]
    ].round(3)
)

method_results_path = RESULTS_DIR / "experiment17_method_comparison.csv"
method_df.to_csv(method_results_path, index=False)

method_summary_path = RESULTS_DIR / "experiment17_method_summary.csv"
method_summary.to_csv(method_summary_path, index=False)

print("Per-case results saved to:", method_results_path)
print("Summary saved to:", method_summary_path)


,task_count,method,cases,validity_pct,mean_order_change_pct,median_runtime_ms
0,10,best_search,100,100.0,18.867,1.063
1,10,fast_search,100,100.0,18.867,0.885
2,10,stable,100,100.0,20.422,0.024
3,15,best_search,100,100.0,25.048,9.389
4,15,fast_search,100,100.0,25.048,6.385
5,15,stable,100,100.0,28.343,0.041
6,20,best_search,100,100.0,27.684,38.143
7,20,fast_search,100,100.0,27.689,37.723
8,20,stable,100,100.0,31.505,0.065
9,30,best_search,100,100.0,32.841,236.897


Per-case results saved to: /content/planproof_results/experiment17_method_comparison.csv
Summary saved to: /content/planproof_results/experiment17_method_summary.csv


In [ ]:

# PlanProof AI — Experiment 18, Step 1
# Define planning tasks for evaluation with an AI model.

ai_planning_tasks = [
    {
        "task_id": "P01",
        "domain": "Research",
        "prompt": (
            "Create a plan for conducting a small research project, "
            "from reviewing literature to submitting a final report."
        ),
        "required_tasks": [
            "review_literature",
            "design_experiment",
            "run_experiment",
            "analyze_results",
            "write_report",
            "submit_report",
        ],
        "precedence_constraints": [
            ("review_literature", "design_experiment"),
            ("design_experiment", "run_experiment"),
            ("run_experiment", "analyze_results"),
            ("analyze_results", "write_report"),
            ("write_report", "submit_report"),
        ],
    },
    {
        "task_id": "P02",
        "domain": "Software Development",
        "prompt": (
            "Create a plan to develop, test, and release a small software "
            "application."
        ),
        "required_tasks": [
            "gather_requirements",
            "design_system",
            "implement_features",
            "run_tests",
            "fix_bugs",
            "deploy_application",
        ],
        "precedence_constraints": [
            ("gather_requirements", "design_system"),
            ("design_system", "implement_features"),
            ("implement_features", "run_tests"),
            ("run_tests", "fix_bugs"),
            ("fix_bugs", "deploy_application"),
        ],
    },
    {
        "task_id": "P03",
        "domain": "Education",
        "prompt": (
            "Create a plan for preparing for an exam, from assessing "
            "the syllabus to taking a practice test."
        ),
        "required_tasks": [
            "review_syllabus",
            "identify_weak_topics",
            "study_topics",
            "solve_practice_questions",
            "review_mistakes",
            "take_mock_test",
        ],
        "precedence_constraints": [
            ("review_syllabus", "identify_weak_topics"),
            ("identify_weak_topics", "study_topics"),
            ("study_topics", "solve_practice_questions"),
            ("solve_practice_questions", "review_mistakes"),
            ("review_mistakes", "take_mock_test"),
        ],
    },
    {
        "task_id": "P04",
        "domain": "Event Planning",
        "prompt": (
            "Create a plan for organizing a small technical workshop."
        ),
        "required_tasks": [
            "define_event_goals",
            "select_venue",
            "confirm_speakers",
            "publish_registration",
            "prepare_materials",
            "host_event",
        ],
        "precedence_constraints": [
            ("define_event_goals", "select_venue"),
            ("define_event_goals", "confirm_speakers"),
            ("select_venue", "publish_registration"),
            ("confirm_speakers", "publish_registration"),
            ("publish_registration", "prepare_materials"),
            ("prepare_materials", "host_event"),
        ],
    },
    {
        "task_id": "P05",
        "domain": "Data Science",
        "prompt": (
            "Create a plan for building and evaluating a machine-learning "
            "model using a dataset."
        ),
        "required_tasks": [
            "inspect_dataset",
            "clean_data",
            "split_dataset",
            "train_model",
            "evaluate_model",
            "document_results",
        ],
        "precedence_constraints": [
            ("inspect_dataset", "clean_data"),
            ("clean_data", "split_dataset"),
            ("split_dataset", "train_model"),
            ("train_model", "evaluate_model"),
            ("evaluate_model", "document_results"),
        ],
    },
]

print("Planning tasks prepared:", len(ai_planning_tasks))

for task in ai_planning_tasks:
    print(
        f"{task['task_id']} | {task['domain']} | "
        f"{len(task['required_tasks'])} required tasks | "
        f"{len(task['precedence_constraints'])} constraints"
    )


Planning tasks prepared: 5
P01 | Research | 6 required tasks | 5 constraints
P02 | Software Development | 6 required tasks | 5 constraints
P03 | Education | 6 required tasks | 5 constraints
P04 | Event Planning | 6 required tasks | 6 constraints
P05 | Data Science | 6 required tasks | 5 constraints


In [ ]:

import json

task = planning_tasks[0]

prompt = f"""
You are an AI planning assistant.

Task:
{task['prompt']}

Required task IDs:
{json.dumps(task['required_tasks'], indent=2)}

Precedence constraints:
{json.dumps(task['precedence_constraints'], indent=2)}

Create a practical execution plan.

Return ONLY a valid JSON object with these fields:
{{
  "task_order": ["task_id_1", "task_id_2"]
}}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Do not include explanations or Markdown.
"""

print(prompt)


NameError: name 'planning_tasks' is not defined

In [ ]:

planning_tasks = [
    {
        "id": "P01",
        "domain": "Research",
        "prompt": "Plan a research project from literature review to final submission.",
        "required_tasks": [
            "review_literature",
            "design_experiment",
            "run_experiment",
            "analyze_results",
            "write_report",
            "submit_report"
        ],
        "precedence_constraints": [
            ["review_literature", "design_experiment"],
            ["design_experiment", "run_experiment"],
            ["run_experiment", "analyze_results"],
            ["analyze_results", "write_report"],
            ["write_report", "submit_report"]
        ]
    },
    {
        "id": "P02",
        "domain": "Software Development",
        "prompt": "Plan a software development project from requirements to deployment.",
        "required_tasks": [
            "gather_requirements",
            "design_system",
            "implement_features",
            "run_tests",
            "fix_bugs",
            "deploy_application"
        ],
        "precedence_constraints": [
            ["gather_requirements", "design_system"],
            ["design_system", "implement_features"],
            ["implement_features", "run_tests"],
            ["run_tests", "fix_bugs"],
            ["fix_bugs", "deploy_application"]
        ]
    },
    {
        "id": "P03",
        "domain": "Education",
        "prompt": "Plan a learning process from syllabus review to a mock test.",
        "required_tasks": [
            "review_syllabus",
            "identify_weak_topics",
            "study_topics",
            "solve_practice_questions",
            "review_mistakes",
            "take_mock_test"
        ],
        "precedence_constraints": [
            ["review_syllabus", "identify_weak_topics"],
            ["identify_weak_topics", "study_topics"],
            ["study_topics", "solve_practice_questions"],
            ["solve_practice_questions", "review_mistakes"],
            ["review_mistakes", "take_mock_test"]
        ]
    },
    {
        "id": "P04",
        "domain": "Event Planning",
        "prompt": "Plan an event from defining its goals to hosting it.",
        "required_tasks": [
            "define_event_goals",
            "select_venue",
            "confirm_speakers",
            "publish_registration",
            "prepare_materials",
            "host_event"
        ],
        "precedence_constraints": [
            ["define_event_goals", "select_venue"],
            ["define_event_goals", "confirm_speakers"],
            ["select_venue", "publish_registration"],
            ["confirm_speakers", "publish_registration"],
            ["publish_registration", "prepare_materials"],
            ["prepare_materials", "host_event"]
        ]
    },
    {
        "id": "P05",
        "domain": "Data Science",
        "prompt": "Plan a data science project from dataset inspection to documentation.",
        "required_tasks": [
            "inspect_dataset",
            "clean_data",
            "split_dataset",
            "train_model",
            "evaluate_model",
            "document_results"
        ],
        "precedence_constraints": [
            ["inspect_dataset", "clean_data"],
            ["clean_data", "split_dataset"],
            ["split_dataset", "train_model"],
            ["train_model", "evaluate_model"],
            ["evaluate_model", "document_results"]
        ]
    }
]

print("Planning tasks prepared:", len(planning_tasks))

for task in planning_tasks:
    print(
        task["id"], "|",
        task["domain"], "|",
        len(task["required_tasks"]), "required tasks |",
        len(task["precedence_constraints"]), "constraints"
    )


Planning tasks prepared: 5
P01 | Research | 6 required tasks | 5 constraints
P02 | Software Development | 6 required tasks | 5 constraints
P03 | Education | 6 required tasks | 5 constraints
P04 | Event Planning | 6 required tasks | 6 constraints
P05 | Data Science | 6 required tasks | 5 constraints


In [ ]:

import json

task = planning_tasks[0]

prompt = f"""
You are an AI planning assistant.

Task:
{task['prompt']}

Required task IDs:
{json.dumps(task['required_tasks'], indent=2)}

Precedence constraints:
{json.dumps(task['precedence_constraints'], indent=2)}

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{{
  "task_order": ["task_id_1", "task_id_2"]
}}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.
"""

print(prompt)



You are an AI planning assistant.

Task:
Plan a research project from literature review to final submission.

Required task IDs:
[
  "review_literature",
  "design_experiment",
  "run_experiment",
  "analyze_results",
  "write_report",
  "submit_report"
]

Precedence constraints:
[
  [
    "review_literature",
    "design_experiment"
  ],
  [
    "design_experiment",
    "run_experiment"
  ],
  [
    "run_experiment",
    "analyze_results"
  ],
  [
    "analyze_results",
    "write_report"
  ],
  [
    "write_report",
    "submit_report"
  ]
]

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{
  "task_order": ["task_id_1", "task_id_2"]
}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.



In [ ]:

{
  "task_order": [
    "review_literature",
    "design_experiment",
    "run_experiment",
    "analyze_results",
    "write_report",
    "submit_report"
  ]
}


{'task_order': ['review_literature',
  'design_experiment',
  'run_experiment',
  'analyze_results',
  'write_report',
  'submit_report']}

In [ ]:

task = planning_tasks[1]

prompt = f"""
You are an AI planning assistant.

Task:
{task['prompt']}

Required task IDs:
{json.dumps(task['required_tasks'], indent=2)}

Precedence constraints:
{json.dumps(task['precedence_constraints'], indent=2)}

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{{
  "task_order": ["task_id_1", "task_id_2"]
}}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.
"""

print(prompt)



You are an AI planning assistant.

Task:
Plan a software development project from requirements to deployment.

Required task IDs:
[
  "gather_requirements",
  "design_system",
  "implement_features",
  "run_tests",
  "fix_bugs",
  "deploy_application"
]

Precedence constraints:
[
  [
    "gather_requirements",
    "design_system"
  ],
  [
    "design_system",
    "implement_features"
  ],
  [
    "implement_features",
    "run_tests"
  ],
  [
    "run_tests",
    "fix_bugs"
  ],
  [
    "fix_bugs",
    "deploy_application"
  ]
]

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{
  "task_order": ["task_id_1", "task_id_2"]
}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.



In [ ]:

{
  "task_order": [
    "gather_requirements",
    "design_system",
    "implement_features",
    "run_tests",
    "fix_bugs",
    "deploy_application"
  ]
}


{'task_order': ['gather_requirements',
  'design_system',
  'implement_features',
  'run_tests',
  'fix_bugs',
  'deploy_application']}

In [ ]:

task = planning_tasks[2]

prompt = f"""
You are an AI planning assistant.

Task:
{task['prompt']}

Required task IDs:
{json.dumps(task['required_tasks'], indent=2)}

Precedence constraints:
{json.dumps(task['precedence_constraints'], indent=2)}

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{{
  "task_order": ["task_id_1", "task_id_2"]
}}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.
"""

print(prompt)



You are an AI planning assistant.

Task:
Plan a learning process from syllabus review to a mock test.

Required task IDs:
[
  "review_syllabus",
  "identify_weak_topics",
  "study_topics",
  "solve_practice_questions",
  "review_mistakes",
  "take_mock_test"
]

Precedence constraints:
[
  [
    "review_syllabus",
    "identify_weak_topics"
  ],
  [
    "identify_weak_topics",
    "study_topics"
  ],
  [
    "study_topics",
    "solve_practice_questions"
  ],
  [
    "solve_practice_questions",
    "review_mistakes"
  ],
  [
    "review_mistakes",
    "take_mock_test"
  ]
]

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{
  "task_order": ["task_id_1", "task_id_2"]
}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.



In [ ]:

{
  "task_order": [
    "review_syllabus",
    "identify_weak_topics",
    "study_topics",
    "solve_practice_questions",
    "review_mistakes",
    "take_mock_test"
  ]
}


{'task_order': ['review_syllabus',
  'identify_weak_topics',
  'study_topics',
  'solve_practice_questions',
  'review_mistakes',
  'take_mock_test']}

In [ ]:

task = planning_tasks[3]

prompt = f"""
You are an AI planning assistant.

Task:
{task['prompt']}

Required task IDs:
{json.dumps(task['required_tasks'], indent=2)}

Precedence constraints:
{json.dumps(task['precedence_constraints'], indent=2)}

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{{
  "task_order": ["task_id_1", "task_id_2"]
}}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.
"""

print(prompt)



You are an AI planning assistant.

Task:
Plan an event from defining its goals to hosting it.

Required task IDs:
[
  "define_event_goals",
  "select_venue",
  "confirm_speakers",
  "publish_registration",
  "prepare_materials",
  "host_event"
]

Precedence constraints:
[
  [
    "define_event_goals",
    "select_venue"
  ],
  [
    "define_event_goals",
    "confirm_speakers"
  ],
  [
    "select_venue",
    "publish_registration"
  ],
  [
    "confirm_speakers",
    "publish_registration"
  ],
  [
    "publish_registration",
    "prepare_materials"
  ],
  [
    "prepare_materials",
    "host_event"
  ]
]

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{
  "task_order": ["task_id_1", "task_id_2"]
}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.



In [ ]:

{
  "task_order": [
    "define_event_goals",
    "confirm_speakers",
    "select_venue",
    "publish_registration",
    "prepare_materials",
    "host_event"
  ]
}


{'task_order': ['define_event_goals',
  'confirm_speakers',
  'select_venue',
  'publish_registration',
  'prepare_materials',
  'host_event']}

In [ ]:

task = planning_tasks[4]

prompt = f"""
You are an AI planning assistant.

Task:
{task['prompt']}

Required task IDs:
{json.dumps(task['required_tasks'], indent=2)}

Precedence constraints:
{json.dumps(task['precedence_constraints'], indent=2)}

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{{
  "task_order": ["task_id_1", "task_id_2"]
}}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.
"""

print(prompt)



You are an AI planning assistant.

Task:
Plan a data science project from dataset inspection to documentation.

Required task IDs:
[
  "inspect_dataset",
  "clean_data",
  "split_dataset",
  "train_model",
  "evaluate_model",
  "document_results"
]

Precedence constraints:
[
  [
    "inspect_dataset",
    "clean_data"
  ],
  [
    "clean_data",
    "split_dataset"
  ],
  [
    "split_dataset",
    "train_model"
  ],
  [
    "train_model",
    "evaluate_model"
  ],
  [
    "evaluate_model",
    "document_results"
  ]
]

Create a practical execution plan.

Return ONLY a valid JSON object with this field:
{
  "task_order": ["task_id_1", "task_id_2"]
}

Rules:
- Use only the required task IDs.
- Include every required task exactly once.
- Try to satisfy all precedence constraints.
- Return no Markdown or additional explanations.



In [ ]:

{
  "task_order": [
    "inspect_dataset",
    "clean_data",
    "split_dataset",
    "train_model",
    "evaluate_model",
    "document_results"
  ]
}



{'task_order': ['inspect_dataset',
  'clean_data',
  'split_dataset',
  'train_model',
  'evaluate_model',
  'document_results']}

In [ ]:

import json
from pathlib import Path

# Four outputs were pasted back as model responses.
# P04 was supplied as an example in our conversation,
# so its provenance is marked honestly.
collected_plans = [
    {
        "id": "P01",
        "source": "pasted_model_response",
        "task_order": [
            "review_literature",
            "design_experiment",
            "run_experiment",
            "analyze_results",
            "write_report",
            "submit_report"
        ]
    },
    {
        "id": "P02",
        "source": "pasted_model_response",
        "task_order": [
            "gather_requirements",
            "design_system",
            "implement_features",
            "run_tests",
            "fix_bugs",
            "deploy_application"
        ]
    },
    {
        "id": "P03",
        "source": "pasted_model_response",
        "task_order": [
            "review_syllabus",
            "identify_weak_topics",
            "study_topics",
            "solve_practice_questions",
            "review_mistakes",
            "take_mock_test"
        ]
    },
    {
        "id": "P04",
        "source": "conversation_example_not_independent_model_sample",
        "task_order": [
            "define_event_goals",
            "confirm_speakers",
            "select_venue",
            "publish_registration",
            "prepare_materials",
            "host_event"
        ]
    },
    {
        "id": "P05",
        "source": "pasted_model_response",
        "task_order": [
            "inspect_dataset",
            "clean_data",
            "split_dataset",
            "train_model",
            "evaluate_model",
            "document_results"
        ]
    }
]

output_dir = Path("/content/planproof_results")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "experiment18_collected_plans.json"
output_file.write_text(
    json.dumps(collected_plans, indent=2),
    encoding="utf-8"
)

print("Saved:", output_file)
print("Plans recorded:", len(collected_plans))
for plan in collected_plans:
    print(plan["id"], "|", plan["source"])


Saved: /content/planproof_results/experiment18_collected_plans.json
Plans recorded: 5
P01 | pasted_model_response
P02 | pasted_model_response
P03 | pasted_model_response
P04 | conversation_example_not_independent_model_sample
P05 | pasted_model_response


In [ ]:

import pandas as pd
from pathlib import Path

output_dir = Path("/content/planproof_results")
task_lookup = {task["id"]: task for task in planning_tasks}
results = []

for item in collected_plans:
    task = task_lookup[item["id"]]

    # Build the complete structure expected by validate_plan()
    plan = {
        "task_order": item["task_order"],
        "required_tasks": task["required_tasks"],
        "precedence_constraints": task["precedence_constraints"]
    }

    validation = validate_plan(plan)
    order = item["task_order"]
    required = task["required_tasks"]

    results.append({
        "plan_id": item["id"],
        "domain": task["domain"],
        "source": item["source"],
        "required_task_count": len(required),
        "present_task_count": sum(t in order for t in required),
        "missing_tasks": len(set(required) - set(order)),
        "duplicate_tasks": len(order) - len(set(order)),
        "invalid_task_ids": len(set(order) - set(required)),
        "precedence_violations": len(validation["violations"]),
        "valid_before_repair": validation["valid"]
    })

results_df = pd.DataFrame(results)
display(results_df)

csv_path = output_dir / "experiment18_initial_evaluation.csv"
results_df.to_csv(csv_path, index=False)

print("Saved evaluation:", csv_path)
print("Plans evaluated:", len(results_df))


,plan_id,domain,source,required_task_count,present_task_count,missing_tasks,duplicate_tasks,invalid_task_ids,precedence_violations,valid_before_repair
0,P01,Research,pasted_model_response,6,6,0,0,0,0,True
1,P02,Software Development,pasted_model_response,6,6,0,0,0,0,True
2,P03,Education,pasted_model_response,6,6,0,0,0,0,True
3,P04,Event Planning,conversation_example_not_independent_model_sample,6,6,0,0,0,0,True
4,P05,Data Science,pasted_model_response,6,6,0,0,0,0,True


Saved evaluation: /content/planproof_results/experiment18_initial_evaluation.csv
Plans evaluated: 5


In [ ]:

import pandas as pd

repair_results = []

repair_methods = {
    "stable": repair_plan,
    "local_search": local_search_repair,
    "fast_local_search": fast_local_search_repair
}

for item in collected_plans:
    task = task_lookup[item["id"]]

    original_plan = {
        "task_order": item["task_order"],
        "required_tasks": task["required_tasks"],
        "precedence_constraints": task["precedence_constraints"]
    }

    original_order = item["task_order"]
    original_validation = validate_plan(original_plan)

    for method_name, method in repair_methods.items():
        # Repair functions expect the plan structure used by this project.
        repaired = method(original_plan)

        if repaired["success"]:
            repaired_plan = repaired["repaired_plan"]
            repaired_order = repaired_plan["task_order"]

            repaired_validation = validate_plan({
                "task_order": repaired_order,
                "required_tasks": task["required_tasks"],
                "precedence_constraints": task["precedence_constraints"]
            })

            change_rate = pairwise_order_change_rate(
                original_order, repaired_order
            )
        else:
            repaired_validation = {"valid": False, "violations": ["repair failed"]}
            change_rate = None

        repair_results.append({
            "plan_id": item["id"],
            "domain": task["domain"],
            "method": method_name,
            "original_valid": original_validation["valid"],
            "repair_success": repaired["success"],
            "repaired_valid": repaired_validation["valid"],
            "pairwise_order_change_rate": change_rate,
            "source": item["source"]
        })

repair_df = pd.DataFrame(repair_results)
display(repair_df)

repair_csv = output_dir / "experiment18_repair_evaluation.csv"
repair_df.to_csv(repair_csv, index=False)
print("Saved:", repair_csv)


,plan_id,domain,method,original_valid,repair_success,repaired_valid,pairwise_order_change_rate,source
0,P01,Research,stable,True,True,True,0.0,pasted_model_response
1,P01,Research,local_search,True,True,True,0.0,pasted_model_response
2,P01,Research,fast_local_search,True,True,True,0.0,pasted_model_response
3,P02,Software Development,stable,True,True,True,0.0,pasted_model_response
4,P02,Software Development,local_search,True,True,True,0.0,pasted_model_response
5,P02,Software Development,fast_local_search,True,True,True,0.0,pasted_model_response
6,P03,Education,stable,True,True,True,0.0,pasted_model_response
7,P03,Education,local_search,True,True,True,0.0,pasted_model_response
8,P03,Education,fast_local_search,True,True,True,0.0,pasted_model_response
9,P04,Event Planning,stable,True,True,True,0.0,conversation_example_not_independent_model_sample


Saved: /content/planproof_results/experiment18_repair_evaluation.csv


In [ ]:

import copy
import pandas as pd

challenge_results = []

for item in collected_plans:
    task = task_lookup[item["id"]]
    original_order = item["task_order"]

    # Create a deliberately invalid order by reversing the plan.
    # Keep the original collected plan unchanged.
    challenge_order = list(reversed(original_order))

    challenge_plan = {
        "task_order": challenge_order,
        "required_tasks": task["required_tasks"],
        "precedence_constraints": task["precedence_constraints"]
    }

    before = validate_plan(challenge_plan)

    repaired = repair_plan(challenge_plan)

    if repaired["success"]:
        repaired_order = repaired["repaired_plan"]["task_order"]

        after = validate_plan({
            "task_order": repaired_order,
            "required_tasks": task["required_tasks"],
            "precedence_constraints": task["precedence_constraints"]
        })
    else:
        after = {"valid": False, "violations": ["repair failed"]}

    challenge_results.append({
        "plan_id": item["id"],
        "domain": task["domain"],
        "valid_before": before["valid"],
        "violations_before": len(before["violations"]),
        "repair_success": repaired["success"],
        "valid_after": after["valid"],
        "violations_after": len(after["violations"])
    })

challenge_df = pd.DataFrame(challenge_results)
display(challenge_df)

challenge_path = output_dir / "experiment18_controlled_challenges.csv"
challenge_df.to_csv(challenge_path, index=False)
print("Saved:", challenge_path)


,plan_id,domain,valid_before,violations_before,repair_success,valid_after,violations_after
0,P01,Research,False,5,True,True,0
1,P02,Software Development,False,5,True,True,0
2,P03,Education,False,5,True,True,0
3,P04,Event Planning,False,6,True,True,0
4,P05,Data Science,False,5,True,True,0


Saved: /content/planproof_results/experiment18_controlled_challenges.csv


In [ ]:

from pathlib import Path
import json
import pandas as pd
from datetime import datetime, timezone

output_dir = Path("/content/planproof_results")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the current result tables.
results_df.to_csv(
    output_dir / "experiment18_initial_evaluation.csv",
    index=False
)
repair_df.to_csv(
    output_dir / "experiment18_repair_evaluation.csv",
    index=False
)
challenge_df.to_csv(
    output_dir / "experiment18_controlled_challenges.csv",
    index=False
)

report = f"""# PlanProof AI — Experiment 18

## Objective
Evaluate five planning examples, test whether the repair methods preserve
valid plans, and test repair on deliberately reversed task orders.

## Data provenance
Four plans were recorded as pasted model responses. P04 was supplied as a
conversation example and is not an independent model sample.

## Initial validation
- Plans evaluated: {len(results_df)}
- Initially valid: {int(results_df["valid_before_repair"].sum())}
- Missing tasks: {int(results_df["missing_tasks"].sum())}
- Duplicate task IDs: {int(results_df["duplicate_tasks"].sum())}
- Invalid task IDs: {int(results_df["invalid_task_ids"].sum())}
- Precedence violations: {int(results_df["precedence_violations"].sum())}

## Repair of initially valid plans
- Repair attempts: {len(repair_df)}
- Successful repairs: {int(repair_df["repair_success"].sum())}
- Valid after repair: {int(repair_df["repaired_valid"].sum())}
- Mean pairwise order-change rate: {repair_df["pairwise_order_change_rate"].mean():.4f}

## Controlled reversed-plan challenge
- Challenge cases: {len(challenge_df)}
- Invalid before repair: {int((~challenge_df["valid_before"]).sum())}
- Repair successes: {int(challenge_df["repair_success"].sum())}
- Valid after repair: {int(challenge_df["valid_after"].sum())}
- Violations before: {int(challenge_df["violations_before"].sum())}
- Violations after: {int(challenge_df["violations_after"].sum())}

## Interpretation
The validator accepted all five collected task orders. The three repair
methods preserved their order in the valid-plan test. On five deliberately
reversed challenge cases, the stable repair method succeeded on all cases
and reduced the observed precedence violations to zero.

These results demonstrate behavior on a small, controlled pilot only.
They do not establish real-world reliability, generalization to other
planning domains, or superiority over other systems. The reversed plans
were manually constructed challenge cases, not observed model failures.
More independent model outputs and varied failure types are needed.

## Reproducibility
Generated at (UTC): {datetime.now(timezone.utc).isoformat()}
"""

report_path = output_dir / "experiment18_report.md"
report_path.write_text(report, encoding="utf-8")

manifest = {
    "experiment": "Experiment 18",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "case_count": 5,
    "model_sample_count_claimed": 4,
    "notes": [
        "P04 is a conversation example, not an independent model sample.",
        "Controlled reversed plans were manually constructed.",
        "Results are pilot-scale."
    ],
    "files": [
        "experiment18_collected_plans.json",
        "experiment18_initial_evaluation.csv",
        "experiment18_repair_evaluation.csv",
        "experiment18_controlled_challenges.csv",
        "experiment18_report.md"
    ]
}

(output_dir / "experiment18_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print(report)
print("\nSaved report:", report_path)
print("Saved manifest:", output_dir / "experiment18_manifest.json")


# PlanProof AI — Experiment 18

## Objective
Evaluate five planning examples, test whether the repair methods preserve
valid plans, and test repair on deliberately reversed task orders.

## Data provenance
Four plans were recorded as pasted model responses. P04 was supplied as a
conversation example and is not an independent model sample.

## Initial validation
- Plans evaluated: 5
- Initially valid: 5
- Missing tasks: 0
- Duplicate task IDs: 0
- Invalid task IDs: 0
- Precedence violations: 0

## Repair of initially valid plans
- Repair attempts: 15
- Successful repairs: 15
- Valid after repair: 15
- Mean pairwise order-change rate: 0.0000

## Controlled reversed-plan challenge
- Challenge cases: 5
- Invalid before repair: 5
- Repair successes: 5
- Valid after repair: 5
- Violations before: 26
- Violations after: 0

## Interpretation
The validator accepted all five collected task orders. The three repair
methods preserved their order in the valid-plan test. On five deliberately
reverse

In [ ]:

from pathlib import Path
import zipfile
import hashlib

output_dir = Path("/content/planproof_results")

files_to_package = [
    "experiment18_collected_plans.json",
    "experiment18_initial_evaluation.csv",
    "experiment18_repair_evaluation.csv",
    "experiment18_controlled_challenges.csv",
    "experiment18_report.md",
    "experiment18_manifest.json",
]

zip_path = output_dir / "PlanProof_AI_Experiment18.zip"

missing = [
    name for name in files_to_package
    if not (output_dir / name).exists()
]

if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in files_to_package:
        archive.write(output_dir / name, arcname=name)

# Verify archive integrity
with zipfile.ZipFile(zip_path, "r") as archive:
    bad_file = archive.testzip()
    if bad_file:
        raise RuntimeError(f"ZIP integrity check failed: {bad_file}")

print("ZIP created and verified:", zip_path)
print("Files included:", len(files_to_package))
print("SHA-256:", hashlib.sha256(zip_path.read_bytes()).hexdigest())


ZIP created and verified: /content/planproof_results/PlanProof_AI_Experiment18.zip
Files included: 6
SHA-256: d0e85e151f24777e4581dff6780e85d63e1500f7032d6126f767ef7c29abbdc5


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone

experiment19_dir = Path("/content/planproof_results/experiment19")
experiment19_dir.mkdir(parents=True, exist_ok=True)

experiment19_config = {
    "experiment_id": "experiment19",
    "title": "Independent AI Planning Evaluation",
    "version": "1.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "target_domains": [
        "Research",
        "Software Development",
        "Education",
        "Data Science"
    ],
    "difficulty_levels": [
        "easy",
        "medium",
        "hard"
    ],
    "tasks_per_domain": 3,
    "independent_generations_per_task": 3,
    "target_plan_count": 36,
    "evaluation_metrics": [
        "missing_tasks",
        "duplicate_tasks",
        "invalid_task_ids",
        "precedence_violations",
        "valid_before_repair",
        "repair_success",
        "valid_after_repair",
        "pairwise_order_change_rate"
    ],
    "provenance_policy": (
        "Store actual model responses unchanged. Record model identity, "
        "prompt, generation settings when available, and collection time. "
        "Do not count manually authored examples as model outputs."
    ),
    "limitations": [
        "Results depend on the selected models, prompts, and benchmark.",
        "Synthetic benchmark constraints do not establish real-world validity.",
        "Do not claim model failures before observing actual outputs."
    ]
}

config_path = experiment19_dir / "experiment19_config.json"
config_path.write_text(
    json.dumps(experiment19_config, indent=2),
    encoding="utf-8"
)

print("Experiment 19 initialized.")
print("Configuration:", config_path)
print("Target domains:", len(experiment19_config["target_domains"]))
print("Target independent plans:", experiment19_config["target_plan_count"])
print("Experiment 18 remains unchanged.")


Experiment 19 initialized.
Configuration: /content/planproof_results/experiment19/experiment19_config.json
Target domains: 4
Target independent plans: 36
Experiment 18 remains unchanged.


In [ ]:

import json
from pathlib import Path

benchmark_tasks = [
    # RESEARCH
    {
        "task_id": "R_EASY",
        "domain": "Research",
        "difficulty": "easy",
        "prompt": "Plan a small research project from reading literature to submitting a report.",
        "required_tasks": [
            "review_literature", "define_question", "design_study",
            "collect_data", "analyze_data", "write_report"
        ],
        "precedence_constraints": [
            ["review_literature", "define_question"],
            ["define_question", "design_study"],
            ["design_study", "collect_data"],
            ["collect_data", "analyze_data"],
            ["analyze_data", "write_report"]
        ]
    },
    {
        "task_id": "R_MEDIUM",
        "domain": "Research",
        "difficulty": "medium",
        "prompt": "Plan a research study with ethics approval, pilot testing, data collection, analysis, and publication.",
        "required_tasks": [
            "review_literature", "formulate_hypothesis", "design_protocol",
            "obtain_ethics_approval", "pilot_study", "collect_data",
            "analyze_data", "write_paper"
        ],
        "precedence_constraints": [
            ["review_literature", "formulate_hypothesis"],
            ["formulate_hypothesis", "design_protocol"],
            ["design_protocol", "obtain_ethics_approval"],
            ["obtain_ethics_approval", "pilot_study"],
            ["pilot_study", "collect_data"],
            ["collect_data", "analyze_data"],
            ["analyze_data", "write_paper"]
        ]
    },
    {
        "task_id": "R_HARD",
        "domain": "Research",
        "difficulty": "hard",
        "prompt": "Plan a multi-stage research project with parallel literature review and dataset preparation, ethics approval before participant recruitment, pilot testing, analysis, replication, and publication.",
        "required_tasks": [
            "review_literature", "prepare_dataset", "design_protocol",
            "obtain_ethics_approval", "recruit_participants", "pilot_study",
            "collect_data", "analyze_data", "replicate_results", "write_paper"
        ],
        "precedence_constraints": [
            ["review_literature", "design_protocol"],
            ["prepare_dataset", "design_protocol"],
            ["design_protocol", "obtain_ethics_approval"],
            ["obtain_ethics_approval", "recruit_participants"],
            ["recruit_participants", "pilot_study"],
            ["pilot_study", "collect_data"],
            ["collect_data", "analyze_data"],
            ["analyze_data", "replicate_results"],
            ["replicate_results", "write_paper"]
        ]
    },

    # SOFTWARE DEVELOPMENT
    {
        "task_id": "S_EASY",
        "domain": "Software Development",
        "difficulty": "easy",
        "prompt": "Plan a small software feature from requirements to release.",
        "required_tasks": [
            "gather_requirements", "design_solution", "implement_feature",
            "test_feature", "fix_defects", "release_feature"
        ],
        "precedence_constraints": [
            ["gather_requirements", "design_solution"],
            ["design_solution", "implement_feature"],
            ["implement_feature", "test_feature"],
            ["test_feature", "fix_defects"],
            ["fix_defects", "release_feature"]
        ]
    },
    {
        "task_id": "S_MEDIUM",
        "domain": "Software Development",
        "difficulty": "medium",
        "prompt": "Plan a web application release including design, implementation, code review, testing, security checks, and deployment.",
        "required_tasks": [
            "gather_requirements", "design_architecture", "implement_backend",
            "implement_frontend", "review_code", "run_tests",
            "security_check", "deploy_application"
        ],
        "precedence_constraints": [
            ["gather_requirements", "design_architecture"],
            ["design_architecture", "implement_backend"],
            ["design_architecture", "implement_frontend"],
            ["implement_backend", "review_code"],
            ["implement_frontend", "review_code"],
            ["review_code", "run_tests"],
            ["run_tests", "security_check"],
            ["security_check", "deploy_application"]
        ]
    },
    {
        "task_id": "S_HARD",
        "domain": "Software Development",
        "difficulty": "hard",
        "prompt": "Plan a production release with parallel frontend and backend work, integration, testing, security approval, staging, and deployment.",
        "required_tasks": [
            "define_requirements", "design_api", "build_backend",
            "build_frontend", "integrate_components", "unit_tests",
            "integration_tests", "security_review", "staging_deployment",
            "production_deployment"
        ],
        "precedence_constraints": [
            ["define_requirements", "design_api"],
            ["design_api", "build_backend"],
            ["design_api", "build_frontend"],
            ["build_backend", "integrate_components"],
            ["build_frontend", "integrate_components"],
            ["build_backend", "unit_tests"],
            ["integrate_components", "integration_tests"],
            ["unit_tests", "integration_tests"],
            ["integration_tests", "security_review"],
            ["security_review", "staging_deployment"],
            ["staging_deployment", "production_deployment"]
        ]
    },

    # EDUCATION
    {
        "task_id": "E_EASY",
        "domain": "Education",
        "difficulty": "easy",
        "prompt": "Plan a study session from topic review to practice and self-testing.",
        "required_tasks": [
            "review_topic", "read_notes", "solve_examples",
            "practice_questions", "check_answers", "self_test"
        ],
        "precedence_constraints": [
            ["review_topic", "read_notes"],
            ["read_notes", "solve_examples"],
            ["solve_examples", "practice_questions"],
            ["practice_questions", "check_answers"],
            ["check_answers", "self_test"]
        ]
    },
    {
        "task_id": "E_MEDIUM",
        "domain": "Education",
        "difficulty": "medium",
        "prompt": "Plan an exam preparation cycle with diagnostic testing, targeted study, practice, feedback, and a final mock test.",
        "required_tasks": [
            "review_syllabus", "take_diagnostic_test", "identify_weak_topics",
            "study_topics", "solve_practice", "review_feedback",
            "revise_weak_topics", "take_mock_test"
        ],
        "precedence_constraints": [
            ["review_syllabus", "take_diagnostic_test"],
            ["take_diagnostic_test", "identify_weak_topics"],
            ["identify_weak_topics", "study_topics"],
            ["study_topics", "solve_practice"],
            ["solve_practice", "review_feedback"],
            ["review_feedback", "revise_weak_topics"],
            ["revise_weak_topics", "take_mock_test"]
        ]
    },
    {
        "task_id": "E_HARD",
        "domain": "Education",
        "difficulty": "hard",
        "prompt": "Plan a learning programme where prerequisite review and diagnostic assessment inform study, practice, feedback, revision, and a final assessment.",
        "required_tasks": [
            "check_prerequisites", "review_syllabus", "diagnostic_assessment",
            "create_study_plan", "study_module_a", "study_module_b",
            "practice_module_a", "practice_module_b", "review_feedback",
            "targeted_revision", "final_assessment"
        ],
        "precedence_constraints": [
            ["check_prerequisites", "create_study_plan"],
            ["review_syllabus", "create_study_plan"],
            ["diagnostic_assessment", "create_study_plan"],
            ["create_study_plan", "study_module_a"],
            ["create_study_plan", "study_module_b"],
            ["study_module_a", "practice_module_a"],
            ["study_module_b", "practice_module_b"],
            ["practice_module_a", "review_feedback"],
            ["practice_module_b", "review_feedback"],
            ["review_feedback", "targeted_revision"],
            ["targeted_revision", "final_assessment"]
        ]
    },

    # DATA SCIENCE
    {
        "task_id": "D_EASY",
        "domain": "Data Science",
        "difficulty": "easy",
        "prompt": "Plan a basic machine learning workflow from inspecting data to evaluating a model.",
        "required_tasks": [
            "inspect_data", "clean_data", "split_data",
            "train_model", "evaluate_model", "document_results"
        ],
        "precedence_constraints": [
            ["inspect_data", "clean_data"],
            ["clean_data", "split_data"],
            ["split_data", "train_model"],
            ["train_model", "evaluate_model"],
            ["evaluate_model", "document_results"]
        ]
    },
    {
        "task_id": "D_MEDIUM",
        "domain": "Data Science",
        "difficulty": "medium",
        "prompt": "Plan a machine learning project with exploratory analysis, leakage-safe preprocessing, training, validation, testing, and reporting.",
        "required_tasks": [
            "inspect_data", "explore_data", "split_data",
            "fit_preprocessing", "train_model", "tune_model",
            "evaluate_test_set", "analyze_errors", "document_results"
        ],
        "precedence_constraints": [
            ["inspect_data", "explore_data"],
            ["explore_data", "split_data"],
            ["split_data", "fit_preprocessing"],
            ["fit_preprocessing", "train_model"],
            ["train_model", "tune_model"],
            ["tune_model", "evaluate_test_set"],
            ["evaluate_test_set", "analyze_errors"],
            ["analyze_errors", "document_results"]
        ]
    },
    {
        "task_id": "D_HARD",
        "domain": "Data Science",
        "difficulty": "hard",
        "prompt": "Plan a robust ML workflow with data quality checks, leakage-safe preprocessing, model comparison, validation, final testing, error analysis, and reproducible documentation.",
        "required_tasks": [
            "inspect_data", "check_data_quality", "define_target",
            "split_data", "build_preprocessing_pipeline", "train_baseline",
            "train_candidate_models", "validate_models", "select_model",
            "evaluate_test_set", "analyze_errors", "document_pipeline"
        ],
        "precedence_constraints": [
            ["inspect_data", "check_data_quality"],
            ["check_data_quality", "define_target"],
            ["define_target", "split_data"],
            ["split_data", "build_preprocessing_pipeline"],
            ["build_preprocessing_pipeline", "train_baseline"],
            ["build_preprocessing_pipeline", "train_candidate_models"],
            ["train_baseline", "validate_models"],
            ["train_candidate_models", "validate_models"],
            ["validate_models", "select_model"],
            ["select_model", "evaluate_test_set"],
            ["evaluate_test_set", "analyze_errors"],
            ["analyze_errors", "document_pipeline"]
        ]
    }
]

assert len(benchmark_tasks) == 12
assert len({t["task_id"] for t in benchmark_tasks}) == 12

# Basic schema checks
for task in benchmark_tasks:
    assert len(task["required_tasks"]) == len(set(task["required_tasks"]))
    known = set(task["required_tasks"])
    for a, b in task["precedence_constraints"]:
        assert a in known and b in known, (task["task_id"], a, b)

benchmark_path = experiment19_dir / "experiment19_benchmark_tasks.json"
benchmark_path.write_text(
    json.dumps(benchmark_tasks, indent=2),
    encoding="utf-8"
)

print("Benchmark tasks created:", len(benchmark_tasks))
print("Unique task IDs:", len({t['task_id'] for t in benchmark_tasks}))
print()
for task in benchmark_tasks:
    print(
        task["task_id"], "|",
        task["domain"], "|",
        task["difficulty"], "|",
        len(task["required_tasks"]), "tasks |",
        len(task["precedence_constraints"]), "constraints"
    )
print("\nSaved:", benchmark_path)


Benchmark tasks created: 12
Unique task IDs: 12

R_EASY | Research | easy | 6 tasks | 5 constraints
R_MEDIUM | Research | medium | 8 tasks | 7 constraints
R_HARD | Research | hard | 10 tasks | 9 constraints
S_EASY | Software Development | easy | 6 tasks | 5 constraints
S_MEDIUM | Software Development | medium | 8 tasks | 8 constraints
S_HARD | Software Development | hard | 10 tasks | 11 constraints
E_EASY | Education | easy | 6 tasks | 5 constraints
E_MEDIUM | Education | medium | 8 tasks | 7 constraints
E_HARD | Education | hard | 11 tasks | 11 constraints
D_EASY | Data Science | easy | 6 tasks | 5 constraints
D_MEDIUM | Data Science | medium | 9 tasks | 8 constraints
D_HARD | Data Science | hard | 12 tasks | 12 constraints

Saved: /content/planproof_results/experiment19/experiment19_benchmark_tasks.json


In [ ]:

import json
from pathlib import Path

experiment19_dir = Path("/content/planproof_results/experiment19")

with open(
    experiment19_dir / "experiment19_benchmark_tasks.json",
    "r",
    encoding="utf-8"
) as f:
    benchmark_tasks = json.load(f)

collection_records = []

for task in benchmark_tasks:
    for generation_number in range(1, 4):
        prompt = f"""
You are an AI planning assistant.

Planning objective:
{task['prompt']}

Required task IDs:
{json.dumps(task['required_tasks'], indent=2)}

Precedence constraints:
{json.dumps(task['precedence_constraints'], indent=2)}

Produce a practical task order.

Return ONLY a valid JSON object with this schema:
{{
  "task_order": ["task_id_1", "task_id_2"]
}}

Requirements:
- Include every required task ID exactly once.
- Do not invent, rename, or duplicate task IDs.
- Satisfy every precedence constraint.
- Do not include explanations or Markdown.
"""

        collection_records.append({
            "sample_id": f"{task['task_id']}_G{generation_number}",
            "task_id": task["task_id"],
            "domain": task["domain"],
            "difficulty": task["difficulty"],
            "generation_number": generation_number,
            "prompt": prompt,
            "model_name": None,
            "generation_settings": None,
            "collection_timestamp_utc": None,
            "raw_response": None,
            "collection_status": "pending"
        })

prompts_path = experiment19_dir / "experiment19_collection_prompts.json"
prompts_path.write_text(
    json.dumps(collection_records, indent=2),
    encoding="utf-8"
)

print("Collection prompts created:", len(collection_records))
print("Unique benchmark tasks:", len(benchmark_tasks))
print("Generations per task: 3")
print("Pending responses:", sum(r["raw_response"] is None for r in collection_records))
print("Saved:", prompts_path)

print("\nExample sample ID:", collection_records[0]["sample_id"])
print("\nExample prompt:\n")
print(collection_records[0]["prompt"])


Collection prompts created: 36
Unique benchmark tasks: 12
Generations per task: 3
Pending responses: 36
Saved: /content/planproof_results/experiment19/experiment19_collection_prompts.json

Example sample ID: R_EASY_G1

Example prompt:


You are an AI planning assistant.

Planning objective:
Plan a small research project from reading literature to submitting a report.

Required task IDs:
[
  "review_literature",
  "define_question",
  "design_study",
  "collect_data",
  "analyze_data",
  "write_report"
]

Precedence constraints:
[
  [
    "review_literature",
    "define_question"
  ],
  [
    "define_question",
    "design_study"
  ],
  [
    "design_study",
    "collect_data"
  ],
  [
    "collect_data",
    "analyze_data"
  ],
  [
    "analyze_data",
    "write_report"
  ]
]

Produce a practical task order.

Return ONLY a valid JSON object with this schema:
{
  "task_order": ["task_id_1", "task_id_2"]
}

Requirements:
- Include every required task ID exactly once.
- Do not invent, re

In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Markdown
import ipywidgets as widgets

experiment19_dir = Path("/content/planproof_results/experiment19")
prompts_path = experiment19_dir / "experiment19_collection_prompts.json"

with open(prompts_path, "r", encoding="utf-8") as f:
    collection_records = json.load(f)

# Find the next response that has not yet been collected.
pending_indices = [
    i for i, record in enumerate(collection_records)
    if not record.get("raw_response")
]

print("Total samples:", len(collection_records))
print("Pending samples:", len(pending_indices))


Total samples: 36
Pending samples: 36


In [ ]:

def show_next_sample():
    pending = [
        i for i, record in enumerate(collection_records)
        if not record.get("raw_response")
    ]

    if not pending:
        print("All responses have been collected!")
        return

    i = pending[0]
    record = collection_records[i]

    print(f"Sample ID: {record['sample_id']}")
    print(f"Domain: {record['domain']}")
    print(f"Difficulty: {record['difficulty']}")
    print("\nCopy the prompt below into a NEW AI conversation:\n")
    print(record["prompt"])

show_next_sample()


Sample ID: R_EASY_G1
Domain: Research
Difficulty: easy

Copy the prompt below into a NEW AI conversation:


You are an AI planning assistant.

Planning objective:
Plan a small research project from reading literature to submitting a report.

Required task IDs:
[
  "review_literature",
  "define_question",
  "design_study",
  "collect_data",
  "analyze_data",
  "write_report"
]

Precedence constraints:
[
  [
    "review_literature",
    "define_question"
  ],
  [
    "define_question",
    "design_study"
  ],
  [
    "design_study",
    "collect_data"
  ],
  [
    "collect_data",
    "analyze_data"
  ],
  [
    "analyze_data",
    "write_report"
  ]
]

Produce a practical task order.

Return ONLY a valid JSON object with this schema:
{
  "task_order": ["task_id_1", "task_id_2"]
}

Requirements:
- Include every required task ID exactly once.
- Do not invent, rename, or duplicate task IDs.
- Satisfy every precedence constraint.
- Do not include explanations or Markdown.



In [ ]:

import json
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets

experiment19_dir = Path("/content/planproof_results/experiment19")
prompts_path = experiment19_dir / "experiment19_collection_prompts.json"

# Load the existing records, preserving any responses already saved.
with open(prompts_path, "r", encoding="utf-8") as f:
    collection_records = json.load(f)

model_input = widgets.Text(
    value="",
    placeholder="Enter model name (e.g., ChatGPT model shown in the UI)",
    description="Model:",
    layout=widgets.Layout(width="80%")
)

response_input = widgets.Textarea(
    value="",
    placeholder="Paste the exact raw model response here",
    description="Response:",
    layout=widgets.Layout(width="95%", height="180px")
)

save_button = widgets.Button(
    description="Save response",
    button_style="success"
)
output = widgets.Output()

def get_pending_indices():
    return [
        i for i, record in enumerate(collection_records)
        if not record.get("raw_response")
    ]

def save_current_response(_):
    with output:
        clear_output()

        pending = get_pending_indices()
        if not pending:
            print("All responses have been collected.")
            return

        if not model_input.value.strip():
            print("Please enter the model name first.")
            return

        if not response_input.value.strip():
            print("Please paste the actual model response first.")
            return

        i = pending[0]
        record = collection_records[i]

        record["model_name"] = model_input.value.strip()
        record["raw_response"] = response_input.value
        record["collection_timestamp_utc"] = (
            datetime.now(timezone.utc).isoformat()
        )
        record["collection_status"] = "collected"

        with open(prompts_path, "w", encoding="utf-8") as f:
            json.dump(collection_records, f, indent=2)

        print("Saved sample:", record["sample_id"])
        print("Responses collected:", len(collection_records) - len(get_pending_indices()))
        print("Responses remaining:", len(get_pending_indices()))

        model_input.value = ""
        response_input.value = ""

        next_pending = get_pending_indices()
        if next_pending:
            next_record = collection_records[next_pending[0]]
            print("Next sample:", next_record["sample_id"])
            print("Run show_next_sample() to display its prompt.")
        else:
            print("All 36 responses have been collected!")

save_button.on_click(save_current_response)

display(model_input, response_input, save_button, output)


Text(value='', description='Model:', layout=Layout(width='80%'), placeholder='Enter model name (e.g., ChatGPT …

Textarea(value='', description='Response:', layout=Layout(height='180px', width='95%'), placeholder='Paste the…

Button(button_style='success', description='Save response', style=ButtonStyle())

Output()

In [ ]:

import json
import random
from pathlib import Path
from datetime import datetime, timezone

experiment19_dir = Path("/content/planproof_results/experiment19")

with open(
    experiment19_dir / "experiment19_benchmark_tasks.json",
    "r",
    encoding="utf-8"
) as f:
    benchmark_tasks = json.load(f)

rng = random.Random(20261019)
controlled_cases = []

for task in benchmark_tasks:
    for attempt in range(1, 4):
        order = list(task["required_tasks"])
        rng.shuffle(order)

        controlled_cases.append({
            "case_id": f"{task['task_id']}_C{attempt}",
            "task_id": task["task_id"],
            "domain": task["domain"],
            "difficulty": task["difficulty"],
            "task_order": order,
            "required_tasks": task["required_tasks"],
            "precedence_constraints": task["precedence_constraints"],
            "case_source": "randomly_shuffled_controlled_order",
            "seed": 20261019
        })

cases_path = experiment19_dir / "experiment19_controlled_cases.json"
cases_path.write_text(
    json.dumps(controlled_cases, indent=2),
    encoding="utf-8"
)

print("Controlled test cases created:", len(controlled_cases))
print("Benchmark tasks:", len(benchmark_tasks))
print("Cases per task: 3")
print("Saved:", cases_path)


FileNotFoundError: [Errno 2] No such file or directory: '/content/planproof_results/experiment19/experiment19_benchmark_tasks.json'

In [ ]:

from pathlib import Path

base_dir = Path("/content/planproof_results")
experiment19_dir = base_dir / "experiment19"

print("Base folder exists:", base_dir.exists())
print("Experiment 19 folder exists:", experiment19_dir.exists())

if experiment19_dir.exists():
    print("\nFiles currently available:")
    for path in sorted(experiment19_dir.iterdir()):
        print("-", path.name)
else:
    print("\nExperiment 19 benchmark file was not found.")

    if base_dir.exists():
        print("\nOther available files:")
        for path in sorted(base_dir.iterdir()):
            print("-", path.name)


Base folder exists: False
Experiment 19 folder exists: False

Experiment 19 benchmark file was not found.


In [ ]:

import json
import random
from pathlib import Path
from datetime import datetime, timezone

# Create the missing directories
base_dir = Path("/content/planproof_results")
experiment19_dir = base_dir / "experiment19"
experiment19_dir.mkdir(parents=True, exist_ok=True)

# Define the 12 benchmark tasks again
benchmark_tasks = [
    ("R_EASY", "Research", "easy",
     ["review_literature", "define_question", "design_study",
      "collect_data", "analyze_data", "write_report"]),
    ("R_MEDIUM", "Research", "medium",
     ["review_literature", "formulate_hypothesis", "design_protocol",
      "obtain_ethics_approval", "pilot_study", "collect_data",
      "analyze_data", "write_paper"]),
    ("R_HARD", "Research", "hard",
     ["review_literature", "prepare_dataset", "design_protocol",
      "obtain_ethics_approval", "recruit_participants", "pilot_study",
      "collect_data", "analyze_data", "replicate_results", "write_paper"]),
    ("S_EASY", "Software Development", "easy",
     ["gather_requirements", "design_solution", "implement_feature",
      "test_feature", "fix_defects", "release_feature"]),
    ("S_MEDIUM", "Software Development", "medium",
     ["gather_requirements", "design_architecture", "implement_backend",
      "implement_frontend", "review_code", "run_tests",
      "security_check", "deploy_application"]),
    ("S_HARD", "Software Development", "hard",
     ["define_requirements", "design_api", "build_backend",
      "build_frontend", "integrate_components", "unit_tests",
      "integration_tests", "security_review", "staging_deployment",
      "production_deployment"]),
    ("E_EASY", "Education", "easy",
     ["review_topic", "read_notes", "solve_examples",
      "practice_questions", "check_answers", "self_test"]),
    ("E_MEDIUM", "Education", "medium",
     ["review_syllabus", "take_diagnostic_test", "identify_weak_topics",
      "study_topics", "solve_practice", "review_feedback",
      "revise_weak_topics", "take_mock_test"]),
    ("E_HARD", "Education", "hard",
     ["check_prerequisites", "review_syllabus", "diagnostic_assessment",
      "create_study_plan", "study_module_a", "study_module_b",
      "practice_module_a", "practice_module_b", "review_feedback",
      "targeted_revision", "final_assessment"]),
    ("D_EASY", "Data Science", "easy",
     ["inspect_data", "clean_data", "split_data",
      "train_model", "evaluate_model", "document_results"]),
    ("D_MEDIUM", "Data Science", "medium",
     ["inspect_data", "explore_data", "split_data",
      "fit_preprocessing", "train_model", "tune_model",
      "evaluate_test_set", "analyze_errors", "document_results"]),
    ("D_HARD", "Data Science", "hard",
     ["inspect_data", "check_data_quality", "define_target",
      "split_data", "build_preprocessing_pipeline", "train_baseline",
      "train_candidate_models", "validate_models", "select_model",
      "evaluate_test_set", "analyze_errors", "document_pipeline"])
]

tasks = []
for task_id, domain, difficulty, required in benchmark_tasks:
    # Default to sequential precedence constraints for this recovery version.
    constraints = [
        [required[i], required[i + 1]]
        for i in range(len(required) - 1)
    ]

    # Restore parallel branches in selected hard tasks.
    if task_id == "S_HARD":
        constraints = [
            ["define_requirements", "design_api"],
            ["design_api", "build_backend"],
            ["design_api", "build_frontend"],
            ["build_backend", "integrate_components"],
            ["build_frontend", "integrate_components"],
            ["build_backend", "unit_tests"],
            ["integrate_components", "integration_tests"],
            ["unit_tests", "integration_tests"],
            ["integration_tests", "security_review"],
            ["security_review", "staging_deployment"],
            ["staging_deployment", "production_deployment"]
        ]
    elif task_id == "E_HARD":
        constraints = [
            ["check_prerequisites", "create_study_plan"],
            ["review_syllabus", "create_study_plan"],
            ["diagnostic_assessment", "create_study_plan"],
            ["create_study_plan", "study_module_a"],
            ["create_study_plan", "study_module_b"],
            ["study_module_a", "practice_module_a"],
            ["study_module_b", "practice_module_b"],
            ["practice_module_a", "review_feedback"],
            ["practice_module_b", "review_feedback"],
            ["review_feedback", "targeted_revision"],
            ["targeted_revision", "final_assessment"]
        ]
    elif task_id == "D_HARD":
        constraints = [
            ["inspect_data", "check_data_quality"],
            ["check_data_quality", "define_target"],
            ["define_target", "split_data"],
            ["split_data", "build_preprocessing_pipeline"],
            ["build_preprocessing_pipeline", "train_baseline"],
            ["build_preprocessing_pipeline", "train_candidate_models"],
            ["train_baseline", "validate_models"],
            ["train_candidate_models", "validate_models"],
            ["validate_models", "select_model"],
            ["select_model", "evaluate_test_set"],
            ["evaluate_test_set", "analyze_errors"],
            ["analyze_errors", "document_pipeline"]
        ]

    tasks.append({
        "task_id": task_id,
        "domain": domain,
        "difficulty": difficulty,
        "prompt": f"Create a practical plan for this {domain.lower()} task.",
        "required_tasks": required,
        "precedence_constraints": constraints
    })

# Save benchmark and configuration
(experiment19_dir / "experiment19_benchmark_tasks.json").write_text(
    json.dumps(tasks, indent=2), encoding="utf-8"
)

config = {
    "experiment_id": "experiment19",
    "restored_utc": datetime.now(timezone.utc).isoformat(),
    "target_domains": ["Research", "Software Development", "Education", "Data Science"],
    "difficulty_levels": ["easy", "medium", "hard"],
    "tasks_per_domain": 3,
    "controlled_cases_per_task": 3,
    "target_controlled_case_count": 36,
    "note": "Benchmark restored after Colab runtime reset; controlled cases are not AI-generated outputs."
}
(experiment19_dir / "experiment19_config.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)

# Generate reproducible shuffled controlled cases
rng = random.Random(20261019)
cases = []

for task in tasks:
    for attempt in range(1, 4):
        order = list(task["required_tasks"])
        rng.shuffle(order)
        cases.append({
            "case_id": f"{task['task_id']}_C{attempt}",
            "task_id": task["task_id"],
            "domain": task["domain"],
            "difficulty": task["difficulty"],
            "task_order": order,
            "required_tasks": task["required_tasks"],
            "precedence_constraints": task["precedence_constraints"],
            "case_source": "randomly_shuffled_controlled_order",
            "seed": 20261019
        })

(experiment19_dir / "experiment19_controlled_cases.json").write_text(
    json.dumps(cases, indent=2), encoding="utf-8"
)

print("Experiment 19 restored.")
print("Benchmark tasks:", len(tasks))
print("Controlled cases:", len(cases))
print("Saved files:")
for path in sorted(experiment19_dir.iterdir()):
    print("-", path.name)


Experiment 19 restored.
Benchmark tasks: 12
Controlled cases: 36
Saved files:
- experiment19_benchmark_tasks.json
- experiment19_config.json
- experiment19_controlled_cases.json


In [ ]:

import json
from pathlib import Path
from collections import Counter

experiment19_dir = Path("/content/planproof_results/experiment19")

with open(experiment19_dir / "experiment19_benchmark_tasks.json",
          encoding="utf-8") as f:
    tasks = json.load(f)

with open(experiment19_dir / "experiment19_controlled_cases.json",
          encoding="utf-8") as f:
    cases = json.load(f)

errors = []

# Check task definitions
task_ids = [task["task_id"] for task in tasks]

if len(task_ids) != len(set(task_ids)):
    errors.append("Duplicate benchmark task IDs")

for task in tasks:
    required = task["required_tasks"]
    required_set = set(required)

    if len(required) != len(required_set):
        errors.append(f"{task['task_id']}: duplicate required task IDs")

    for before, after in task["precedence_constraints"]:
        if before not in required_set or after not in required_set:
            errors.append(
                f"{task['task_id']}: constraint references unknown task"
            )

# Check controlled cases
case_ids = [case["case_id"] for case in cases]

if len(case_ids) != len(set(case_ids)):
    errors.append("Duplicate controlled case IDs")

if len(cases) != 36:
    errors.append(f"Expected 36 cases, found {len(cases)}")

for case in cases:
    if set(case["task_order"]) != set(case["required_tasks"]):
        errors.append(f"{case['case_id']}: task order mismatch")

    if len(case["task_order"]) != len(set(case["task_order"])):
        errors.append(f"{case['case_id']}: duplicate task IDs")

print("Benchmark tasks:", len(tasks))
print("Controlled cases:", len(cases))
print("Cases per task:", Counter(c["task_id"] for c in cases).most_common(1)[0][1])
print("Structural errors:", len(errors))

if errors:
    for error in errors:
        print("ERROR:", error)
else:
    print("PASS: benchmark structure is consistent.")


Benchmark tasks: 12
Controlled cases: 36
Cases per task: 3
Structural errors: 0
PASS: benchmark structure is consistent.


In [ ]:

import json
import pandas as pd
from pathlib import Path

experiment19_dir = Path("/content/planproof_results/experiment19")

with open(
    experiment19_dir / "experiment19_controlled_cases.json",
    "r",
    encoding="utf-8"
) as f:
    cases = json.load(f)

rows = []

for case in cases:
    original_plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    before = validate_plan(original_plan)
    repaired = repair_plan(original_plan)

    if repaired["success"]:
        repaired_order = repaired["repaired_plan"]["task_order"]

        after = validate_plan({
            "task_order": repaired_order,
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        })

        change_rate = pairwise_order_change_rate(
            case["task_order"], repaired_order
        )
    else:
        after = {"valid": False, "violations": ["repair failed"]}
        change_rate = None

    rows.append({
        "case_id": case["case_id"],
        "task_id": case["task_id"],
        "domain": case["domain"],
        "difficulty": case["difficulty"],
        "case_source": case["case_source"],
        "valid_before_repair": before["valid"],
        "violations_before": len(before["violations"]),
        "repair_success": repaired["success"],
        "valid_after_repair": after["valid"],
        "violations_after": len(after["violations"]),
        "pairwise_order_change_rate": change_rate
    })

results19 = pd.DataFrame(rows)

results_path = experiment19_dir / "experiment19_controlled_results.csv"
results19.to_csv(results_path, index=False)

print("Cases evaluated:", len(results19))
print("Initially valid:", int(results19["valid_before_repair"].sum()))
print("Initially invalid:", int((~results19["valid_before_repair"]).sum()))
print("Repair successes:", int(results19["repair_success"].sum()))
print("Valid after repair:", int(results19["valid_after_repair"].sum()))
print("Remaining violations:", int(results19["violations_after"].sum()))
print("Saved:", results_path)

display(results19.groupby("difficulty").agg(
    cases=("case_id", "count"),
    initially_valid=("valid_before_repair", "sum"),
    repair_successes=("repair_success", "sum"),
    valid_after_repair=("valid_after_repair", "sum"),
    violations_before=("violations_before", "sum"),
    violations_after=("violations_after", "sum")
).reset_index())


NameError: name 'validate_plan' is not defined

In [ ]:

for name in [
    "validate_plan",
    "repair_plan",
    "local_search_repair",
    "fast_local_search_repair",
    "pairwise_order_change_rate"
]:
    print(f"{name}: {'Available' if name in globals() else 'Missing'}")


validate_plan: Missing
repair_plan: Missing
local_search_repair: Missing
fast_local_search_repair: Missing
pairwise_order_change_rate: Missing


In [ ]:

import heapq

def validate_plan(plan):
    """Validate task IDs and precedence constraints."""
    order = plan["task_order"]
    required = plan["required_tasks"]
    constraints = plan["precedence_constraints"]

    violations = []

    if len(order) != len(set(order)):
        violations.append("Duplicate task IDs")

    order_set = set(order)
    required_set = set(required)

    missing = required_set - order_set
    unknown = order_set - required_set

    if missing:
        violations.append(f"Missing tasks: {sorted(missing)}")

    if unknown:
        violations.append(f"Unknown tasks: {sorted(unknown)}")

    # Check precedence constraints using first occurrence positions.
    positions = {}
    for i, task_id in enumerate(order):
        positions.setdefault(task_id, i)

    for before, after in constraints:
        if before not in positions or after not in positions:
            violations.append(
                f"Cannot check constraint {before} -> {after}: task missing"
            )
        elif positions[before] >= positions[after]:
            violations.append(f"Precedence violation: {before} -> {after}")

    return {"valid": len(violations) == 0, "violations": violations}


def repair_plan(plan):
    """Repair a task order using stable topological sorting."""
    required = list(plan["required_tasks"])
    constraints = plan["precedence_constraints"]
    original = list(plan["task_order"])

    required_set = set(required)

    if len(required) != len(required_set):
        return {
            "success": False,
            "repaired_plan": None,
            "message": "Required task IDs contain duplicates"
        }

    # Validate constraint endpoints.
    for before, after in constraints:
        if before not in required_set or after not in required_set:
            return {
                "success": False,
                "repaired_plan": None,
                "message": "Constraint references an unknown task"
            }

    # Stable tie-breaking follows the original order.
    first_position = {}
    for i, task_id in enumerate(original):
        if task_id in required_set and task_id not in first_position:
            first_position[task_id] = i

    next_position = len(original)
    for task_id in required:
        if task_id not in first_position:
            first_position[task_id] = next_position
            next_position += 1

    adjacency = {task_id: set() for task_id in required}
    indegree = {task_id: 0 for task_id in required}

    for before, after in constraints:
        if after not in adjacency[before]:
            adjacency[before].add(after)
            indegree[after] += 1

    ready = [
        (first_position[task_id], task_id)
        for task_id in required
        if indegree[task_id] == 0
    ]
    heapq.heapify(ready)

    repaired_order = []

    while ready:
        _, current = heapq.heappop(ready)
        repaired_order.append(current)

        for neighbour in adjacency[current]:
            indegree[neighbour] -= 1
            if indegree[neighbour] == 0:
                heapq.heappush(
                    ready, (first_position[neighbour], neighbour)
                )

    if len(repaired_order) != len(required):
        return {
            "success": False,
            "repaired_plan": None,
            "message": "Constraints contain a cycle; no valid order exists"
        }

    repaired = {
        "task_order": repaired_order,
        "required_tasks": required,
        "precedence_constraints": constraints
    }

    check = validate_plan(repaired)

    if not check["valid"]:
        return {
            "success": False,
            "repaired_plan": None,
            "message": f"Post-repair validation failed: {check['violations']}"
        }

    return {
        "success": True,
        "repaired_plan": repaired,
        "message": "Plan repaired and validated"
    }


def pairwise_order_change_rate(original, repaired):
    """Fraction of task pairs whose relative order changed."""
    original = list(original)
    repaired = list(repaired)

    common = set(original) & set(repaired)
    original = [x for x in original if x in common]
    repaired = [x for x in repaired if x in common]

    if len(common) < 2:
        return 0.0

    original_positions = {x: i for i, x in enumerate(original)}
    repaired_positions = {x: i for i, x in enumerate(repaired)}

    pairs = 0
    changed = 0

    items = list(common)
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            a, b = items[i], items[j]
            pairs += 1
            if (
                original_positions[a] < original_positions[b]
            ) != (
                repaired_positions[a] < repaired_positions[b]
            ):
                changed += 1

    return changed / pairs if pairs else 0.0


print("Core functions restored:")
print("validate_plan:", callable(validate_plan))
print("repair_plan:", callable(repair_plan))
print("pairwise_order_change_rate:", callable(pairwise_order_change_rate))


Core functions restored:
validate_plan: True
repair_plan: True
pairwise_order_change_rate: True


In [ ]:

# A deliberately invalid plan: C must come after B, and B after A.
test_plan = {
    "task_order": ["C", "B", "A"],
    "required_tasks": ["A", "B", "C"],
    "precedence_constraints": [
        ["A", "B"],
        ["B", "C"]
    ]
}

before = validate_plan(test_plan)
repaired = repair_plan(test_plan)

print("Before repair:")
print("Valid:", before["valid"])
print("Violations:", before["violations"])

print("\nRepair result:")
print("Success:", repaired["success"])
print("Message:", repaired["message"])

if repaired["success"]:
    repaired_plan = repaired["repaired_plan"]
    after = validate_plan(repaired_plan)

    print("Repaired order:", repaired_plan["task_order"])
    print("Valid after repair:", after["valid"])
    print("Remaining violations:", after["violations"])

    assert after["valid"], "Repaired plan should be valid"
    assert repaired_plan["task_order"] == ["A", "B", "C"]

print("\nPASS: core function test completed.")


Before repair:
Valid: False
Violations: ['Precedence violation: A -> B', 'Precedence violation: B -> C']

Repair result:
Success: True
Message: Plan repaired and validated
Repaired order: ['A', 'B', 'C']
Valid after repair: True
Remaining violations: []

PASS: core function test completed.


In [ ]:

import json
import pandas as pd
from pathlib import Path

experiment19_dir = Path("/content/planproof_results/experiment19")

with open(
    experiment19_dir / "experiment19_controlled_cases.json",
    "r",
    encoding="utf-8"
) as f:
    cases = json.load(f)

rows = []

for case in cases:
    original_plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    before = validate_plan(original_plan)
    repaired = repair_plan(original_plan)

    if repaired["success"]:
        repaired_plan = repaired["repaired_plan"]
        after = validate_plan(repaired_plan)
        change_rate = pairwise_order_change_rate(
            case["task_order"],
            repaired_plan["task_order"]
        )
    else:
        after = {"valid": False, "violations": ["repair failed"]}
        change_rate = None

    rows.append({
        "case_id": case["case_id"],
        "task_id": case["task_id"],
        "domain": case["domain"],
        "difficulty": case["difficulty"],
        "valid_before_repair": before["valid"],
        "violations_before": len(before["violations"]),
        "repair_success": repaired["success"],
        "valid_after_repair": after["valid"],
        "violations_after": len(after["violations"]),
        "pairwise_order_change_rate": change_rate
    })

results19 = pd.DataFrame(rows)

results_path = experiment19_dir / "experiment19_controlled_results.csv"
results19.to_csv(results_path, index=False)

print("Cases evaluated:", len(results19))
print("Initially valid:", int(results19["valid_before_repair"].sum()))
print("Initially invalid:", int((~results19["valid_before_repair"]).sum()))
print("Repair successes:", int(results19["repair_success"].sum()))
print("Valid after repair:", int(results19["valid_after_repair"].sum()))
print("Remaining violations:", int(results19["violations_after"].sum()))
print("Saved:", results_path)

print("\nResults by difficulty:")
display(
    results19.groupby("difficulty", sort=False).agg(
        cases=("case_id", "count"),
        initially_valid=("valid_before_repair", "sum"),
        repair_successes=("repair_success", "sum"),
        valid_after_repair=("valid_after_repair", "sum"),
        violations_before=("violations_before", "sum"),
        violations_after=("violations_after", "sum")
    ).reset_index()
)


Cases evaluated: 36
Initially valid: 0
Initially invalid: 36
Repair successes: 36
Valid after repair: 36
Remaining violations: 0
Saved: /content/planproof_results/experiment19/experiment19_controlled_results.csv

Results by difficulty:


,difficulty,cases,initially_valid,repair_successes,valid_after_repair,violations_before,violations_after
0,easy,12,0,12,12,28,0
1,medium,12,0,12,12,41,0
2,hard,12,0,12,12,63,0


In [ ]:

import pandas as pd
from pathlib import Path

results_path = Path(
    "/content/planproof_results/experiment19/"
    "experiment19_controlled_results.csv"
)

results19 = pd.read_csv(results_path)

print("PAIRWISE ORDER PRESERVATION")
print("----------------------------")

print(
    "Mean change rate:",
    round(results19["pairwise_order_change_rate"].mean(), 4)
)

print(
    "Median change rate:",
    round(results19["pairwise_order_change_rate"].median(), 4)
)

print(
    "Minimum change rate:",
    round(results19["pairwise_order_change_rate"].min(), 4)
)

print(
    "Maximum change rate:",
    round(results19["pairwise_order_change_rate"].max(), 4)
)

print("\nBy difficulty:")

display(
    results19.groupby("difficulty", sort=False)[
        "pairwise_order_change_rate"
    ].agg(
        ["count", "mean", "median", "min", "max"]
    ).round(4)
)


PAIRWISE ORDER PRESERVATION
----------------------------
Mean change rate: 0.4563
Median change rate: 0.4444
Minimum change rate: 0.2182
Maximum change rate: 0.7333

By difficulty:


,count,mean,median,min,max
difficulty,,,,,
easy,12,0.5000,0.5333,0.2667,0.7333
medium,12,0.4358,0.4286,0.2500,0.7143
hard,12,0.4330,0.4333,0.2182,0.6222


In [ ]:

import pandas as pd
from pathlib import Path

results_path = Path(
    "/content/planproof_results/experiment19/"
    "experiment19_controlled_results.csv"
)

results19 = pd.read_csv(results_path)

# Show the 5 cases with the largest order changes
largest_changes = results19.sort_values(
    "pairwise_order_change_rate",
    ascending=False
).head(5)

display(
    largest_changes[
        [
            "case_id",
            "task_id",
            "difficulty",
            "violations_before",
            "violations_after",
            "pairwise_order_change_rate"
        ]
    ].reset_index(drop=True)
)


,case_id,task_id,difficulty,violations_before,violations_after,pairwise_order_change_rate
0,E_EASY_C3,E_EASY,easy,3,0,0.733333
1,E_MEDIUM_C3,E_MEDIUM,medium,4,0,0.714286
2,R_HARD_C3,R_HARD,hard,6,0,0.622222
3,R_EASY_C3,R_EASY,easy,1,0,0.600000
4,E_EASY_C1,E_EASY,easy,3,0,0.600000


In [ ]:

import json
from pathlib import Path

experiment19_dir = Path(
    "/content/planproof_results/experiment19"
)

with open(
    experiment19_dir / "experiment19_controlled_cases.json",
    "r",
    encoding="utf-8"
) as f:
    cases = json.load(f)

case = next(
    item for item in cases
    if item["case_id"] == "E_EASY_C3"
)

original_plan = {
    "task_order": case["task_order"],
    "required_tasks": case["required_tasks"],
    "precedence_constraints": case["precedence_constraints"]
}

result = repair_plan(original_plan)

print("CASE:", case["case_id"])
print("\nOriginal order:")
print(original_plan["task_order"])

print("\nRequired precedence constraints:")
for before, after in original_plan["precedence_constraints"]:
    print(f"{before} -> {after}")

print("\nViolations before repair:")
for violation in validate_plan(original_plan)["violations"]:
    print("-", violation)

print("\nRepaired order:")
if result["success"]:
    print(result["repaired_plan"]["task_order"])
    print("\nViolations after repair:")
    print(validate_plan(result["repaired_plan"])["violations"])
else:
    print("Repair failed:", result["message"])


CASE: E_EASY_C3

Original order:
['check_answers', 'self_test', 'solve_examples', 'read_notes', 'review_topic', 'practice_questions']

Required precedence constraints:
review_topic -> read_notes
read_notes -> solve_examples
solve_examples -> practice_questions
practice_questions -> check_answers
check_answers -> self_test

Violations before repair:
- Precedence violation: review_topic -> read_notes
- Precedence violation: read_notes -> solve_examples
- Precedence violation: practice_questions -> check_answers

Repaired order:
['review_topic', 'read_notes', 'solve_examples', 'practice_questions', 'check_answers', 'self_test']

Violations after repair:
[]


In [ ]:

import inspect

print(inspect.getsource(repair_plan))


def repair_plan(plan):
    """Repair a task order using stable topological sorting."""
    required = list(plan["required_tasks"])
    constraints = plan["precedence_constraints"]
    original = list(plan["task_order"])

    required_set = set(required)

    if len(required) != len(required_set):
        return {
            "success": False,
            "repaired_plan": None,
            "message": "Required task IDs contain duplicates"
        }

    # Validate constraint endpoints.
    for before, after in constraints:
        if before not in required_set or after not in required_set:
            return {
                "success": False,
                "repaired_plan": None,
                "message": "Constraint references an unknown task"
            }

    # Stable tie-breaking follows the original order.
    first_position = {}
    for i, task_id in enumerate(original):
        if task_id in required_set and task_id not in first_position:
            first_position[task_id] =

In [ ]:

from itertools import permutations

def minimum_change_repair(plan):
    """
    Find a valid task ordering with the fewest pairwise
    reversals relative to the original ordering.

    Exhaustive search: intended only for small plans.
    """
    required = list(plan["required_tasks"])
    original = list(plan["task_order"])
    constraints = plan["precedence_constraints"]

    if len(required) != len(set(required)):
        return {"success": False, "message": "Duplicate required task IDs"}

    required_set = set(required)

    if any(a not in required_set or b not in required_set
           for a, b in constraints):
        return {"success": False, "message": "Unknown task in constraints"}

    if len(required) > 9:
        return {
            "success": False,
            "message": "Exhaustive search limited to 9 tasks"
        }

    # Record the first occurrence of each required task in the input.
    original_positions = {
        task: i for i, task in enumerate(original)
        if task in required_set
    }

    # Missing tasks are placed after the original tasks for tie-breaking.
    next_position = len(original)
    for task in required:
        if task not in original_positions:
            original_positions[task] = next_position
            next_position += 1

    # Count pairwise order reversals, including required tasks
    # that were missing from the original plan.
    def change_cost(candidate):
        positions = {task: i for i, task in enumerate(candidate)}
        common = set(original) & required_set
        changed = 0
        pairs = 0

        for a, b in permutations(sorted(common), 2):
            if a >= b:
                continue
            pairs += 1
            original_a_before_b = (
                original_positions[a] < original_positions[b]
            )
            candidate_a_before_b = positions[a] < positions[b]

            if original_a_before_b != candidate_a_before_b:
                changed += 1

        return changed

    best_order = None
    best_cost = float("inf")

    for candidate in permutations(required):
        positions = {task: i for i, task in enumerate(candidate)}

        if any(positions[a] >= positions[b] for a, b in constraints):
            continue

        cost = change_cost(candidate)

        if cost < best_cost:
            best_cost = cost
            best_order = list(candidate)

    if best_order is None:
        return {"success": False, "message": "No valid ordering exists"}

    repaired = {
        "task_order": best_order,
        "required_tasks": required,
        "precedence_constraints": constraints
    }

    return {
        "success": True,
        "repaired_plan": repaired,
        "change_count": best_cost,
        "message": "Minimum-change valid ordering found by exhaustive search"
    }


# Test it on the high-change example.
test_plan = {
    "task_order": case["task_order"],
    "required_tasks": case["required_tasks"],
    "precedence_constraints": case["precedence_constraints"]
}

minimum_result = minimum_change_repair(test_plan)

print("Case:", case["case_id"])
print("Minimum-change search successful:", minimum_result["success"])

if minimum_result["success"]:
    print("Minimum-change order:")
    print(minimum_result["repaired_plan"]["task_order"])
    print(
        "Validation:",
        validate_plan(minimum_result["repaired_plan"])
    )
    print("Pairwise change count:", minimum_result["change_count"])


Case: E_EASY_C3
Minimum-change search successful: True
Minimum-change order:
['review_topic', 'read_notes', 'solve_examples', 'practice_questions', 'check_answers', 'self_test']
Validation: {'valid': True, 'violations': []}
Pairwise change count: 11


In [ ]:

import json
import pandas as pd
from pathlib import Path

experiment19_dir = Path(
    "/content/planproof_results/experiment19"
)

with open(
    experiment19_dir / "experiment19_controlled_cases.json",
    "r",
    encoding="utf-8"
) as f:
    cases = json.load(f)

comparison_rows = []

for item in cases:
    plan = {
        "task_order": item["task_order"],
        "required_tasks": item["required_tasks"],
        "precedence_constraints": item["precedence_constraints"]
    }

    stable = repair_plan(plan)
    minimum = minimum_change_repair(plan)

    if not stable["success"] or not minimum["success"]:
        comparison_rows.append({
            "case_id": item["case_id"],
            "stable_success": stable["success"],
            "minimum_success": minimum["success"],
            "stable_change_rate": None,
            "minimum_change_rate": None,
            "minimum_is_better_or_equal": None
        })
        continue

    stable_order = stable["repaired_plan"]["task_order"]
    minimum_order = minimum["repaired_plan"]["task_order"]

    stable_rate = pairwise_order_change_rate(
        plan["task_order"], stable_order
    )
    minimum_rate = pairwise_order_change_rate(
        plan["task_order"], minimum_order
    )

    comparison_rows.append({
        "case_id": item["case_id"],
        "stable_success": stable["success"],
        "minimum_success": minimum["success"],
        "stable_change_rate": stable_rate,
        "minimum_change_rate": minimum_rate,
        "minimum_is_better_or_equal": (
            minimum_rate <= stable_rate + 1e-12
        )
    })

comparison = pd.DataFrame(comparison_rows)

comparison_path = (
    experiment19_dir / "experiment19_repair_comparison.csv"
)
comparison.to_csv(comparison_path, index=False)

successful = comparison[
    comparison["stable_success"] & comparison["minimum_success"]
]

print("Cases compared:", len(comparison))
print("Both methods successful:", len(successful))
print(
    "Stable mean change rate:",
    round(successful["stable_change_rate"].mean(), 4)
)
print(
    "Minimum-change mean change rate:",
    round(successful["minimum_change_rate"].mean(), 4)
)
print(
    "Cases where minimum-change is strictly better:",
    int((
        successful["minimum_change_rate"]
        < successful["stable_change_rate"] - 1e-12
    ).sum())
)
print(
    "Cases with equal change rates:",
    int((
        abs(
            successful["minimum_change_rate"]
            - successful["stable_change_rate"]
        ) <= 1e-12
    ).sum())
)
print("Saved:", comparison_path)

display(
    successful[
        [
            "case_id",
            "stable_change_rate",
            "minimum_change_rate"
        ]
    ].head(10).round(4)
)


Cases compared: 36
Both methods successful: 24
Stable mean change rate: 0.4679
Minimum-change mean change rate: 0.4679
Cases where minimum-change is strictly better: 0
Cases with equal change rates: 24
Saved: /content/planproof_results/experiment19/experiment19_repair_comparison.csv


,case_id,stable_change_rate,minimum_change_rate
0,R_EASY_C1,0.5333,0.5333
1,R_EASY_C2,0.2667,0.2667
2,R_EASY_C3,0.6000,0.6000
3,R_MEDIUM_C1,0.2500,0.2500
4,R_MEDIUM_C2,0.4286,0.4286
5,R_MEDIUM_C3,0.3571,0.3571
9,S_EASY_C1,0.5333,0.5333
10,S_EASY_C2,0.6000,0.6000
11,S_EASY_C3,0.3333,0.3333
12,S_MEDIUM_C1,0.5357,0.5357


In [ ]:

failed_comparisons = comparison[
    (~comparison["stable_success"])
    | (~comparison["minimum_success"])
]

print("Cases not successfully compared:", len(failed_comparisons))

display(
    failed_comparisons[
        ["case_id", "stable_success", "minimum_success"]
    ].reset_index(drop=True)
)


Cases not successfully compared: 12


,case_id,stable_success,minimum_success
0,R_HARD_C1,True,False
1,R_HARD_C2,True,False
2,R_HARD_C3,True,False
3,S_HARD_C1,True,False
4,S_HARD_C2,True,False
5,S_HARD_C3,True,False
6,E_HARD_C1,True,False
7,E_HARD_C2,True,False
8,E_HARD_C3,True,False
9,D_HARD_C1,True,False


In [ ]:

import json
from pathlib import Path

experiment19_dir = Path(
    "/content/planproof_results/experiment19"
)

with open(
    experiment19_dir / "experiment19_controlled_cases.json",
    "r",
    encoding="utf-8"
) as f:
    cases = json.load(f)

hard_cases = [
    case for case in cases
    if case["difficulty"] == "hard"
]

for case in hard_cases:
    print(
        case["case_id"],
        "| Required tasks:", len(case["required_tasks"]),
        "| Original order:", len(case["task_order"])
    )


R_HARD_C1 | Required tasks: 10 | Original order: 10
R_HARD_C2 | Required tasks: 10 | Original order: 10
R_HARD_C3 | Required tasks: 10 | Original order: 10
S_HARD_C1 | Required tasks: 10 | Original order: 10
S_HARD_C2 | Required tasks: 10 | Original order: 10
S_HARD_C3 | Required tasks: 10 | Original order: 10
E_HARD_C1 | Required tasks: 11 | Original order: 11
E_HARD_C2 | Required tasks: 11 | Original order: 11
E_HARD_C3 | Required tasks: 11 | Original order: 11
D_HARD_C1 | Required tasks: 12 | Original order: 12
D_HARD_C2 | Required tasks: 12 | Original order: 12
D_HARD_C3 | Required tasks: 12 | Original order: 12


In [ ]:

%pip -q install ortools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 8.2 MB/s eta 0:00:00


In [ ]:

import ortools
from ortools.linear_solver import pywraplp

print("OR-Tools version:", ortools.__version__)
print("Linear solver module imported successfully.")


OR-Tools version: 9.15.6755
Linear solver module imported successfully.


In [ ]:

from ortools.sat.python import cp_model

def ortools_minimum_change_repair(plan, time_limit_seconds=30):
    required = list(plan["required_tasks"])
    original = list(plan["task_order"])
    constraints = plan["precedence_constraints"]

    if len(required) != len(set(required)):
        return {"success": False, "message": "Duplicate required task IDs"}

    task_set = set(required)
    if any(a not in task_set or b not in task_set for a, b in constraints):
        return {"success": False, "message": "Unknown task in constraints"}

    n = len(required)
    if n == 0:
        return {
            "success": True,
            "optimal": True,
            "repaired_plan": {
                "task_order": [],
                "required_tasks": required,
                "precedence_constraints": constraints
            },
            "change_rate": 0.0
        }

    # Position of each required task in the original order.
    # Missing tasks receive positions after the original list.
    original_position = {}
    for i, task in enumerate(original):
        if task in task_set and task not in original_position:
            original_position[task] = i

    next_position = len(original)
    for task in required:
        if task not in original_position:
            original_position[task] = next_position
            next_position += 1

    model = cp_model.CpModel()
    position = {
        task: model.new_int_var(0, n - 1, f"pos_{i}")
        for i, task in enumerate(required)
    }
    model.add_all_different(list(position.values()))

    # Enforce the precedence constraints.
    for before, after in constraints:
        model.add(position[before] < position[after])

    # Penalize reversing the original order of every pair
    # that appeared in the original plan.
    changes = []

    original_tasks = [
        task for task in original
        if task in task_set
    ]
    original_tasks = list(dict.fromkeys(original_tasks))

    for i in range(len(original_tasks)):
        for j in range(i + 1, len(original_tasks)):
            a = original_tasks[i]
            b = original_tasks[j]

            # If a originally appeared before b, penalize b before a.
            reversed_pair = model.new_bool_var(
                f"reverse_{len(changes)}"
            )
            model.add(
                position[b] < position[a]
            ).only_enforce_if(reversed_pair)
            model.add(
                position[a] < position[b]
            ).only_enforce_if(reversed_pair.Not())
            changes.append(reversed_pair)

    model.minimize(sum(changes))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit_seconds
    status = solver.solve(model)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return {
            "success": False,
            "message": f"No solution returned. Solver status: {solver.status_name(status)}"
        }

    repaired_order = sorted(
        required,
        key=lambda task: solver.value(position[task])
    )

    repaired_plan = {
        "task_order": repaired_order,
        "required_tasks": required,
        "precedence_constraints": constraints
    }

    validation = validate_plan(repaired_plan)
    if not validation["valid"]:
        return {
            "success": False,
            "message": f"Validation failed: {validation['violations']}"
        }

    common = set(original_tasks)
    total_pairs = len(common) * (len(common) - 1) // 2
    rate = (
        solver.objective_value / total_pairs
        if total_pairs else 0.0
    )

    return {
        "success": True,
        "optimal": status == cp_model.OPTIMAL,
        "solver_status": solver.status_name(status),
        "repaired_plan": repaired_plan,
        "change_count": int(round(solver.objective_value)),
        "change_rate": rate
    }


print("OR-Tools minimum-change optimizer defined.")


OR-Tools minimum-change optimizer defined.


In [ ]:

test_case = next(
    item for item in cases
    if item["case_id"] == "E_EASY_C3"
)

test_plan = {
    "task_order": test_case["task_order"],
    "required_tasks": test_case["required_tasks"],
    "precedence_constraints": test_case["precedence_constraints"]
}

optimized = ortools_minimum_change_repair(test_plan)

print("Success:", optimized["success"])

if optimized["success"]:
    print("Solver status:", optimized["solver_status"])
    print("Optimality certified:", optimized["optimal"])
    print("Repaired order:", optimized["repaired_plan"]["task_order"])
    print("Pairwise change count:", optimized["change_count"])
    print("Pairwise change rate:", round(optimized["change_rate"], 4))
    print("Validation:", validate_plan(optimized["repaired_plan"]))
else:
    print("Message:", optimized["message"])


Success: True
Solver status: OPTIMAL
Optimality certified: True
Repaired order: ['review_topic', 'read_notes', 'solve_examples', 'practice_questions', 'check_answers', 'self_test']
Pairwise change count: 11
Pairwise change rate: 0.7333
Validation: {'valid': True, 'violations': []}


In [ ]:

import pandas as pd
from pathlib import Path

comparison_rows = []

for item in cases:
    plan = {
        "task_order": item["task_order"],
        "required_tasks": item["required_tasks"],
        "precedence_constraints": item["precedence_constraints"]
    }

    stable = repair_plan(plan)
    optimized = ortools_minimum_change_repair(
        plan,
        time_limit_seconds=30
    )

    row = {
        "case_id": item["case_id"],
        "difficulty": item["difficulty"],
        "stable_success": stable["success"],
        "optimizer_success": optimized["success"],
        "optimality_certified": optimized.get("optimal", False),
        "stable_change_rate": None,
        "optimal_change_rate": optimized.get("change_rate"),
        "stable_valid": False,
        "optimizer_valid": False
    }

    if stable["success"]:
        stable_check = validate_plan(stable["repaired_plan"])
        row["stable_valid"] = stable_check["valid"]
        row["stable_change_rate"] = pairwise_order_change_rate(
            plan["task_order"],
            stable["repaired_plan"]["task_order"]
        )

    if optimized["success"]:
        row["optimizer_valid"] = validate_plan(
            optimized["repaired_plan"]
        )["valid"]

    comparison_rows.append(row)

ortools_comparison = pd.DataFrame(comparison_rows)

comparison_path = (
    Path("/content/planproof_results/experiment19")
    / "experiment19_ortools_comparison.csv"
)

ortools_comparison.to_csv(comparison_path, index=False)

successful = ortools_comparison[
    ortools_comparison["stable_success"]
    & ortools_comparison["optimizer_success"]
]

print("Total cases:", len(ortools_comparison))
print("Both methods successful:", len(successful))
print(
    "Optimizer optimality certified:",
    int(successful["optimality_certified"].sum()),
    "/",
    len(successful)
)
print(
    "Stable mean change rate:",
    round(successful["stable_change_rate"].mean(), 4)
)
print(
    "Optimizer mean change rate:",
    round(successful["optimal_change_rate"].mean(), 4)
)
print(
    "Cases optimizer improved:",
    int((
        successful["optimal_change_rate"]
        < successful["stable_change_rate"] - 1e-12
    ).sum())
)
print(
    "Cases with equal change rates:",
    int((
        abs(
            successful["optimal_change_rate"]
            - successful["stable_change_rate"]
        ) <= 1e-12
    ).sum())
)
print("Saved:", comparison_path)

display(
    ortools_comparison.groupby("difficulty", sort=False).agg(
        cases=("case_id", "count"),
        optimizer_successes=("optimizer_success", "sum"),
        optimality_proofs=("optimality_certified", "sum")
    ).reset_index()
)


Total cases: 36
Both methods successful: 36
Optimizer optimality certified: 36 / 36
Stable mean change rate: 0.4563
Optimizer mean change rate: 0.4563
Cases optimizer improved: 0
Cases with equal change rates: 36
Saved: /content/planproof_results/experiment19/experiment19_ortools_comparison.csv


,difficulty,cases,optimizer_successes,optimality_proofs
0,easy,12,12,12
1,medium,12,12,12
2,hard,12,12,12


In [ ]:

import pandas as pd
from pathlib import Path

path = Path(
    "/content/planproof_results/experiment19/"
    "experiment19_ortools_comparison.csv"
)

df = pd.read_csv(path)

print("Total records:", len(df))
print("Unique case IDs:", df["case_id"].nunique())
print("All stable repairs successful:", df["stable_success"].all())
print("All optimizer repairs successful:", df["optimizer_success"].all())
print("All stable repairs valid:", df["stable_valid"].all())
print("All optimizer repairs valid:", df["optimizer_valid"].all())
print("All optimality proofs certified:", df["optimality_certified"].all())

differences = (
    df["stable_change_rate"] - df["optimal_change_rate"]
).abs()

print(
    "Cases with different change rates:",
    int((differences > 1e-9).sum())
)

print("Maximum absolute rate difference:", round(differences.max(), 10))

assert len(df) == 36
assert df["case_id"].nunique() == 36
assert df["stable_valid"].all()
assert df["optimizer_valid"].all()
assert df["optimality_certified"].all()
assert (differences <= 1e-9).all()

print("\nPASS: All comparison checks passed.")


Total records: 36
Unique case IDs: 36
All stable repairs successful: True
All optimizer repairs successful: True
All stable repairs valid: True
All optimizer repairs valid: True
All optimality proofs certified: True
Cases with different change rates: 0
Maximum absolute rate difference: 0.0

PASS: All comparison checks passed.


In [ ]:

from pathlib import Path
import zipfile
import json
import pandas as pd

base = Path("/content/planproof_results/experiment19")
assert base.exists(), "Experiment 19 directory not found."

required_files = [
    "experiment19_benchmark_tasks.json",
    "experiment19_config.json",
    "experiment19_controlled_cases.json",
    "experiment19_controlled_results.csv",
    "experiment19_repair_comparison.csv",
    "experiment19_ortools_comparison.csv",
]

missing = [name for name in required_files if not (base / name).exists()]
if missing:
    print("Missing files:", missing)
    print("Stop here and send me this output.")
else:
    df = pd.read_csv(base / "experiment19_ortools_comparison.csv")

    report = f"""# PlanProof AI — Experiment 19

## Experiment type
Controlled synthetic benchmark using shuffled task orders and predefined
precedence constraints. These are not independently collected AI outputs.

## Benchmark
- Cases: {len(df)}
- Unique case IDs: {df['case_id'].nunique()}
- Easy, medium, hard: 12 cases each

## Results
- Stable repair successes: {int(df['stable_success'].sum())}/{len(df)}
- Optimizer successes: {int(df['optimizer_success'].sum())}/{len(df)}
- Stable repairs valid: {int(df['stable_valid'].sum())}/{len(df)}
- Optimizer repairs valid: {int(df['optimizer_valid'].sum())}/{len(df)}
- OR-Tools optimality certificates: {int(df['optimality_certified'].sum())}/{len(df)}
- Stable mean pairwise order-change rate: {df['stable_change_rate'].mean():.4f}
- Optimizer mean pairwise order-change rate: {df['optimal_change_rate'].mean():.4f}
- Cases with different change rates: {int(((df['stable_change_rate'] - df['optimal_change_rate']).abs() > 1e-9).sum())}

## Conclusion
For these 36 controlled cases, stable topological repair matched the
minimum pairwise-order-change objective certified by OR-Tools in every case.

## Limitations
The cases were generated by shuffling task orders with predefined constraints.
The results do not establish generalization to arbitrary plans or real
AI-generated responses. The objective measures pairwise order reversals,
not practical plan quality or user satisfaction.

## Reproducibility
See the included JSON inputs, configuration, and CSV outputs. The restored
implementation should be documented alongside the code before publication.
"""

    report_path = base / "experiment19_report.md"
    report_path.write_text(report, encoding="utf-8")

    zip_path = Path("/content/PlanProof_AI_Experiment19.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for name in required_files:
            z.write(base / name, arcname=f"experiment19/{name}")
        z.write(report_path, arcname="experiment19/experiment19_report.md")

    print("Report saved:", report_path)
    print("Archive saved:", zip_path)
    print("Archive size (bytes):", zip_path.stat().st_size)
    print("Files packaged:", len(required_files) + 1)


Report saved: /content/planproof_results/experiment19/experiment19_report.md
Archive saved: /content/PlanProof_AI_Experiment19.zip
Archive size (bytes): 7439
Files packaged: 7


In [ ]:

from google.colab import files
files.download("/content/PlanProof_AI_Experiment19.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import json
from pathlib import Path

experiment20_dir = Path(
    "/content/planproof_results/experiment20"
)
experiment20_dir.mkdir(parents=True, exist_ok=True)

benchmark = [
    {
        "case_id": "E20_01_MISSING_TASK",
        "category": "missing_task",
        "required_tasks": ["A", "B", "C", "D"],
        "task_order": ["A", "C", "D"],
        "precedence_constraints": [["A", "B"], ["B", "C"]],
        "expected_repair_success": True,
        "expected_valid_after_repair": True
    },
    {
        "case_id": "E20_02_DUPLICATE_TASK",
        "category": "duplicate_task",
        "required_tasks": ["A", "B", "C"],
        "task_order": ["A", "B", "B", "C"],
        "precedence_constraints": [["A", "B"], ["B", "C"]],
        "expected_repair_success": True,
        "expected_valid_after_repair": True
    },
    {
        "case_id": "E20_03_UNKNOWN_TASK",
        "category": "unknown_task",
        "required_tasks": ["A", "B", "C"],
        "task_order": ["A", "X", "B", "C"],
        "precedence_constraints": [["A", "B"], ["B", "C"]],
        "expected_repair_success": True,
        "expected_valid_after_repair": True
    },
    {
        "case_id": "E20_04_BRANCHING_DEPENDENCIES",
        "category": "branching_dependencies",
        "required_tasks": ["A", "B", "C", "D", "E"],
        "task_order": ["D", "C", "B", "A", "E"],
        "precedence_constraints": [
            ["A", "B"], ["A", "C"], ["B", "D"], ["C", "D"], ["D", "E"]
        ],
        "expected_repair_success": True,
        "expected_valid_after_repair": True
    },
    {
        "case_id": "E20_05_CYCLIC_CONSTRAINTS",
        "category": "cyclic_constraints",
        "required_tasks": ["A", "B", "C"],
        "task_order": ["A", "B", "C"],
        "precedence_constraints": [["A", "B"], ["B", "C"], ["C", "A"]],
        "expected_repair_success": False,
        "expected_valid_after_repair": False
    },
    {
        "case_id": "E20_06_UNKNOWN_CONSTRAINT_TASK",
        "category": "unknown_constraint_task",
        "required_tasks": ["A", "B", "C"],
        "task_order": ["A", "B", "C"],
        "precedence_constraints": [["A", "X"]],
        "expected_repair_success": False,
        "expected_valid_after_repair": False
    }
]

benchmark_path = (
    experiment20_dir / "experiment20_benchmark.json"
)

with open(benchmark_path, "w", encoding="utf-8") as f:
    json.dump(benchmark, f, indent=2)

print("Experiment 20 benchmark specification created.")
print("Cases:", len(benchmark))
print("Categories:", sorted({item["category"] for item in benchmark}))
print("Saved:", benchmark_path)


Experiment 20 benchmark specification created.
Cases: 6
Categories: ['branching_dependencies', 'cyclic_constraints', 'duplicate_task', 'missing_task', 'unknown_constraint_task', 'unknown_task']
Saved: /content/planproof_results/experiment20/experiment20_benchmark.json


In [ ]:

import heapq

def validate_plan(plan):
    order = plan["task_order"]
    required = plan["required_tasks"]
    constraints = plan["precedence_constraints"]

    violations = []

    if len(order) != len(set(order)):
        violations.append("Duplicate task IDs")

    order_set = set(order)
    required_set = set(required)

    missing = required_set - order_set
    unknown = order_set - required_set

    if missing:
        violations.append(f"Missing tasks: {sorted(missing)}")

    if unknown:
        violations.append(f"Unknown tasks: {sorted(unknown)}")

    positions = {}
    for i, task_id in enumerate(order):
        positions.setdefault(task_id, i)

    for before, after in constraints:
        if before not in positions or after not in positions:
            violations.append(
                f"Cannot check constraint {before} -> {after}: task missing"
            )
        elif positions[before] >= positions[after]:
            violations.append(
                f"Precedence violation: {before} -> {after}"
            )

    return {
        "valid": len(violations) == 0,
        "violations": violations
    }


def repair_plan(plan):
    required = list(plan["required_tasks"])
    constraints = plan["precedence_constraints"]
    original = list(plan["task_order"])

    required_set = set(required)

    if len(required) != len(required_set):
        return {
            "success": False,
            "repaired_plan": None,
            "message": "Required task IDs contain duplicates"
        }

    for before, after in constraints:
        if before not in required_set or after not in required_set:
            return {
                "success": False,
                "repaired_plan": None,
                "message": "Constraint references an unknown task"
            }

    first_position = {}
    for i, task_id in enumerate(original):
        if task_id in required_set and task_id not in first_position:
            first_position[task_id] = i

    next_position = len(original)
    for task_id in required:
        if task_id not in first_position:
            first_position[task_id] = next_position
            next_position += 1

    adjacency = {task_id: set() for task_id in required}
    indegree = {task_id: 0 for task_id in required}

    for before, after in constraints:
        if after not in adjacency[before]:
            adjacency[before].add(after)
            indegree[after] += 1

    ready = [
        (first_position[task_id], task_id)
        for task_id in required
        if indegree[task_id] == 0
    ]
    heapq.heapify(ready)

    repaired_order = []

    while ready:
        _, current = heapq.heappop(ready)
        repaired_order.append(current)

        for neighbour in adjacency[current]:
            indegree[neighbour] -= 1
            if indegree[neighbour] == 0:
                heapq.heappush(
                    ready, (first_position[neighbour], neighbour)
                )

    if len(repaired_order) != len(required):
        return {
            "success": False,
            "repaired_plan": None,
            "message": "Constraints contain a cycle; no valid order exists"
        }

    repaired = {
        "task_order": repaired_order,
        "required_tasks": required,
        "precedence_constraints": constraints
    }

    check = validate_plan(repaired)

    if not check["valid"]:
        return {
            "success": False,
            "repaired_plan": None,
            "message": f"Post-repair validation failed: {check['violations']}"
        }

    return {
        "success": True,
        "repaired_plan": repaired,
        "message": "Plan repaired and validated"
    }


print("Validator and repair functions restored.")
print("Validator test:", validate_plan({
    "task_order": ["A", "B"],
    "required_tasks": ["A", "B"],
    "precedence_constraints": [["A", "B"]]
}))


Validator and repair functions restored.
Validator test: {'valid': True, 'violations': []}


In [ ]:

import json
import pandas as pd
from pathlib import Path

experiment20_dir = Path(
    "/content/planproof_results/experiment20"
)

with open(
    experiment20_dir / "experiment20_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    benchmark = json.load(f)

rows = []

for case in benchmark:
    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    before = validate_plan(plan)
    repaired = repair_plan(plan)

    if repaired["success"]:
        after = validate_plan(repaired["repaired_plan"])
        valid_after = after["valid"]
        violations_after = len(after["violations"])
    else:
        valid_after = False
        violations_after = None

    rows.append({
        "case_id": case["case_id"],
        "category": case["category"],
        "valid_before": before["valid"],
        "repair_success": repaired["success"],
        "expected_repair_success": case["expected_repair_success"],
        "repair_outcome_correct": (
            repaired["success"] == case["expected_repair_success"]
        ),
        "valid_after": valid_after,
        "expected_valid_after": case["expected_valid_after_repair"],
        "validity_outcome_correct": (
            valid_after == case["expected_valid_after_repair"]
        ),
        "violations_after": violations_after,
        "repair_message": repaired["message"]
    })

results20 = pd.DataFrame(rows)

results_path = (
    experiment20_dir / "experiment20_pilot_results.csv"
)
results20.to_csv(results_path, index=False)

display(results20[
    [
        "case_id",
        "valid_before",
        "repair_success",
        "repair_outcome_correct",
        "valid_after",
        "validity_outcome_correct"
    ]
])

print("\nTotal cases:", len(results20))
print(
    "Correct repair outcomes:",
    int(results20["repair_outcome_correct"].sum()),
    "/",
    len(results20)
)
print(
    "Correct validity outcomes:",
    int(results20["validity_outcome_correct"].sum()),
    "/",
    len(results20)
)
print("Saved:", results_path)


,case_id,valid_before,repair_success,repair_outcome_correct,valid_after,validity_outcome_correct
0,E20_01_MISSING_TASK,False,True,True,True,True
1,E20_02_DUPLICATE_TASK,False,True,True,True,True
2,E20_03_UNKNOWN_TASK,False,True,True,True,True
3,E20_04_BRANCHING_DEPENDENCIES,False,True,True,True,True
4,E20_05_CYCLIC_CONSTRAINTS,False,False,True,False,True
5,E20_06_UNKNOWN_CONSTRAINT_TASK,False,False,True,False,True



Total cases: 6
Correct repair outcomes: 6 / 6
Correct validity outcomes: 6 / 6
Saved: /content/planproof_results/experiment20/experiment20_pilot_results.csv


In [ ]:

import json
from pathlib import Path

experiment20_dir = Path(
    "/content/planproof_results/experiment20"
)

with open(
    experiment20_dir / "experiment20_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    benchmark = json.load(f)

details = []

for case in benchmark:
    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    result = repair_plan(plan)

    if result["success"]:
        repaired = result["repaired_plan"]
        order = repaired["task_order"]

        details.append({
            "case_id": case["case_id"],
            "repair_success": True,
            "all_required_tasks_present": (
                set(order) == set(case["required_tasks"])
                and len(order) == len(set(order))
            ),
            "no_unknown_tasks": (
                set(order) <= set(case["required_tasks"])
            ),
            "precedence_valid": validate_plan(repaired)["valid"],
            "rejection_correct": None
        })
    else:
        details.append({
            "case_id": case["case_id"],
            "repair_success": False,
            "all_required_tasks_present": None,
            "no_unknown_tasks": None,
            "precedence_valid": None,
            "rejection_correct": (
                not case["expected_repair_success"]
            )
        })

import pandas as pd
details_df = pd.DataFrame(details)
display(details_df)

print("\nSuccessful repairs:", int(details_df["repair_success"].sum()))
print(
    "Successful repairs containing every required task:",
    int(details_df["all_required_tasks_present"].fillna(False).sum())
)
print(
    "Successful repairs with valid precedence:",
    int(details_df["precedence_valid"].fillna(False).sum())
)
print(
    "Correct rejections:",
    int(details_df["rejection_correct"].fillna(False).sum())
)


,case_id,repair_success,all_required_tasks_present,no_unknown_tasks,precedence_valid,rejection_correct
0,E20_01_MISSING_TASK,True,True,True,True,None
1,E20_02_DUPLICATE_TASK,True,True,True,True,None
2,E20_03_UNKNOWN_TASK,True,True,True,True,None
3,E20_04_BRANCHING_DEPENDENCIES,True,True,True,True,None
4,E20_05_CYCLIC_CONSTRAINTS,False,None,None,None,True
5,E20_06_UNKNOWN_CONSTRAINT_TASK,False,None,None,None,True



Successful repairs: 4
Successful repairs containing every required task: 4
Successful repairs with valid precedence: 4
Correct rejections: 2


/tmp/ipykernel_1799/2400346040.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  int(details_df["all_required_tasks_present"].fillna(False).sum())
/tmp/ipykernel_1799/2400346040.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  int(details_df["precedence_valid"].fillna(False).sum())
/tmp/ipykernel_1799/2400346040.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future

In [ ]:

from pathlib import Path
import json
import pandas as pd

experiment20_dir = Path(
    "/content/planproof_results/experiment20"
)

# Save the detailed structural verification table.
details_path = (
    experiment20_dir / "experiment20_structural_verification.csv"
)
details_df.to_csv(details_path, index=False)

successful = details_df[details_df["repair_success"] == True]
rejected = details_df[details_df["repair_success"] == False]

report = f"""# PlanProof AI — Experiment 20 Pilot

## Experiment type
Six-case controlled robustness pilot.

## Results
- Total cases: {len(details_df)}
- Successful repairs: {len(successful)}
- Successful repairs containing every required task:
  {int(successful["all_required_tasks_present"].eq(True).sum())}/{len(successful)}
- Successful repairs with valid precedence:
  {int(successful["precedence_valid"].eq(True).sum())}/{len(successful)}
- Correctly rejected impossible cases:
  {int(rejected["rejection_correct"].eq(True).sum())}/{len(rejected)}

## Tested scenarios
Missing tasks, duplicate task IDs, unknown task IDs, branching
dependencies, cyclic constraints, and constraints referencing unknown tasks.

## Interpretation
The recovered validator and repair implementation passed the six
designed pilot cases. Four cases were repaired into valid plans, and two
cases with impossible or invalid constraints were rejected.

## Limitations
This is a small, hand-designed pilot, not a broad robustness evaluation.
The implementation was restored after a runtime reset and should be
versioned with the source code. More varied cases and independent tests
are required before making general claims.
"""

report_path = experiment20_dir / "experiment20_pilot_report.md"
report_path.write_text(report, encoding="utf-8")

print("Structural verification saved:", details_path)
print("Pilot report saved:", report_path)
print("Cases:", len(details_df))
print("All four repairable cases verified:",
      len(successful) == 4
      and successful["all_required_tasks_present"].eq(True).all()
      and successful["precedence_valid"].eq(True).all())
print("Both invalid cases correctly rejected:",
      len(rejected) == 2
      and rejected["rejection_correct"].eq(True).all())


Structural verification saved: /content/planproof_results/experiment20/experiment20_structural_verification.csv
Pilot report saved: /content/planproof_results/experiment20/experiment20_pilot_report.md
Cases: 6
All four repairable cases verified: True
Both invalid cases correctly rejected: True


In [ ]:

import json
from pathlib import Path

experiment20_dir = Path(
    "/content/planproof_results/experiment20"
)
experiment20_dir.mkdir(parents=True, exist_ok=True)

cases20 = []

def add_case(case_id, category, required, order, constraints,
             expected_success):
    cases20.append({
        "case_id": case_id,
        "category": category,
        "required_tasks": required,
        "task_order": order,
        "precedence_constraints": constraints,
        "expected_repair_success": expected_success,
        "expected_valid_after_repair": expected_success
    })

# 1. Independent tasks: any ordering is valid.
for i in range(1, 4):
    tasks = [f"T{j}" for j in range(1, 7)]
    add_case(
        f"E20B_{i:02d}_INDEPENDENT",
        "independent_tasks",
        tasks,
        tasks[i:] + tasks[:i],
        [],
        True
    )

# 2. Branches that merge into one final task.
for i in range(1, 4):
    tasks = ["A", "B", "C", "D", "E", "F"]
    constraints = [
        ["A", "B"], ["A", "C"], ["B", "D"],
        ["C", "E"], ["D", "F"], ["E", "F"]
    ]
    order = tasks[i:] + tasks[:i]
    add_case(
        f"E20B_{i+3:02d}_BRANCH_MERGE",
        "branching_and_merging",
        tasks, order, constraints, True
    )

# 3. Missing tasks combined with duplicated tasks.
for i in range(1, 4):
    tasks = ["A", "B", "C", "D", "E"]
    constraints = [["A", "C"], ["B", "C"], ["C", "D"], ["D", "E"]]
    order = ["B", "A", "B", "D"] if i == 1 else (
        ["D", "A", "C", "C"] if i == 2 else ["E", "B", "A", "B"]
    )
    add_case(
        f"E20B_{i+6:02d}_MISSING_DUPLICATE",
        "missing_and_duplicate",
        tasks, order, constraints, True
    )

# 4. Unknown tasks combined with ordering defects.
for i in range(1, 4):
    tasks = ["A", "B", "C", "D", "E"]
    constraints = [["A", "B"], ["B", "C"], ["C", "D"], ["D", "E"]]
    order = (
        ["X", "C", "A", "D"] if i == 1 else
        ["E", "X", "B", "A"] if i == 2 else
        ["D", "A", "Y", "C", "A"]
    )
    add_case(
        f"E20B_{i+9:02d}_UNKNOWN_MIXED",
        "unknown_task_and_ordering",
        tasks, order, constraints, True
    )

# 5. Cycles of different lengths: repair must reject.
for i, constraints in enumerate([
    [["A", "B"], ["B", "A"]],
    [["A", "B"], ["B", "C"], ["C", "A"]],
    [["A", "B"], ["B", "C"], ["C", "D"], ["D", "A"]]
], start=1):
    tasks = sorted({task for edge in constraints for task in edge})
    add_case(
        f"E20B_{i+12:02d}_CYCLE",
        "cyclic_constraints",
        tasks, tasks, constraints, False
    )

# 6. Constraint references an unknown task: reject.
for i in range(1, 4):
    tasks = ["A", "B", "C", "D"]
    constraints = [["A", "B"], ["B", "X"]]
    if i == 2:
        constraints = [["X", "A"]]
    elif i == 3:
        constraints = [["C", "D"], ["D", "Y"]]

    add_case(
        f"E20B_{i+15:02d}_UNKNOWN_CONSTRAINT",
        "unknown_constraint_task",
        tasks, tasks, constraints, False
    )

# 7. Larger valid plans with branching and merging.
for i, n in enumerate([8, 10, 12, 15, 18, 20], start=1):
    tasks = [f"T{j:02d}" for j in range(1, n + 1)]
    constraints = []

    # A forward chain with additional forward edges.
    for j in range(n - 1):
        constraints.append([tasks[j], tasks[j + 1]])

    for j in range(n - 2):
        constraints.append([tasks[j], tasks[j + 2]])

    order = tasks[::2][::-1] + tasks[1::2][::-1]
    add_case(
        f"E20B_{i+18:02d}_LARGE_DAG",
        "larger_dependency_graph",
        tasks, order, constraints, True
    )

# 8. Combined defects in branching plans.
for i in range(1, 4):
    tasks = ["A", "B", "C", "D", "E", "F", "G"]
    constraints = [
        ["A", "C"], ["B", "C"], ["C", "D"],
        ["C", "E"], ["D", "F"], ["E", "F"], ["F", "G"]
    ]
    order = (
        ["B", "B", "D", "A", "X"] if i == 1 else
        ["G", "C", "A", "C", "E"] if i == 2 else
        ["X", "F", "B", "A", "B", "D"]
    )
    add_case(
        f"E20B_{i+24:02d}_COMBINED_DEFECTS",
        "combined_defects",
        tasks, order, constraints, True
    )

benchmark_path = (
    experiment20_dir / "experiment20_expanded_benchmark.json"
)

with open(benchmark_path, "w", encoding="utf-8") as f:
    json.dump(cases20, f, indent=2)

print("Expanded benchmark created.")
print("Total cases:", len(cases20))
print("Categories:", len(set(c["category"] for c in cases20)))
print("Saved:", benchmark_path)

from collections import Counter
print("\nCases per category:")
for category, count in sorted(
    Counter(c["category"] for c in cases20).items()
):
    print(f"{category}: {count}")


Expanded benchmark created.
Total cases: 27
Categories: 8
Saved: /content/planproof_results/experiment20/experiment20_expanded_benchmark.json

Cases per category:
branching_and_merging: 3
combined_defects: 3
cyclic_constraints: 3
independent_tasks: 3
larger_dependency_graph: 6
missing_and_duplicate: 3
unknown_constraint_task: 3
unknown_task_and_ordering: 3


In [ ]:

import json
from pathlib import Path
from collections import Counter, defaultdict, deque

experiment20_dir = Path(
    "/content/planproof_results/experiment20"
)

benchmark_path = (
    experiment20_dir / "experiment20_expanded_benchmark.json"
)

with open(benchmark_path, "r", encoding="utf-8") as f:
    cases20 = json.load(f)

audit_rows = []
case_ids = [case["case_id"] for case in cases20]
duplicate_case_ids = [
    cid for cid, count in Counter(case_ids).items() if count > 1
]

for case in cases20:
    required = case["required_tasks"]
    required_set = set(required)
    constraints = case["precedence_constraints"]

    errors = []

    if len(required) != len(required_set):
        errors.append("Duplicate required task IDs")

    if not case["case_id"]:
        errors.append("Missing case ID")

    for edge in constraints:
        if not isinstance(edge, list) or len(edge) != 2:
            errors.append(f"Malformed constraint: {edge}")
            continue

        before, after = edge
        if before not in required_set or after not in required_set:
            errors.append(f"Constraint references unknown task: {edge}")

    # Check whether the precedence graph is acyclic.
    adjacency = defaultdict(set)
    indegree = {task: 0 for task in required}

    for edge in constraints:
        if (
            isinstance(edge, list)
            and len(edge) == 2
            and edge[0] in required_set
            and edge[1] in required_set
        ):
            before, after = edge
            if after not in adjacency[before]:
                adjacency[before].add(after)
                indegree[after] += 1

    queue = deque(
        task for task in required if indegree[task] == 0
    )
    visited = 0

    while queue:
        task = queue.popleft()
        visited += 1
        for neighbour in adjacency[task]:
            indegree[neighbour] -= 1
            if indegree[neighbour] == 0:
                queue.append(neighbour)

    acyclic = visited == len(required)

    # The expected repair outcome should match graph feasibility
    # and constraint endpoint validity.
    endpoints_valid = all(
        isinstance(edge, list)
        and len(edge) == 2
        and edge[0] in required_set
        and edge[1] in required_set
        for edge in constraints
    )

    expected_success = endpoints_valid and acyclic
    expectation_consistent = (
        case["expected_repair_success"] == expected_success
    )

    if not acyclic:
        errors.append("Dependency cycle detected")

    if not endpoints_valid:
        errors.append("Invalid constraint endpoint")

    if not expectation_consistent:
        errors.append("Expected outcome disagrees with graph feasibility")

    audit_rows.append({
        "case_id": case["case_id"],
        "category": case["category"],
        "task_count": len(required),
        "constraints_acyclic": acyclic,
        "constraint_endpoints_valid": endpoints_valid,
        "expected_success": case["expected_repair_success"],
        "expectation_consistent": expectation_consistent,
        "audit_pass": len(errors) == 0,
        "audit_notes": "; ".join(errors)
    })

import pandas as pd

audit20 = pd.DataFrame(audit_rows)
audit_path = (
    experiment20_dir / "experiment20_expanded_audit.csv"
)
audit20.to_csv(audit_path, index=False)

print("Total cases:", len(audit20))
print("Unique case IDs:", len(set(case_ids)))
print("Duplicate case IDs:", duplicate_case_ids)
print("Audit passes:", int(audit20["audit_pass"].sum()))
print("Audit failures:", int((~audit20["audit_pass"]).sum()))
print("Saved:", audit_path)

display(
    audit20.loc[
        ~audit20["audit_pass"],
        [
            "case_id",
            "category",
            "expected_success",
            "audit_notes"
        ]
    ]
)


Total cases: 27
Unique case IDs: 27
Duplicate case IDs: []
Audit passes: 21
Audit failures: 6
Saved: /content/planproof_results/experiment20/experiment20_expanded_audit.csv


,case_id,category,expected_success,audit_notes
12,E20B_13_CYCLE,cyclic_constraints,False,Dependency cycle detected
13,E20B_14_CYCLE,cyclic_constraints,False,Dependency cycle detected
14,E20B_15_CYCLE,cyclic_constraints,False,Dependency cycle detected
15,E20B_16_UNKNOWN_CONSTRAINT,unknown_constraint_task,False,"Constraint references unknown task: ['B', 'X']..."
16,E20B_17_UNKNOWN_CONSTRAINT,unknown_constraint_task,False,"Constraint references unknown task: ['X', 'A']..."
17,E20B_18_UNKNOWN_CONSTRAINT,unknown_constraint_task,False,"Constraint references unknown task: ['D', 'Y']..."


In [ ]:

import json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict, deque

experiment20_dir = Path(
    "/content/planproof_results/experiment20"
)

with open(
    experiment20_dir / "experiment20_expanded_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    cases20 = json.load(f)

rows = []

for case in cases20:
    required = case["required_tasks"]
    required_set = set(required)
    constraints = case["precedence_constraints"]
    notes = []

    if len(required) != len(required_set):
        notes.append("Duplicate required task IDs")

    endpoints_valid = all(
        isinstance(edge, list)
        and len(edge) == 2
        and edge[0] in required_set
        and edge[1] in required_set
        for edge in constraints
    )

    adjacency = defaultdict(set)
    indegree = {task: 0 for task in required}

    for edge in constraints:
        if (
            isinstance(edge, list)
            and len(edge) == 2
            and edge[0] in required_set
            and edge[1] in required_set
        ):
            before, after = edge
            if after not in adjacency[before]:
                adjacency[before].add(after)
                indegree[after] += 1

    queue = deque(
        task for task in required if indegree[task] == 0
    )
    visited = 0

    while queue:
        task = queue.popleft()
        visited += 1
        for neighbour in adjacency[task]:
            indegree[neighbour] -= 1
            if indegree[neighbour] == 0:
                queue.append(neighbour)

    acyclic = visited == len(required)

    expected_success = endpoints_valid and acyclic
    expected_matches_graph = (
        case["expected_repair_success"] == expected_success
    )

    if not expected_matches_graph:
        notes.append("Expected outcome conflicts with graph feasibility")

    # A cycle or unknown endpoint is not itself an audit failure:
    # it is valid when the case explicitly expects rejection.
    if len(required) != len(required_set):
        passed = False
    else:
        passed = expected_matches_graph

    rows.append({
        "case_id": case["case_id"],
        "category": case["category"],
        "task_count": len(required),
        "endpoints_valid": endpoints_valid,
        "acyclic": acyclic,
        "expected_repair_success": case["expected_repair_success"],
        "expectation_matches_graph": expected_matches_graph,
        "audit_pass": passed,
        "audit_notes": "; ".join(notes)
    })

audit20 = pd.DataFrame(rows)
audit_path = (
    experiment20_dir / "experiment20_expanded_audit.csv"
)
audit20.to_csv(audit_path, index=False)

print("Total cases:", len(audit20))
print("Unique case IDs:", audit20["case_id"].nunique())
print("Audit passes:", int(audit20["audit_pass"].sum()))
print("Audit failures:", int((~audit20["audit_pass"]).sum()))
print("Saved:", audit_path)

display(
    audit20.loc[
        ~audit20["audit_pass"],
        ["case_id", "audit_notes"]
    ]
)


Total cases: 27
Unique case IDs: 27
Audit passes: 27
Audit failures: 0
Saved: /content/planproof_results/experiment20/experiment20_expanded_audit.csv


,case_id,audit_notes


In [ ]:

import json
import pandas as pd
from pathlib import Path

experiment20_dir = Path(
    "/content/planproof_results/experiment20"
)

with open(
    experiment20_dir / "experiment20_expanded_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    cases20 = json.load(f)

rows = []

for case in cases20:
    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    before = validate_plan(plan)
    result = repair_plan(plan)

    expected_success = case["expected_repair_success"]
    success_correct = result["success"] == expected_success

    if result["success"]:
        repaired = result["repaired_plan"]
        after = validate_plan(repaired)
        order = repaired["task_order"]

        all_required_present = (
            set(order) == set(case["required_tasks"])
            and len(order) == len(set(order))
        )
        no_unknown_tasks = set(order) <= set(case["required_tasks"])
        precedence_valid = after["valid"]
        violations_after = len(after["violations"])
    else:
        all_required_present = None
        no_unknown_tasks = None
        precedence_valid = None
        violations_after = None

    rows.append({
        "case_id": case["case_id"],
        "category": case["category"],
        "task_count": len(case["required_tasks"]),
        "valid_before": before["valid"],
        "repair_success": result["success"],
        "expected_success": expected_success,
        "success_outcome_correct": success_correct,
        "all_required_tasks_present": all_required_present,
        "no_unknown_tasks": no_unknown_tasks,
        "precedence_valid": precedence_valid,
        "violations_after": violations_after,
        "repair_message": result["message"]
    })

results20_expanded = pd.DataFrame(rows)

results_path = (
    experiment20_dir / "experiment20_expanded_results.csv"
)
results20_expanded.to_csv(results_path, index=False)

print("Total cases:", len(results20_expanded))
print(
    "Correct repair/rejection outcomes:",
    int(results20_expanded["success_outcome_correct"].sum()),
    "/", len(results20_expanded)
)

successful = results20_expanded[
    results20_expanded["repair_success"]
]

rejected = results20_expanded[
    ~results20_expanded["repair_success"]
]

print("Successful repairs:", len(successful))
print(
    "Successful repairs preserving all required tasks:",
    int(successful["all_required_tasks_present"].eq(True).sum()),
    "/", len(successful)
)
print(
    "Successful repairs with valid precedence:",
    int(successful["precedence_valid"].eq(True).sum()),
    "/", len(successful)
)
print(
    "Expected rejections:",
    int((~results20_expanded["expected_success"]).sum())
)
print("Saved:", results_path)

print("\nResults by category:")
display(
    results20_expanded.groupby("category", sort=False).agg(
        cases=("case_id", "count"),
        successful_repairs=("repair_success", "sum"),
        correct_outcomes=("success_outcome_correct", "sum")
    ).reset_index()
)

print("\nCases with unexpected outcomes:")
display(
    results20_expanded.loc[
        ~results20_expanded["success_outcome_correct"],
        ["case_id", "category", "expected_success",
         "repair_success", "repair_message"]
    ]
)


Total cases: 27
Correct repair/rejection outcomes: 27 / 27
Successful repairs: 21
Successful repairs preserving all required tasks: 21 / 21
Successful repairs with valid precedence: 21 / 21
Expected rejections: 6
Saved: /content/planproof_results/experiment20/experiment20_expanded_results.csv

Results by category:


,category,cases,successful_repairs,correct_outcomes
0,independent_tasks,3,3,3
1,branching_and_merging,3,3,3
2,missing_and_duplicate,3,3,3
3,unknown_task_and_ordering,3,3,3
4,cyclic_constraints,3,0,3
5,unknown_constraint_task,3,0,3
6,larger_dependency_graph,6,6,6
7,combined_defects,3,3,3



Cases with unexpected outcomes:


,case_id,category,expected_success,repair_success,repair_message


In [ ]:

from pathlib import Path
import zipfile

experiment20_dir = Path(
    "/content/planproof_results/experiment20"
)

files_to_package = [
    "experiment20_benchmark.json",
    "experiment20_pilot_results.csv",
    "experiment20_structural_verification.csv",
    "experiment20_pilot_report.md",
    "experiment20_expanded_benchmark.json",
    "experiment20_expanded_audit.csv",
    "experiment20_expanded_results.csv",
]

missing = [
    name for name in files_to_package
    if not (experiment20_dir / name).exists()
]

if missing:
    print("Missing files:", missing)
else:
    archive_path = Path(
        "/content/PlanProof_AI_Experiment20.zip"
    )

    with zipfile.ZipFile(
        archive_path, "w", zipfile.ZIP_DEFLATED
    ) as archive:
        for name in files_to_package:
            archive.write(
                experiment20_dir / name,
                arcname=f"experiment20/{name}"
            )

    print("Archive created:", archive_path)
    print("Files packaged:", len(files_to_package))
    print("Archive size (bytes):", archive_path.stat().st_size)


Archive created: /content/PlanProof_AI_Experiment20.zip
Files packaged: 7
Archive size (bytes): 4764


In [ ]:

import json
import random
from pathlib import Path

SEED21 = 20261021
NUM_CASES21 = 100

rng21 = random.Random(SEED21)

experiment21_dir = Path(
    "/content/planproof_results/experiment21"
)
experiment21_dir.mkdir(parents=True, exist_ok=True)

cases21 = []

for i in range(NUM_CASES21):
    # Vary graph size between 5 and 20 tasks.
    n = rng21.randint(5, 20)
    tasks = [f"T{j:02d}" for j in range(n)]

    # A random topological order provides a guaranteed DAG.
    topological_order = tasks[:]
    rng21.shuffle(topological_order)

    # Add forward edges only, guaranteeing no cycles.
    constraints = set()

    # Ensure every task after the first has at least one predecessor.
    for j in range(1, n):
        predecessor_index = rng21.randrange(j)
        constraints.add((
            topological_order[predecessor_index],
            topological_order[j]
        ))

    # Add extra edges to create varied dependency structures.
    for j in range(n):
        for k in range(j + 1, n):
            if rng21.random() < 0.12:
                constraints.add((
                    topological_order[j],
                    topological_order[k]
                ))

    constraints = sorted(constraints)

    # Create a shuffled candidate plan.
    candidate = tasks[:]
    rng21.shuffle(candidate)

    # Inject one controlled defect type per case.
    defect = rng21.choice([
        "shuffled_order",
        "missing_task",
        "duplicate_task",
        "unknown_task",
        "mixed_defects"
    ])

    if defect in ("missing_task", "mixed_defects"):
        candidate.remove(rng21.choice(candidate))

    if defect in ("duplicate_task", "mixed_defects"):
        task_to_duplicate = rng21.choice(candidate)
        candidate.insert(
            rng21.randrange(len(candidate) + 1),
            task_to_duplicate
        )

    if defect in ("unknown_task", "mixed_defects"):
        candidate.insert(
            rng21.randrange(len(candidate) + 1),
            f"UNKNOWN_{i:03d}"
        )

    cases21.append({
        "case_id": f"E21_{i+1:03d}",
        "seed": SEED21,
        "task_count": n,
        "defect_type": defect,
        "required_tasks": tasks,
        "task_order": candidate,
        "precedence_constraints": [
            [before, after]
            for before, after in constraints
        ]
    })

benchmark21_path = (
    experiment21_dir / "experiment21_randomized_benchmark.json"
)

with open(benchmark21_path, "w", encoding="utf-8") as f:
    json.dump(cases21, f, indent=2)

print("Experiment 21 benchmark generated.")
print("Cases:", len(cases21))
print("Random seed:", SEED21)
print("Minimum task count:", min(c["task_count"] for c in cases21))
print("Maximum task count:", max(c["task_count"] for c in cases21))
print("Saved:", benchmark21_path)

from collections import Counter
print("\nDefect distribution:")
for defect, count in sorted(
    Counter(c["defect_type"] for c in cases21).items()
):
    print(f"{defect}: {count}")


Experiment 21 benchmark generated.
Cases: 100
Random seed: 20261021
Minimum task count: 5
Maximum task count: 20
Saved: /content/planproof_results/experiment21/experiment21_randomized_benchmark.json

Defect distribution:
duplicate_task: 16
missing_task: 24
mixed_defects: 13
shuffled_order: 25
unknown_task: 22


In [ ]:

import json
from pathlib import Path
from collections import Counter, defaultdict, deque

experiment21_dir = Path(
    "/content/planproof_results/experiment21"
)

benchmark_path = (
    experiment21_dir / "experiment21_randomized_benchmark.json"
)

with open(benchmark_path, "r", encoding="utf-8") as f:
    cases21 = json.load(f)

audit_rows21 = []

case_ids = [case["case_id"] for case in cases21]

for case in cases21:
    required = case["required_tasks"]
    order = case["task_order"]
    constraints = case["precedence_constraints"]
    defect = case["defect_type"]

    required_set = set(required)
    order_set = set(order)
    issues = []

    if len(required) != len(required_set):
        issues.append("Duplicate required task IDs")

    if len(order) != len(order_set):
        has_duplicate = True
    else:
        has_duplicate = False

    missing = required_set - order_set
    unknown = order_set - required_set

    # Check all constraint endpoints.
    endpoints_valid = all(
        len(edge) == 2
        and edge[0] in required_set
        and edge[1] in required_set
        for edge in constraints
    )

    # Independently check acyclicity with Kahn's algorithm.
    adjacency = defaultdict(set)
    indegree = {task: 0 for task in required}

    for before, after in constraints:
        if before in required_set and after in required_set:
            if after not in adjacency[before]:
                adjacency[before].add(after)
                indegree[after] += 1

    queue = deque(
        task for task in required if indegree[task] == 0
    )
    visited = 0

    while queue:
        task = queue.popleft()
        visited += 1

        for neighbour in adjacency[task]:
            indegree[neighbour] -= 1
            if indegree[neighbour] == 0:
                queue.append(neighbour)

    acyclic = visited == len(required)

    # Verify the defect label.
    label_correct = {
        "shuffled_order": (
            not missing and not unknown and not has_duplicate
        ),
        "missing_task": (
            bool(missing) and not unknown and not has_duplicate
        ),
        "duplicate_task": (
            has_duplicate and not missing and not unknown
        ),
        "unknown_task": (
            bool(unknown) and not missing and not has_duplicate
        ),
        "mixed_defects": (
            bool(missing) and bool(unknown) and has_duplicate
        )
    }.get(defect, False)

    if not endpoints_valid:
        issues.append("Invalid constraint endpoint")

    if not acyclic:
        issues.append("Cycle detected")

    if not label_correct:
        issues.append("Defect label does not match actual order")

    audit_rows21.append({
        "case_id": case["case_id"],
        "task_count": len(required),
        "defect_type": defect,
        "has_missing": bool(missing),
        "has_duplicate": has_duplicate,
        "has_unknown": bool(unknown),
        "endpoints_valid": endpoints_valid,
        "acyclic": acyclic,
        "label_correct": label_correct,
        "audit_pass": len(issues) == 0,
        "audit_notes": "; ".join(issues)
    })

duplicate_case_ids = [
    case_id for case_id, count in Counter(case_ids).items()
    if count > 1
]

import pandas as pd

audit21 = pd.DataFrame(audit_rows21)

audit_path = (
    experiment21_dir / "experiment21_randomized_audit.csv"
)
audit21.to_csv(audit_path, index=False)

print("Total cases:", len(audit21))
print("Unique case IDs:", audit21["case_id"].nunique())
print("Duplicate case IDs:", duplicate_case_ids)
print("Acyclic graphs:", int(audit21["acyclic"].sum()))
print("Valid constraint endpoints:", int(audit21["endpoints_valid"].sum()))
print("Correct defect labels:", int(audit21["label_correct"].sum()))
print("Audit passes:", int(audit21["audit_pass"].sum()))
print("Audit failures:", int((~audit21["audit_pass"]).sum()))
print("Saved:", audit_path)

display(
    audit21.loc[
        ~audit21["audit_pass"],
        [
            "case_id", "defect_type",
            "has_missing", "has_duplicate", "has_unknown",
            "audit_notes"
        ]
    ].head(20)
)


Total cases: 100
Unique case IDs: 100
Duplicate case IDs: []
Acyclic graphs: 100
Valid constraint endpoints: 100
Correct defect labels: 100
Audit passes: 100
Audit failures: 0
Saved: /content/planproof_results/experiment21/experiment21_randomized_audit.csv


,case_id,defect_type,has_missing,has_duplicate,has_unknown,audit_notes


In [ ]:

import json
import pandas as pd
from pathlib import Path
from collections import Counter

# Experiment 21 paths
experiment21_dir = Path("/content/planproof_results/experiment21")
benchmark_path = experiment21_dir / "experiment21_randomized_benchmark.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    cases21 = json.load(f)

results21 = []

for case in cases21:
    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    # Validate before repair
    before = validate_plan(plan)

    # Attempt repair
    repair_result = repair_plan(plan)

    # Extract the repaired plan
    repaired = repair_result.get("repaired_plan")
    repair_success = (
        repair_result.get("success", False)
        and isinstance(repaired, dict)
    )

    # Independently validate the returned plan
    if repair_success:
        after = validate_plan(repaired)
        repaired_order = repaired.get("task_order", [])
        required = case["required_tasks"]

        structural_ok = (
            len(repaired_order) == len(set(repaired_order))
            and set(repaired_order) == set(required)
            and len(repaired_order) == len(required)
        )

        final_valid = after.get("valid", False)
        violations_after = len(after.get("violations", []))
    else:
        structural_ok = False
        final_valid = False
        violations_after = None

    results21.append({
        "case_id": case["case_id"],
        "defect_type": case["defect_type"],
        "task_count": case["task_count"],
        "valid_before": before.get("valid", False),
        "violations_before": len(before.get("violations", [])),
        "repair_success": repair_success,
        "structural_ok": structural_ok,
        "valid_after": final_valid,
        "violations_after": violations_after
    })

results_df21 = pd.DataFrame(results21)

# Save detailed results
results_path = experiment21_dir / "experiment21_repair_results.csv"
results_df21.to_csv(results_path, index=False)

# Overall summary
total = len(results_df21)
print("EXPERIMENT 21 — REPAIR EVALUATION")
print("=" * 45)
print("Total cases:", total)
print("Initially valid:", int(results_df21["valid_before"].sum()))
print("Initially invalid:", int((~results_df21["valid_before"]).sum()))
print("Repair successes:", int(results_df21["repair_success"].sum()))
print("Structurally correct repairs:", int(results_df21["structural_ok"].sum()))
print("Valid after repair:", int(results_df21["valid_after"].sum()))
print("Remaining violations:",
      int(results_df21["violations_after"].fillna(0).sum()))
print("Results saved:", results_path)

# Breakdown by injected defect category
summary21 = results_df21.groupby("defect_type").agg(
    cases=("case_id", "count"),
    initially_valid=("valid_before", "sum"),
    repair_successes=("repair_success", "sum"),
    structurally_correct=("structural_ok", "sum"),
    valid_after_repair=("valid_after", "sum")
).reset_index()

print("\nRESULTS BY DEFECT CATEGORY")
display(summary21)

# Show any cases that need investigation
problems21 = results_df21[
    (~results_df21["repair_success"])
    | (~results_df21["structural_ok"])
    | (~results_df21["valid_after"])
]

print("\nCASES REQUIRING INVESTIGATION:", len(problems21))
display(problems21.head(20))


EXPERIMENT 21 — REPAIR EVALUATION
Total cases: 100
Initially valid: 0
Initially invalid: 100
Repair successes: 100
Structurally correct repairs: 100
Valid after repair: 100
Remaining violations: 0
Results saved: /content/planproof_results/experiment21/experiment21_repair_results.csv

RESULTS BY DEFECT CATEGORY


,defect_type,cases,initially_valid,repair_successes,structurally_correct,valid_after_repair
0,duplicate_task,16,0,16,16,16
1,missing_task,24,0,24,24,24
2,mixed_defects,13,0,13,13,13
3,shuffled_order,25,0,25,25,25
4,unknown_task,22,0,22,22,22



CASES REQUIRING INVESTIGATION: 0


,case_id,defect_type,task_count,valid_before,violations_before,repair_success,structural_ok,valid_after,violations_after


In [ ]:

from pathlib import Path
import json
import zipfile
import pandas as pd

base = Path("/content/planproof_results/experiment21")
benchmark = pd.read_json(base / "experiment21_randomized_benchmark.json")
audit = pd.read_csv(base / "experiment21_randomized_audit.csv")
results = pd.read_csv(base / "experiment21_repair_results.csv")

total = len(results)
successes = int(results["repair_success"].sum())
structural = int(results["structural_ok"].sum())
valid_after = int(results["valid_after"].sum())
remaining = int(results["violations_after"].fillna(0).sum())

by_defect = results.groupby("defect_type").agg(
    cases=("case_id", "count"),
    repairs_succeeded=("repair_success", "sum"),
    valid_after_repair=("valid_after", "sum")
)

report = f"""# PlanProof AI — Experiment 21

## Randomized Robustness Evaluation

### Objective
Evaluate the plan-repair implementation on a reproducible,
synthetically generated benchmark with randomized task orders
and dependency graphs.

### Experimental setup
- Random seed: 20261021
- Cases generated: {total}
- Tasks per case: 5–20
- Benchmark audit passes: {int(audit["audit_pass"].sum())}
- Defect categories: {results["defect_type"].nunique()}

### Overall results
- Repair success: {successes}/{total}
- Structurally correct outputs: {structural}/{total}
- Valid plans after repair: {valid_after}/{total}
- Remaining violations: {remaining}

### Results by defect category

{by_defect.to_markdown()}

### Interpretation
The implementation repaired all {total} cases in this synthetic
benchmark. All returned plans were structurally correct and passed
post-repair validation, with zero remaining reported violations.

These findings support performance on this particular benchmark.
They do not establish performance on real-world AI-generated plans,
all possible dependency graphs, or superiority over competing methods.

### Reproducibility and limitations
The benchmark uses a fixed random seed. Preserve the benchmark JSON,
audit CSV, evaluation CSV, and implementation version to reproduce
the experiment. Record the exact code version and software environment
before publication.

The benchmark's defect categories describe injected structural
defects and randomized task ordering. They should not be interpreted
as representative of every real-world planning failure.
"""

report_path = base / "experiment21_report.md"
report_path.write_text(report, encoding="utf-8")

zip_path = Path("/content/PlanProof_AI_Experiment21.zip")
files_to_package = [
    base / "experiment21_randomized_benchmark.json",
    base / "experiment21_randomized_audit.csv",
    base / "experiment21_repair_results.csv",
    report_path
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in files_to_package:
        if path.exists():
            zf.write(path, arcname=path.name)

print("Experiment 21 report saved:", report_path)
print("Research package saved:", zip_path)
print("Files included:", len(files_to_package))
print("\nDefect category summary:")
display(by_defect.reset_index())


Experiment 21 report saved: /content/planproof_results/experiment21/experiment21_report.md
Research package saved: /content/PlanProof_AI_Experiment21.zip
Files included: 4

Defect category summary:


,defect_type,cases,repairs_succeeded,valid_after_repair
0,duplicate_task,16,16,16
1,missing_task,24,24,24
2,mixed_defects,13,13,13
3,shuffled_order,25,25,25
4,unknown_task,22,22,22


In [ ]:
print("validate_plan available:", callable(globals().get("validate_plan")))
print("repair_plan available:", callable(globals().get("repair_plan")))
print(
    "pairwise_order_change_rate available:",
    callable(globals().get("pairwise_order_change_rate"))
)

validate_plan available: True
repair_plan available: True
pairwise_order_change_rate available: False


In [ ]:

def pairwise_order_change_rate(original_order, repaired_order):
    """
    Measures how many pairwise relative orders change
    between the original and repaired task sequences.

    Only tasks present in both sequences are compared.
    """
    original = list(original_order)
    repaired = list(repaired_order)

    common = set(original) & set(repaired)
    original = [task for task in original if task in common]
    repaired = [task for task in repaired if task in common]

    n = len(common)
    if n < 2:
        return 0.0

    original_position = {
        task: index for index, task in enumerate(original)
    }
    repaired_position = {
        task: index for index, task in enumerate(repaired)
    }

    changed_pairs = 0
    total_pairs = n * (n - 1) // 2

    for i, task_a in enumerate(original):
        for task_b in original[i + 1:]:
            if repaired_position[task_a] > repaired_position[task_b]:
                changed_pairs += 1

    return changed_pairs / total_pairs


# Quick sanity checks
print("Metric available:", callable(pairwise_order_change_rate))
print("Identical order:", pairwise_order_change_rate(
    ["A", "B", "C"], ["A", "B", "C"]
))
print("Completely reversed:", pairwise_order_change_rate(
    ["A", "B", "C"], ["C", "B", "A"]
))


Metric available: True
Identical order: 0.0
Completely reversed: 1.0


In [ ]:

import json
import pandas as pd
from pathlib import Path

experiment21_dir = Path("/content/planproof_results/experiment21")
experiment22_dir = Path("/content/planproof_results/experiment22")
experiment22_dir.mkdir(parents=True, exist_ok=True)

with open(
    experiment21_dir / "experiment21_randomized_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    cases22 = json.load(f)

results22 = []

for case in cases22:
    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    repair_result = repair_plan(plan)
    repaired = repair_result.get("repaired_plan")

    if (
        not repair_result.get("success", False)
        or not isinstance(repaired, dict)
    ):
        results22.append({
            "case_id": case["case_id"],
            "defect_type": case["defect_type"],
            "task_count": case["task_count"],
            "repair_success": False,
            "order_change_rate": None
        })
        continue

    rate = pairwise_order_change_rate(
        case["task_order"],
        repaired["task_order"]
    )

    results22.append({
        "case_id": case["case_id"],
        "defect_type": case["defect_type"],
        "task_count": case["task_count"],
        "repair_success": True,
        "order_change_rate": rate
    })

results22_df = pd.DataFrame(results22)

output_path = experiment22_dir / "experiment22_order_preservation.csv"
results22_df.to_csv(output_path, index=False)

successful = results22_df[
    results22_df["repair_success"]
    & results22_df["order_change_rate"].notna()
]

print("EXPERIMENT 22 — ORDER PRESERVATION")
print("=" * 42)
print("Total cases:", len(results22_df))
print("Successful repairs:", len(successful))
print("Failed repairs:", int((~results22_df["repair_success"]).sum()))

if len(successful):
    print(
        "Mean order-change rate:",
        round(successful["order_change_rate"].mean(), 4)
    )
    print(
        "Median order-change rate:",
        round(successful["order_change_rate"].median(), 4)
    )
    print(
        "Minimum order-change rate:",
        round(successful["order_change_rate"].min(), 4)
    )
    print(
        "Maximum order-change rate:",
        round(successful["order_change_rate"].max(), 4)
    )

print("Results saved:", output_path)

print("\nRESULTS BY DEFECT CATEGORY")
display(
    successful.groupby("defect_type")["order_change_rate"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)


EXPERIMENT 22 — ORDER PRESERVATION
Total cases: 100
Successful repairs: 100
Failed repairs: 0
Mean order-change rate: 0.4229
Median order-change rate: 0.4026
Minimum order-change rate: 0.0
Maximum order-change rate: 1.0667
Results saved: /content/planproof_results/experiment22/experiment22_order_preservation.csv

RESULTS BY DEFECT CATEGORY


,defect_type,count,mean,median,min,max
0,duplicate_task,16,0.452847,0.438734,0.333333,0.700000
1,missing_task,24,0.429213,0.408425,0.142857,0.800000
2,mixed_defects,13,0.584044,0.523810,0.380117,1.066667
3,shuffled_order,25,0.381290,0.380952,0.136364,0.555556
4,unknown_task,22,0.346352,0.368904,0.000000,0.600000


In [ ]:

def pairwise_order_change_rate(original_order, repaired_order):
    # Keep each task's first occurrence
    original_unique = list(dict.fromkeys(original_order))
    repaired_unique = list(dict.fromkeys(repaired_order))

    # Compare only tasks present in both sequences
    common = set(original_unique) & set(repaired_unique)
    original = [t for t in original_unique if t in common]
    repaired = [t for t in repaired_unique if t in common]

    n = len(original)
    if n < 2:
        return 0.0

    repaired_position = {
        task: i for i, task in enumerate(repaired)
    }

    changed_pairs = 0
    total_pairs = n * (n - 1) // 2

    for i in range(n):
        for j in range(i + 1, n):
            a, b = original[i], original[j]

            if repaired_position[a] > repaired_position[b]:
                changed_pairs += 1

    return changed_pairs / total_pairs


# Sanity checks
print("Identical:", pairwise_order_change_rate(
    ["A", "B", "C"], ["A", "B", "C"]
))
print("Reversed:", pairwise_order_change_rate(
    ["A", "B", "C"], ["C", "B", "A"]
))
print("Duplicate input:", pairwise_order_change_rate(
    ["A", "A", "B", "C"], ["B", "A", "C"]
))


Identical: 0.0
Reversed: 1.0
Duplicate input: 0.3333333333333333


In [ ]:

import json
import pandas as pd
from pathlib import Path

benchmark_dir = Path("/content/planproof_results/experiment21")
results_dir = Path("/content/planproof_results/experiment22")

with open(
    benchmark_dir / "experiment21_randomized_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    cases22 = json.load(f)

corrected_rows = []

for case in cases22:
    original_order = case["task_order"]

    plan = {
        "task_order": original_order,
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    result = repair_plan(plan)
    repaired = result.get("repaired_plan")

    success = (
        result.get("success", False)
        and isinstance(repaired, dict)
    )

    if success:
        rate = pairwise_order_change_rate(
            original_order,
            repaired["task_order"]
        )
        final_validation = validate_plan(repaired)
        valid_after = final_validation.get("valid", False)
    else:
        rate = None
        valid_after = False

    corrected_rows.append({
        "case_id": case["case_id"],
        "defect_type": case["defect_type"],
        "task_count": case["task_count"],
        "repair_success": success,
        "valid_after_repair": valid_after,
        "order_change_rate": rate
    })

corrected_df = pd.DataFrame(corrected_rows)

output_path = results_dir / "experiment22_order_preservation.csv"
corrected_df.to_csv(output_path, index=False)

successful = corrected_df[
    corrected_df["repair_success"]
    & corrected_df["valid_after_repair"]
    & corrected_df["order_change_rate"].notna()
]

print("EXPERIMENT 22 — CORRECTED RESULTS")
print("=" * 42)
print("Total cases:", len(corrected_df))
print("Successful and valid repairs:", len(successful))
print("Failed or invalid repairs:", len(corrected_df) - len(successful))
print("Mean order-change rate:", round(
    successful["order_change_rate"].mean(), 4
))
print("Median order-change rate:", round(
    successful["order_change_rate"].median(), 4
))
print("Minimum:", round(successful["order_change_rate"].min(), 4))
print("Maximum:", round(successful["order_change_rate"].max(), 4))
print("All rates within [0, 1]:", bool(
    successful["order_change_rate"].between(0, 1).all()
))
print("Saved:", output_path)

print("\nBY DEFECT CATEGORY")
display(
    successful.groupby("defect_type")["order_change_rate"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)


EXPERIMENT 22 — CORRECTED RESULTS
Total cases: 100
Successful and valid repairs: 100
Failed or invalid repairs: 0
Mean order-change rate: 0.393
Median order-change rate: 0.381
Minimum: 0.0
Maximum: 0.8
All rates within [0, 1]: True
Saved: /content/planproof_results/experiment22/experiment22_order_preservation.csv

BY DEFECT CATEGORY


,defect_type,count,mean,median,min,max
0,duplicate_task,16,0.372154,0.373483,0.247619,0.573529
1,missing_task,24,0.429213,0.408425,0.142857,0.800000
2,mixed_defects,13,0.453370,0.438095,0.238095,0.800000
3,shuffled_order,25,0.381290,0.380952,0.136364,0.555556
4,unknown_task,22,0.346352,0.368904,0.000000,0.600000


In [ ]:

from pathlib import Path
import json
import ortools

benchmark_path = Path(
    "/content/planproof_results/experiment21/"
    "experiment21_randomized_benchmark.json"
)

with open(benchmark_path, "r", encoding="utf-8") as f:
    cases23 = json.load(f)

print("OR-Tools version:", ortools.__version__)
print("Benchmark loaded:", benchmark_path.exists())
print("Total cases:", len(cases23))
print("First case ID:", cases23[0]["case_id"])
print("Tasks in first case:", len(cases23[0]["required_tasks"]))
print("Pairwise metric available:",
      callable(globals().get("pairwise_order_change_rate")))


ModuleNotFoundError: No module named 'ortools'

In [ ]:

%pip install ortools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:

import ortools
print("OR-Tools version:", ortools.__version__)

from pathlib import Path
benchmark_path = Path(
    "/content/planproof_results/experiment21/"
    "experiment21_randomized_benchmark.json"
)

print("Benchmark exists:", benchmark_path.exists())


OR-Tools version: 9.15.6755
Benchmark exists: True


In [ ]:

import json
from pathlib import Path
from ortools.sat.python import cp_model

benchmark_path = Path(
    "/content/planproof_results/experiment21/"
    "experiment21_randomized_benchmark.json"
)

with open(benchmark_path, "r", encoding="utf-8") as f:
    cases23 = json.load(f)

print("OR-Tools CP-SAT ready:", cp_model is not None)
print("Cases loaded:", len(cases23))
print("Unique case IDs:", len({
    case["case_id"] for case in cases23
}))
print("Pairwise metric available:",
      callable(globals().get("pairwise_order_change_rate")))
print("Repair algorithm available:",
      callable(globals().get("repair_plan")))


OR-Tools CP-SAT ready: True
Cases loaded: 100
Unique case IDs: 100
Pairwise metric available: True
Repair algorithm available: True


In [ ]:

from ortools.sat.python import cp_model

def exact_optimize_order(case, time_limit_seconds=30):
    required = list(case["required_tasks"])
    original = list(dict.fromkeys(case["task_order"]))

    # Only required tasks participate in the ordering objective.
    original = [t for t in original if t in set(required)]
    original_rank = {t: i for i, t in enumerate(original)}

    # Missing tasks are appended in required-task order for objective ranking.
    for task in required:
        if task not in original_rank:
            original_rank[task] = len(original_rank)

    n = len(required)
    model = cp_model.CpModel()

    position = {
        task: model.NewIntVar(0, n - 1, f"pos_{i}")
        for i, task in enumerate(required)
    }
    model.AddAllDifferent(list(position.values()))

    # Enforce precedence constraints.
    required_set = set(required)
    for before, after in case["precedence_constraints"]:
        if before in required_set and after in required_set:
            model.Add(position[before] < position[after])

    # For each pair, indicate whether its relative order is reversed
    # compared with the original first-occurrence ranking.
    reversal_vars = []
    for i, a in enumerate(required):
        for b in required[i + 1:]:
            if original_rank[a] < original_rank[b]:
                reversed_pair = model.NewBoolVar(f"rev_{a}_{b}")
                model.Add(position[a] > position[b]).OnlyEnforceIf(reversed_pair)
                model.Add(position[a] < position[b]).OnlyEnforceIf(reversed_pair.Not())
            else:
                reversed_pair = model.NewBoolVar(f"rev_{b}_{a}")
                model.Add(position[b] > position[a]).OnlyEnforceIf(reversed_pair)
                model.Add(position[b] < position[a]).OnlyEnforceIf(reversed_pair.Not())

            reversal_vars.append(reversed_pair)

    model.Minimize(sum(reversal_vars))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit_seconds
    solver.parameters.num_search_workers = 8

    status = solver.Solve(model)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return {
            "status": solver.StatusName(status),
            "optimal": False,
            "order": None,
            "reversals": None
        }

    optimized_order = sorted(
        required, key=lambda task: solver.Value(position[task])
    )

    return {
        "status": solver.StatusName(status),
        "optimal": status == cp_model.OPTIMAL,
        "order": optimized_order,
        "reversals": int(round(solver.ObjectiveValue()))
    }


# Test one benchmark case
test_case23 = cases23[0]
test_result23 = exact_optimize_order(test_case23)

plan23 = {
    "task_order": test_case23["task_order"],
    "required_tasks": test_case23["required_tasks"],
    "precedence_constraints": test_case23["precedence_constraints"]
}
stable_result23 = repair_plan(plan23)

print("Case:", test_case23["case_id"])
print("Optimizer status:", test_result23["status"])
print("Optimality proven:", test_result23["optimal"])
print("Optimizer produced order:", test_result23["order"] is not None)
print("Stable repair succeeded:", stable_result23.get("success", False))

if test_result23["order"] is not None:
    print("Optimizer order:", test_result23["order"])
    print("Optimizer pairwise change:", round(
        pairwise_order_change_rate(
            test_case23["task_order"], test_result23["order"]
        ), 4
    ))

if stable_result23.get("success", False):
    print("Stable repair pairwise change:", round(
        pairwise_order_change_rate(
            test_case23["task_order"],
            stable_result23["repaired_plan"]["task_order"]
        ), 4
    ))


Case: E21_001
Optimizer status: OPTIMAL
Optimality proven: True
Optimizer produced order: True
Stable repair succeeded: True
Optimizer order: ['T13', 'T01', 'T11', 'T02', 'T09', 'T16', 'T08', 'T00', 'T03', 'T07', 'T15', 'T10', 'T05', 'T14', 'T04', 'T06', 'T12']
Optimizer pairwise change: 0.425
Stable repair pairwise change: 0.6083


In [ ]:

import pandas as pd
from pathlib import Path

experiment23_dir = Path("/content/planproof_results/experiment23")
experiment23_dir.mkdir(parents=True, exist_ok=True)

rows23 = []

for i, case in enumerate(cases23, start=1):
    # Stable repair
    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    stable = repair_plan(plan)
    stable_ok = (
        stable.get("success", False)
        and isinstance(stable.get("repaired_plan"), dict)
    )

    stable_rate = None
    if stable_ok:
        stable_plan = stable["repaired_plan"]
        stable_valid = validate_plan(stable_plan).get("valid", False)
        if stable_valid:
            stable_rate = pairwise_order_change_rate(
                case["task_order"], stable_plan["task_order"]
            )

    # Exact optimization
    exact = exact_optimize_order(case, time_limit_seconds=30)
    exact_rate = None
    exact_valid = False

    if exact["order"] is not None:
        exact_plan = {
            "task_order": exact["order"],
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        }
        exact_valid = validate_plan(exact_plan).get("valid", False)

        if exact_valid:
            exact_rate = pairwise_order_change_rate(
                case["task_order"], exact["order"]
            )

    comparable = (
        stable_rate is not None
        and exact_rate is not None
    )

    rows23.append({
        "case_id": case["case_id"],
        "defect_type": case["defect_type"],
        "task_count": case["task_count"],
        "stable_valid": bool(stable_ok and stable_valid),
        "stable_change_rate": stable_rate,
        "solver_status": exact["status"],
        "optimality_proven": exact["optimal"],
        "exact_valid": exact_valid,
        "exact_change_rate": exact_rate,
        "comparable": comparable,
        "stable_minus_exact": (
            stable_rate - exact_rate if comparable else None
        )
    })

    if i % 10 == 0:
        print(f"Processed {i}/{len(cases23)} cases")

results23 = pd.DataFrame(rows23)
csv_path = experiment23_dir / "experiment23_comparison.csv"
results23.to_csv(csv_path, index=False)

comparable_df = results23[results23["comparable"]]
proven_df = comparable_df[comparable_df["optimality_proven"]]

print("\nEXPERIMENT 23 — SUMMARY")
print("=" * 40)
print("Total cases:", len(results23))
print("Comparable valid results:", len(comparable_df))
print("Optimality proven:", len(proven_df))
print("Stable repair valid:", int(results23["stable_valid"].sum()))
print("Exact optimizer valid:", int(results23["exact_valid"].sum()))
print("Mean stable change rate:",
      round(comparable_df["stable_change_rate"].mean(), 4))
print("Mean exact change rate:",
      round(comparable_df["exact_change_rate"].mean(), 4))
print("Mean stable-minus-exact:",
      round(comparable_df["stable_minus_exact"].mean(), 4))
print("Cases stable matched exact:",
      int((comparable_df["stable_minus_exact"].abs() < 1e-9).sum()))
print("Cases stable worse than exact:",
      int((comparable_df["stable_minus_exact"] > 1e-9).sum()))
print("Cases stable better than exact:",
      int((comparable_df["stable_minus_exact"] < -1e-9).sum()))
print("Saved:", csv_path)

print("\nSolver statuses:")
display(results23["solver_status"].value_counts().rename_axis(
    "status"
).reset_index(name="cases"))

print("\nCases requiring investigation:")
display(results23[
    (~results23["stable_valid"])
    | (~results23["exact_valid"])
    | (~results23["comparable"])
].head(20))


Processed 10/100 cases
Processed 20/100 cases
Processed 30/100 cases
Processed 40/100 cases
Processed 50/100 cases
Processed 60/100 cases
Processed 70/100 cases
Processed 80/100 cases
Processed 90/100 cases
Processed 100/100 cases

EXPERIMENT 23 — SUMMARY
Total cases: 100
Comparable valid results: 100
Optimality proven: 100
Stable repair valid: 100
Exact optimizer valid: 100
Mean stable change rate: 0.393
Mean exact change rate: 0.3673
Mean stable-minus-exact: 0.0257
Cases stable matched exact: 48
Cases stable worse than exact: 52
Cases stable better than exact: 0
Saved: /content/planproof_results/experiment23/experiment23_comparison.csv

Solver statuses:


,status,cases
0,OPTIMAL,100



Cases requiring investigation:


,case_id,defect_type,task_count,stable_valid,stable_change_rate,solver_status,optimality_proven,exact_valid,exact_change_rate,comparable,stable_minus_exact


In [ ]:

import pandas as pd
from pathlib import Path

comparison_path = Path(
    "/content/planproof_results/experiment23/"
    "experiment23_comparison.csv"
)

df24 = pd.read_csv(comparison_path)

valid = df24[
    df24["stable_valid"].astype(bool)
    & df24["exact_valid"].astype(bool)
    & df24["optimality_proven"].astype(bool)
].copy()

valid["recomputed_difference"] = (
    valid["stable_change_rate"] - valid["exact_change_rate"]
)

valid["difference_matches"] = (
    (valid["recomputed_difference"] - valid["stable_minus_exact"]).abs()
    < 1e-9
)

print("EXPERIMENT 24 — COMPARISON AUDIT")
print("=" * 42)
print("Rows in file:", len(df24))
print("Valid optimal comparisons:", len(valid))
print("Difference calculations consistent:",
      int(valid["difference_matches"].sum()), "/", len(valid))
print("Negative differences:", int(
    (valid["recomputed_difference"] < -1e-9).sum()
))
print("Zero differences:", int(
    (valid["recomputed_difference"].abs() < 1e-9).sum()
))
print("Positive differences:", int(
    (valid["recomputed_difference"] > 1e-9).sum()
))
print("Mean stable change:", round(valid["stable_change_rate"].mean(), 4))
print("Mean exact change:", round(valid["exact_change_rate"].mean(), 4))
print("Mean absolute reduction in rate:", round(
    valid["recomputed_difference"].mean(), 4
))

if len(valid) > 0:
    print("Relative reduction vs stable repair (%):", round(
        100 * valid["recomputed_difference"].mean()
        / valid["stable_change_rate"].mean(), 2
    ))

print("\nInconsistent rows:")
display(valid[~valid["difference_matches"]].head(10))


EXPERIMENT 24 — COMPARISON AUDIT
Rows in file: 100
Valid optimal comparisons: 100
Difference calculations consistent: 100 / 100
Negative differences: 0
Zero differences: 48
Positive differences: 52
Mean stable change: 0.393
Mean exact change: 0.3673
Mean absolute reduction in rate: 0.0257
Relative reduction vs stable repair (%): 6.55

Inconsistent rows:


,case_id,defect_type,task_count,stable_valid,stable_change_rate,solver_status,optimality_proven,exact_valid,exact_change_rate,comparable,stable_minus_exact,recomputed_difference,difference_matches


In [ ]:

from pathlib import Path
import zipfile

root = Path("/content/planproof_results")
output_zip = Path("/content/PlanProof_AI_Experiments_21_to_24.zip")

files = [
    root / "experiment21/experiment21_randomized_benchmark.json",
    root / "experiment21/experiment21_randomized_audit.csv",
    root / "experiment21/experiment21_repair_results.csv",
    root / "experiment21/experiment21_report.md",
    root / "experiment22/experiment22_order_preservation.csv",
    root / "experiment23/experiment23_comparison.csv",
]

missing = [str(path) for path in files if not path.exists()]

if missing:
    print("Missing files:")
    for path in missing:
        print("-", path)
else:
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, arcname=str(path.relative_to(root)))

    print("Research archive created:", output_zip)
    print("Files packaged:", len(files))
    print("Archive size (MB):", round(output_zip.stat().st_size / (1024 * 1024), 3))


Research archive created: /content/PlanProof_AI_Experiments_21_to_24.zip
Files packaged: 6
Archive size (MB): 0.014


In [ ]:

import random
import json
from pathlib import Path

SEED25 = 20261025
rng25 = random.Random(SEED25)

experiment25_dir = Path("/content/planproof_results/experiment25")
experiment25_dir.mkdir(parents=True, exist_ok=True)

task_sizes25 = [10, 20, 30, 40, 50]
REPEATS25 = 3
cases25 = []

for n in task_sizes25:
    for repeat in range(1, REPEATS25 + 1):
        tasks = [f"T{i:03d}" for i in range(n)]

        # A known topological order used to construct an acyclic graph
        topo = tasks[:]
        rng25.shuffle(topo)

        edges = set()

        # Ensure every task after the first has a predecessor
        for j in range(1, n):
            predecessor = rng25.choice(topo[:j])
            edges.add((predecessor, topo[j]))

        # Add additional forward edges
        for j in range(n):
            for k in range(j + 1, n):
                if rng25.random() < 0.06:
                    edges.add((topo[j], topo[k]))

        original_order = tasks[:]
        rng25.shuffle(original_order)

        cases25.append({
            "case_id": f"E25_N{n}_R{repeat}",
            "task_count": n,
            "repeat": repeat,
            "required_tasks": tasks,
            "task_order": original_order,
            "precedence_constraints": [list(edge) for edge in sorted(edges)]
        })

benchmark25_path = experiment25_dir / "experiment25_scalability_benchmark.json"
with open(benchmark25_path, "w", encoding="utf-8") as f:
    json.dump(cases25, f, indent=2)

print("Experiment 25 benchmark created.")
print("Total cases:", len(cases25))
print("Task sizes:", task_sizes25)
print("Repeats per size:", REPEATS25)
print("Random seed:", SEED25)
print("Saved:", benchmark25_path)


Experiment 25 benchmark created.
Total cases: 15
Task sizes: [10, 20, 30, 40, 50]
Repeats per size: 3
Random seed: 20261025
Saved: /content/planproof_results/experiment25/experiment25_scalability_benchmark.json


In [ ]:

import json
import pandas as pd
from pathlib import Path
from collections import defaultdict, deque

benchmark_path = Path(
    "/content/planproof_results/experiment25/"
    "experiment25_scalability_benchmark.json"
)

with open(benchmark_path, "r", encoding="utf-8") as f:
    cases25 = json.load(f)

audit_rows25 = []

for case in cases25:
    tasks = case["required_tasks"]
    task_set = set(tasks)
    edges = case["precedence_constraints"]

    endpoints_valid = all(
        len(edge) == 2
        and edge[0] in task_set
        and edge[1] in task_set
        for edge in edges
    )

    adjacency = defaultdict(list)
    indegree = {task: 0 for task in tasks}

    for before, after in edges:
        if before in task_set and after in task_set:
            adjacency[before].append(after)
            indegree[after] += 1

    queue = deque(
        task for task in tasks if indegree[task] == 0
    )
    visited = 0

    while queue:
        task = queue.popleft()
        visited += 1
        for neighbour in adjacency[task]:
            indegree[neighbour] -= 1
            if indegree[neighbour] == 0:
                queue.append(neighbour)

    acyclic = visited == len(tasks)

    audit_rows25.append({
        "case_id": case["case_id"],
        "task_count": len(tasks),
        "edge_count": len(edges),
        "valid_endpoints": endpoints_valid,
        "acyclic": acyclic,
        "audit_pass": endpoints_valid and acyclic
    })

audit25 = pd.DataFrame(audit_rows25)
audit_path = benchmark_path.parent / "experiment25_graph_audit.csv"
audit25.to_csv(audit_path, index=False)

print("Total cases:", len(audit25))
print("Valid dependency endpoints:", int(audit25["valid_endpoints"].sum()))
print("Acyclic graphs:", int(audit25["acyclic"].sum()))
print("Audit passes:", int(audit25["audit_pass"].sum()))
print("Audit failures:", int((~audit25["audit_pass"]).sum()))
print("Saved:", audit_path)

display(audit25.groupby("task_count").agg(
    cases=("case_id", "count"),
    mean_edges=("edge_count", "mean"),
    audit_passes=("audit_pass", "sum")
).reset_index())


Total cases: 15
Valid dependency endpoints: 15
Acyclic graphs: 15
Audit passes: 15
Audit failures: 0
Saved: /content/planproof_results/experiment25/experiment25_graph_audit.csv


,task_count,cases,mean_edges,audit_passes
0,10,3,12.333333,3
1,20,3,28.333333,3
2,30,3,57.000000,3
3,40,3,83.666667,3
4,50,3,126.000000,3


In [ ]:

import json
import time
import statistics
import pandas as pd
from pathlib import Path
from ortools.sat.python import cp_model

base25 = Path("/content/planproof_results/experiment25")
with open(base25 / "experiment25_scalability_benchmark.json",
          "r", encoding="utf-8") as f:
    cases25 = json.load(f)

TIMING_REPEATS25 = 5
rows25 = []

for case in cases25:
    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    # Warm up stable repair once
    warmup = repair_plan(plan)
    if not warmup.get("success", False):
        raise RuntimeError(f"Stable repair failed: {case['case_id']}")

    stable_times = []
    stable_result = None

    for _ in range(TIMING_REPEATS25):
        start = time.perf_counter()
        stable_result = repair_plan(plan)
        stable_times.append(time.perf_counter() - start)

    stable_plan = stable_result["repaired_plan"]
    stable_valid = validate_plan(stable_plan).get("valid", False)
    stable_rate = pairwise_order_change_rate(
        case["task_order"], stable_plan["task_order"]
    )

    # Exact optimizer timing
    exact_times = []
    exact_result = None

    for _ in range(TIMING_REPEATS25):
        start = time.perf_counter()
        exact_result = exact_optimize_order(
            case, time_limit_seconds=30
        )
        exact_times.append(time.perf_counter() - start)

    exact_order = exact_result["order"]
    exact_valid = False
    exact_rate = None

    if exact_order is not None:
        exact_valid = validate_plan({
            "task_order": exact_order,
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        }).get("valid", False)

        exact_rate = pairwise_order_change_rate(
            case["task_order"], exact_order
        )

    rows25.append({
        "case_id": case["case_id"],
        "task_count": case["task_count"],
        "edge_count": len(case["precedence_constraints"]),
        "stable_median_seconds": statistics.median(stable_times),
        "exact_median_seconds": statistics.median(exact_times),
        "stable_valid": stable_valid,
        "exact_valid": exact_valid,
        "stable_order_change": stable_rate,
        "exact_order_change": exact_rate,
        "solver_status": exact_result["status"],
        "optimality_proven": exact_result["optimal"]
    })

    print(
        f'{case["case_id"]}: n={case["task_count"]}, '
        f'stable={statistics.median(stable_times):.6f}s, '
        f'exact={statistics.median(exact_times):.6f}s, '
        f'status={exact_result["status"]}'
    )

timing_df25 = pd.DataFrame(rows25)
timing_path = base25 / "experiment25_runtime_results.csv"
timing_df25.to_csv(timing_path, index=False)

print("\nEXPERIMENT 25 — RUNTIME SUMMARY")
print("=" * 42)
print("Cases:", len(timing_df25))
print("Stable valid:", int(timing_df25["stable_valid"].sum()))
print("Exact valid:", int(timing_df25["exact_valid"].sum()))
print("Optimality proven:", int(timing_df25["optimality_proven"].sum()))
print("Saved:", timing_path)

display(
    timing_df25.groupby("task_count").agg(
        cases=("case_id", "count"),
        stable_median_seconds=("stable_median_seconds", "median"),
        exact_median_seconds=("exact_median_seconds", "median"),
        mean_stable_change=("stable_order_change", "mean"),
        mean_exact_change=("exact_order_change", "mean"),
        optimal_cases=("optimality_proven", "sum")
    ).reset_index()
)


E25_N10_R1: n=10, stable=0.000043s, exact=0.237976s, status=OPTIMAL
E25_N10_R2: n=10, stable=0.000043s, exact=0.114007s, status=OPTIMAL
E25_N10_R3: n=10, stable=0.000043s, exact=0.215968s, status=OPTIMAL
E25_N20_R1: n=20, stable=0.000096s, exact=0.940637s, status=OPTIMAL
E25_N20_R2: n=20, stable=0.000096s, exact=1.000470s, status=OPTIMAL
E25_N20_R3: n=20, stable=0.000080s, exact=1.255106s, status=OPTIMAL
E25_N30_R1: n=30, stable=0.000155s, exact=5.397212s, status=OPTIMAL
E25_N30_R2: n=30, stable=0.000083s, exact=3.592566s, status=OPTIMAL
E25_N30_R3: n=30, stable=0.000083s, exact=1.953772s, status=OPTIMAL
E25_N40_R1: n=40, stable=0.000113s, exact=15.054939s, status=OPTIMAL
E25_N40_R2: n=40, stable=0.000122s, exact=21.806533s, status=OPTIMAL
E25_N40_R3: n=40, stable=0.000124s, exact=11.252337s, status=OPTIMAL
E25_N50_R1: n=50, stable=0.000160s, exact=30.065034s, status=FEASIBLE
E25_N50_R2: n=50, stable=0.000284s, exact=30.050834s, status=FEASIBLE
E25_N50_R3: n=50, stable=0.000161s, exact

,task_count,cases,stable_median_seconds,exact_median_seconds,mean_stable_change,mean_exact_change,optimal_cases
0,10,3,0.000043,0.215968,0.511111,0.422222,3
1,20,3,0.000096,1.000470,0.321053,0.270175,3
2,30,3,0.000083,3.592566,0.340230,0.281992,3
3,40,3,0.000122,15.054939,0.405128,0.361111,3
4,50,3,0.000161,30.065034,0.458231,0.406531,0


In [ ]:

import pandas as pd
from pathlib import Path

path26 = Path(
    "/content/planproof_results/experiment25/"
    "experiment25_runtime_results.csv"
)
df26 = pd.read_csv(path26)

df26["order_change_reduction"] = (
    df26["stable_order_change"] - df26["exact_order_change"]
)

df26["runtime_ratio"] = (
    df26["exact_median_seconds"]
    / df26["stable_median_seconds"]
)

df26["solution_status"] = df26["optimality_proven"].map({
    True: "Optimality proven",
    False: "Feasible only / not proven optimal"
})

summary26 = df26.groupby("task_count").agg(
    cases=("case_id", "count"),
    mean_stable_change=("stable_order_change", "mean"),
    mean_exact_change=("exact_order_change", "mean"),
    mean_change_reduction=("order_change_reduction", "mean"),
    median_runtime_ratio=("runtime_ratio", "median"),
    optimal_cases=("optimality_proven", "sum")
).reset_index()

output26 = Path(
    "/content/planproof_results/experiment25/"
    "experiment26_quality_runtime_tradeoff.csv"
)
df26.to_csv(output26, index=False)

print("EXPERIMENT 26 — QUALITY/RUNTIME TRADE-OFF")
print("=" * 46)
print("Total cases:", len(df26))
print("Order-change reduction is positive when exact is better.")
print("Runtime ratio compares measured median runtimes.")
print("Saved:", output26)
display(summary26)

print("\n50-TASK SOLVER STATUS")
display(df26[df26["task_count"] == 50][[
    "case_id",
    "exact_median_seconds",
    "order_change_reduction",
    "solution_status"
]])


EXPERIMENT 26 — QUALITY/RUNTIME TRADE-OFF
Total cases: 15
Order-change reduction is positive when exact is better.
Runtime ratio compares measured median runtimes.
Saved: /content/planproof_results/experiment25/experiment26_quality_runtime_tradeoff.csv


,task_count,cases,mean_stable_change,mean_exact_change,mean_change_reduction,median_runtime_ratio,optimal_cases
0,10,3,0.511111,0.422222,0.088889,5052.483662,3
1,20,3,0.321053,0.270175,0.050877,10461.555980,3
2,30,3,0.340230,0.281992,0.058238,34931.603631,3
3,40,3,0.405128,0.361111,0.044017,133517.853394,3
4,50,3,0.458231,0.406531,0.051701,187233.069598,0



50-TASK SOLVER STATUS


,case_id,exact_median_seconds,order_change_reduction,solution_status
12,E25_N50_R1,30.065034,0.082449,Feasible only / not proven optimal
13,E25_N50_R2,30.050834,0.026939,Feasible only / not proven optimal
14,E25_N50_R3,30.172422,0.045714,Feasible only / not proven optimal


In [ ]:

import pandas as pd
from pathlib import Path

base27 = Path("/content/planproof_results/experiment25")
df27 = pd.read_csv(base27 / "experiment25_runtime_results.csv")

df27["rate_in_range"] = (
    df27["stable_order_change"].between(0, 1)
    & df27["exact_order_change"].between(0, 1)
)

df27["optimality_label"] = df27["optimality_proven"].map({
    True: "Proven optimal",
    False: "Optimality not proven"
})

print("EXPERIMENT 27 — OBJECTIVE AND STATUS AUDIT")
print("=" * 48)
print("Total cases:", len(df27))
print("Both rates within [0, 1]:", int(df27["rate_in_range"].sum()))
print("Optimality proven:", int(df27["optimality_proven"].sum()))
print("Optimality not proven:", int((~df27["optimality_proven"]).sum()))

print("\nSTATUS BY TASK SIZE")
display(df27.groupby("task_count").agg(
    cases=("case_id", "count"),
    valid_rates=("rate_in_range", "sum"),
    proven_optimal=("optimality_proven", "sum")
).reset_index())

print("\nCASES WITHOUT PROVEN OPTIMALITY")
display(df27.loc[
    ~df27["optimality_proven"],
    ["case_id", "task_count", "solver_status",
     "exact_order_change", "exact_median_seconds"]
])


EXPERIMENT 27 — OBJECTIVE AND STATUS AUDIT
Total cases: 15
Both rates within [0, 1]: 15
Optimality proven: 12
Optimality not proven: 3

STATUS BY TASK SIZE


,task_count,cases,valid_rates,proven_optimal
0,10,3,3,3
1,20,3,3,3
2,30,3,3,3
3,40,3,3,3
4,50,3,3,0



CASES WITHOUT PROVEN OPTIMALITY


,case_id,task_count,solver_status,exact_order_change,exact_median_seconds
12,E25_N50_R1,50,FEASIBLE,0.384490,30.065034
13,E25_N50_R2,50,FEASIBLE,0.421224,30.050834
14,E25_N50_R3,50,FEASIBLE,0.413878,30.172422


In [ ]:

import json
from pathlib import Path

path28 = Path(
    "/content/planproof_results/experiment21/"
    "experiment21_randomized_benchmark.json"
)

with open(path28, "r", encoding="utf-8") as f:
    cases28 = json.load(f)

# Inspect a case containing missing tasks
case28 = next(
    c for c in cases28 if c["defect_type"] == "missing_task"
)

result28 = exact_optimize_order(case28, time_limit_seconds=30)

print("Case:", case28["case_id"])
print("Defect type:", case28["defect_type"])
print("Required task count:", len(case28["required_tasks"]))
print("Unique original task count:", len(set(case28["task_order"])))
print("Missing required tasks:", sorted(
    set(case28["required_tasks"]) - set(case28["task_order"])
))
print("Optimizer status:", result28["status"])
print("Optimality proven:", result28["optimal"])

if result28["order"] is not None:
    print("Optimized order length:", len(result28["order"]))
    print("Optimized order has unique tasks:",
          len(result28["order"]) == len(set(result28["order"])))
    print("Correct required task set:",
          set(result28["order"]) == set(case28["required_tasks"]))
    print("Reported order-change rate:",
          pairwise_order_change_rate(
              case28["task_order"], result28["order"]
          ))


Case: E21_001
Defect type: missing_task
Required task count: 17
Unique original task count: 16
Missing required tasks: ['T07']
Optimizer status: OPTIMAL
Optimality proven: True
Optimized order length: 17
Optimized order has unique tasks: True
Correct required task set: True
Reported order-change rate: 0.425


In [ ]:

case = case28

plan = {
    "task_order": case["task_order"],
    "required_tasks": case["required_tasks"],
    "precedence_constraints": case["precedence_constraints"]
}

stable = repair_plan(plan)
exact = exact_optimize_order(case, time_limit_seconds=30)

if stable.get("success") and exact["order"] is not None:
    stable_order = stable["repaired_plan"]["task_order"]
    exact_order = exact["order"]

    stable_rate = pairwise_order_change_rate(
        case["task_order"], stable_order
    )
    exact_rate = pairwise_order_change_rate(
        case["task_order"], exact_order
    )

    print("Case:", case["case_id"])
    print("Stable repair valid:", validate_plan(
        stable["repaired_plan"]
    )["valid"])
    print("Exact optimizer valid:", validate_plan({
        "task_order": exact_order,
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    })["valid"])
    print("Stable repair order-change rate:", round(stable_rate, 4))
    print("Exact optimizer order-change rate:", round(exact_rate, 4))
    print("Difference (stable - exact):", round(
        stable_rate - exact_rate, 4
    ))
    print("Exact solver status:", exact["status"])
    print("Optimality proven:", exact["optimal"])
else:
    print("A method failed; inspect its result before comparing.")
    print("Stable result:", stable)
    print("Exact result:", exact)


Case: E21_001
Stable repair valid: True
Exact optimizer valid: True
Stable repair order-change rate: 0.6083
Exact optimizer order-change rate: 0.425
Difference (stable - exact): 0.1833
Exact solver status: OPTIMAL
Optimality proven: True


In [ ]:

from math import comb

def shared_task_order_change_rate(original_order, repaired_order):
    """Pairwise relative-order changes among shared unique task IDs."""
    original = list(dict.fromkeys(original_order))
    repaired = list(dict.fromkeys(repaired_order))

    common = set(original) & set(repaired)
    original = [t for t in original if t in common]
    repaired = [t for t in repaired if t in common]

    n = len(original)
    if n < 2:
        return 0.0

    repaired_position = {
        task: i for i, task in enumerate(repaired)
    }

    reversed_pairs = 0

    for i in range(n):
        for j in range(i + 1, n):
            a, b = original[i], original[j]
            if repaired_position[a] > repaired_position[b]:
                reversed_pairs += 1

    return reversed_pairs / comb(n, 2)


def full_plan_repair_cost(original_order, required_tasks):
    """
    Count structural edits needed to make the task sequence contain
    each required task exactly once, excluding precedence reordering.

    Returns separate counts and their unweighted total.
    """
    required = set(required_tasks)

    original_set = set(original_order)
    missing_count = len(required - original_set)

    duplicate_count = len(original_order) - len(original_set)

    unknown_count = sum(
        1 for task in original_order if task not in required
    )

    return {
        "missing_tasks_to_restore": missing_count,
        "duplicate_occurrences_to_remove": duplicate_count,
        "unknown_occurrences_to_remove": unknown_count,
        "structural_edit_count": (
            missing_count + duplicate_count + unknown_count
        )
    }


# Sanity checks
print("Identical order:",
      shared_task_order_change_rate(["A", "B", "C"], ["A", "B", "C"]))
print("Reversed order:",
      shared_task_order_change_rate(["A", "B", "C"], ["C", "B", "A"]))

example_order = ["A", "A", "B", "X"]
example_required = ["A", "B", "C"]

print("Example structural cost:",
      full_plan_repair_cost(example_order, example_required))


Identical order: 0.0
Reversed order: 1.0
Example structural cost: {'missing_tasks_to_restore': 1, 'duplicate_occurrences_to_remove': 1, 'unknown_occurrences_to_remove': 1, 'structural_edit_count': 3}


In [ ]:

import json
import pandas as pd
from pathlib import Path

base29 = Path("/content/planproof_results/experiment21")
output29 = Path("/content/planproof_results/experiment29")
output29.mkdir(parents=True, exist_ok=True)

with open(
    base29 / "experiment21_randomized_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    cases29 = json.load(f)

rows29 = []

for case in cases29:
    original = case["task_order"]

    plan = {
        "task_order": original,
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    result = repair_plan(plan)
    repaired = result.get("repaired_plan")
    success = (
        result.get("success", False)
        and isinstance(repaired, dict)
    )

    row = {
        "case_id": case["case_id"],
        "defect_type": case["defect_type"],
        "task_count": case["task_count"],
        "repair_success": success
    }

    if success:
        validation = validate_plan(repaired)
        order = repaired["task_order"]

        structural = full_plan_repair_cost(
            original, case["required_tasks"]
        )

        row.update({
            "valid_after_repair": validation.get("valid", False),
            "shared_task_order_change_rate":
                shared_task_order_change_rate(original, order),
            **structural
        })
    else:
        row.update({
            "valid_after_repair": False,
            "shared_task_order_change_rate": None,
            "missing_tasks_to_restore": None,
            "duplicate_occurrences_to_remove": None,
            "unknown_occurrences_to_remove": None,
            "structural_edit_count": None
        })

    rows29.append(row)

df29 = pd.DataFrame(rows29)
csv29 = output29 / "experiment29_dual_metrics.csv"
df29.to_csv(csv29, index=False)

successful29 = df29[
    df29["repair_success"] & df29["valid_after_repair"]
]

print("EXPERIMENT 29 — DUAL METRIC EVALUATION")
print("=" * 46)
print("Total cases:", len(df29))
print("Successful repairs:", int(df29["repair_success"].sum()))
print("Valid after repair:", int(df29["valid_after_repair"].sum()))
print("Mean shared-task order-change rate:",
      round(successful29["shared_task_order_change_rate"].mean(), 4))
print("Mean structural edit count:",
      round(successful29["structural_edit_count"].mean(), 2))
print("Total missing-task restorations:",
      int(successful29["missing_tasks_to_restore"].sum()))
print("Total duplicate removals:",
      int(successful29["duplicate_occurrences_to_remove"].sum()))
print("Total unknown-task removals:",
      int(successful29["unknown_occurrences_to_remove"].sum()))
print("Saved:", csv29)

print("\nBY DEFECT CATEGORY")
display(successful29.groupby("defect_type").agg(
    cases=("case_id", "count"),
    mean_order_change=("shared_task_order_change_rate", "mean"),
    mean_structural_edits=("structural_edit_count", "mean"),
    total_structural_edits=("structural_edit_count", "sum")
).reset_index())


EXPERIMENT 29 — DUAL METRIC EVALUATION
Total cases: 100
Successful repairs: 100
Valid after repair: 100
Mean shared-task order-change rate: 0.393
Mean structural edit count: 1.01
Total missing-task restorations: 37
Total duplicate removals: 29
Total unknown-task removals: 35
Saved: /content/planproof_results/experiment29/experiment29_dual_metrics.csv

BY DEFECT CATEGORY


,defect_type,cases,mean_order_change,mean_structural_edits,total_structural_edits
0,duplicate_task,16,0.372154,1.0,16
1,missing_task,24,0.429213,1.0,24
2,mixed_defects,13,0.453370,3.0,39
3,shuffled_order,25,0.381290,0.0,0
4,unknown_task,22,0.346352,1.0,22


In [ ]:

import json
import pandas as pd
from pathlib import Path

base30 = Path("/content/planproof_results")
with open(
    base30 / "experiment21/experiment21_randomized_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    benchmark30 = json.load(f)

results30 = pd.read_csv(
    base30 / "experiment29/experiment29_dual_metrics.csv"
)

benchmark_by_id = {
    case["case_id"]: case for case in benchmark30
}

audit_rows30 = []

for _, row in results30.iterrows():
    case = benchmark_by_id[row["case_id"]]

    expected = full_plan_repair_cost(
        case["task_order"],
        case["required_tasks"]
    )

    matches = all(
        int(row[key]) == expected[key]
        for key in [
            "missing_tasks_to_restore",
            "duplicate_occurrences_to_remove",
            "unknown_occurrences_to_remove",
            "structural_edit_count"
        ]
    )

    audit_rows30.append({
        "case_id": row["case_id"],
        "structural_counts_match": matches,
        "order_change_in_range": (
            0 <= row["shared_task_order_change_rate"] <= 1
        ),
        "repair_success": bool(row["repair_success"]),
        "valid_after_repair": bool(row["valid_after_repair"])
    })

audit30 = pd.DataFrame(audit_rows30)
audit_path30 = base30 / "experiment29/experiment30_consistency_audit.csv"
audit30.to_csv(audit_path30, index=False)

print("EXPERIMENT 30 — CONSISTENCY AUDIT")
print("=" * 42)
print("Cases checked:", len(audit30))
print("Structural counts match:", int(
    audit30["structural_counts_match"].sum()
))
print("Order-change rates in range:", int(
    audit30["order_change_in_range"].sum()
))
print("Successful repairs:", int(audit30["repair_success"].sum()))
print("Valid after repair:", int(audit30["valid_after_repair"].sum()))
print("Audit failures:", int(
    (~audit30.drop(columns=["case_id"]).all(axis=1)).sum()
))
print("Saved:", audit_path30)



EXPERIMENT 30 — CONSISTENCY AUDIT
Cases checked: 100
Structural counts match: 100
Order-change rates in range: 100
Successful repairs: 100
Valid after repair: 100
Audit failures: 0
Saved: /content/planproof_results/experiment29/experiment30_consistency_audit.csv


In [ ]:

import json
import pandas as pd
from pathlib import Path

base31 = Path("/content/planproof_results")
out31 = base31 / "experiment31"
out31.mkdir(parents=True, exist_ok=True)

with open(
    base31 / "experiment21/experiment21_randomized_benchmark.json",
    "r", encoding="utf-8"
) as f:
    cases31 = json.load(f)

rows31 = []

for case in cases31:
    original = case["task_order"]
    required = case["required_tasks"]
    constraints = case["precedence_constraints"]

    plan = {
        "task_order": original,
        "required_tasks": required,
        "precedence_constraints": constraints
    }

    # Stable repair
    stable_result = repair_plan(plan)
    stable_ok = (
        stable_result.get("success", False)
        and isinstance(stable_result.get("repaired_plan"), dict)
    )

    # Exact optimization
    exact_result = exact_optimize_order(
        case, time_limit_seconds=30
    )

    stable_rate = None
    stable_valid = False
    stable_structural = None

    if stable_ok:
        stable_plan = stable_result["repaired_plan"]
        stable_valid = validate_plan(stable_plan).get("valid", False)

        if stable_valid:
            stable_rate = shared_task_order_change_rate(
                original, stable_plan["task_order"]
            )
            stable_structural = full_plan_repair_cost(
                original, required
            )["structural_edit_count"]

    exact_rate = None
    exact_valid = False
    exact_structural = None

    if exact_result["order"] is not None:
        exact_plan = {
            "task_order": exact_result["order"],
            "required_tasks": required,
            "precedence_constraints": constraints
        }
        exact_valid = validate_plan(exact_plan).get("valid", False)

        if exact_valid:
            exact_rate = shared_task_order_change_rate(
                original, exact_result["order"]
            )
            exact_structural = full_plan_repair_cost(
                original, required
            )["structural_edit_count"]

    comparable = (
        stable_rate is not None and exact_rate is not None
    )

    rows31.append({
        "case_id": case["case_id"],
        "defect_type": case["defect_type"],
        "task_count": case["task_count"],
        "stable_valid": stable_valid,
        "stable_order_change": stable_rate,
        "stable_structural_edits": stable_structural,
        "exact_valid": exact_valid,
        "exact_order_change": exact_rate,
        "exact_structural_edits": exact_structural,
        "solver_status": exact_result["status"],
        "optimality_proven": exact_result["optimal"],
        "comparable": comparable,
        "order_change_reduction": (
            stable_rate - exact_rate if comparable else None
        ),
        "structural_counts_match": (
            stable_structural == exact_structural
            if comparable else None
        )
    })

df31 = pd.DataFrame(rows31)
csv31 = out31 / "experiment31_fair_comparison.csv"
df31.to_csv(csv31, index=False)

comp31 = df31[df31["comparable"]].copy()
optimal31 = comp31[comp31["optimality_proven"]]

print("EXPERIMENT 31 — FAIR COMPARISON")
print("=" * 44)
print("Total cases:", len(df31))
print("Comparable valid cases:", len(comp31))
print("Stable valid:", int(df31["stable_valid"].sum()))
print("Exact valid:", int(df31["exact_valid"].sum()))
print("Exact optimality proven:", int(df31["optimality_proven"].sum()))
print("Mean stable order-change:",
      round(comp31["stable_order_change"].mean(), 4))
print("Mean exact order-change:",
      round(comp31["exact_order_change"].mean(), 4))
print("Mean order-change reduction:",
      round(comp31["order_change_reduction"].mean(), 4))
print("Structural counts match:",
      int(comp31["structural_counts_match"].sum()), "/", len(comp31))
print("Mean reduction among proven-optimal cases:",
      round(optimal31["order_change_reduction"].mean(), 4)
      if len(optimal31) else "No proven-optimal cases")
print("Saved:", csv31)

print("\nRESULTS BY DEFECT CATEGORY")
display(comp31.groupby("defect_type").agg(
    cases=("case_id", "count"),
    stable_mean_change=("stable_order_change", "mean"),
    exact_mean_change=("exact_order_change", "mean"),
    mean_reduction=("order_change_reduction", "mean"),
    optimal_cases=("optimality_proven", "sum")
).reset_index())

print("\nCASES REQUIRING INVESTIGATION")
display(df31[
    (~df31["stable_valid"])
    | (~df31["exact_valid"])
    | (~df31["comparable"])
    | (~df31["structural_counts_match"].fillna(False))
].head(20))


EXPERIMENT 31 — FAIR COMPARISON
Total cases: 100
Comparable valid cases: 100
Stable valid: 100
Exact valid: 100
Exact optimality proven: 100
Mean stable order-change: 0.393
Mean exact order-change: 0.3621
Mean order-change reduction: 0.0309
Structural counts match: 100 / 100
Mean reduction among proven-optimal cases: 0.0309
Saved: /content/planproof_results/experiment31/experiment31_fair_comparison.csv

RESULTS BY DEFECT CATEGORY


,defect_type,cases,stable_mean_change,exact_mean_change,mean_reduction,optimal_cases
0,duplicate_task,16,0.372154,0.356806,0.015348,16
1,missing_task,24,0.429213,0.359280,0.069933,24
2,mixed_defects,13,0.453370,0.416373,0.036997,13
3,shuffled_order,25,0.381290,0.360817,0.020473,25
4,unknown_task,22,0.346352,0.338630,0.007722,22



CASES REQUIRING INVESTIGATION


,case_id,defect_type,task_count,stable_valid,stable_order_change,stable_structural_edits,exact_valid,exact_order_change,exact_structural_edits,solver_status,optimality_proven,comparable,order_change_reduction,structural_counts_match


In [ ]:

from ortools.sat.python import cp_model

def exact_optimize_order_fair(case, time_limit_seconds=30):
    """
    Exact optimization aligned with shared-task order-change rate.

    Only pairs of known tasks present in the original order contribute
    to the objective. Missing tasks must still be included and all
    precedence constraints must still be satisfied.
    """
    required = list(dict.fromkeys(case["required_tasks"]))
    required_set = set(required)
    original = case["task_order"]
    constraints = case["precedence_constraints"]

    # Retain the first occurrence of each known task in the original plan.
    observed = list(dict.fromkeys(
        task for task in original if task in required_set
    ))

    model = cp_model.CpModel()
    n = len(required)

    if n == 0:
        return {
            "status": "EMPTY",
            "optimal": True,
            "order": [],
            "reversals": 0
        }

    positions = {
        task: model.NewIntVar(0, n - 1, f"pos_{i}")
        for i, task in enumerate(required)
    }
    model.AddAllDifferent(list(positions.values()))

    # Enforce precedence constraints.
    for edge in constraints:
        before, after = edge[0], edge[1]
        if before in required_set and after in required_set:
            model.Add(positions[before] < positions[after])

    # Count reversals only between tasks observed in the original plan.
    reversal_vars = []

    for i in range(len(observed)):
        for j in range(i + 1, len(observed)):
            first = observed[i]
            second = observed[j]

            reversed_pair = model.NewBoolVar(
                f"reversed_{i}_{j}"
            )

            # Original order is first before second.
            # reversed_pair = 1 exactly when second comes before first.
            model.Add(
                positions[second] < positions[first]
            ).OnlyEnforceIf(reversed_pair)

            model.Add(
                positions[second] > positions[first]
            ).OnlyEnforceIf(reversed_pair.Not())

            reversal_vars.append(reversed_pair)

    model.Minimize(sum(reversal_vars))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit_seconds
    solver.parameters.num_search_workers = 1

    status_code = solver.Solve(model)
    status = solver.StatusName(status_code)

    if status_code not in (
        cp_model.OPTIMAL,
        cp_model.FEASIBLE
    ):
        return {
            "status": status,
            "optimal": False,
            "order": None,
            "reversals": None
        }

    repaired_order = sorted(
        required,
        key=lambda task: solver.Value(positions[task])
    )

    return {
        "status": status,
        "optimal": status_code == cp_model.OPTIMAL,
        "order": repaired_order,
        "reversals": sum(
            solver.Value(var) for var in reversal_vars
        )
    }


print("Corrected fair optimizer defined.")
print("Objective: reversals among tasks observed in the original plan.")


Corrected fair optimizer defined.
Objective: reversals among tasks observed in the original plan.


In [ ]:

# Load the Experiment 21 benchmark
import json
from pathlib import Path

base32 = Path("/content/planproof_results")

with open(
    base32 / "experiment21/experiment21_randomized_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    cases32 = json.load(f)

# Select the first case containing a missing required task
test_case32 = next(
    case for case in cases32
    if any(
        task not in case["task_order"]
        for task in case["required_tasks"]
    )
)

# Run the corrected optimizer
result32 = exact_optimize_order_fair(
    test_case32,
    time_limit_seconds=30
)

print("Case ID:", test_case32["case_id"])
print("Defect type:", test_case32["defect_type"])
print("Solver status:", result32["status"])
print("Optimality proven:", result32["optimal"])
print("Original task count:", len(test_case32["task_order"]))
print("Required unique tasks:", len(set(test_case32["required_tasks"])))
print("Repaired task count:",
      len(result32["order"]) if result32["order"] is not None else None)

if result32["order"] is not None:
    repaired_plan32 = {
        "task_order": result32["order"],
        "required_tasks": test_case32["required_tasks"],
        "precedence_constraints": test_case32["precedence_constraints"]
    }

    validation32 = validate_plan(repaired_plan32)

    measured_rate32 = shared_task_order_change_rate(
        test_case32["task_order"],
        result32["order"]
    )

    print("Plan valid:", validation32["valid"])
    print("Violation count:", len(validation32["violations"]))
    print("Optimizer reversal count:", result32["reversals"])
    print("Metric reversal count:",
          round(
              measured_rate32
              * (
                  len(set(
                      task for task in test_case32["task_order"]
                      if task in set(test_case32["required_tasks"])
                  ))
                  * (
                      len(set(
                          task for task in test_case32["task_order"]
                          if task in set(test_case32["required_tasks"])
                      )) - 1
                  ) / 2
              )
          ))
    print("Measured order-change rate:", round(measured_rate32, 4))


Case ID: E21_001
Defect type: missing_task
Solver status: OPTIMAL
Optimality proven: True
Original task count: 16
Required unique tasks: 17
Repaired task count: 17
Plan valid: True
Violation count: 0
Optimizer reversal count: 51
Metric reversal count: 51
Measured order-change rate: 0.425


In [ ]:

import pandas as pd
from pathlib import Path

out32 = Path("/content/planproof_results/experiment32")
out32.mkdir(parents=True, exist_ok=True)

rows32 = []

for case in cases32:
    original = case["task_order"]
    required = case["required_tasks"]

    plan = {
        "task_order": original,
        "required_tasks": required,
        "precedence_constraints": case["precedence_constraints"]
    }

    # Stable repair baseline
    stable_result = repair_plan(plan)
    stable_order = None
    stable_rate = None
    stable_valid = False

    if stable_result.get("success") and isinstance(
        stable_result.get("repaired_plan"), dict
    ):
        stable_plan = stable_result["repaired_plan"]
        stable_valid = validate_plan(stable_plan).get("valid", False)

        if stable_valid:
            stable_order = stable_plan["task_order"]
            stable_rate = shared_task_order_change_rate(
                original, stable_order
            )

    # Objective-aligned exact optimizer
    exact_result = exact_optimize_order_fair(
        case, time_limit_seconds=30
    )

    exact_order = exact_result["order"]
    exact_valid = False
    exact_rate = None

    if exact_order is not None:
        exact_plan = {
            "task_order": exact_order,
            "required_tasks": required,
            "precedence_constraints": case["precedence_constraints"]
        }
        exact_valid = validate_plan(exact_plan).get("valid", False)

        if exact_valid:
            exact_rate = shared_task_order_change_rate(
                original, exact_order
            )

    comparable = (
        stable_rate is not None and exact_rate is not None
    )

    rows32.append({
        "case_id": case["case_id"],
        "defect_type": case["defect_type"],
        "task_count": case["task_count"],
        "stable_valid": stable_valid,
        "stable_order_change": stable_rate,
        "exact_valid": exact_valid,
        "exact_order_change": exact_rate,
        "solver_status": exact_result["status"],
        "optimality_proven": exact_result["optimal"],
        "comparable": comparable,
        "order_change_reduction": (
            stable_rate - exact_rate if comparable else None
        )
    })

df32 = pd.DataFrame(rows32)
df32.to_csv(out32 / "experiment32_fair_comparison.csv", index=False)

comp32 = df32[df32["comparable"]]
optimal32 = comp32[comp32["optimality_proven"]]

print("EXPERIMENT 32 — OBJECTIVE-ALIGNED COMPARISON")
print("=" * 48)
print("Total cases:", len(df32))
print("Comparable cases:", len(comp32))
print("Stable valid:", int(df32["stable_valid"].sum()))
print("Exact valid:", int(df32["exact_valid"].sum()))
print("Optimality proven:", int(df32["optimality_proven"].sum()))

if len(comp32):
    print("Mean stable order-change:",
          round(comp32["stable_order_change"].mean(), 4))
    print("Mean exact order-change:",
          round(comp32["exact_order_change"].mean(), 4))
    print("Mean reduction:",
          round(comp32["order_change_reduction"].mean(), 4))

if len(optimal32):
    print("Mean reduction on proven-optimal cases:",
          round(optimal32["order_change_reduction"].mean(), 4))

print("Saved:", out32 / "experiment32_fair_comparison.csv")

print("\nRESULTS BY DEFECT CATEGORY")
display(comp32.groupby("defect_type").agg(
    cases=("case_id", "count"),
    stable_mean_change=("stable_order_change", "mean"),
    exact_mean_change=("exact_order_change", "mean"),
    mean_reduction=("order_change_reduction", "mean"),
    optimal_cases=("optimality_proven", "sum")
).reset_index())

print("\nCASES NEEDING INVESTIGATION")
display(df32[
    (~df32["stable_valid"])
    | (~df32["exact_valid"])
    | (~df32["comparable"])
    | (~df32["optimality_proven"])
].head(20))


EXPERIMENT 32 — OBJECTIVE-ALIGNED COMPARISON
Total cases: 100
Comparable cases: 100
Stable valid: 100
Exact valid: 100
Optimality proven: 100
Mean stable order-change: 0.393
Mean exact order-change: 0.3596
Mean reduction: 0.0334
Mean reduction on proven-optimal cases: 0.0334
Saved: /content/planproof_results/experiment32/experiment32_fair_comparison.csv

RESULTS BY DEFECT CATEGORY


,defect_type,cases,stable_mean_change,exact_mean_change,mean_reduction,optimal_cases
0,duplicate_task,16,0.372154,0.356806,0.015348,16
1,missing_task,24,0.429213,0.357607,0.071606,24
2,mixed_defects,13,0.453370,0.400090,0.053279,13
3,shuffled_order,25,0.381290,0.360817,0.020473,25
4,unknown_task,22,0.346352,0.338630,0.007722,22



CASES NEEDING INVESTIGATION


,case_id,defect_type,task_count,stable_valid,stable_order_change,exact_valid,exact_order_change,solver_status,optimality_proven,comparable,order_change_reduction


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

# Load Experiment 32 results
file33 = Path(
    "/content/planproof_results/experiment32/"
    "experiment32_fair_comparison.csv"
)
df33 = pd.read_csv(file33)

# Keep only cases with valid, comparable measurements
paired33 = df33[
    df33["comparable"].astype(str).str.lower().eq("true")
].copy()

stable = paired33["stable_order_change"].astype(float).to_numpy()
exact = paired33["exact_order_change"].astype(float).to_numpy()

# Positive difference means exact optimization improved the result
improvement = stable - exact

n = len(improvement)
mean_improvement = float(np.mean(improvement))
median_improvement = float(np.median(improvement))
sd_improvement = float(np.std(improvement, ddof=1))

# 95% paired t confidence interval for the mean improvement
se = stats.sem(improvement)
critical_t = stats.t.ppf(0.975, df=n - 1)
ci_low = mean_improvement - critical_t * se
ci_high = mean_improvement + critical_t * se

# Two-sided paired t-test
t_stat, t_pvalue = stats.ttest_rel(stable, exact)

# Wilcoxon signed-rank test: paired, distribution-free alternative
# Handle the possibility that every difference is zero.
if np.allclose(improvement, 0):
    wilcoxon_stat, wilcoxon_pvalue = 0.0, 1.0
else:
    wilcoxon_result = stats.wilcoxon(
        stable,
        exact,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto"
    )
    wilcoxon_stat = float(wilcoxon_result.statistic)
    wilcoxon_pvalue = float(wilcoxon_result.pvalue)

wins = int(np.sum(improvement > 1e-12))
ties = int(np.sum(np.abs(improvement) <= 1e-12))
losses = int(np.sum(improvement < -1e-12))

relative_reduction = (
    mean_improvement / np.mean(stable) * 100
    if np.mean(stable) != 0 else np.nan
)

print("EXPERIMENT 33 — PAIRED STATISTICAL ANALYSIS")
print("=" * 48)
print("Paired cases:", n)
print("Stable mean order-change:", round(float(np.mean(stable)), 4))
print("Exact mean order-change:", round(float(np.mean(exact)), 4))
print("Mean improvement:", round(mean_improvement, 4))
print("Median improvement:", round(median_improvement, 4))
print("Relative reduction (%):", round(relative_reduction, 2))
print("95% CI for mean improvement:",
      (round(ci_low, 4), round(ci_high, 4)))
print()
print("Exact optimizer wins:", wins)
print("Ties:", ties)
print("Exact optimizer losses:", losses)
print()
print("Paired t-test statistic:", round(float(t_stat), 4))
print("Paired t-test p-value:", float(t_pvalue))
print("Wilcoxon statistic:", round(wilcoxon_stat, 4))
print("Wilcoxon p-value:", float(wilcoxon_pvalue))

# Save case-level differences for reproducibility
paired33["paired_improvement"] = improvement
out33 = Path("/content/planproof_results/experiment33")
out33.mkdir(parents=True, exist_ok=True)

output33 = out33 / "experiment33_paired_analysis.csv"
paired33.to_csv(output33, index=False)

print("\nSaved:", output33)


EXPERIMENT 33 — PAIRED STATISTICAL ANALYSIS
Paired cases: 100
Stable mean order-change: 0.393
Exact mean order-change: 0.3596
Mean improvement: 0.0334
Median improvement: 0.0112
Relative reduction (%): 8.49
95% CI for mean improvement: (np.float64(0.0232), np.float64(0.0436))

Exact optimizer wins: 57
Ties: 43
Exact optimizer losses: 0

Paired t-test statistic: 6.504
Paired t-test p-value: 3.2132894068805702e-09
Wilcoxon statistic: 0.0
Wilcoxon p-value: 5.140176622341345e-11

Saved: /content/planproof_results/experiment33/experiment33_paired_analysis.csv


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

# Load the paired results from Experiment 33
file34 = Path(
    "/content/planproof_results/experiment33/"
    "experiment33_paired_analysis.csv"
)
df34 = pd.read_csv(file34)

# Recalculate differences to avoid relying on a saved derived column
df34["improvement"] = (
    df34["stable_order_change"] - df34["exact_order_change"]
)

valid34 = df34[
    df34["stable_order_change"].notna()
    & df34["exact_order_change"].notna()
].copy()

def summarize_improvement(group):
    """Summarize paired improvements without assuming all are positive."""
    x = group["improvement"].to_numpy(dtype=float)
    n = len(x)

    if n == 0:
        return pd.Series({
            "cases": 0,
            "mean_improvement": np.nan,
            "median_improvement": np.nan,
            "wins": 0,
            "ties": 0,
            "losses": 0,
            "min_improvement": np.nan,
            "max_improvement": np.nan
        })

    return pd.Series({
        "cases": n,
        "mean_improvement": np.mean(x),
        "median_improvement": np.median(x),
        "wins": int(np.sum(x > 1e-12)),
        "ties": int(np.sum(np.abs(x) <= 1e-12)),
        "losses": int(np.sum(x < -1e-12)),
        "min_improvement": np.min(x),
        "max_improvement": np.max(x)
    })

print("EXPERIMENT 34 — ROBUSTNESS ANALYSIS")
print("=" * 44)
print("Comparable cases:", len(valid34))

# Overall robustness summary
display(summarize_improvement(valid34).to_frame("overall").T.round(4))

# Breakdown by defect category
print("\nRESULTS BY DEFECT CATEGORY")
category34 = (
    valid34.groupby("defect_type")
    .apply(summarize_improvement, include_groups=False)
    .reset_index()
)
display(category34.round(4))

# Check whether the result depends on a small number of large improvements
print("\nSENSITIVITY CHECKS")
x34 = valid34["improvement"].to_numpy(dtype=float)

print("Mean improvement, all comparable cases:",
      round(float(np.mean(x34)), 4))

if len(x34) > 1:
    # Leave-one-out means: recompute after removing each case in turn
    loo_means34 = np.array([
        np.mean(np.delete(x34, i)) for i in range(len(x34))
    ])
    print("Leave-one-out mean range:",
          (round(float(loo_means34.min()), 4),
           round(float(loo_means34.max()), 4)))

# Results restricted to cases with proven optimality, if the column exists
if "optimality_proven" in valid34.columns:
    proven34 = valid34[
        valid34["optimality_proven"].astype(str).str.lower().eq("true")
    ]
    print("Cases with proven optimality:", len(proven34))
    if len(proven34):
        print("Mean improvement on proven-optimal cases:",
              round(float(proven34["improvement"].mean()), 4))

# Save detailed results
out34 = Path("/content/planproof_results/experiment34")
out34.mkdir(parents=True, exist_ok=True)

valid34.to_csv(out34 / "experiment34_case_level.csv", index=False)
category34.to_csv(out34 / "experiment34_category_summary.csv", index=False)

print("\nSaved files:")
print(out34 / "experiment34_case_level.csv")
print(out34 / "experiment34_category_summary.csv")


EXPERIMENT 34 — ROBUSTNESS ANALYSIS
Comparable cases: 100


,cases,mean_improvement,median_improvement,wins,ties,losses,min_improvement,max_improvement
overall,100.0,0.0334,0.0112,57.0,43.0,0.0,0.0,0.3333



RESULTS BY DEFECT CATEGORY


,defect_type,cases,mean_improvement,median_improvement,wins,ties,losses,min_improvement,max_improvement
0,duplicate_task,16.0,0.0153,0.0065,9.0,7.0,0.0,0.0,0.0571
1,missing_task,24.0,0.0716,0.0523,18.0,6.0,0.0,0.0,0.3333
2,mixed_defects,13.0,0.0533,0.0606,10.0,3.0,0.0,0.0,0.1538
3,shuffled_order,25.0,0.0205,0.0000,12.0,13.0,0.0,0.0,0.0735
4,unknown_task,22.0,0.0077,0.0000,8.0,14.0,0.0,0.0,0.0750



SENSITIVITY CHECKS
Mean improvement, all comparable cases: 0.0334
Leave-one-out mean range: (0.0304, 0.0337)
Cases with proven optimality: 100
Mean improvement on proven-optimal cases: 0.0334

Saved files:
/content/planproof_results/experiment34/experiment34_case_level.csv
/content/planproof_results/experiment34/experiment34_category_summary.csv


In [ ]:

import json
from pathlib import Path

path35 = Path(
    "/content/planproof_results/experiment25/"
    "experiment25_runtime_results.csv"
)

print("Experiment 25 runtime file exists:", path35.exists())

# Check whether the original scalability cases are available.
possible_files35 = [
    Path("/content/planproof_results/experiment25/experiment25_cases.json"),
    Path("/content/planproof_results/experiment25/scalability_cases.json"),
    Path("/content/planproof_results/experiment25/experiment25_benchmark.json"),
]

for p in possible_files35:
    print(p.name, "exists:", p.exists())


Experiment 25 runtime file exists: True
experiment25_cases.json exists: False
scalability_cases.json exists: False
experiment25_benchmark.json exists: False


In [ ]:

from pathlib import Path

root35 = Path("/content/planproof_results/experiment25")

print("Experiment 25 directory exists:", root35.exists())

if root35.exists():
    print("\nFiles currently available:")
    for file in sorted(root35.iterdir()):
        print(f"{file.name} ({file.stat().st_size} bytes)")

expected35 = root35 / "experiment25_scalability_benchmark.json"
print("\nExpected benchmark exists:", expected35.exists())

# Search the experiment results directory for JSON benchmark files.
results_root35 = Path("/content/planproof_results")
print("\nMatching benchmark files:")
for file in sorted(results_root35.rglob("*benchmark*.json")):
    print(file)


Experiment 25 directory exists: True

Files currently available:
experiment25_graph_audit.csv (548 bytes)
experiment25_runtime_results.csv (1952 bytes)
experiment25_scalability_benchmark.json (59408 bytes)
experiment26_quality_runtime_tradeoff.csv (2848 bytes)

Expected benchmark exists: True

Matching benchmark files:
/content/planproof_results/experiment20/experiment20_benchmark.json
/content/planproof_results/experiment20/experiment20_expanded_benchmark.json
/content/planproof_results/experiment21/experiment21_randomized_benchmark.json
/content/planproof_results/experiment25/experiment25_scalability_benchmark.json


In [ ]:

import json
from pathlib import Path
from collections import Counter

benchmark35 = Path(
    "/content/planproof_results/experiment25/"
    "experiment25_scalability_benchmark.json"
)

with open(benchmark35, "r", encoding="utf-8") as f:
    cases35 = json.load(f)

print("EXPERIMENT 35 — SOURCE DATA AUDIT")
print("=" * 42)
print("File:", benchmark35.name)
print("Total cases:", len(cases35))

sizes35 = Counter(case.get("task_count") for case in cases35)
print("\nCases by task size:")
for size, count in sorted(sizes35.items()):
    print(f"{size} tasks: {count} cases")

defects35 = Counter(case.get("defect_type", "MISSING_LABEL")
                    for case in cases35)
print("\nDefect distribution:")
for defect, count in sorted(defects35.items()):
    print(f"{defect}: {count}")

required_fields35 = {
    "case_id",
    "task_count",
    "task_order",
    "required_tasks",
    "precedence_constraints"
}

missing_fields35 = [
    case.get("case_id", f"case_{i}")
    for i, case in enumerate(cases35)
    if not required_fields35.issubset(case.keys())
]

print("\nCases missing required fields:", len(missing_fields35))
print("Missing-field case IDs:", missing_fields35[:10])

actual_sizes35 = {
    len(set(case.get("required_tasks", [])))
    for case in cases35
}

print("Unique required-task counts:", sorted(actual_sizes35))
print("Dataset audit complete.")


EXPERIMENT 35 — SOURCE DATA AUDIT
File: experiment25_scalability_benchmark.json
Total cases: 15

Cases by task size:
10 tasks: 3 cases
20 tasks: 3 cases
30 tasks: 3 cases
40 tasks: 3 cases
50 tasks: 3 cases

Defect distribution:
MISSING_LABEL: 15

Cases missing required fields: 0
Missing-field case IDs: []
Unique required-task counts: [10, 20, 30, 40, 50]
Dataset audit complete.


In [ ]:

import time
import pandas as pd
import numpy as np
from pathlib import Path

out35 = Path("/content/planproof_results/experiment35")
out35.mkdir(parents=True, exist_ok=True)

rows35 = []

for case in cases35:
    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    # Stable repair: five timing repetitions
    stable_times = []
    stable_result = None

    for _ in range(5):
        start = time.perf_counter()
        result = repair_plan(plan)
        stable_times.append(time.perf_counter() - start)
        stable_result = result

    stable_order = None
    stable_valid = False
    stable_rate = None

    if stable_result.get("success") and isinstance(
        stable_result.get("repaired_plan"), dict
    ):
        stable_plan = stable_result["repaired_plan"]
        stable_valid = validate_plan(stable_plan).get("valid", False)

        if stable_valid:
            stable_order = stable_plan["task_order"]
            stable_rate = shared_task_order_change_rate(
                case["task_order"], stable_order
            )

    # Exact optimization: five repetitions
    exact_times = []
    exact_result = None

    for _ in range(5):
        start = time.perf_counter()
        result = exact_optimize_order_fair(
            case, time_limit_seconds=30
        )
        exact_times.append(time.perf_counter() - start)
        exact_result = result

    exact_order = exact_result["order"]
    exact_valid = False
    exact_rate = None

    if exact_order is not None:
        exact_plan = {
            "task_order": exact_order,
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        }
        exact_valid = validate_plan(exact_plan).get("valid", False)

        if exact_valid:
            exact_rate = shared_task_order_change_rate(
                case["task_order"], exact_order
            )

    rows35.append({
        "case_id": case["case_id"],
        "task_count": case["task_count"],
        "stable_median_seconds": float(np.median(stable_times)),
        "stable_valid": stable_valid,
        "stable_order_change": stable_rate,
        "exact_median_seconds": float(np.median(exact_times)),
        "exact_valid": exact_valid,
        "exact_order_change": exact_rate,
        "solver_status": exact_result["status"],
        "optimality_proven": exact_result["optimal"],
        "runtime_ratio": (
            float(np.median(exact_times) / np.median(stable_times))
            if np.median(stable_times) > 0 else np.nan
        )
    })

    print(
        f"{case['case_id']}: {case['task_count']} tasks | "
        f"stable={np.median(stable_times):.6f}s | "
        f"exact={np.median(exact_times):.3f}s | "
        f"status={exact_result['status']}"
    )

df35 = pd.DataFrame(rows35)
df35.to_csv(out35 / "experiment35_scalability_results.csv", index=False)

print("\nEXPERIMENT 35 — BENCHMARK SUMMARY")
print("=" * 44)
print("Cases:", len(df35))
print("Stable valid:", int(df35["stable_valid"].sum()))
print("Exact valid:", int(df35["exact_valid"].sum()))
print("Optimality proven:", int(df35["optimality_proven"].sum()))
print("Saved:", out35 / "experiment35_scalability_results.csv")

print("\nSUMMARY BY TASK SIZE")
display(df35.groupby("task_count").agg(
    cases=("case_id", "count"),
    stable_median_seconds=("stable_median_seconds", "median"),
    exact_median_seconds=("exact_median_seconds", "median"),
    median_runtime_ratio=("runtime_ratio", "median"),
    stable_mean_order_change=("stable_order_change", "mean"),
    exact_mean_order_change=("exact_order_change", "mean"),
    optimal_cases=("optimality_proven", "sum")
).reset_index().round(6))


E25_N10_R1: 10 tasks | stable=0.000043s | exact=0.034s | status=OPTIMAL
E25_N10_R2: 10 tasks | stable=0.000037s | exact=0.087s | status=OPTIMAL
E25_N10_R3: 10 tasks | stable=0.000038s | exact=0.095s | status=OPTIMAL
E25_N20_R1: 20 tasks | stable=0.000083s | exact=0.348s | status=OPTIMAL
E25_N20_R2: 20 tasks | stable=0.000091s | exact=0.339s | status=OPTIMAL
E25_N20_R3: 20 tasks | stable=0.000077s | exact=1.434s | status=OPTIMAL
E25_N30_R1: 30 tasks | stable=0.000157s | exact=30.020s | status=FEASIBLE
E25_N30_R2: 30 tasks | stable=0.000094s | exact=30.022s | status=FEASIBLE
E25_N30_R3: 30 tasks | stable=0.000174s | exact=2.487s | status=OPTIMAL
E25_N40_R1: 40 tasks | stable=0.000205s | exact=30.029s | status=FEASIBLE
E25_N40_R2: 40 tasks | stable=0.000122s | exact=30.024s | status=FEASIBLE
E25_N40_R3: 40 tasks | stable=0.000125s | exact=30.154s | status=FEASIBLE
E25_N50_R1: 50 tasks | stable=0.000176s | exact=30.036s | status=FEASIBLE
E25_N50_R2: 50 tasks | stable=0.000168s | exact=30.0

,task_count,cases,stable_median_seconds,exact_median_seconds,median_runtime_ratio,stable_mean_order_change,exact_mean_order_change,optimal_cases
0,10,3,0.000038,0.087037,2350.518651,0.511111,0.422222,3
1,20,3,0.000083,0.348143,4200.569485,0.321053,0.270175,3
2,30,3,0.000157,30.020456,191151.003500,0.340230,0.281992,1
3,40,3,0.000125,30.029122,241047.396781,0.405128,0.369231,0
4,50,3,0.000170,30.039525,176929.977878,0.458231,0.410340,0


In [ ]:

from pathlib import Path

benchmark36 = Path(
    "/content/planproof_results/experiment25/"
    "experiment25_scalability_benchmark.json"
)

print("Benchmark exists:", benchmark36.exists())
print("Cases loaded:", len(cases35) if "cases35" in globals() else "Not loaded")
print("Corrected optimizer available:",
      callable(globals().get("exact_optimize_order_fair")))
print("Order-change metric available:",
      callable(globals().get("shared_task_order_change_rate")))
print("Plan validator available:",
      callable(globals().get("validate_plan")))

if benchmark36.exists():
    import json
    with open(benchmark36, "r", encoding="utf-8") as f:
        check36 = json.load(f)
    print("Cases in benchmark file:", len(check36))
    print("Task sizes:", sorted(set(c["task_count"] for c in check36)))


Benchmark exists: True
Cases loaded: 15
Corrected optimizer available: True
Order-change metric available: True
Plan validator available: True
Cases in benchmark file: 15
Task sizes: [10, 20, 30, 40, 50]


In [ ]:

import time
import pandas as pd
import numpy as np
from pathlib import Path

budgets36 = [1, 5, 10, 30]
rows36 = []

out36 = Path("/content/planproof_results/experiment36")
out36.mkdir(parents=True, exist_ok=True)

for case in cases35:
    for budget in budgets36:
        start36 = time.perf_counter()

        result36 = exact_optimize_order_fair(
            case,
            time_limit_seconds=budget
        )

        elapsed36 = time.perf_counter() - start36
        order36 = result36["order"]
        valid36 = False
        change36 = None

        if order36 is not None:
            plan36 = {
                "task_order": order36,
                "required_tasks": case["required_tasks"],
                "precedence_constraints": case["precedence_constraints"]
            }
            valid36 = validate_plan(plan36).get("valid", False)

            if valid36:
                change36 = shared_task_order_change_rate(
                    case["task_order"], order36
                )

        rows36.append({
            "case_id": case["case_id"],
            "task_count": case["task_count"],
            "time_budget_seconds": budget,
            "actual_runtime_seconds": elapsed36,
            "solver_status": result36["status"],
            "optimality_proven": result36["optimal"],
            "valid_solution": valid36,
            "order_change_rate": change36,
            "reversal_count": result36["reversals"]
        })

        print(
            f"{case['case_id']} | budget={budget:2d}s | "
            f"actual={elapsed36:6.2f}s | "
            f"status={result36['status']} | "
            f"valid={valid36} | "
            f"change={change36}"
        )

df36 = pd.DataFrame(rows36)
csv36 = out36 / "experiment36_anytime_results.csv"
df36.to_csv(csv36, index=False)

print("\nEXPERIMENT 36 — SUMMARY")
print("=" * 42)
print("Total solver runs:", len(df36))
print("Valid solutions:", int(df36["valid_solution"].sum()))
print("Optimality proven:", int(df36["optimality_proven"].sum()))
print("Saved:", csv36)

print("\nRESULTS BY TIME BUDGET")
display(df36.groupby("time_budget_seconds").agg(
    runs=("case_id", "count"),
    valid_solutions=("valid_solution", "sum"),
    optimal_cases=("optimality_proven", "sum"),
    mean_runtime=("actual_runtime_seconds", "mean"),
    median_runtime=("actual_runtime_seconds", "median"),
    mean_order_change=("order_change_rate", "mean")
).reset_index().round(4))

print("\nRESULTS BY TASK SIZE AND TIME BUDGET")
display(df36.groupby(
    ["task_count", "time_budget_seconds"]
).agg(
    valid_solutions=("valid_solution", "sum"),
    optimal_cases=("optimality_proven", "sum"),
    mean_order_change=("order_change_rate", "mean"),
    median_runtime=("actual_runtime_seconds", "median")
).reset_index().round(4))


E25_N10_R1 | budget= 1s | actual=  0.10s | status=OPTIMAL | valid=True | change=0.4
E25_N10_R1 | budget= 5s | actual=  0.05s | status=OPTIMAL | valid=True | change=0.4
E25_N10_R1 | budget=10s | actual=  0.04s | status=OPTIMAL | valid=True | change=0.4
E25_N10_R1 | budget=30s | actual=  0.05s | status=OPTIMAL | valid=True | change=0.4
E25_N10_R2 | budget= 1s | actual=  0.04s | status=OPTIMAL | valid=True | change=0.4222222222222222
E25_N10_R2 | budget= 5s | actual=  0.03s | status=OPTIMAL | valid=True | change=0.4222222222222222
E25_N10_R2 | budget=10s | actual=  0.05s | status=OPTIMAL | valid=True | change=0.4222222222222222
E25_N10_R2 | budget=30s | actual=  0.08s | status=OPTIMAL | valid=True | change=0.4222222222222222
E25_N10_R3 | budget= 1s | actual=  0.05s | status=OPTIMAL | valid=True | change=0.4444444444444444
E25_N10_R3 | budget= 5s | actual=  0.05s | status=OPTIMAL | valid=True | change=0.4444444444444444
E25_N10_R3 | budget=10s | actual=  0.06s | status=OPTIMAL | valid=True

,time_budget_seconds,runs,valid_solutions,optimal_cases,mean_runtime,median_runtime,mean_order_change
0,1,15,10,5,0.7443,0.9307,0.3713
1,5,15,15,7,3.2597,5.0119,0.3607
2,10,15,15,7,5.8865,10.0139,0.3548
3,30,15,15,7,16.5887,30.0220,0.3508



RESULTS BY TASK SIZE AND TIME BUDGET


,task_count,time_budget_seconds,valid_solutions,optimal_cases,mean_order_change,median_runtime
0,10,1,3,3,0.4222,0.0480
1,10,5,3,3,0.4222,0.0452
2,10,10,3,3,0.4222,0.0545
3,10,30,3,3,0.4222,0.0824
4,20,1,3,2,0.2754,0.5758
5,20,5,3,3,0.2702,0.8204
6,20,10,3,3,0.2702,0.8193
7,20,30,3,3,0.2702,0.8477
8,30,1,2,0,0.3345,1.0129
9,30,5,3,1,0.2881,5.0119


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

file37 = Path(
    "/content/planproof_results/experiment36/"
    "experiment36_anytime_results.csv"
)

df37 = pd.read_csv(file37)

# Keep only valid solutions with a measured order-change rate.
valid37 = df37[
    df37["valid_solution"].astype(str).str.lower().eq("true")
    & df37["order_change_rate"].notna()
].copy()

# Pivot results so each case has one column per time budget.
quality37 = valid37.pivot(
    index=["case_id", "task_count"],
    columns="time_budget_seconds",
    values="order_change_rate"
).reset_index()

quality37.columns.name = None
quality37 = quality37.rename(columns={
    1: "change_1s",
    5: "change_5s",
    10: "change_10s",
    30: "change_30s"
})

# Compare adjacent budgets only when both measurements exist.
def classify_change37(previous, current, tolerance=1e-12):
    if pd.isna(previous) or pd.isna(current):
        return "Not comparable"
    difference = current - previous
    if difference < -tolerance:
        return "Improved"
    if difference > tolerance:
        return "Worsened"
    return "Unchanged"

for earlier, later in [(1, 5), (5, 10), (10, 30)]:
    quality37[f"{earlier}s_to_{later}s"] = [
        classify_change37(a, b)
        for a, b in zip(
            quality37[f"change_{earlier}s"],
            quality37[f"change_{later}s"]
        )
    ]

print("EXPERIMENT 37 — ANYTIME IMPROVEMENT AUDIT")
print("=" * 48)
print("Unique cases:", quality37["case_id"].nunique())
print("Valid measurements at 1s:", int(quality37["change_1s"].notna().sum()))
print("Valid measurements at 5s:", int(quality37["change_5s"].notna().sum()))
print("Valid measurements at 10s:", int(quality37["change_10s"].notna().sum()))
print("Valid measurements at 30s:", int(quality37["change_30s"].notna().sum()))

print("\nADJACENT-BUDGET COMPARISONS")
for earlier, later in [(1, 5), (5, 10), (10, 30)]:
    column = f"{earlier}s_to_{later}s"
    print(f"\n{earlier}s to {later}s:")
    print(quality37[column].value_counts(dropna=False).to_dict())

print("\nQUALITY BY BUDGET")
for budget in [1, 5, 10, 30]:
    col = f"change_{budget}s"
    available = quality37[col].dropna()
    print(
        f"{budget}s: n={len(available)}, "
        f"mean={available.mean():.4f}" if len(available)
        else f"{budget}s: no valid solutions"
    )

out37 = Path("/content/planproof_results/experiment37")
out37.mkdir(parents=True, exist_ok=True)

quality37.to_csv(
    out37 / "experiment37_anytime_audit.csv",
    index=False
)

print("\nSaved:", out37 / "experiment37_anytime_audit.csv")

print("\nCASE-LEVEL RESULTS")
display(quality37.round(4))


EXPERIMENT 37 — ANYTIME IMPROVEMENT AUDIT
Unique cases: 15
Valid measurements at 1s: 10
Valid measurements at 5s: 15
Valid measurements at 10s: 15
Valid measurements at 30s: 15

ADJACENT-BUDGET COMPARISONS

1s to 5s:
{'Unchanged': 5, 'Improved': 5, 'Not comparable': 5}

5s to 10s:
{'Improved': 8, 'Unchanged': 7}

10s to 30s:
{'Unchanged': 10, 'Improved': 5}

QUALITY BY BUDGET
1s: n=10, mean=0.3713
5s: n=15, mean=0.3607
10s: n=15, mean=0.3548
30s: n=15, mean=0.3508

Saved: /content/planproof_results/experiment37/experiment37_anytime_audit.csv

CASE-LEVEL RESULTS


,case_id,task_count,change_1s,change_5s,change_10s,change_30s,1s_to_5s,5s_to_10s,10s_to_30s
0,E25_N10_R1,10,0.4000,0.4000,0.4000,0.4000,Unchanged,Unchanged,Unchanged
1,E25_N10_R2,10,0.4222,0.4222,0.4222,0.4222,Unchanged,Unchanged,Unchanged
2,E25_N10_R3,10,0.4444,0.4444,0.4444,0.4444,Unchanged,Unchanged,Unchanged
3,E25_N20_R1,20,0.1632,0.1632,0.1632,0.1632,Unchanged,Unchanged,Unchanged
4,E25_N20_R2,20,0.3211,0.3211,0.3211,0.3211,Unchanged,Unchanged,Unchanged
5,E25_N20_R3,20,0.3421,0.3263,0.3263,0.3263,Improved,Unchanged,Unchanged
6,E25_N30_R1,30,NaN,0.2207,0.2046,0.2046,Not comparable,Improved,Unchanged
7,E25_N30_R2,30,0.3816,0.3655,0.3632,0.3632,Improved,Improved,Unchanged
8,E25_N30_R3,30,0.2874,0.2782,0.2782,0.2782,Improved,Unchanged,Unchanged
9,E25_N40_R1,40,0.4974,0.3769,0.3564,0.3462,Improved,Improved,Improved


In [ ]:

import time

def adaptive_repair_plan(
    case,
    time_budget_seconds=5
):
    """
    Return a valid plan quickly using stable repair, then try
    exact optimization within a time budget.

    The stable repair is retained as a fallback. Candidates are
    compared using shared-task order-change rate.
    """
    start_time = time.perf_counter()

    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    # Step 1: Obtain the stable-repair fallback.
    stable_result = repair_plan(plan)

    if (
        not stable_result.get("success", False)
        or not isinstance(stable_result.get("repaired_plan"), dict)
    ):
        return {
            "success": False,
            "order": None,
            "method": "stable_repair_failed",
            "order_change_rate": None,
            "elapsed_seconds": time.perf_counter() - start_time,
            "optimality_proven": False
        }

    stable_order = stable_result["repaired_plan"]["task_order"]

    stable_plan = {
        "task_order": stable_order,
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    if not validate_plan(stable_plan).get("valid", False):
        return {
            "success": False,
            "order": None,
            "method": "stable_repair_invalid",
            "order_change_rate": None,
            "elapsed_seconds": time.perf_counter() - start_time,
            "optimality_proven": False
        }

    best_order = stable_order
    best_rate = shared_task_order_change_rate(
        case["task_order"], stable_order
    )
    best_method = "stable_repair"
    optimality_proven = False

    # Step 2: Try exact optimization if the budget permits.
    remaining = time_budget_seconds - (
        time.perf_counter() - start_time
    )

    if remaining > 0:
        exact_result = exact_optimize_order_fair(
            case,
            time_limit_seconds=remaining
        )

        candidate = exact_result["order"]

        if candidate is not None:
            candidate_plan = {
                "task_order": candidate,
                "required_tasks": case["required_tasks"],
                "precedence_constraints": case["precedence_constraints"]
            }

            if validate_plan(candidate_plan).get("valid", False):
                candidate_rate = shared_task_order_change_rate(
                    case["task_order"], candidate
                )

                # Keep the exact candidate only if it is no worse.
                if candidate_rate < best_rate - 1e-12:
                    best_order = candidate
                    best_rate = candidate_rate
                    best_method = "exact_optimizer"

                # This flag describes the exact solver's objective,
                # not global optimality across every possible metric.
                optimality_proven = exact_result["optimal"]

    return {
        "success": True,
        "order": best_order,
        "method": best_method,
        "order_change_rate": best_rate,
        "elapsed_seconds": time.perf_counter() - start_time,
        "optimality_proven": optimality_proven
    }


print("Adaptive repair function defined.")
print("Fallback: stable repair.")
print("Candidate acceptance: valid and no worse on order-change rate.")


Adaptive repair function defined.
Fallback: stable repair.
Candidate acceptance: valid and no worse on order-change rate.


In [ ]:

# Select one 30-task case from Experiment 25
test_case38 = next(
    case for case in cases35
    if case["task_count"] == 30
)

result38 = adaptive_repair_plan(
    test_case38,
    time_budget_seconds=5
)

print("EXPERIMENT 38 — SINGLE-CASE TEST")
print("=" * 40)
print("Case:", test_case38["case_id"])
print("Task count:", test_case38["task_count"])
print("Success:", result38["success"])
print("Selected method:", result38["method"])
print("Order-change rate:", result38["order_change_rate"])
print("Elapsed seconds:", round(result38["elapsed_seconds"], 3))
print("Optimality proven:", result38["optimality_proven"])

if result38["order"] is not None:
    final_plan38 = {
        "task_order": result38["order"],
        "required_tasks": test_case38["required_tasks"],
        "precedence_constraints": test_case38["precedence_constraints"]
    }
    validation38 = validate_plan(final_plan38)
    print("Final plan valid:", validation38["valid"])
    print("Violation count:", len(validation38["violations"]))


EXPERIMENT 38 — SINGLE-CASE TEST
Case: E25_N30_R1
Task count: 30
Success: True
Selected method: exact_optimizer
Order-change rate: 0.22298850574712645
Elapsed seconds: 5.036
Optimality proven: False
Final plan valid: True
Violation count: 0


In [ ]:

import time
import pandas as pd
import numpy as np
from pathlib import Path

out38 = Path("/content/planproof_results/experiment38")
out38.mkdir(parents=True, exist_ok=True)

rows38 = []

for case in cases35:
    original = case["task_order"]
    required = case["required_tasks"]
    constraints = case["precedence_constraints"]

    plan = {
        "task_order": original,
        "required_tasks": required,
        "precedence_constraints": constraints
    }

    # 1. Stable repair
    start = time.perf_counter()
    stable_result = repair_plan(plan)
    stable_time = time.perf_counter() - start

    stable_order = None
    stable_rate = None
    stable_valid = False

    if stable_result.get("success") and isinstance(
        stable_result.get("repaired_plan"), dict
    ):
        stable_order = stable_result["repaired_plan"]["task_order"]
        stable_plan = {
            "task_order": stable_order,
            "required_tasks": required,
            "precedence_constraints": constraints
        }
        stable_valid = validate_plan(stable_plan).get("valid", False)

        if stable_valid:
            stable_rate = shared_task_order_change_rate(
                original, stable_order
            )

    # 2. Exact optimization with a 5-second solver limit
    start = time.perf_counter()
    exact_result = exact_optimize_order_fair(
        case, time_limit_seconds=5
    )
    exact_time = time.perf_counter() - start

    exact_order = exact_result["order"]
    exact_valid = False
    exact_rate = None

    if exact_order is not None:
        exact_plan = {
            "task_order": exact_order,
            "required_tasks": required,
            "precedence_constraints": constraints
        }
        exact_valid = validate_plan(exact_plan).get("valid", False)

        if exact_valid:
            exact_rate = shared_task_order_change_rate(
                original, exact_order
            )

    # 3. Adaptive repair with a 5-second solver budget
    adaptive_result = adaptive_repair_plan(
        case, time_budget_seconds=5
    )
    adaptive_order = adaptive_result["order"]
    adaptive_valid = False
    adaptive_rate = None

    if adaptive_order is not None:
        adaptive_plan = {
            "task_order": adaptive_order,
            "required_tasks": required,
            "precedence_constraints": constraints
        }
        adaptive_valid = validate_plan(adaptive_plan).get("valid", False)

        if adaptive_valid:
            adaptive_rate = shared_task_order_change_rate(
                original, adaptive_order
            )

    rows38.append({
        "case_id": case["case_id"],
        "task_count": case["task_count"],
        "stable_valid": stable_valid,
        "stable_seconds": stable_time,
        "stable_order_change": stable_rate,
        "exact_valid": exact_valid,
        "exact_seconds": exact_time,
        "exact_status": exact_result["status"],
        "exact_optimal": exact_result["optimal"],
        "exact_order_change": exact_rate,
        "adaptive_valid": adaptive_valid,
        "adaptive_seconds": adaptive_result["elapsed_seconds"],
        "adaptive_method": adaptive_result["method"],
        "adaptive_order_change": adaptive_rate,
        "adaptive_optimal": adaptive_result["optimality_proven"]
    })

df38 = pd.DataFrame(rows38)
df38.to_csv(out38 / "experiment38_policy_comparison.csv", index=False)

print("EXPERIMENT 38 — POLICY COMPARISON")
print("=" * 42)
print("Cases:", len(df38))
print("Stable valid:", int(df38["stable_valid"].sum()))
print("Exact valid:", int(df38["exact_valid"].sum()))
print("Adaptive valid:", int(df38["adaptive_valid"].sum()))
print("Adaptive selected exact:", int(
    (df38["adaptive_method"] == "exact_optimizer").sum()
))
print("Saved:", out38 / "experiment38_policy_comparison.csv")

print("\nMEAN RESULTS")
display(df38[[
    "stable_seconds", "stable_order_change",
    "exact_seconds", "exact_order_change",
    "adaptive_seconds", "adaptive_order_change"
]].mean().to_frame("mean").T.round(4))

print("\nRESULTS BY TASK SIZE")
display(df38.groupby("task_count").agg(
    cases=("case_id", "count"),
    stable_mean_change=("stable_order_change", "mean"),
    exact_mean_change=("exact_order_change", "mean"),
    adaptive_mean_change=("adaptive_order_change", "mean"),
    adaptive_median_seconds=("adaptive_seconds", "median"),
    adaptive_exact_selections=(
        "adaptive_method",
        lambda x: int((x == "exact_optimizer").sum())
    )
).reset_index().round(4))


EXPERIMENT 38 — POLICY COMPARISON
Cases: 15
Stable valid: 15
Exact valid: 15
Adaptive valid: 15
Adaptive selected exact: 12
Saved: /content/planproof_results/experiment38/experiment38_policy_comparison.csv

MEAN RESULTS


,stable_seconds,stable_order_change,exact_seconds,exact_order_change,adaptive_seconds,adaptive_order_change
mean,0.0003,0.4072,3.0043,0.3617,3.049,0.3601



RESULTS BY TASK SIZE


,task_count,cases,stable_mean_change,exact_mean_change,adaptive_mean_change,adaptive_median_seconds,adaptive_exact_selections
0,10,3,0.5111,0.4222,0.4222,0.0377,2
1,20,3,0.3211,0.2702,0.2702,0.4121,3
2,30,3,0.3402,0.2889,0.2843,5.0224,3
3,40,3,0.4051,0.3906,0.3902,5.0338,2
4,50,3,0.4582,0.4365,0.4335,5.0534,2


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

file39 = Path(
    "/content/planproof_results/experiment38/"
    "experiment38_policy_comparison.csv"
)
df39 = pd.read_csv(file39)

# Check whether the adaptive result is no worse than both methods.
df39["adaptive_vs_stable"] = (
    df39["adaptive_order_change"] - df39["stable_order_change"]
)
df39["adaptive_vs_exact"] = (
    df39["adaptive_order_change"] - df39["exact_order_change"]
)

tol39 = 1e-12

df39["adaptive_no_worse_than_stable"] = (
    df39["adaptive_order_change"]
    <= df39["stable_order_change"] + tol39
)
df39["adaptive_no_worse_than_exact"] = (
    df39["adaptive_order_change"]
    <= df39["exact_order_change"] + tol39
)

df39["selected_exact_but_worse_than_stable"] = (
    df39["adaptive_method"].eq("exact_optimizer")
    & (df39["adaptive_order_change"]
       > df39["stable_order_change"] + tol39)
) if "adaptive_method" in df39.columns else False

print("EXPERIMENT 39 — ADAPTIVE DECISION AUDIT")
print("=" * 46)
print("Cases:", len(df39))
print("Adaptive no worse than stable:",
      int(df39["adaptive_no_worse_than_stable"].sum()), "/", len(df39))
print("Adaptive no worse than exact:",
      int(df39["adaptive_no_worse_than_exact"].sum()), "/", len(df39))
print("Adaptive worse than stable:",
      int((df39["adaptive_vs_stable"] > tol39).sum()))
print("Adaptive worse than exact:",
      int((df39["adaptive_vs_exact"] > tol39).sum()))

print("\nCASES WHERE STABLE REPAIR WAS RETAINED")
if "adaptive_method" in df39.columns:
    display(df39.loc[
        df39["adaptive_method"] != "exact_optimizer",
        [
            "case_id", "task_count",
            "stable_order_change", "exact_order_change",
            "adaptive_order_change", "adaptive_method",
            "adaptive_vs_stable", "adaptive_vs_exact"
        ]
    ].round(4))

print("\nALL CASES")
display(df39[[
    "case_id", "task_count",
    "stable_order_change", "exact_order_change",
    "adaptive_order_change",
    "adaptive_vs_stable", "adaptive_vs_exact",
    "adaptive_no_worse_than_stable",
    "adaptive_no_worse_than_exact"
]].round(4))

out39 = Path("/content/planproof_results/experiment39")
out39.mkdir(parents=True, exist_ok=True)
df39.to_csv(out39 / "experiment39_decision_audit.csv", index=False)
print("\nSaved:", out39 / "experiment39_decision_audit.csv")


EXPERIMENT 39 — ADAPTIVE DECISION AUDIT
Cases: 15
Adaptive no worse than stable: 15 / 15
Adaptive no worse than exact: 14 / 15
Adaptive worse than stable: 0
Adaptive worse than exact: 1

CASES WHERE STABLE REPAIR WAS RETAINED


,case_id,task_count,stable_order_change,exact_order_change,adaptive_order_change,adaptive_method,adaptive_vs_stable,adaptive_vs_exact
1,E25_N10_R2,10,0.4222,0.4222,0.4222,stable_repair,0.0,0.0000
10,E25_N40_R2,40,0.4346,0.4397,0.4346,stable_repair,0.0,-0.0051
13,E25_N50_R2,50,0.4482,0.4571,0.4482,stable_repair,0.0,-0.0090



ALL CASES


,case_id,task_count,stable_order_change,exact_order_change,adaptive_order_change,adaptive_vs_stable,adaptive_vs_exact,adaptive_no_worse_than_stable,adaptive_no_worse_than_exact
0,E25_N10_R1,10,0.4444,0.4000,0.4000,-0.0444,0.0000,True,True
1,E25_N10_R2,10,0.4222,0.4222,0.4222,0.0000,0.0000,True,True
2,E25_N10_R3,10,0.6667,0.4444,0.4444,-0.2222,0.0000,True,True
3,E25_N20_R1,20,0.2211,0.1632,0.1632,-0.0579,0.0000,True,True
4,E25_N20_R2,20,0.3737,0.3211,0.3211,-0.0526,0.0000,True,True
5,E25_N20_R3,20,0.3684,0.3263,0.3263,-0.0421,0.0000,True,True
6,E25_N30_R1,30,0.2874,0.2138,0.2092,-0.0782,-0.0046,True,True
7,E25_N30_R2,30,0.3862,0.3747,0.3655,-0.0207,-0.0092,True,True
8,E25_N30_R3,30,0.3471,0.2782,0.2782,-0.0690,0.0000,True,True
9,E25_N40_R1,40,0.4154,0.3769,0.3769,-0.0385,0.0000,True,True



Saved: /content/planproof_results/experiment39/experiment39_decision_audit.csv


In [ ]:

# Experiment 40: investigate E25_N40_R3

case40 = next(
    case for case in cases35
    if case["case_id"] == "E25_N40_R3"
)

original40 = case40["task_order"]
required40 = case40["required_tasks"]
constraints40 = case40["precedence_constraints"]

# Recompute stable repair
stable40 = repair_plan({
    "task_order": original40,
    "required_tasks": required40,
    "precedence_constraints": constraints40
})

stable_order40 = stable40["repaired_plan"]["task_order"]
stable_rate40 = shared_task_order_change_rate(
    original40, stable_order40
)

# Run the corrected exact optimizer again
exact40 = exact_optimize_order_fair(
    case40, time_limit_seconds=5
)

exact_rate40 = None
if exact40["order"] is not None:
    exact_rate40 = shared_task_order_change_rate(
        original40, exact40["order"]
    )

# Run adaptive repair independently
adaptive40 = adaptive_repair_plan(
    case40, time_budget_seconds=5
)

print("EXPERIMENT 40 — CASE DIAGNOSIS")
print("=" * 40)
print("Case:", case40["case_id"])
print("Stable rate:", round(stable_rate40, 6))
print("Exact status:", exact40["status"])
print("Exact optimality proven:", exact40["optimal"])
print("Exact rate:", (
    round(exact_rate40, 6)
    if exact_rate40 is not None else None
))
print("Adaptive method:", adaptive40["method"])
print("Adaptive rate:", (
    round(adaptive40["order_change_rate"], 6)
    if adaptive40["order_change_rate"] is not None else None
))
print("Adaptive success:", adaptive40["success"])


EXPERIMENT 40 — CASE DIAGNOSIS
Case: E25_N40_R3
Stable rate: 0.365385
Exact status: FEASIBLE
Exact optimality proven: False
Exact rate: 0.362821
Adaptive method: exact_optimizer
Adaptive rate: 0.362821
Adaptive success: True


In [ ]:
case41 = next(
    case for case in cases35
    if case["case_id"] == "E25_N40_R3"
)

stable41 = repair_plan(case41)
stable_order41 = stable41["repaired_plan"]["task_order"]

adaptive41 = adaptive_repair_plan(
    case41,
    time_budget_seconds=5
)

adaptive_order41 = adaptive41["order"]

print("EXPERIMENT 41 — ADAPTIVE POLICY CHECK")
print("=" * 45)
print("Case:", case41["case_id"])
print("Stable rate:", pairwise_order_change_rate(
    case41["task_order"], stable_order41
))
print("Adaptive rate:", adaptive41["order_change_rate"])
print("Adaptive method:", adaptive41["method"])
print("Adaptive success:", adaptive41["success"])
print("Adaptive order valid:", validate_plan({
    **case41,
    "task_order": adaptive_order41
})["valid"])
print("Adaptive never-worse-than-stable:",
      adaptive41["order_change_rate"] <=
      pairwise_order_change_rate(
          case41["task_order"], stable_order41
      ) + 1e-12)

EXPERIMENT 41 — ADAPTIVE POLICY CHECK
Case: E25_N40_R3
Stable rate: 0.36538461538461536
Adaptive rate: 0.3628205128205128
Adaptive method: exact_optimizer
Adaptive success: True
Adaptive order valid: True
Adaptive never-worse-than-stable: True


In [ ]:
# EXPERIMENT 42 — FULL ADAPTIVE POLICY AUDIT

results42 = []

for case in cases35:
    stable_result = repair_plan(case)
    stable_order = stable_result["repaired_plan"]["task_order"]

    stable_rate = pairwise_order_change_rate(
        case["task_order"], stable_order
    )

    adaptive_result = adaptive_repair_plan(
        case, time_budget_seconds=5
    )

    adaptive_order = adaptive_result["order"]

    validation = validate_plan({
        **case,
        "task_order": adaptive_order
    })

    adaptive_rate = adaptive_result["order_change_rate"]

    results42.append({
        "case_id": case["case_id"],
        "stable_rate": stable_rate,
        "adaptive_rate": adaptive_rate,
        "method": adaptive_result["method"],
        "success": adaptive_result["success"],
        "valid": validation["valid"],
        "never_worse_than_stable": (
            adaptive_rate <= stable_rate + 1e-12
        )
    })

print("EXPERIMENT 42 — FULL ADAPTIVE POLICY AUDIT")
print("=" * 50)
print("Total cases:", len(results42))
print("Successful:", sum(r["success"] for r in results42))
print("Valid plans:", sum(r["valid"] for r in results42))
print("Never worse than stable:", sum(
    r["never_worse_than_stable"] for r in results42
))
print("Stable selected:", sum(
    r["method"] == "stable" for r in results42
))
print("Exact optimizer selected:", sum(
    r["method"] == "exact_optimizer" for r in results42
))

failures42 = [
    r for r in results42
    if not r["success"]
    or not r["valid"]
    or not r["never_worse_than_stable"]
]

print("Audit failures:", len(failures42))

if failures42:
    for row in failures42:
        print(row)
else:
    print("All cases passed the audit.")

EXPERIMENT 42 — FULL ADAPTIVE POLICY AUDIT
Total cases: 15
Successful: 15
Valid plans: 15
Never worse than stable: 15
Stable selected: 0
Exact optimizer selected: 13
Audit failures: 0
All cases passed the audit.


In [ ]:
from collections import Counter

print("EXPERIMENT 43 — METHOD SELECTION AUDIT")
print("=" * 45)

print("Method counts:")
print(Counter(r["method"] for r in results42))

print("\nCases not labelled stable or exact_optimizer:")
for row in results42:
    if row["method"] not in ("stable", "exact_optimizer"):
        print(row)

print("\nAll case IDs and selected methods:")
for row in results42:
    print(row["case_id"], "->", row["method"])

EXPERIMENT 43 — METHOD SELECTION AUDIT
Method counts:
Counter({'exact_optimizer': 13, 'stable_repair': 2})

Cases not labelled stable or exact_optimizer:
{'case_id': 'E25_N10_R2', 'stable_rate': 0.4222222222222222, 'adaptive_rate': 0.4222222222222222, 'method': 'stable_repair', 'success': True, 'valid': True, 'never_worse_than_stable': True}
{'case_id': 'E25_N40_R2', 'stable_rate': 0.4346153846153846, 'adaptive_rate': 0.4346153846153846, 'method': 'stable_repair', 'success': True, 'valid': True, 'never_worse_than_stable': True}

All case IDs and selected methods:
E25_N10_R1 -> exact_optimizer
E25_N10_R2 -> stable_repair
E25_N10_R3 -> exact_optimizer
E25_N20_R1 -> exact_optimizer
E25_N20_R2 -> exact_optimizer
E25_N20_R3 -> exact_optimizer
E25_N30_R1 -> exact_optimizer
E25_N30_R2 -> exact_optimizer
E25_N30_R3 -> exact_optimizer
E25_N40_R1 -> exact_optimizer
E25_N40_R2 -> stable_repair
E25_N40_R3 -> exact_optimizer
E25_N50_R1 -> exact_optimizer
E25_N50_R2 -> exact_optimizer
E25_N50_R3 -> 

In [ ]:
import inspect

print(inspect.getsource(adaptive_repair_plan))

def adaptive_repair_plan(
    case,
    time_budget_seconds=5
):
    """
    Return a valid plan quickly using stable repair, then try
    exact optimization within a time budget.

    The stable repair is retained as a fallback. Candidates are
    compared using shared-task order-change rate.
    """
    start_time = time.perf_counter()

    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    # Step 1: Obtain the stable-repair fallback.
    stable_result = repair_plan(plan)

    if (
        not stable_result.get("success", False)
        or not isinstance(stable_result.get("repaired_plan"), dict)
    ):
        return {
            "success": False,
            "order": None,
            "method": "stable_repair_failed",
            "order_change_rate": None,
            "elapsed_seconds": time.perf_counter() - start_time,
            "optimality_proven": False

In [ ]:
import inspect

print(inspect.getsource(adaptive_repair_plan))

def adaptive_repair_plan(
    case,
    time_budget_seconds=5
):
    """
    Return a valid plan quickly using stable repair, then try
    exact optimization within a time budget.

    The stable repair is retained as a fallback. Candidates are
    compared using shared-task order-change rate.
    """
    start_time = time.perf_counter()

    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    # Step 1: Obtain the stable-repair fallback.
    stable_result = repair_plan(plan)

    if (
        not stable_result.get("success", False)
        or not isinstance(stable_result.get("repaired_plan"), dict)
    ):
        return {
            "success": False,
            "order": None,
            "method": "stable_repair_failed",
            "order_change_rate": None,
            "elapsed_seconds": time.perf_counter() - start_time,
            "optimality_proven": False

In [ ]:

def adaptive_repair_plan(
    case,
    time_budget_seconds=5
):
    """
    Return a valid plan using stable repair first, then try
    exact optimization within the remaining time budget.

    The exact candidate is selected only if it strictly improves
    the shared-task order-change rate.

    optimality_proven refers to the SELECTED plan's objective.
    """
    start_time = time.perf_counter()

    plan = {
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    # Step 1: Construct the stable-repair fallback.
    stable_result = repair_plan(plan)

    if (
        not stable_result.get("success", False)
        or not isinstance(stable_result.get("repaired_plan"), dict)
    ):
        return {
            "success": False,
            "order": None,
            "method": "stable_repair_failed",
            "order_change_rate": None,
            "elapsed_seconds": time.perf_counter() - start_time,
            "optimality_proven": False,
            "solver_status": None,
            "solver_optimality_proven": False
        }

    stable_order = stable_result["repaired_plan"]["task_order"]

    stable_plan = {
        "task_order": stable_order,
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    if not validate_plan(stable_plan).get("valid", False):
        return {
            "success": False,
            "order": None,
            "method": "stable_repair_invalid",
            "order_change_rate": None,
            "elapsed_seconds": time.perf_counter() - start_time,
            "optimality_proven": False,
            "solver_status": None,
            "solver_optimality_proven": False
        }

    best_order = stable_order
    best_rate = shared_task_order_change_rate(
        case["task_order"], stable_order
    )
    best_method = "stable_repair"

    solver_status = None
    solver_optimality_proven = False

    # Step 2: Try exact optimization within the remaining budget.
    remaining = time_budget_seconds - (
        time.perf_counter() - start_time
    )

    if remaining > 0:
        exact_result = exact_optimize_order_fair(
            case,
            time_limit_seconds=remaining
        )

        solver_status = exact_result.get("status")
        solver_optimality_proven = bool(
            exact_result.get("optimal", False)
        )

        candidate = exact_result.get("order")

        if candidate is not None:
            candidate_plan = {
                "task_order": candidate,
                "required_tasks": case["required_tasks"],
                "precedence_constraints": case["precedence_constraints"]
            }

            if validate_plan(candidate_plan).get("valid", False):
                candidate_rate = shared_task_order_change_rate(
                    case["task_order"], candidate
                )

                # Select only a strictly better candidate.
                if candidate_rate < best_rate - 1e-12:
                    best_order = candidate
                    best_rate = candidate_rate
                    best_method = "exact_optimizer"

    # Only claim optimality when the selected plan is the exact
    # candidate and the solver proved that candidate optimal.
    selected_plan_optimality_proven = (
        best_method == "exact_optimizer"
        and solver_optimality_proven
    )

    return {
        "success": True,
        "order": best_order,
        "method": best_method,
        "order_change_rate": best_rate,
        "elapsed_seconds": time.perf_counter() - start_time,
        "optimality_proven": selected_plan_optimality_proven,
        "solver_status": solver_status,
        "solver_optimality_proven": solver_optimality_proven
    }


In [ ]:

case45 = next(
    case for case in cases35
    if case["case_id"] == "E25_N40_R2"
)

result45 = adaptive_repair_plan(
    case45,
    time_budget_seconds=5
)

print("EXPERIMENT 45 — OPTIMALITY FLAG CHECK")
print("=" * 45)
print("Case:", case45["case_id"])
print("Selected method:", result45["method"])
print("Selected rate:", result45["order_change_rate"])
print("Success:", result45["success"])
print("Solver status:", result45["solver_status"])
print("Solver optimality proven:",
      result45["solver_optimality_proven"])
print("Selected plan optimality proven:",
      result45["optimality_proven"])


EXPERIMENT 45 — OPTIMALITY FLAG CHECK
Case: E25_N40_R2
Selected method: stable_repair
Selected rate: 0.4346153846153846
Success: True
Solver status: FEASIBLE
Solver optimality proven: False
Selected plan optimality proven: False


In [ ]:

# EXPERIMENT 46 — ADAPTIVE POLICY REGRESSION TEST

checks46 = []

for case in cases35:
    stable = repair_plan(case)
    stable_order = stable["repaired_plan"]["task_order"]

    stable_rate = shared_task_order_change_rate(
        case["task_order"], stable_order
    )

    adaptive = adaptive_repair_plan(
        case, time_budget_seconds=5
    )

    validation = validate_plan({
        "task_order": adaptive["order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }) if adaptive["success"] else {"valid": False}

    checks46.append({
        "case_id": case["case_id"],
        "success": adaptive["success"],
        "valid": validation["valid"],
        "never_worse": (
            adaptive["order_change_rate"] <= stable_rate + 1e-12
            if adaptive["success"] else False
        ),
        "flag_consistent": (
            not adaptive["optimality_proven"]
            if adaptive["method"] == "stable_repair"
            else (
                adaptive["optimality_proven"]
                == adaptive["solver_optimality_proven"]
                if adaptive["method"] == "exact_optimizer"
                else not adaptive["optimality_proven"]
            )
        )
    })

print("EXPERIMENT 46 — REGRESSION TEST")
print("=" * 45)
print("Total cases:", len(checks46))
print("Successful:", sum(x["success"] for x in checks46))
print("Valid plans:", sum(x["valid"] for x in checks46))
print("Never worse than stable:", sum(x["never_worse"] for x in checks46))
print("Optimality flags consistent:", sum(
    x["flag_consistent"] for x in checks46
))
print("Failures:", sum(
    not all([
        x["success"], x["valid"],
        x["never_worse"], x["flag_consistent"]
    ])
    for x in checks46
))


EXPERIMENT 46 — REGRESSION TEST
Total cases: 15
Successful: 15
Valid plans: 15
Never worse than stable: 15
Optimality flags consistent: 15
Failures: 0


In [ ]:

# EXPERIMENT 47 — CONSOLIDATED BENCHMARK COMPARISON

import os
import time
import pandas as pd

results47 = []

for case in cases35:
    # 1. Stable repair
    t0 = time.perf_counter()
    stable_result = repair_plan(case)
    stable_seconds = time.perf_counter() - t0

    stable_order = stable_result["repaired_plan"]["task_order"]
    stable_validation = validate_plan({
        "task_order": stable_order,
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    })
    stable_rate = shared_task_order_change_rate(
        case["task_order"], stable_order
    )

    # 2. Fair exact optimizer (separate run)
    t0 = time.perf_counter()
    exact_result = exact_optimize_order_fair(
        case, time_limit_seconds=5
    )
    exact_seconds = time.perf_counter() - t0

    exact_order = exact_result.get("order")
    exact_valid = False
    exact_rate = None

    if exact_order is not None:
        exact_valid = validate_plan({
            "task_order": exact_order,
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        }).get("valid", False)

        if exact_valid:
            exact_rate = shared_task_order_change_rate(
                case["task_order"], exact_order
            )

    # 3. Adaptive policy (its own solver run)
    t0 = time.perf_counter()
    adaptive_result = adaptive_repair_plan(
        case, time_budget_seconds=5
    )
    adaptive_seconds = time.perf_counter() - t0

    adaptive_order = adaptive_result.get("order")
    adaptive_valid = False

    if adaptive_order is not None:
        adaptive_valid = validate_plan({
            "task_order": adaptive_order,
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        }).get("valid", False)

    results47.append({
        "case_id": case["case_id"],
        "task_count": len(case["required_tasks"]),

        "stable_valid": stable_validation.get("valid", False),
        "stable_rate": stable_rate,
        "stable_seconds": stable_seconds,

        "exact_status": exact_result.get("status"),
        "exact_optimality_proven": exact_result.get("optimal", False),
        "exact_valid": exact_valid,
        "exact_rate": exact_rate,
        "exact_seconds": exact_seconds,

        "adaptive_success": adaptive_result.get("success", False),
        "adaptive_method": adaptive_result.get("method"),
        "adaptive_valid": adaptive_valid,
        "adaptive_rate": adaptive_result.get("order_change_rate"),
        "adaptive_seconds": adaptive_seconds,
        "adaptive_optimality_proven": adaptive_result.get(
            "optimality_proven", False
        )
    })

df47 = pd.DataFrame(results47)

output_dir47 = "/content/planproof_results/experiment47"
os.makedirs(output_dir47, exist_ok=True)

csv_path47 = os.path.join(
    output_dir47,
    "experiment47_consolidated_comparison.csv"
)
df47.to_csv(csv_path47, index=False)

print("EXPERIMENT 47 — CONSOLIDATED COMPARISON")
print("=" * 50)
print("Cases:", len(df47))
print("Stable valid:", int(df47["stable_valid"].sum()))
print("Exact valid:", int(df47["exact_valid"].sum()))
print("Adaptive successful:", int(df47["adaptive_success"].sum()))
print("Adaptive valid:", int(df47["adaptive_valid"].sum()))

print("\nMean order-change rates:")
print("Stable:", round(df47["stable_rate"].mean(), 6))
print("Exact (valid results only):",
      round(df47["exact_rate"].mean(), 6))
print("Adaptive:",
      round(df47["adaptive_rate"].mean(), 6))

print("\nMean runtime (seconds):")
print("Stable:", round(df47["stable_seconds"].mean(), 6))
print("Exact:", round(df47["exact_seconds"].mean(), 4))
print("Adaptive:", round(df47["adaptive_seconds"].mean(), 4))

print("\nAdaptive selected methods:")
print(df47["adaptive_method"].value_counts().to_dict())

print("\nExact optimality proven:",
      int(df47["exact_optimality_proven"].sum()),
      "/", len(df47))

print("\nSaved:", csv_path47)
print("\nFirst five rows:")
display(df47.head())


EXPERIMENT 47 — CONSOLIDATED COMPARISON
Cases: 15
Stable valid: 15
Exact valid: 15
Adaptive successful: 15
Adaptive valid: 15

Mean order-change rates:
Stable: 0.407151
Exact (valid results only): 0.360787
Adaptive: 0.359953

Mean runtime (seconds):
Stable: 0.000317
Exact: 3.1167
Adaptive: 3.121

Adaptive selected methods:
{'exact_optimizer': 13, 'stable_repair': 2}

Exact optimality proven: 7 / 15

Saved: /content/planproof_results/experiment47/experiment47_consolidated_comparison.csv

First five rows:


,case_id,task_count,stable_valid,stable_rate,stable_seconds,exact_status,exact_optimality_proven,exact_valid,exact_rate,exact_seconds,adaptive_success,adaptive_method,adaptive_valid,adaptive_rate,adaptive_seconds,adaptive_optimality_proven
0,E25_N10_R1,10,True,0.444444,0.000103,OPTIMAL,True,True,0.400000,0.043972,True,exact_optimizer,True,0.400000,0.077874,True
1,E25_N10_R2,10,True,0.422222,0.000064,OPTIMAL,True,True,0.422222,0.074879,True,stable_repair,True,0.422222,0.040939,False
2,E25_N10_R3,10,True,0.666667,0.000069,OPTIMAL,True,True,0.444444,0.051480,True,exact_optimizer,True,0.444444,0.054948,True
3,E25_N20_R1,20,True,0.221053,0.000187,OPTIMAL,True,True,0.163158,0.403247,True,exact_optimizer,True,0.163158,0.428636,True
4,E25_N20_R2,20,True,0.373684,0.000119,OPTIMAL,True,True,0.321053,0.440435,True,exact_optimizer,True,0.321053,0.724352,True


In [ ]:

# EXPERIMENT 48 — OPTIMALITY METADATA AUDIT

import pandas as pd

audit48 = []

for row in results47:
    method = row["adaptive_method"]
    selected_optimal = bool(row["adaptive_optimality_proven"])

    # In results47, the adaptive solver's raw optimality flag
    # is not stored separately. The corrected function returns it
    # as solver_optimality_proven, so use a fresh run to inspect it.
    case = next(
        c for c in cases35
        if c["case_id"] == row["case_id"]
    )

    adaptive = adaptive_repair_plan(
        case,
        time_budget_seconds=5
    )

    solver_optimal = bool(
        adaptive.get("solver_optimality_proven", False)
    )

    expected_selected_optimal = (
        adaptive.get("method") == "exact_optimizer"
        and solver_optimal
    )

    audit48.append({
        "case_id": row["case_id"],
        "selected_method": adaptive.get("method"),
        "solver_status": adaptive.get("solver_status"),
        "solver_optimality_proven": solver_optimal,
        "selected_plan_optimality_proven": selected_optimal,
        "expected_selected_optimality": expected_selected_optimal,
        "flag_consistent": (
            adaptive.get("optimality_proven", False)
            == expected_selected_optimal
        )
    })

df48 = pd.DataFrame(audit48)

print("EXPERIMENT 48 — OPTIMALITY METADATA AUDIT")
print("=" * 50)
print("Cases audited:", len(df48))
print("Consistent flags:", int(df48["flag_consistent"].sum()))
print("Inconsistent flags:", int((~df48["flag_consistent"]).sum()))

print("\nSolver status counts:")
print(df48["solver_status"].value_counts(dropna=False).to_dict())

print("\nSelected method counts:")
print(df48["selected_method"].value_counts().to_dict())

print("\nCases where solver proved optimal but stable repair was selected:")
print(df48[
    (df48["solver_optimality_proven"])
    & (df48["selected_method"] == "stable_repair")
].to_string(index=False))

output48 = "/content/planproof_results/experiment48_optimality_audit.csv"
import os
os.makedirs(os.path.dirname(output48), exist_ok=True)
df48.to_csv(output48, index=False)

print("\nSaved:", output48)


EXPERIMENT 48 — OPTIMALITY METADATA AUDIT
Cases audited: 15
Consistent flags: 15
Inconsistent flags: 0

Solver status counts:
{'FEASIBLE': 8, 'OPTIMAL': 7}

Selected method counts:
{'exact_optimizer': 12, 'stable_repair': 3}

Cases where solver proved optimal but stable repair was selected:
   case_id selected_method solver_status  solver_optimality_proven  selected_plan_optimality_proven  expected_selected_optimality  flag_consistent
E25_N10_R2   stable_repair       OPTIMAL                      True                            False                         False             True

Saved: /content/planproof_results/experiment48_optimality_audit.csv


In [ ]:

# EXPERIMENT 49 — FINAL ADAPTIVE POLICY AUDIT

final49 = []

for case in cases35:
    stable = repair_plan(case)
    stable_order = stable["repaired_plan"]["task_order"]
    stable_rate = shared_task_order_change_rate(
        case["task_order"], stable_order
    )

    adaptive = adaptive_repair_plan(
        case, time_budget_seconds=5
    )

    valid = False
    never_worse = False

    if adaptive["success"] and adaptive["order"] is not None:
        validation = validate_plan({
            "task_order": adaptive["order"],
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        })
        valid = validation.get("valid", False)

        never_worse = (
            adaptive["order_change_rate"] <= stable_rate + 1e-12
        )

    solver_optimal = adaptive.get(
        "solver_optimality_proven", False
    )
    expected_flag = (
        adaptive.get("method") == "exact_optimizer"
        and solver_optimal
    )
    flag_correct = (
        adaptive.get("optimality_proven", False) == expected_flag
    )

    final49.append({
        "case_id": case["case_id"],
        "success": adaptive["success"],
        "valid": valid,
        "never_worse_than_stable": never_worse,
        "optimality_flag_correct": flag_correct
    })

df49 = pd.DataFrame(final49)

print("EXPERIMENT 49 — FINAL AUDIT")
print("=" * 45)
print("Cases:", len(df49))
print("Successful:", int(df49["success"].sum()))
print("Valid:", int(df49["valid"].sum()))
print("Never worse than stable:",
      int(df49["never_worse_than_stable"].sum()))
print("Correct optimality flags:",
      int(df49["optimality_flag_correct"].sum()))

all_pass = df49[
    ["success", "valid", "never_worse_than_stable",
     "optimality_flag_correct"]
].all(axis=1)

print("Total audit failures:", int((~all_pass).sum()))

out49 = "/content/planproof_results/experiment49_final_audit.csv"
os.makedirs(os.path.dirname(out49), exist_ok=True)
df49.to_csv(out49, index=False)
print("Saved:", out49)


EXPERIMENT 49 — FINAL AUDIT
Cases: 15
Successful: 15
Valid: 15
Never worse than stable: 15
Correct optimality flags: 15
Total audit failures: 0
Saved: /content/planproof_results/experiment49_final_audit.csv


In [ ]:

# EXPERIMENT 50 — STEP 1: INSPECT AVAILABLE GENERATORS

import inspect

print("Available benchmark variables:")
for name in ["cases35", "cases21", "generate_synthetic_case",
             "generate_benchmark_case", "generate_dag"]:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}: {type(obj).__name__}")
        if callable(obj):
            try:
                print(inspect.signature(obj))
                print(inspect.getsource(obj)[:5000])
            except (OSError, TypeError) as exc:
                print("Source unavailable:", exc)

print("\nExisting case count:",
      len(cases35) if "cases35" in globals() else "cases35 not found")

if "cases35" in globals() and cases35:
    print("\nExample case keys:", list(cases35[0].keys()))
    print("Example case ID:", cases35[0].get("case_id"))
    print("Required task count:", len(cases35[0]["required_tasks"]))
    print("Original order length:", len(cases35[0]["task_order"]))
    print("Precedence constraints:",
          len(cases35[0]["precedence_constraints"]))


Available benchmark variables:

cases35: list

cases21: list

Existing case count: 15

Example case keys: ['case_id', 'task_count', 'repeat', 'required_tasks', 'task_order', 'precedence_constraints']
Example case ID: E25_N10_R1
Required task count: 10
Original order length: 10
Precedence constraints: 14


In [ ]:

# EXPERIMENT 50 — STEP 2: FIND THE BENCHMARK GENERATOR

ip = get_ipython()

keywords = [
    "experiment25_scalability_benchmark",
    "precedence_constraints",
    "random.seed",
    "np.random.seed",
    "E25_N10_R1"
]

history = ip.history_manager.input_hist_raw

matches = []

for cell_number, source in enumerate(history):
    if not source:
        continue

    score = sum(keyword in source for keyword in keywords)

    if score >= 2:
        matches.append((cell_number, score, source))

matches.sort(key=lambda item: (-item[1], -item[0]))

print("EXPERIMENT 50 — GENERATOR SEARCH")
print("=" * 45)
print("Matching notebook cells:", len(matches))

for cell_number, score, source in matches[:8]:
    print(f"\n--- Cell {cell_number} | Match score: {score} ---")
    print(source[:4000])


EXPERIMENT 50 — GENERATOR SEARCH
Matching notebook cells: 6

--- Cell 71 | Match score: 5 ---

# EXPERIMENT 50 — STEP 2: FIND THE BENCHMARK GENERATOR

ip = get_ipython()

keywords = [
    "experiment25_scalability_benchmark",
    "precedence_constraints",
    "random.seed",
    "np.random.seed",
    "E25_N10_R1"
]

history = ip.history_manager.input_hist_raw

matches = []

for cell_number, source in enumerate(history):
    if not source:
        continue

    score = sum(keyword in source for keyword in keywords)

    if score >= 2:
        matches.append((cell_number, score, source))

matches.sort(key=lambda item: (-item[1], -item[0]))

print("EXPERIMENT 50 — GENERATOR SEARCH")
print("=" * 45)
print("Matching notebook cells:", len(matches))

for cell_number, score, source in matches[:8]:
    print(f"\n--- Cell {cell_number} | Match score: {score} ---")
    print(source[:4000])

--- Cell 70 | Match score: 5 ---

# EXPERIMENT 50 — STEP 2: FIND THE BENCHMARK GENERATOR

ip = get_ipython()

In [ ]:

# EXPERIMENT 50 — INDEPENDENT SYNTHETIC BENCHMARK
# Step 3: Generate and audit 100 new cases

import random
import json
import os
from pathlib import Path
from collections import defaultdict, deque

SEED50 = 20261050
rng50 = random.Random(SEED50)

task_sizes50 = [10, 20, 30, 40, 50]
REPEATS50 = 20
cases50 = []

for n in task_sizes50:
    for repeat in range(1, REPEATS50 + 1):
        tasks = [f"T{i:03d}" for i in range(n)]

        # Generate a random topological order.
        topo = tasks[:]
        rng50.shuffle(topo)

        edges = set()

        # Ensure each task after the first has a predecessor.
        for j in range(1, n):
            predecessor = rng50.choice(topo[:j])
            edges.add((predecessor, topo[j]))

        # Add random forward edges while preserving acyclicity.
        for j in range(n):
            for k in range(j + 1, n):
                if rng50.random() < 0.06:
                    edges.add((topo[j], topo[k]))

        # Generate an observed order independent of the DAG order.
        original_order = tasks[:]
        rng50.shuffle(original_order)

        cases50.append({
            "case_id": f"E50_N{n}_R{repeat:02d}",
            "task_count": n,
            "repeat": repeat,
            "required_tasks": tasks,
            "task_order": original_order,
            "precedence_constraints": [
                list(edge) for edge in sorted(edges)
            ]
        })

# Audit fields, unique IDs, edge endpoints, and acyclicity.
audit50 = []

for case in cases50:
    tasks = case["required_tasks"]
    task_set = set(tasks)
    edges = case["precedence_constraints"]

    fields_ok = all(
        key in case for key in [
            "case_id", "task_count", "repeat",
            "required_tasks", "task_order",
            "precedence_constraints"
        ]
    )

    unique_tasks = len(tasks) == len(task_set)
    endpoints_ok = all(
        len(edge) == 2
        and edge[0] in task_set
        and edge[1] in task_set
        for edge in edges
    )

    adjacency = defaultdict(list)
    indegree = {task: 0 for task in tasks}

    for before, after in edges:
        adjacency[before].append(after)
        indegree[after] += 1

    queue = deque(
        task for task in tasks if indegree[task] == 0
    )
    visited = 0

    while queue:
        task = queue.popleft()
        visited += 1

        for neighbour in adjacency[task]:
            indegree[neighbour] -= 1
            if indegree[neighbour] == 0:
                queue.append(neighbour)

    acyclic = visited == len(tasks)

    audit50.append({
        "case_id": case["case_id"],
        "fields_ok": fields_ok,
        "unique_tasks": unique_tasks,
        "endpoints_ok": endpoints_ok,
        "acyclic": acyclic,
        "audit_pass": (
            fields_ok and unique_tasks
            and endpoints_ok and acyclic
        )
    })

# Save independently from all previous experiments.
out_dir50 = Path("/content/planproof_results/experiment50")
out_dir50.mkdir(parents=True, exist_ok=True)

benchmark_path50 = out_dir50 / "experiment50_benchmark.json"
audit_path50 = out_dir50 / "experiment50_graph_audit.json"

with open(benchmark_path50, "w", encoding="utf-8") as f:
    json.dump(cases50, f, indent=2)

with open(audit_path50, "w", encoding="utf-8") as f:
    json.dump(audit50, f, indent=2)

print("EXPERIMENT 50 — BENCHMARK GENERATION")
print("=" * 48)
print("Random seed:", SEED50)
print("Total cases:", len(cases50))
print("Task sizes:", task_sizes50)
print("Cases per size:", REPEATS50)

print("\nGraph audit:")
print("Fields present:", sum(x["fields_ok"] for x in audit50), "/ 100")
print("Unique task IDs:", sum(x["unique_tasks"] for x in audit50), "/ 100")
print("Valid edge endpoints:", sum(x["endpoints_ok"] for x in audit50), "/ 100")
print("Acyclic graphs:", sum(x["acyclic"] for x in audit50), "/ 100")
print("Audit passes:", sum(x["audit_pass"] for x in audit50), "/ 100")
print("Audit failures:", sum(not x["audit_pass"] for x in audit50))

print("\nSaved benchmark:", benchmark_path50)
print("Saved graph audit:", audit_path50)


EXPERIMENT 50 — BENCHMARK GENERATION
Random seed: 20261050
Total cases: 100
Task sizes: [10, 20, 30, 40, 50]
Cases per size: 20

Graph audit:
Fields present: 100 / 100
Unique task IDs: 100 / 100
Valid edge endpoints: 100 / 100
Acyclic graphs: 100 / 100
Audit passes: 100 / 100
Audit failures: 0

Saved benchmark: /content/planproof_results/experiment50/experiment50_benchmark.json
Saved graph audit: /content/planproof_results/experiment50/experiment50_graph_audit.json


In [ ]:

# EXPERIMENT 51 — STABLE REPAIR ON 100 INDEPENDENT CASES

import json
import os
import time
import pandas as pd
from pathlib import Path

# Reload the saved benchmark to ensure we use the actual saved data.
benchmark_path51 = Path(
    "/content/planproof_results/experiment50/experiment50_benchmark.json"
)

with open(benchmark_path51, "r", encoding="utf-8") as f:
    cases51 = json.load(f)

rows51 = []

for case in cases51:
    start = time.perf_counter()
    result = repair_plan({
        "task_order": case["task_order"],
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    })
    elapsed = time.perf_counter() - start

    success = result.get("success", False)
    order = (
        result["repaired_plan"]["task_order"]
        if success and isinstance(result.get("repaired_plan"), dict)
        else None
    )

    valid = False
    rate = None
    violations = None

    if order is not None:
        validation = validate_plan({
            "task_order": order,
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        })
        valid = validation.get("valid", False)
        violations = len(validation.get("violations", []))
        rate = shared_task_order_change_rate(
            case["task_order"], order
        )

    rows51.append({
        "case_id": case["case_id"],
        "task_count": case["task_count"],
        "success": success,
        "valid": valid,
        "violations": violations,
        "order_change_rate": rate,
        "runtime_seconds": elapsed
    })

df51 = pd.DataFrame(rows51)

out_dir51 = Path("/content/planproof_results/experiment51")
out_dir51.mkdir(parents=True, exist_ok=True)

csv_path51 = out_dir51 / "experiment51_stable_baseline.csv"
df51.to_csv(csv_path51, index=False)

print("EXPERIMENT 51 — STABLE REPAIR BASELINE")
print("=" * 48)
print("Cases:", len(df51))
print("Successful:", int(df51["success"].sum()))
print("Valid:", int(df51["valid"].sum()))
print("Cases with zero violations:",
      int((df51["violations"] == 0).sum()))
print("Mean order-change rate:",
      round(df51["order_change_rate"].mean(), 6))
print("Median runtime (seconds):",
      round(df51["runtime_seconds"].median(), 6))
print("Mean runtime (seconds):",
      round(df51["runtime_seconds"].mean(), 6))

print("\nResults by task size:")
display(
    df51.groupby("task_count").agg(
        cases=("case_id", "count"),
        valid=("valid", "sum"),
        mean_order_change=("order_change_rate", "mean"),
        median_runtime=("runtime_seconds", "median")
    ).reset_index()
)

print("\nSaved:", csv_path51)


EXPERIMENT 51 — STABLE REPAIR BASELINE
Cases: 100
Successful: 100
Valid: 100
Cases with zero violations: 100
Mean order-change rate: 0.376916
Median runtime (seconds): 0.000161
Mean runtime (seconds): 0.000171

Results by task size:


,task_count,cases,valid,mean_order_change,median_runtime
0,10,20,20,0.356667,0.000048
1,20,20,20,0.349211,0.000101
2,30,20,20,0.379310,0.000161
3,40,20,20,0.380128,0.000235
4,50,20,20,0.419265,0.000297



Saved: /content/planproof_results/experiment51/experiment51_stable_baseline.csv


In [ ]:

# EXPERIMENT 52 — FAIR EXACT OPTIMIZER ON 100 CASES

import os
import time
import pandas as pd
from pathlib import Path

rows52 = []

for case in cases51:
    start = time.perf_counter()

    result = exact_optimize_order_fair(
        case,
        time_limit_seconds=5
    )

    elapsed = time.perf_counter() - start
    order = result.get("order")

    valid = False
    rate = None
    violations = None

    if order is not None:
        validation = validate_plan({
            "task_order": order,
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        })

        valid = validation.get("valid", False)
        violations = len(validation.get("violations", []))

        if valid:
            rate = shared_task_order_change_rate(
                case["task_order"], order
            )

    rows52.append({
        "case_id": case["case_id"],
        "task_count": case["task_count"],
        "solver_status": result.get("status"),
        "optimality_proven": bool(result.get("optimal", False)),
        "valid": valid,
        "violations": violations,
        "order_change_rate": rate,
        "runtime_seconds": elapsed
    })

df52 = pd.DataFrame(rows52)

out_dir52 = Path("/content/planproof_results/experiment52")
out_dir52.mkdir(parents=True, exist_ok=True)

csv_path52 = out_dir52 / "experiment52_fair_exact_results.csv"
df52.to_csv(csv_path52, index=False)

print("EXPERIMENT 52 — FAIR EXACT OPTIMIZER")
print("=" * 48)
print("Cases:", len(df52))
print("Valid solutions:", int(df52["valid"].sum()))
print("Optimality proven:", int(df52["optimality_proven"].sum()))
print("Feasible status:", int((df52["solver_status"] == "FEASIBLE").sum()))
print("Optimal status:", int((df52["solver_status"] == "OPTIMAL").sum()))
print("Mean order-change rate (valid solutions):",
      round(df52["order_change_rate"].mean(), 6))
print("Median runtime (seconds):",
      round(df52["runtime_seconds"].median(), 4))
print("Mean runtime (seconds):",
      round(df52["runtime_seconds"].mean(), 4))

print("\nResults by task size:")
display(
    df52.groupby("task_count").agg(
        cases=("case_id", "count"),
        valid=("valid", "sum"),
        optimality_proven=("optimality_proven", "sum"),
        mean_order_change=("order_change_rate", "mean"),
        median_runtime=("runtime_seconds", "median")
    ).reset_index()
)

print("\nSaved:", csv_path52)


EXPERIMENT 52 — FAIR EXACT OPTIMIZER
Cases: 100
Valid solutions: 100
Optimality proven: 42
Feasible status: 58
Optimal status: 42
Mean order-change rate (valid solutions): 0.34448
Median runtime (seconds): 5.0203
Mean runtime (seconds): 3.1486

Results by task size:


,task_count,cases,valid,optimality_proven,mean_order_change,median_runtime
0,10,20,20,20,0.350000,0.056034
1,20,20,20,20,0.301316,0.488418
2,30,20,20,2,0.334138,5.020281
3,40,20,20,0,0.346090,5.024240
4,50,20,20,0,0.390857,5.036904



Saved: /content/planproof_results/experiment52/experiment52_fair_exact_results.csv


In [ ]:

# EXPERIMENT 53 — THREE-METHOD COMPARISON ON 100 CASES

import os
import pandas as pd
from pathlib import Path

# Load saved results from Experiments 51 and 52.
df51 = pd.read_csv(
    "/content/planproof_results/experiment51/"
    "experiment51_stable_baseline.csv"
)
df52 = pd.read_csv(
    "/content/planproof_results/experiment52/"
    "experiment52_fair_exact_results.csv"
)

rows53 = []

for case in cases51:
    adaptive = adaptive_repair_plan(
        case, time_budget_seconds=5
    )

    order = adaptive.get("order")
    valid = False

    if adaptive.get("success") and order is not None:
        valid = validate_plan({
            "task_order": order,
            "required_tasks": case["required_tasks"],
            "precedence_constraints": case["precedence_constraints"]
        }).get("valid", False)

    rows53.append({
        "case_id": case["case_id"],
        "adaptive_success": adaptive.get("success", False),
        "adaptive_valid": valid,
        "adaptive_method": adaptive.get("method"),
        "adaptive_rate": adaptive.get("order_change_rate"),
        "adaptive_runtime": adaptive.get("elapsed_seconds"),
        "adaptive_optimality_proven": adaptive.get(
            "optimality_proven", False
        ),
        "solver_status": adaptive.get("solver_status"),
        "solver_optimality_proven": adaptive.get(
            "solver_optimality_proven", False
        )
    })

df53 = pd.DataFrame(rows53)

# Join all methods by case ID.
comparison53 = (
    df51.merge(df52, on=["case_id", "task_count"], suffixes=("_stable", "_exact"))
        .merge(df53, on="case_id")
)

comparison53["adaptive_no_worse_than_stable"] = (
    comparison53["adaptive_rate"]
    <= comparison53["order_change_rate"] + 1e-12
)

comparison53["adaptive_vs_exact_rate_difference"] = (
    comparison53["adaptive_rate"]
    - comparison53["order_change_rate_exact"]
)

out_dir53 = Path("/content/planproof_results/experiment53")
out_dir53.mkdir(parents=True, exist_ok=True)

csv_path53 = out_dir53 / "experiment53_three_method_comparison.csv"
comparison53.to_csv(csv_path53, index=False)

print("EXPERIMENT 53 — THREE-METHOD COMPARISON")
print("=" * 50)
print("Cases:", len(comparison53))
print("Adaptive successful:", int(comparison53["adaptive_success"].sum()))
print("Adaptive valid:", int(comparison53["adaptive_valid"].sum()))
print("Adaptive no worse than stable:",
      int(comparison53["adaptive_no_worse_than_stable"].sum()))

print("\nMean order-change rates:")
print("Stable:", round(
    comparison53["order_change_rate_stable"].mean(), 6
))
print("Exact (valid solutions):", round(
    comparison53["order_change_rate_exact"].mean(), 6
))
print("Adaptive:", round(
    comparison53["adaptive_rate"].mean(), 6
))

print("\nAdaptive selected methods:")
print(comparison53["adaptive_method"].value_counts().to_dict())

print("\nAdaptive mean runtime (seconds):",
      round(comparison53["adaptive_runtime"].mean(), 4))

print("\nAdaptive vs separate exact run:")
print("Adaptive lower rate:",
      int((comparison53["adaptive_vs_exact_rate_difference"] < -1e-12).sum()))
print("Tied:",
      int((comparison53["adaptive_vs_exact_rate_difference"].abs() <= 1e-12).sum()))
print("Adaptive higher rate:",
      int((comparison53["adaptive_vs_exact_rate_difference"] > 1e-12).sum()))

print("\nSaved:", csv_path53)


KeyError: 'order_change_rate'

In [ ]:

# EXPERIMENT 53 — FIXED THREE-METHOD COMPARISON

import pandas as pd
from pathlib import Path

df51 = pd.read_csv(
    "/content/planproof_results/experiment51/"
    "experiment51_stable_baseline.csv"
)

df52 = pd.read_csv(
    "/content/planproof_results/experiment52/"
    "experiment52_fair_exact_results.csv"
)

# Rename metrics before merging to prevent ambiguous column names.
stable53 = df51.rename(columns={
    "order_change_rate": "stable_rate",
    "runtime_seconds": "stable_runtime",
    "valid": "stable_valid"
})

exact53 = df52.rename(columns={
    "order_change_rate": "exact_rate",
    "runtime_seconds": "exact_runtime",
    "valid": "exact_valid"
})

adaptive53 = pd.DataFrame(rows53)

comparison53 = (
    stable53.merge(exact53, on=["case_id", "task_count"])
    .merge(adaptive53, on="case_id")
)

comparison53["adaptive_no_worse_than_stable"] = (
    comparison53["adaptive_rate"]
    <= comparison53["stable_rate"] + 1e-12
)

comparison53["adaptive_vs_exact_rate_difference"] = (
    comparison53["adaptive_rate"] - comparison53["exact_rate"]
)

out_dir53 = Path("/content/planproof_results/experiment53")
out_dir53.mkdir(parents=True, exist_ok=True)

csv_path53 = out_dir53 / "experiment53_three_method_comparison.csv"
comparison53.to_csv(csv_path53, index=False)

print("EXPERIMENT 53 — THREE-METHOD COMPARISON")
print("=" * 50)
print("Cases:", len(comparison53))
print("Stable valid:", int(comparison53["stable_valid"].sum()))
print("Exact valid:", int(comparison53["exact_valid"].sum()))
print("Adaptive successful:", int(comparison53["adaptive_success"].sum()))
print("Adaptive valid:", int(comparison53["adaptive_valid"].sum()))
print("Adaptive no worse than stable:",
      int(comparison53["adaptive_no_worse_than_stable"].sum()))

print("\nMean order-change rates:")
print("Stable:", round(comparison53["stable_rate"].mean(), 6))
print("Exact (valid results):",
      round(comparison53["exact_rate"].mean(), 6))
print("Adaptive:", round(comparison53["adaptive_rate"].mean(), 6))

print("\nMean runtime (seconds):")
print("Stable:", round(comparison53["stable_runtime"].mean(), 6))
print("Exact:", round(comparison53["exact_runtime"].mean(), 4))
print("Adaptive:", round(comparison53["adaptive_runtime"].mean(), 4))

print("\nAdaptive selected methods:")
print(comparison53["adaptive_method"].value_counts().to_dict())

print("\nAdaptive vs separate exact run:")
print("Adaptive lower rate:", int(
    (comparison53["adaptive_vs_exact_rate_difference"] < -1e-12).sum()
))
print("Tied:", int(
    comparison53["adaptive_vs_exact_rate_difference"].abs().le(1e-12).sum()
))
print("Adaptive higher rate:", int(
    (comparison53["adaptive_vs_exact_rate_difference"] > 1e-12).sum()
))

print("\nSaved:", csv_path53)


EXPERIMENT 53 — THREE-METHOD COMPARISON
Cases: 100
Stable valid: 100
Exact valid: 100
Adaptive successful: 100
Adaptive valid: 100
Adaptive no worse than stable: 100

Mean order-change rates:
Stable: 0.376916
Exact (valid results): 0.34448
Adaptive: 0.34462

Mean runtime (seconds):
Stable: 0.000171
Exact: 3.1486
Adaptive: 3.1777

Adaptive selected methods:
{'exact_optimizer': 80, 'stable_repair': 20}

Adaptive vs separate exact run:
Adaptive lower rate: 3
Tied: 92
Adaptive higher rate: 5

Saved: /content/planproof_results/experiment53/experiment53_three_method_comparison.csv


In [ ]:

# EXPERIMENT 54 — STATISTICAL ANALYSIS

import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

df54 = comparison53.copy()

# Pairwise improvements: positive means adaptive is better.
df54["improvement"] = (
    df54["stable_rate"] - df54["adaptive_rate"]
)

improvements = df54["improvement"].dropna().to_numpy()
n54 = len(improvements)

mean_improvement = float(np.mean(improvements))
median_improvement = float(np.median(improvements))
wins54 = int(np.sum(improvements > 1e-12))
ties54 = int(np.sum(np.abs(improvements) <= 1e-12))
losses54 = int(np.sum(improvements < -1e-12))

if n54 > 1:
    sem54 = stats.sem(improvements)
    critical54 = stats.t.ppf(0.975, df=n54 - 1)
    ci_low54 = mean_improvement - critical54 * sem54
    ci_high54 = mean_improvement + critical54 * sem54

    if np.any(np.abs(improvements) > 1e-12):
        wilcoxon54 = stats.wilcoxon(
            improvements,
            alternative="two-sided",
            zero_method="wilcox"
        )
        wilcoxon_stat = float(wilcoxon54.statistic)
        wilcoxon_p = float(wilcoxon54.pvalue)
    else:
        wilcoxon_stat = 0.0
        wilcoxon_p = 1.0
else:
    ci_low54 = ci_high54 = np.nan
    wilcoxon_stat = wilcoxon_p = np.nan

print("EXPERIMENT 54 — PAIRED STATISTICAL ANALYSIS")
print("=" * 50)
print("Paired cases:", n54)
print("Mean absolute improvement:", round(mean_improvement, 6))
print("Median absolute improvement:", round(median_improvement, 6))
print("95% paired t confidence interval:",
      (round(ci_low54, 6), round(ci_high54, 6)))
print("Adaptive wins / ties / losses:",
      wins54, "/", ties54, "/", losses54)

baseline_mean = df54["stable_rate"].mean()
adaptive_mean = df54["adaptive_rate"].mean()

print("Relative mean reduction (%):",
      round(100 * (baseline_mean - adaptive_mean) / baseline_mean, 3))
print("Wilcoxon statistic:", round(wilcoxon_stat, 4))
print("Wilcoxon p-value:", f"{wilcoxon_p:.6g}")

print("\nBy task size:")
display(
    df54.groupby("task_count").agg(
        cases=("case_id", "count"),
        stable_mean=("stable_rate", "mean"),
        adaptive_mean=("adaptive_rate", "mean"),
        mean_improvement=("improvement", "mean"),
        wins=("improvement", lambda x: int((x > 1e-12).sum())),
        ties=("improvement", lambda x: int((x.abs() <= 1e-12).sum())),
        losses=("improvement", lambda x: int((x < -1e-12).sum()))
    ).reset_index().round(6)
)

summary54 = pd.DataFrame([{
    "n": n54,
    "mean_improvement": mean_improvement,
    "median_improvement": median_improvement,
    "ci95_low": ci_low54,
    "ci95_high": ci_high54,
    "wins": wins54,
    "ties": ties54,
    "losses": losses54,
    "wilcoxon_statistic": wilcoxon_stat,
    "wilcoxon_p_value": wilcoxon_p
}])

out54 = Path("/content/planproof_results/experiment54")
out54.mkdir(parents=True, exist_ok=True)
summary54.to_csv(out54 / "experiment54_summary.csv", index=False)
df54.to_csv(out54 / "experiment54_case_level.csv", index=False)

print("\nSaved:", out54 / "experiment54_summary.csv")
print("Saved:", out54 / "experiment54_case_level.csv")


EXPERIMENT 54 — PAIRED STATISTICAL ANALYSIS
Paired cases: 100
Mean absolute improvement: 0.032296
Median absolute improvement: 0.026219
95% paired t confidence interval: (np.float64(0.026633), np.float64(0.03796))
Adaptive wins / ties / losses: 80 / 20 / 0
Relative mean reduction (%): 8.569
Wilcoxon statistic: 45.0
Wilcoxon p-value: 1.15528e-15

By task size:


,task_count,cases,stable_mean,adaptive_mean,mean_improvement,wins,ties,losses
0,10,20,0.356667,0.350000,0.006667,3,17,0
1,20,20,0.349211,0.301316,0.047895,20,0,0
2,30,20,0.379310,0.334943,0.044368,19,1,0
3,40,20,0.380128,0.346026,0.034103,19,1,0
4,50,20,0.419265,0.390816,0.028449,19,1,0



Saved: /content/planproof_results/experiment54/experiment54_summary.csv
Saved: /content/planproof_results/experiment54/experiment54_case_level.csv


In [ ]:

# EXPERIMENT 55 — FINAL BENCHMARK AUDIT AND SUMMARY

import json
import pandas as pd
from pathlib import Path

base55 = Path("/content/planproof_results")

benchmark55 = base55 / "experiment50" / "experiment50_benchmark.json"
stable_path55 = base55 / "experiment51" / "experiment51_stable_baseline.csv"
exact_path55 = base55 / "experiment52" / "experiment52_fair_exact_results.csv"
compare_path55 = base55 / "experiment53" / "experiment53_three_method_comparison.csv"
stats_path55 = base55 / "experiment54" / "experiment54_summary.csv"
case_stats_path55 = base55 / "experiment54" / "experiment54_case_level.csv"

required_files55 = [
    benchmark55, stable_path55, exact_path55,
    compare_path55, stats_path55, case_stats_path55
]

print("EXPERIMENT 55 — FINAL AUDIT")
print("=" * 48)

missing_files55 = [str(p) for p in required_files55 if not p.exists()]
print("Required files present:",
      len(required_files55) - len(missing_files55),
      "/", len(required_files55))

if missing_files55:
    print("Missing files:")
    for path in missing_files55:
        print(path)
else:
    with open(benchmark55, "r", encoding="utf-8") as f:
        cases = json.load(f)

    stable = pd.read_csv(stable_path55)
    exact = pd.read_csv(exact_path55)
    comparison = pd.read_csv(compare_path55)
    summary = pd.read_csv(stats_path55)
    case_stats = pd.read_csv(case_stats_path55)

    print("Benchmark cases:", len(cases))
    print("Unique benchmark IDs:",
          len({c["case_id"] for c in cases}))
    print("Stable rows:", len(stable))
    print("Exact rows:", len(exact))
    print("Comparison rows:", len(comparison))
    print("Statistical case rows:", len(case_stats))

    print("\nDuplicate IDs:")
    print("Benchmark:", len(cases) - len({c["case_id"] for c in cases}))
    print("Stable:", int(stable["case_id"].duplicated().sum()))
    print("Exact:", int(exact["case_id"].duplicated().sum()))
    print("Comparison:", int(comparison["case_id"].duplicated().sum()))

    print("\nMissing metric values:")
    print("Stable order-change rates:", int(stable["order_change_rate"].isna().sum()))
    print("Exact order-change rates:", int(exact["order_change_rate"].isna().sum()))
    print("Adaptive order-change rates:", int(comparison["adaptive_rate"].isna().sum()))

    print("\nValidity:")
    print("Stable valid:", int(stable["valid"].sum()), "/", len(stable))
    print("Exact valid:", int(exact["valid"].sum()), "/", len(exact))
    print("Adaptive valid:", int(comparison["adaptive_valid"].sum()), "/", len(comparison))

    print("\nStatistical summary:")
    display(summary)

    # Save a compact final summary without modifying raw results.
    final_summary55 = pd.DataFrame([{
        "benchmark": "Experiment 50 independently seeded synthetic cases",
        "case_count": len(cases),
        "stable_mean_order_change": stable["order_change_rate"].mean(),
        "exact_mean_order_change_valid": exact.loc[
            exact["valid"], "order_change_rate"
        ].mean(),
        "adaptive_mean_order_change": comparison["adaptive_rate"].mean(),
        "stable_valid_count": int(stable["valid"].sum()),
        "exact_valid_count": int(exact["valid"].sum()),
        "adaptive_valid_count": int(comparison["adaptive_valid"].sum()),
        "adaptive_never_worse_count": int(
            comparison["adaptive_no_worse_than_stable"].sum()
        ),
        "adaptive_mean_runtime_seconds": comparison["adaptive_runtime"].mean(),
        "interpretation_scope": "Synthetic benchmark only"
    }])

    final_path55 = base55 / "experiment55_final_summary.csv"
    final_summary55.to_csv(final_path55, index=False)

    print("\nSaved final summary:", final_path55)


EXPERIMENT 55 — FINAL AUDIT
Required files present: 6 / 6
Benchmark cases: 100
Unique benchmark IDs: 100
Stable rows: 100
Exact rows: 100
Comparison rows: 100
Statistical case rows: 100

Duplicate IDs:
Benchmark: 0
Stable: 0
Exact: 0
Comparison: 0

Missing metric values:
Stable order-change rates: 0
Exact order-change rates: 0
Adaptive order-change rates: 0

Validity:
Stable valid: 100 / 100
Exact valid: 100 / 100
Adaptive valid: 100 / 100

Statistical summary:


,n,mean_improvement,median_improvement,ci95_low,ci95_high,wins,ties,losses,wilcoxon_statistic,wilcoxon_p_value
0,100,0.032296,0.026219,0.026633,0.03796,80,20,0,45.0,1.155276e-15



Saved final summary: /content/planproof_results/experiment55_final_summary.csv


In [ ]:

# EXPERIMENT 56 — STEP 1
# Inspect the existing benchmark schema

import json
from pathlib import Path

benchmark_path = Path(
    "/content/planproof_results/experiment50/experiment50_benchmark.json"
)

if not benchmark_path.exists():
    print("ERROR: Experiment 50 benchmark file not found.")
else:
    with open(benchmark_path, "r", encoding="utf-8") as f:
        existing_cases = json.load(f)

    print("EXPERIMENT 56 — BENCHMARK SCHEMA")
    print("=" * 45)
    print("Total existing cases:", len(existing_cases))

    if existing_cases:
        first_case = existing_cases[0]

        print("\nFirst case keys:")
        print(list(first_case.keys()))

        print("\nFirst case example:")
        print(json.dumps(first_case, indent=2)[:5000])

        print("\nTask sizes:")
        sizes = {}
        for case in existing_cases:
            size = len(case.get("required_tasks", []))
            sizes[size] = sizes.get(size, 0) + 1

        print(dict(sorted(sizes.items())))


EXPERIMENT 56 — BENCHMARK SCHEMA
Total existing cases: 100

First case keys:
['case_id', 'task_count', 'repeat', 'required_tasks', 'task_order', 'precedence_constraints']

First case example:
{
  "case_id": "E50_N10_R01",
  "task_count": 10,
  "repeat": 1,
  "required_tasks": [
    "T000",
    "T001",
    "T002",
    "T003",
    "T004",
    "T005",
    "T006",
    "T007",
    "T008",
    "T009"
  ],
  "task_order": [
    "T000",
    "T003",
    "T008",
    "T005",
    "T007",
    "T006",
    "T002",
    "T009",
    "T001",
    "T004"
  ],
  "precedence_constraints": [
    [
      "T000",
      "T001"
    ],
    [
      "T000",
      "T003"
    ],
    [
      "T002",
      "T005"
    ],
    [
      "T004",
      "T000"
    ],
    [
      "T004",
      "T009"
    ],
    [
      "T005",
      "T004"
    ],
    [
      "T006",
      "T007"
    ],
    [
      "T007",
      "T008"
    ],
    [
      "T009",
      "T006"
    ]
  ]
}

Task sizes:
{10: 20, 20: 20, 30: 20, 40: 20, 50: 20}


In [ ]:

# EXPERIMENT 56 — STEP 2
# Generate an independent synthetic benchmark

import json
import random
from pathlib import Path
from collections import defaultdict, deque

SEED56 = 20261056
rng56 = random.Random(SEED56)

output_dir56 = Path("/content/planproof_results/experiment56")
output_dir56.mkdir(parents=True, exist_ok=True)

def is_dag56(tasks, edges):
    indegree = {task: 0 for task in tasks}
    children = defaultdict(list)

    for u, v in edges:
        children[u].append(v)
        indegree[v] += 1

    queue = deque(
        task for task in tasks if indegree[task] == 0
    )
    visited = 0

    while queue:
        u = queue.popleft()
        visited += 1
        for v in children[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)

    return visited == len(tasks)

def generate_case56(task_count, repeat):
    tasks = [f"T{i:03d}" for i in range(task_count)]

    # Sample candidate edges, then retain the graph only if acyclic.
    # This differs from generating edges only in a known topological order.
    max_attempts = 500
    edges = []

    for attempt in range(max_attempts):
        candidate_edges = []

        for u in tasks:
            for v in tasks:
                if u != v and rng56.random() < 0.045:
                    candidate_edges.append((u, v))

        if is_dag56(tasks, candidate_edges):
            edges = candidate_edges
            break
    else:
        # Safe fallback: an acyclic chain.
        edges = [
            (tasks[i], tasks[i + 1])
            for i in range(task_count - 1)
        ]

    # Keep the benchmark reasonably connected by adding edges
    # only when they do not create a cycle.
    shuffled_pairs = [
        (u, v) for u in tasks for v in tasks if u != v
    ]
    rng56.shuffle(shuffled_pairs)

    for u, v in shuffled_pairs:
        if (u, v) in edges:
            continue
        if rng56.random() < 0.015:
            trial_edges = edges + [(u, v)]
            if is_dag56(tasks, trial_edges):
                edges = trial_edges

    # Observed order is an arbitrary permutation and may violate constraints.
    observed_order = tasks.copy()
    rng56.shuffle(observed_order)

    return {
        "case_id": f"E56_N{task_count}_R{repeat:02d}",
        "task_count": task_count,
        "repeat": repeat,
        "required_tasks": tasks,
        "task_order": observed_order,
        "precedence_constraints": [list(edge) for edge in edges]
    }

cases56 = []

for task_count in [10, 20, 30, 40, 50]:
    for repeat in range(1, 21):
        cases56.append(
            generate_case56(task_count, repeat)
        )

# Audit IDs, task sets, endpoints, and graph acyclicity.
audit_errors56 = []

for case in cases56:
    tasks = case["required_tasks"]
    edges = [tuple(edge) for edge in case["precedence_constraints"]]

    if len(tasks) != len(set(tasks)):
        audit_errors56.append((case["case_id"], "duplicate task IDs"))

    if len(case["task_order"]) != len(tasks):
        audit_errors56.append((case["case_id"], "wrong order length"))

    if set(case["task_order"]) != set(tasks):
        audit_errors56.append((case["case_id"], "task order mismatch"))

    if any(u not in tasks or v not in tasks for u, v in edges):
        audit_errors56.append((case["case_id"], "invalid edge endpoint"))

    if len(edges) != len(set(edges)):
        audit_errors56.append((case["case_id"], "duplicate edges"))

    if not is_dag56(tasks, edges):
        audit_errors56.append((case["case_id"], "cycle detected"))

benchmark_path56 = output_dir56 / "experiment56_benchmark.json"
audit_path56 = output_dir56 / "experiment56_graph_audit.json"

with open(benchmark_path56, "w", encoding="utf-8") as f:
    json.dump(cases56, f, indent=2)

audit56 = {
    "seed": SEED56,
    "case_count": len(cases56),
    "unique_case_ids": len({c["case_id"] for c in cases56}),
    "audit_error_count": len(audit_errors56),
    "audit_errors": audit_errors56,
    "task_sizes": {
        str(n): sum(c["task_count"] == n for c in cases56)
        for n in [10, 20, 30, 40, 50]
    }
}

with open(audit_path56, "w", encoding="utf-8") as f:
    json.dump(audit56, f, indent=2)

print("EXPERIMENT 56 — INDEPENDENT BENCHMARK")
print("=" * 48)
print("Seed:", SEED56)
print("Cases generated:", len(cases56))
print("Unique case IDs:", audit56["unique_case_ids"])
print("Audit errors:", audit56["audit_error_count"])
print("Task-size distribution:", audit56["task_sizes"])
print("Benchmark saved:", benchmark_path56)
print("Audit saved:", audit_path56)

if audit_errors56:
    print("\nFirst audit errors:")
    print(audit_errors56[:10])
else:
    print("\nRESULT: All structural audits passed.")


EXPERIMENT 56 — INDEPENDENT BENCHMARK
Seed: 20261056
Cases generated: 100
Unique case IDs: 100
Audit errors: 0
Task-size distribution: {'10': 20, '20': 20, '30': 20, '40': 20, '50': 20}
Benchmark saved: /content/planproof_results/experiment56/experiment56_benchmark.json
Audit saved: /content/planproof_results/experiment56/experiment56_graph_audit.json

RESULT: All structural audits passed.


In [ ]:

# EXPERIMENT 56 — STEP 3
# Evaluate stable repair on the new benchmark

import json
import time
import pandas as pd
from pathlib import Path

benchmark_path = Path(
    "/content/planproof_results/experiment56/experiment56_benchmark.json"
)
output_dir = Path("/content/planproof_results/experiment57")
output_dir.mkdir(parents=True, exist_ok=True)

with open(benchmark_path, "r", encoding="utf-8") as f:
    cases57 = json.load(f)

results57 = []

for case in cases57:
    original_order = case["task_order"]

    plan = {
        "task_order": original_order,
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    }

    start = time.perf_counter()
    repair_result = repair_plan(plan)
    elapsed = time.perf_counter() - start

    if not repair_result.get("success", False):
        results57.append({
            "case_id": case["case_id"],
            "task_count": case["task_count"],
            "success": False,
            "valid": False,
            "violations": None,
            "order_change_rate": None,
            "runtime_seconds": elapsed
        })
        continue

    repaired = repair_result["repaired_plan"]
    repaired_order = repaired["task_order"]

    validation = validate_plan({
        "task_order": repaired_order,
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"]
    })

    order_rate = shared_task_order_change_rate(
        original_order, repaired_order
    )

    results57.append({
        "case_id": case["case_id"],
        "task_count": case["task_count"],
        "success": True,
        "valid": bool(validation["valid"]),
        "violations": len(validation.get("violations", [])),
        "order_change_rate": order_rate,
        "runtime_seconds": elapsed
    })

df57 = pd.DataFrame(results57)
csv_path57 = output_dir / "experiment57_stable_baseline.csv"
df57.to_csv(csv_path57, index=False)

print("EXPERIMENT 57 — STABLE REPAIR BASELINE")
print("=" * 48)
print("Cases:", len(df57))
print("Repair success:", int(df57["success"].sum()), "/", len(df57))
print("Valid after repair:", int(df57["valid"].sum()), "/", len(df57))
print("Cases with violations:", int((df57["violations"].fillna(0) > 0).sum()))
print("Missing order-change metrics:", int(df57["order_change_rate"].isna().sum()))

print("\nMean order-change rate:",
      round(df57["order_change_rate"].mean(), 6))
print("Median order-change rate:",
      round(df57["order_change_rate"].median(), 6))
print("Mean runtime (seconds):",
      round(df57["runtime_seconds"].mean(), 6))

print("\nResults by task size:")
display(
    df57.groupby("task_count").agg(
        cases=("case_id", "count"),
        valid=("valid", "sum"),
        mean_order_change=("order_change_rate", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean")
    ).round(6)
)

print("\nSaved:", csv_path57)


EXPERIMENT 57 — STABLE REPAIR BASELINE
Cases: 100
Repair success: 100 / 100
Valid after repair: 100 / 100
Cases with violations: 0
Missing order-change metrics: 0

Mean order-change rate: 0.365861
Median order-change rate: 0.379274
Mean runtime (seconds): 0.00026

Results by task size:


,cases,valid,mean_order_change,mean_runtime_seconds
task_count,,,,
10,20,20,0.198889,0.000075
20,20,20,0.317105,0.000091
30,20,20,0.384828,0.000443
40,20,20,0.425256,0.000408
50,20,20,0.503224,0.000284



Saved: /content/planproof_results/experiment57/experiment57_stable_baseline.csv


In [ ]:

# EXPERIMENT 58 — FAIR EXACT OPTIMIZER

import json
import time
import pandas as pd
from pathlib import Path

benchmark_path58 = Path(
    "/content/planproof_results/experiment56/experiment56_benchmark.json"
)
output_dir58 = Path("/content/planproof_results/experiment58")
output_dir58.mkdir(parents=True, exist_ok=True)

with open(benchmark_path58, "r", encoding="utf-8") as f:
    cases58 = json.load(f)

results58 = []

for i, case in enumerate(cases58, start=1):
    start = time.perf_counter()

    try:
        result = exact_optimize_order_fair(
            case,
            time_limit_seconds=5
        )
        elapsed = time.perf_counter() - start

        order = result.get("order")
        status = result.get("status", "UNKNOWN")
        optimal = bool(result.get("optimal", False))

        valid = False
        order_rate = None
        violations_count = None

        if order is not None:
            validation = validate_plan({
                "task_order": order,
                "required_tasks": case["required_tasks"],
                "precedence_constraints": case["precedence_constraints"]
            })
            valid = bool(validation["valid"])
            violations_count = len(validation.get("violations", []))

            if valid:
                order_rate = shared_task_order_change_rate(
                    case["task_order"], order
                )

        results58.append({
            "case_id": case["case_id"],
            "task_count": case["task_count"],
            "solver_status": status,
            "optimality_proven": optimal,
            "valid": valid,
            "violations": violations_count,
            "order_change_rate": order_rate,
            "runtime_seconds": elapsed
        })

    except Exception as exc:
        results58.append({
            "case_id": case["case_id"],
            "task_count": case["task_count"],
            "solver_status": "ERROR",
            "optimality_proven": False,
            "valid": False,
            "violations": None,
            "order_change_rate": None,
            "runtime_seconds": time.perf_counter() - start,
            "error": str(exc)
        })

    if i % 10 == 0:
        print(f"Processed {i}/{len(cases58)} cases")

df58 = pd.DataFrame(results58)
csv_path58 = output_dir58 / "experiment58_fair_exact_results.csv"
df58.to_csv(csv_path58, index=False)

print("\nEXPERIMENT 58 — SUMMARY")
print("=" * 45)
print("Cases:", len(df58))
print("Valid solutions:", int(df58["valid"].sum()), "/", len(df58))
print("Optimality proven:", int(df58["optimality_proven"].sum()), "/", len(df58))
print("OPTIMAL status:", int((df58["solver_status"] == "OPTIMAL").sum()))
print("FEASIBLE status:", int((df58["solver_status"] == "FEASIBLE").sum()))
print("Missing order-change rates:", int(df58["order_change_rate"].isna().sum()))
print("Mean order-change rate (valid solutions):",
      round(df58.loc[df58["valid"], "order_change_rate"].mean(), 6))
print("Mean runtime (seconds):",
      round(df58["runtime_seconds"].mean(), 6))

print("\nResults by task size:")
display(
    df58.groupby("task_count").agg(
        cases=("case_id", "count"),
        valid=("valid", "sum"),
        optimality_proven=("optimality_proven", "sum"),
        mean_order_change=("order_change_rate", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean")
    ).round(6)
)

print("\nSaved:", csv_path58)


Processed 10/100 cases
Processed 20/100 cases
Processed 30/100 cases
Processed 40/100 cases
Processed 50/100 cases
Processed 60/100 cases
Processed 70/100 cases
Processed 80/100 cases
Processed 90/100 cases
Processed 100/100 cases

EXPERIMENT 58 — SUMMARY
Cases: 100
Valid solutions: 100 / 100
Optimality proven: 71 / 100
OPTIMAL status: 71
FEASIBLE status: 29
Missing order-change rates: 0
Mean order-change rate (valid solutions): 0.336665
Mean runtime (seconds): 2.106789

Results by task size:


,cases,valid,optimality_proven,mean_order_change,mean_runtime_seconds
task_count,,,,,
10,20,20,20,0.178889,0.050397
20,20,20,19,0.266316,1.845406
30,20,20,9,0.344253,4.018529
40,20,20,3,0.390641,4.472882
50,20,20,20,0.503224,0.146733



Saved: /content/planproof_results/experiment58/experiment58_fair_exact_results.csv


In [ ]:

# EXPERIMENT 59 — PAIRED RESULT CONSISTENCY AUDIT

import pandas as pd
from pathlib import Path

base59 = Path("/content/planproof_results")

stable_path59 = base59 / "experiment57/experiment57_stable_baseline.csv"
exact_path59 = base59 / "experiment58/experiment58_fair_exact_results.csv"

stable59 = pd.read_csv(stable_path59)
exact59 = pd.read_csv(exact_path59)

merged59 = stable59.merge(
    exact59,
    on=["case_id", "task_count"],
    how="outer",
    suffixes=("_stable", "_exact"),
    indicator=True
)

print("EXPERIMENT 59 — PAIRED AUDIT")
print("=" * 45)
print("Stable rows:", len(stable59))
print("Exact rows:", len(exact59))
print("Matched cases:", int((merged59["_merge"] == "both").sum()))
print("Unmatched cases:", int((merged59["_merge"] != "both").sum()))

paired59 = merged59[
    (merged59["_merge"] == "both")
    & merged59["valid_stable"].astype(bool)
    & merged59["valid_exact"].astype(bool)
].copy()

paired59["improvement"] = (
    paired59["order_change_rate_stable"]
    - paired59["order_change_rate_exact"]
)

print("\nPaired valid cases:", len(paired59))
print("Mean stable rate:",
      round(paired59["order_change_rate_stable"].mean(), 6))
print("Mean exact rate:",
      round(paired59["order_change_rate_exact"].mean(), 6))
print("Mean improvement:",
      round(paired59["improvement"].mean(), 6))
print("Wins:", int((paired59["improvement"] > 1e-12).sum()))
print("Ties:", int((paired59["improvement"].abs() <= 1e-12).sum()))
print("Losses:", int((paired59["improvement"] < -1e-12).sum()))

print("\nBy task size:")
display(
    paired59.groupby("task_count").agg(
        cases=("case_id", "count"),
        stable_mean=("order_change_rate_stable", "mean"),
        exact_mean=("order_change_rate_exact", "mean"),
        mean_improvement=("improvement", "mean"),
        optimal_cases=("optimality_proven_exact", "sum")
    ).round(6)
)

print("\n50-task exact-solver statuses:")
print(
    exact59.loc[
        exact59["task_count"] == 50, "solver_status"
    ].value_counts(dropna=False).to_dict()
)

print("\n50-task case-level comparison:")
display(
    paired59.loc[
        paired59["task_count"] == 50,
        [
            "case_id",
            "order_change_rate_stable",
            "order_change_rate_exact",
            "solver_status",
            "optimality_proven_exact",
            "improvement"
        ]
    ].sort_values("case_id")
)


EXPERIMENT 59 — PAIRED AUDIT
Stable rows: 100
Exact rows: 100
Matched cases: 100
Unmatched cases: 0

Paired valid cases: 100
Mean stable rate: 0.365861
Mean exact rate: 0.336665
Mean improvement: 0.029196
Wins: 67
Ties: 33
Losses: 0

By task size:


KeyError: "Column(s) ['optimality_proven_exact'] do not exist"

In [ ]:

# EXPERIMENT 59 — FIXED PAIRED AUDIT
# Reuse existing results; do not rerun the optimizer.

print("Available merged columns:")
print(paired59.columns.tolist())

# Find the actual optimality column after the merge.
optimality_col59 = next(
    (
        col for col in paired59.columns
        if col.startswith("optimality_proven")
        and col.endswith("_exact")
    ),
    None
)

agg59 = {
    "cases": ("case_id", "count"),
    "stable_mean": ("order_change_rate_stable", "mean"),
    "exact_mean": ("order_change_rate_exact", "mean"),
    "mean_improvement": ("improvement", "mean")
}

if optimality_col59 is not None:
    agg59["optimal_cases"] = (optimality_col59, "sum")
else:
    print("No merged optimality column found; checking exact results directly.")
    exact_optimality59 = exact59[
        ["case_id", "optimality_proven"]
    ].drop_duplicates("case_id")

    paired59 = paired59.merge(
        exact_optimality59,
        on="case_id",
        how="left"
    )
    agg59["optimal_cases"] = ("optimality_proven", "sum")

print("\nResults by task size:")
display(
    paired59.groupby("task_count").agg(**agg59).round(6)
)

print("\n50-task exact-solver statuses:")
print(
    exact59.loc[
        exact59["task_count"] == 50, "solver_status"
    ].value_counts(dropna=False).to_dict()
)

print("\n50-task case-level comparison:")
display(
    paired59.loc[
        paired59["task_count"] == 50,
        [
            "case_id",
            "order_change_rate_stable",
            "order_change_rate_exact",
            "improvement"
        ]
    ].sort_values("case_id")
)


Available merged columns:
['case_id', 'task_count', 'success', 'valid_stable', 'violations_stable', 'order_change_rate_stable', 'runtime_seconds_stable', 'solver_status', 'optimality_proven', 'valid_exact', 'violations_exact', 'order_change_rate_exact', 'runtime_seconds_exact', '_merge', 'improvement']
No merged optimality column found; checking exact results directly.

Results by task size:


KeyError: "Column(s) ['optimality_proven'] do not exist"

In [ ]:

# EXPERIMENT 59 — FINAL FIX
# Uses the existing merged results; no optimizer rerun.

import pandas as pd

# Rebuild the paired table from the original result files.
stable_clean59 = stable59[
    [
        "case_id", "task_count", "valid",
        "order_change_rate", "runtime_seconds"
    ]
].rename(columns={
    "valid": "stable_valid",
    "order_change_rate": "stable_rate",
    "runtime_seconds": "stable_runtime"
})

exact_clean59 = exact59[
    [
        "case_id", "valid", "order_change_rate",
        "runtime_seconds", "solver_status", "optimality_proven"
    ]
].rename(columns={
    "valid": "exact_valid",
    "order_change_rate": "exact_rate",
    "runtime_seconds": "exact_runtime"
})

paired_fixed59 = stable_clean59.merge(
    exact_clean59,
    on="case_id",
    how="inner",
    validate="one_to_one"
)

paired_fixed59["improvement"] = (
    paired_fixed59["stable_rate"] - paired_fixed59["exact_rate"]
)

print("EXPERIMENT 59 — CORRECTED AUDIT")
print("=" * 45)
print("Matched cases:", len(paired_fixed59))
print("Stable valid:", int(paired_fixed59["stable_valid"].sum()))
print("Exact valid:", int(paired_fixed59["exact_valid"].sum()))
print("Mean improvement:",
      round(paired_fixed59["improvement"].mean(), 6))
print("Wins:",
      int((paired_fixed59["improvement"] > 1e-12).sum()))
print("Ties:",
      int((paired_fixed59["improvement"].abs() <= 1e-12).sum()))
print("Losses:",
      int((paired_fixed59["improvement"] < -1e-12).sum()))

print("\nResults by task size:")
display(
    paired_fixed59.groupby("task_count").agg(
        cases=("case_id", "count"),
        stable_mean=("stable_rate", "mean"),
        exact_mean=("exact_rate", "mean"),
        mean_improvement=("improvement", "mean"),
        optimal_cases=("optimality_proven", "sum")
    ).round(6)
)

print("\n50-task solver statuses:")
print(
    paired_fixed59.loc[
        paired_fixed59["task_count"] == 50,
        "solver_status"
    ].value_counts().to_dict()
)

print("\n50-task case-level comparison:")
display(
    paired_fixed59.loc[
        paired_fixed59["task_count"] == 50,
        [
            "case_id", "stable_rate", "exact_rate",
            "solver_status", "optimality_proven", "improvement"
        ]
    ].sort_values("case_id")
)


EXPERIMENT 59 — CORRECTED AUDIT
Matched cases: 100
Stable valid: 100
Exact valid: 100
Mean improvement: 0.029196
Wins: 67
Ties: 33
Losses: 0

Results by task size:


,cases,stable_mean,exact_mean,mean_improvement,optimal_cases
task_count,,,,,
10,20,0.198889,0.178889,0.020000,20
20,20,0.317105,0.266316,0.050789,19
30,20,0.384828,0.344253,0.040575,9
40,20,0.425256,0.390641,0.034615,3
50,20,0.503224,0.503224,0.000000,20



50-task solver statuses:
{'OPTIMAL': 20}

50-task case-level comparison:


,case_id,stable_rate,exact_rate,solver_status,optimality_proven,improvement
80,E56_N50_R01,0.560816,0.560816,OPTIMAL,True,0.0
81,E56_N50_R02,0.492245,0.492245,OPTIMAL,True,0.0
82,E56_N50_R03,0.403265,0.403265,OPTIMAL,True,0.0
83,E56_N50_R04,0.541224,0.541224,OPTIMAL,True,0.0
84,E56_N50_R05,0.466939,0.466939,OPTIMAL,True,0.0
85,E56_N50_R06,0.478367,0.478367,OPTIMAL,True,0.0
86,E56_N50_R07,0.553469,0.553469,OPTIMAL,True,0.0
87,E56_N50_R08,0.439184,0.439184,OPTIMAL,True,0.0
88,E56_N50_R09,0.525714,0.525714,OPTIMAL,True,0.0
89,E56_N50_R10,0.506939,0.506939,OPTIMAL,True,0.0


In [ ]:

# EXPERIMENT 60 — OPTIMIZER OUTPUT DIAGNOSTICS

import pandas as pd
from pathlib import Path

path60 = Path(
    "/content/planproof_results/experiment58/"
    "experiment58_fair_exact_results.csv"
)
exact60 = pd.read_csv(path60)

print("EXPERIMENT 60 — SOLVER DIAGNOSTICS")
print("=" * 45)

print("\nColumns available:")
print(exact60.columns.tolist())

print("\nAll solver statuses:")
print(exact60["solver_status"].value_counts().to_dict())

print("\nOptimality by task size:")
display(
    exact60.groupby("task_count").agg(
        cases=("case_id", "count"),
        optimality_proven=("optimality_proven", "sum"),
        mean_rate=("order_change_rate", "mean"),
        mean_runtime=("runtime_seconds", "mean")
    ).round(6)
)

print("\nAvailable objective/reversal-related columns:")
objective_cols60 = [
    c for c in exact60.columns
    if any(term in c.lower() for term in
           ["objective", "reversal", "order", "status", "optimal"])
]
print(objective_cols60)

print("\nFirst five 50-task records:")
display(
    exact60.loc[
        exact60["task_count"] == 50
    ].head(5)
)


EXPERIMENT 60 — SOLVER DIAGNOSTICS

Columns available:
['case_id', 'task_count', 'solver_status', 'optimality_proven', 'valid', 'violations', 'order_change_rate', 'runtime_seconds']

All solver statuses:
{'OPTIMAL': 71, 'FEASIBLE': 29}

Optimality by task size:


,cases,optimality_proven,mean_rate,mean_runtime
task_count,,,,
10,20,20,0.178889,0.050397
20,20,19,0.266316,1.845406
30,20,9,0.344253,4.018529
40,20,3,0.390641,4.472882
50,20,20,0.503224,0.146733



Available objective/reversal-related columns:
['solver_status', 'optimality_proven', 'order_change_rate']

First five 50-task records:


,case_id,task_count,solver_status,optimality_proven,valid,violations,order_change_rate,runtime_seconds
80,E56_N50_R01,50,OPTIMAL,True,True,0,0.560816,0.131023
81,E56_N50_R02,50,OPTIMAL,True,True,0,0.492245,0.124022
82,E56_N50_R03,50,OPTIMAL,True,True,0,0.403265,0.116091
83,E56_N50_R04,50,OPTIMAL,True,True,0,0.541224,0.122486
84,E56_N50_R05,50,OPTIMAL,True,True,0,0.466939,0.125563


In [ ]:

# EXPERIMENT 61 — ADAPTIVE REPAIR COMPARISON

import json
import time
import pandas as pd
from pathlib import Path

base61 = Path("/content/planproof_results")
benchmark61 = base61 / "experiment56/experiment56_benchmark.json"
stable_path61 = base61 / "experiment57/experiment57_stable_baseline.csv"
exact_path61 = base61 / "experiment58/experiment58_fair_exact_results.csv"
output_dir61 = base61 / "experiment61"
output_dir61.mkdir(parents=True, exist_ok=True)

with open(benchmark61, "r", encoding="utf-8") as f:
    cases61 = json.load(f)

stable61 = pd.read_csv(stable_path61)
exact61 = pd.read_csv(exact_path61)

rows61 = []

for i, case in enumerate(cases61, start=1):
    start = time.perf_counter()

    try:
        result = adaptive_repair_plan(case, time_budget_seconds=5)
        elapsed = time.perf_counter() - start

        order = result.get("order")
        valid = False
        rate = None
        violations = None

        if order is not None:
            validation = validate_plan({
                "task_order": order,
                "required_tasks": case["required_tasks"],
                "precedence_constraints": case["precedence_constraints"]
            })
            valid = bool(validation["valid"])
            violations = len(validation.get("violations", []))

            if valid:
                rate = shared_task_order_change_rate(
                    case["task_order"], order
                )

        rows61.append({
            "case_id": case["case_id"],
            "task_count": case["task_count"],
            "success": bool(result.get("success", False)),
            "valid": valid,
            "violations": violations,
            "adaptive_rate": rate,
            "method": result.get("method"),
            "runtime_seconds": elapsed,
            "optimality_proven": bool(
                result.get("optimality_proven", False)
            ),
            "solver_status": result.get("solver_status")
        })

    except Exception as exc:
        rows61.append({
            "case_id": case["case_id"],
            "task_count": case["task_count"],
            "success": False,
            "valid": False,
            "violations": None,
            "adaptive_rate": None,
            "method": "ERROR",
            "runtime_seconds": time.perf_counter() - start,
            "optimality_proven": False,
            "solver_status": str(exc)
        })

    if i % 10 == 0:
        print(f"Processed {i}/{len(cases61)} cases")

adaptive61 = pd.DataFrame(rows61)

comparison61 = (
    stable61[["case_id", "order_change_rate", "valid"]]
    .rename(columns={
        "order_change_rate": "stable_rate",
        "valid": "stable_valid"
    })
    .merge(
        exact61[["case_id", "order_change_rate", "valid",
                 "optimality_proven", "solver_status"]]
        .rename(columns={
            "order_change_rate": "exact_rate",
            "valid": "exact_valid",
            "optimality_proven": "exact_optimality_proven",
            "solver_status": "exact_solver_status"
        }),
        on="case_id",
        validate="one_to_one"
    )
    .merge(adaptive61, on=["case_id", "task_count"], validate="one_to_one")
)

comparison61["improvement_vs_stable"] = (
    comparison61["stable_rate"] - comparison61["adaptive_rate"]
)
comparison61["adaptive_no_worse_than_stable"] = (
    comparison61["adaptive_rate"]
    <= comparison61["stable_rate"] + 1e-12
)

csv_path61 = output_dir61 / "experiment61_adaptive_comparison.csv"
comparison61.to_csv(csv_path61, index=False)

print("\nEXPERIMENT 61 — SUMMARY")
print("=" * 45)
print("Cases:", len(comparison61))
print("Adaptive success:", int(comparison61["success"].sum()), "/", len(comparison61))
print("Adaptive valid:", int(comparison61["valid"].sum()), "/", len(comparison61))
print("Missing adaptive rates:", int(comparison61["adaptive_rate"].isna().sum()))
print("Adaptive no worse than stable:",
      int(comparison61["adaptive_no_worse_than_stable"].sum()), "/", len(comparison61))
print("Stable mean rate:", round(comparison61["stable_rate"].mean(), 6))
print("Exact mean rate:", round(comparison61["exact_rate"].mean(), 6))
print("Adaptive mean rate:", round(comparison61["adaptive_rate"].mean(), 6))
print("Adaptive mean runtime:",
      round(comparison61["runtime_seconds"].mean(), 6), "seconds")

print("\nAdaptive methods selected:")
print(comparison61["method"].value_counts(dropna=False).to_dict())

print("\nBy task size:")
display(
    comparison61.groupby("task_count").agg(
        cases=("case_id", "count"),
        stable_mean=("stable_rate", "mean"),
        exact_mean=("exact_rate", "mean"),
        adaptive_mean=("adaptive_rate", "mean"),
        adaptive_valid=("valid", "sum")
    ).round(6)
)

print("\nSaved:", csv_path61)


Processed 10/100 cases
Processed 20/100 cases
Processed 30/100 cases
Processed 40/100 cases
Processed 50/100 cases
Processed 60/100 cases
Processed 70/100 cases
Processed 80/100 cases
Processed 90/100 cases
Processed 100/100 cases


KeyError: 'task_count'

In [ ]:

# EXPERIMENT 61 — FIX MERGE AND SAVE RESULTS
# Reuse completed adaptive runs.

import json
import pandas as pd
from pathlib import Path

base61 = Path("/content/planproof_results")

with open(
    base61 / "experiment56/experiment56_benchmark.json",
    "r",
    encoding="utf-8"
) as f:
    cases61 = json.load(f)

task_lookup61 = pd.DataFrame([
    {"case_id": c["case_id"], "task_count": c["task_count"]}
    for c in cases61
])

# Restore task_count if it was not included in adaptive61.
if "task_count" not in adaptive61.columns:
    adaptive61 = adaptive61.merge(
        task_lookup61,
        on="case_id",
        how="left",
        validate="one_to_one"
    )

stable_clean61 = stable61[
    ["case_id", "order_change_rate", "valid"]
].rename(columns={
    "order_change_rate": "stable_rate",
    "valid": "stable_valid"
})

exact_clean61 = exact61[
    ["case_id", "order_change_rate", "valid",
     "optimality_proven", "solver_status"]
].rename(columns={
    "order_change_rate": "exact_rate",
    "valid": "exact_valid",
    "optimality_proven": "exact_optimality_proven",
    "solver_status": "exact_solver_status"
})

comparison61 = (
    stable_clean61
    .merge(exact_clean61, on="case_id", validate="one_to_one")
    .merge(adaptive61, on="case_id", suffixes=("", "_adaptive"),
           validate="one_to_one")
)

comparison61["improvement_vs_stable"] = (
    comparison61["stable_rate"] - comparison61["adaptive_rate"]
)
comparison61["adaptive_no_worse_than_stable"] = (
    comparison61["adaptive_rate"]
    <= comparison61["stable_rate"] + 1e-12
)

out61 = base61 / "experiment61"
out61.mkdir(parents=True, exist_ok=True)
csv_path61 = out61 / "experiment61_adaptive_comparison.csv"
comparison61.to_csv(csv_path61, index=False)

print("EXPERIMENT 61 — RECOVERED SUMMARY")
print("=" * 45)
print("Cases:", len(comparison61))
print("Adaptive success:", int(comparison61["success"].sum()), "/", len(comparison61))
print("Adaptive valid:", int(comparison61["valid"].sum()), "/", len(comparison61))
print("Missing adaptive rates:", int(comparison61["adaptive_rate"].isna().sum()))
print("No worse than stable:",
      int(comparison61["adaptive_no_worse_than_stable"].sum()), "/", len(comparison61))
print("Stable mean rate:", round(comparison61["stable_rate"].mean(), 6))
print("Exact mean rate:", round(comparison61["exact_rate"].mean(), 6))
print("Adaptive mean rate:", round(comparison61["adaptive_rate"].mean(), 6))
print("Adaptive mean runtime:",
      round(comparison61["runtime_seconds"].mean(), 6), "seconds")

print("\nMethods selected:")
print(comparison61["method"].value_counts(dropna=False).to_dict())

print("\nResults by task size:")
display(
    comparison61.groupby("task_count").agg(
        cases=("case_id", "count"),
        stable_mean=("stable_rate", "mean"),
        exact_mean=("exact_rate", "mean"),
        adaptive_mean=("adaptive_rate", "mean"),
        adaptive_valid=("valid", "sum")
    ).round(6)
)

print("\nSaved:", csv_path61)


EXPERIMENT 61 — RECOVERED SUMMARY
Cases: 100
Adaptive success: 100 / 100
Adaptive valid: 100 / 100
Missing adaptive rates: 0
No worse than stable: 100 / 100
Stable mean rate: 0.365861
Exact mean rate: 0.336665
Adaptive mean rate: 0.336883
Adaptive mean runtime: 2.209486 seconds

Methods selected:
{'exact_optimizer': 67, 'stable_repair': 33}

Results by task size:


,cases,stable_mean,exact_mean,adaptive_mean,adaptive_valid
task_count,,,,,
10,20,0.198889,0.178889,0.178889,20
20,20,0.317105,0.266316,0.267368,20
30,20,0.384828,0.344253,0.343908,20
40,20,0.425256,0.390641,0.391026,20
50,20,0.503224,0.503224,0.503224,20



Saved: /content/planproof_results/experiment61/experiment61_adaptive_comparison.csv


In [ ]:

# EXPERIMENT 62 — PAIRED STATISTICAL ANALYSIS

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

path62 = Path(
    "/content/planproof_results/experiment61/"
    "experiment61_adaptive_comparison.csv"
)
df62 = pd.read_csv(path62)

paired62 = df62[
    ["case_id", "task_count", "stable_rate", "adaptive_rate"]
].dropna().copy()

paired62["improvement"] = (
    paired62["stable_rate"] - paired62["adaptive_rate"]
)

values62 = paired62["improvement"].to_numpy()
n62 = len(values62)
mean62 = float(np.mean(values62))
median62 = float(np.median(values62))
sd62 = float(np.std(values62, ddof=1))
se62 = sd62 / np.sqrt(n62)

critical62 = stats.t.ppf(0.975, df=n62 - 1)
ci_low62 = mean62 - critical62 * se62
ci_high62 = mean62 + critical62 * se62

wins62 = int((values62 > 1e-12).sum())
ties62 = int((np.abs(values62) <= 1e-12).sum())
losses62 = int((values62 < -1e-12).sum())

wilcoxon_values62 = values62[np.abs(values62) > 1e-12]
if len(wilcoxon_values62):
    wilcoxon62 = stats.wilcoxon(wilcoxon_values62)
    wilcoxon_stat62 = float(wilcoxon62.statistic)
    wilcoxon_p62 = float(wilcoxon62.pvalue)
else:
    wilcoxon_stat62 = 0.0
    wilcoxon_p62 = 1.0

relative_reduction62 = (
    mean62 / paired62["stable_rate"].mean() * 100
    if paired62["stable_rate"].mean() else np.nan
)

summary62 = pd.DataFrame([{
    "n": n62,
    "mean_improvement": mean62,
    "median_improvement": median62,
    "ci95_low": ci_low62,
    "ci95_high": ci_high62,
    "relative_mean_reduction_percent": relative_reduction62,
    "wins": wins62,
    "ties": ties62,
    "losses": losses62,
    "wilcoxon_statistic": wilcoxon_stat62,
    "wilcoxon_p_value": wilcoxon_p62
}])

out62 = Path("/content/planproof_results/experiment62")
out62.mkdir(parents=True, exist_ok=True)

summary62.to_csv(out62 / "experiment62_summary.csv", index=False)
paired62.to_csv(out62 / "experiment62_paired_cases.csv", index=False)

print("EXPERIMENT 62 — PAIRED STATISTICS")
print("=" * 45)
print("Paired cases:", n62)
print("Mean improvement:", round(mean62, 6))
print("Median improvement:", round(median62, 6))
print("95% t-confidence interval:",
      (round(ci_low62, 6), round(ci_high62, 6)))
print("Relative mean reduction (%):", round(relative_reduction62, 3))
print("Wins / ties / losses:", wins62, "/", ties62, "/", losses62)
print("Wilcoxon statistic:", wilcoxon_stat62)
print("Wilcoxon p-value:", wilcoxon_p62)
print("\nSaved summary:", out62 / "experiment62_summary.csv")
print("Saved paired cases:", out62 / "experiment62_paired_cases.csv")


EXPERIMENT 62 — PAIRED STATISTICS
Paired cases: 100
Mean improvement: 0.028977
Median improvement: 0.022222
95% t-confidence interval: (np.float64(0.02275), np.float64(0.035205))
Relative mean reduction (%): 7.92
Wins / ties / losses: 67 / 33 / 0
Wilcoxon statistic: 0.0
Wilcoxon p-value: 1.1168862442963136e-12

Saved summary: /content/planproof_results/experiment62/experiment62_summary.csv
Saved paired cases: /content/planproof_results/experiment62/experiment62_paired_cases.csv


In [ ]:

# EXPERIMENT 63 — ROBUSTNESS BY TASK SIZE

import pandas as pd
from pathlib import Path

input63 = Path(
    "/content/planproof_results/experiment62/"
    "experiment62_paired_cases.csv"
)
output_dir63 = Path("/content/planproof_results/experiment63")
output_dir63.mkdir(parents=True, exist_ok=True)

df63 = pd.read_csv(input63)

df63["win"] = df63["improvement"] > 1e-12
df63["tie"] = df63["improvement"].abs() <= 1e-12
df63["loss"] = df63["improvement"] < -1e-12

summary63 = (
    df63.groupby("task_count")
    .agg(
        cases=("case_id", "count"),
        stable_mean=("stable_rate", "mean"),
        adaptive_mean=("adaptive_rate", "mean"),
        mean_improvement=("improvement", "mean"),
        median_improvement=("improvement", "median"),
        wins=("win", "sum"),
        ties=("tie", "sum"),
        losses=("loss", "sum")
    )
    .reset_index()
)

summary63["relative_reduction_percent"] = (
    summary63["mean_improvement"]
    / summary63["stable_mean"] * 100
)

summary63 = summary63.round(6)

summary_path63 = output_dir63 / "experiment63_size_robustness.csv"
case_path63 = output_dir63 / "experiment63_case_level.csv"

summary63.to_csv(summary_path63, index=False)
df63.to_csv(case_path63, index=False)

print("EXPERIMENT 63 — SIZE ROBUSTNESS")
print("=" * 45)
display(summary63)

print("\nOverall checks:")
print("Cases:", len(df63))
print("Wins:", int(df63["win"].sum()))
print("Ties:", int(df63["tie"].sum()))
print("Losses:", int(df63["loss"].sum()))
print("Missing improvements:", int(df63["improvement"].isna().sum()))

print("\nSaved summary:", summary_path63)
print("Saved case-level results:", case_path63)


EXPERIMENT 63 — SIZE ROBUSTNESS


,task_count,cases,stable_mean,adaptive_mean,mean_improvement,median_improvement,wins,ties,losses,relative_reduction_percent
0,10,20,0.198889,0.178889,0.020000,0.000000,9,11,0,10.055866
1,20,20,0.317105,0.267368,0.049737,0.036842,19,1,0,15.684647
2,30,20,0.384828,0.343908,0.040920,0.040230,20,0,0,10.633214
3,40,20,0.425256,0.391026,0.034231,0.037179,19,1,0,8.049442
4,50,20,0.503224,0.503224,0.000000,0.000000,0,20,0,0.000000



Overall checks:
Cases: 100
Wins: 67
Ties: 33
Losses: 0
Missing improvements: 0

Saved summary: /content/planproof_results/experiment63/experiment63_size_robustness.csv
Saved case-level results: /content/planproof_results/experiment63/experiment63_case_level.csv


In [ ]:

# EXPERIMENT 64 — CONSOLIDATED RESEARCH REPORT

import json
import pandas as pd
from pathlib import Path

base64 = Path("/content/planproof_results")
report_dir64 = base64 / "experiment64"
report_dir64.mkdir(parents=True, exist_ok=True)

# Load saved results
stable_a64 = pd.read_csv(
    base64 / "experiment51/experiment51_stable_baseline.csv"
)
exact_a64 = pd.read_csv(
    base64 / "experiment52/experiment52_fair_exact_results.csv"
)
adaptive_a64 = pd.read_csv(
    base64 / "experiment53/experiment53_three_method_comparison.csv"
)
stats_a64 = pd.read_csv(
    base64 / "experiment54/experiment54_summary.csv"
)

stable_b64 = pd.read_csv(
    base64 / "experiment57/experiment57_stable_baseline.csv"
)
exact_b64 = pd.read_csv(
    base64 / "experiment58/experiment58_fair_exact_results.csv"
)
adaptive_b64 = pd.read_csv(
    base64 / "experiment61/experiment61_adaptive_comparison.csv"
)
stats_b64 = pd.read_csv(
    base64 / "experiment62/experiment62_summary.csv"
)
size_b64 = pd.read_csv(
    base64 / "experiment63/experiment63_size_robustness.csv"
)

def mean_safe64(df, column):
    return float(df[column].mean())

# Recalculate benchmark A's adaptive paired outcome from saved data.
adaptive_a_rate = "adaptive_rate"
stable_a_rate = "stable_rate" if "stable_rate" in adaptive_a64.columns else None

if stable_a_rate is None:
    raise ValueError(
        "Expected stable_rate column missing from Experiment 53 comparison."
    )

a_improvement = (
    adaptive_a64[stable_a_rate] - adaptive_a64[adaptive_a_rate]
)

# Benchmark B paired outcomes
b_improvement = (
    adaptive_b64["stable_rate"] - adaptive_b64["adaptive_rate"]
)

summary64 = {
    "benchmark_A": {
        "cases": len(adaptive_a64),
        "stable_mean_order_change": mean_safe64(adaptive_a64, "stable_rate"),
        "exact_mean_order_change": mean_safe64(adaptive_a64, "exact_rate"),
        "adaptive_mean_order_change": mean_safe64(adaptive_a64, "adaptive_rate"),
        "adaptive_valid_count": int(adaptive_a64["adaptive_valid"].sum()),
        "adaptive_no_worse_than_stable": int(
            adaptive_a64["adaptive_no_worse_than_stable"].sum()
        ),
        "paired_mean_improvement": float(a_improvement.mean())
    },
    "benchmark_B": {
        "cases": len(adaptive_b64),
        "stable_mean_order_change": mean_safe64(adaptive_b64, "stable_rate"),
        "exact_mean_order_change": mean_safe64(adaptive_b64, "exact_rate"),
        "adaptive_mean_order_change": mean_safe64(adaptive_b64, "adaptive_rate"),
        "adaptive_valid_count": int(adaptive_b64["valid"].sum()),
        "adaptive_no_worse_than_stable": int(
            adaptive_b64["adaptive_no_worse_than_stable"].sum()
        ),
        "paired_mean_improvement": float(b_improvement.mean())
    },
    "scope": "Synthetic benchmarks only"
}

# Markdown report
report64 = f"""# PlanProof AI — Synthetic Benchmark Research Report

## 1. Objective

Evaluate whether adaptive plan repair can reduce changes to the relative
ordering of shared tasks compared with stable repair, while producing valid
task orders under the benchmark precedence constraints.

## 2. Methods

- **Stable repair:** deterministic precedence-aware repair baseline.
- **Fair exact optimization:** minimizes reversals among task pairs observed
  in the original order, subject to the plan constraints.
- **Adaptive repair:** compares the stable result with its own exact-solver
  candidate and retains the candidate only when its measured order-change
  rate is strictly lower.

The order-change rate measures relative ordering changes among shared tasks.
It is not a complete measure of semantic quality or total repair cost.

## 3. Benchmark A — Experiment 50

- Cases: {summary64['benchmark_A']['cases']}
- Stable mean order-change rate: {summary64['benchmark_A']['stable_mean_order_change']:.6f}
- Exact mean order-change rate: {summary64['benchmark_A']['exact_mean_order_change']:.6f}
- Adaptive mean order-change rate: {summary64['benchmark_A']['adaptive_mean_order_change']:.6f}
- Adaptive valid cases: {summary64['benchmark_A']['adaptive_valid_count']}
- Adaptive cases no worse than stable: {summary64['benchmark_A']['adaptive_no_worse_than_stable']}
- Paired mean improvement over stable: {summary64['benchmark_A']['paired_mean_improvement']:.6f}

## 4. Benchmark B — Experiment 56

- Cases: {summary64['benchmark_B']['cases']}
- Stable mean order-change rate: {summary64['benchmark_B']['stable_mean_order_change']:.6f}
- Exact mean order-change rate: {summary64['benchmark_B']['exact_mean_order_change']:.6f}
- Adaptive mean order-change rate: {summary64['benchmark_B']['adaptive_mean_order_change']:.6f}
- Adaptive valid cases: {summary64['benchmark_B']['adaptive_valid_count']}
- Adaptive cases no worse than stable: {summary64['benchmark_B']['adaptive_no_worse_than_stable']}
- Paired mean improvement over stable: {summary64['benchmark_B']['paired_mean_improvement']:.6f}

## 5. Paired statistical results

### Benchmark A
{stats_a64.to_markdown(index=False)}

### Benchmark B
{stats_b64.to_markdown(index=False)}

These tests quantify paired differences on the generated cases. Their
p-values should not be interpreted as evidence of performance on real-world
plans. Inference depends on the benchmark-generation process and sampling
assumptions.

## 6. Robustness by task size — Benchmark B

{size_b64.to_markdown(index=False)}

The recorded results show improvements for 10-, 20-, 30-, and 40-task groups.
The 50-task group showed ties across all 20 cases. The reason for those ties
has not yet been independently established.

## 7. Exact-solver limitations

Under the five-second per-case budget in Benchmark B, the exact optimizer
proved optimality for {int(exact_b64['optimality_proven'].sum())} of
{len(exact_b64)} cases. A valid FEASIBLE solution is not the same as a
solution whose optimality has been proven.

## 8. Limitations

1. Both benchmarks are synthetic; neither establishes real-world performance.
2. The benchmarks use particular graph-generation procedures and may not
   represent realistic AI-generated plans.
3. The order-change metric does not measure semantic correctness, task
   usefulness, execution success, or user satisfaction.
4. Results from related cases generated by one procedure should not be
   treated as equivalent to independent real-world evaluations.
5. The exact optimizer's returned orders and objective values should be
   preserved in future runs for stronger auditability.

## 9. Conclusion

Across two synthetic benchmarks, adaptive repair achieved lower mean
order-change rates than stable repair, and the recorded adaptive results
were never worse than stable repair on the tested cases. These are
promising preliminary results for the defined metric.

The next research priority is evaluation on held-out plans generated by
different systems or collected from realistic planning tasks, with a
predefined evaluation protocol and human or rule-based quality checks.

**Evidence scope:** synthetic benchmarks only.
"""

report_path64 = report_dir64 / "planproof_research_report.md"
json_path64 = report_dir64 / "planproof_summary.json"

report_path64.write_text(report64, encoding="utf-8")
json_path64.write_text(
    json.dumps(summary64, indent=2),
    encoding="utf-8"
)

print("EXPERIMENT 64 — REPORT CREATED")
print("=" * 45)
print("Benchmark A cases:", summary64["benchmark_A"]["cases"])
print("Benchmark B cases:", summary64["benchmark_B"]["cases"])
print("Benchmark A adaptive mean:",
      round(summary64["benchmark_A"]["adaptive_mean_order_change"], 6))
print("Benchmark B adaptive mean:",
      round(summary64["benchmark_B"]["adaptive_mean_order_change"], 6))
print("Benchmark B size groups:", len(size_b64))
print("\nMarkdown report:", report_path64)
print("Machine-readable summary:", json_path64)
print("\nEvidence scope: synthetic benchmarks only")


EXPERIMENT 64 — REPORT CREATED
Benchmark A cases: 100
Benchmark B cases: 100
Benchmark A adaptive mean: 0.34462
Benchmark B adaptive mean: 0.336883
Benchmark B size groups: 5

Markdown report: /content/planproof_results/experiment64/planproof_research_report.md
Machine-readable summary: /content/planproof_results/experiment64/planproof_summary.json

Evidence scope: synthetic benchmarks only


In [ ]:

# EXPERIMENT 65 — STEP 1
# Create a real-plan evaluation dataset template

import json
import csv
from pathlib import Path

base65 = Path("/content/planproof_results/experiment65")
base65.mkdir(parents=True, exist_ok=True)

fields65 = [
    "case_id",
    "source_model",
    "collection_date",
    "prompt",
    "original_plan_text",
    "required_tasks_json",
    "precedence_constraints_json",
    "task_order_json",
    "domain",
    "independent_validation_notes",
    "included_in_evaluation"
]

csv_path65 = base65 / "real_plan_dataset_template.csv"
json_path65 = base65 / "dataset_protocol.json"

with open(csv_path65, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields65)
    writer.writeheader()

protocol65 = {
    "experiment": "Experiment 65",
    "purpose": "Prepare a dataset for evaluating plan repair on AI-generated plans",
    "target_initial_cases": 30,
    "suggested_domains": [
        "study planning",
        "software project planning",
        "travel planning",
        "research workflow planning",
        "event planning"
    ],
    "collection_rules": [
        "Preserve each original AI response unchanged",
        "Record the model or service and collection date",
        "Extract atomic tasks and task order before applying any repair",
        "Record precedence constraints with an independent justification",
        "Do not invent constraints merely to make a repair succeed",
        "Mark ambiguous cases for review rather than silently guessing",
        "Keep original plans separate from corrupted evaluation copies",
        "Do not claim that collected plans are representative of all AI systems"
    ],
    "validation_warning": (
        "Precedence constraints need an independent source of justification. "
        "An AI-generated plan and constraints inferred from that same plan "
        "are not sufficient independent evidence of correctness."
    ),
    "evaluation_scope": "Real AI-generated plans; results may still be limited by the sample"
}

with open(json_path65, "w", encoding="utf-8") as f:
    json.dump(protocol65, f, indent=2)

print("EXPERIMENT 65 — DATASET TEMPLATE")
print("=" * 45)
print("Target initial cases:", protocol65["target_initial_cases"])
print("CSV template:", csv_path65)
print("Protocol:", json_path65)
print("Rows collected so far: 0")
print("\nNext: collect original AI-generated plans before creating defects.")


EXPERIMENT 65 — DATASET TEMPLATE
Target initial cases: 30
CSV template: /content/planproof_results/experiment65/real_plan_dataset_template.csv
Protocol: /content/planproof_results/experiment65/dataset_protocol.json
Rows collected so far: 0

Next: collect original AI-generated plans before creating defects.


In [ ]:

from pathlib import Path
import pandas as pd
import json
from datetime import date

base = Path("/content/planproof_results/experiment65")
base.mkdir(parents=True, exist_ok=True)

csv_path = base / "real_plan_dataset_template.csv"
raw_path = base / "case_001_original_plan.txt"

original_plan = """## 10-Step Software Project Plan

Here is a structured, end-to-end plan for executing a small software project. Each step lists its required execution order and dependencies to ensure a smooth workflow from concept to launch.

### Phase 1: Initiation and Definition

#### 1. Requirements Gathering and Scope Definition
Execution Order: Step 1 (First)
Dependencies: None.
Details: Define what the software needs to do, who the target users are, and what the core features (MVP) will be. Establish the project boundaries to prevent scope creep.

#### 2. Architecture and Technology Stack Selection
Execution Order: Step 2
Dependencies: Requires completed Step 1 (Requirements).
Details: Choose the programming languages, frameworks, databases, and hosting infrastructure based on the project requirements. Design the high-level system architecture and data flow.

### Phase 2: Design and Setup

#### 3. UI/UX Design and Prototyping
Execution Order: Step 3
Dependencies: Requires finalized Step 1 (Requirements). Can run concurrently with Step 2.
Details: Create wireframes, user flows, and mockups for the interface. This ensures alignment on the user experience before writing code.

#### 4. Project Initialization and Environment Setup
Execution Order: Step 4
Dependencies: Requires completed Step 2 (Tech Stack).
Details: Set up the version control repository (e.g., Git), configure development environments, initialize the project boilerplate, and set up CI/CD pipeline skeletons.

### Phase 3: Development

#### 5. Core Backend and Database Development
Execution Order: Step 5
Dependencies: Requires completed Step 4 (Environment Setup) and finalized database schemas from Step 2.
Details: Build the server-side logic, database connections, API endpoints, and authentication systems.

#### 6. Frontend Implementation
Execution Order: Step 6
Dependencies: Requires functional API endpoints or mocked data from Step 5 and designs from Step 3.
Details: Develop the user-facing interfaces, integrate state management, and connect the frontend to the backend APIs.

### Phase 4: Quality Assurance

#### 7. Testing and Quality Assurance
Execution Order: Step 7
Dependencies: Requires completed feature implementations from Step 5 and Step 6.
Details: Execute unit tests, integration tests, and user acceptance testing (UAT). Identify, log, and fix bugs, security vulnerabilities, and performance bottlenecks.

### Phase 5: Deployment and Beyond

#### 8. Production Deployment
Execution Order: Step 8
Dependencies: Requires passing tests from Step 7 and a stable production environment from Step 4.
Details: Provision production servers or serverless environments, configure domain names and SSL certificates, and release the software to the live environment.

#### 9. Monitoring and Post-Launch Verification
Execution Order: Step 9
Dependencies: Requires completed Step 8 (Deployment).
Details: Monitor application performance, error rates, and server health using logging and monitoring tools. Verify that core user journeys function correctly in production.

#### 10. Documentation and Project Handover
Execution Order: Step 10 (Final)
Dependencies: Can begin after deployment and stability verification in Step 9.
Details: Finalize technical documentation, API guides, and user manuals. Conduct a retrospective to review project performance and archive the repository.

Would you like to dive deeper into any specific phase, such as setting up the CI/CD pipeline or structuring the testing phase?
"""

tasks = [f"T{i}" for i in range(1, 11)]

# Each pair means: the first task must precede the second.
dependencies = [
    ["T1", "T2"],
    ["T1", "T3"],
    ["T2", "T4"],
    ["T2", "T5"],
    ["T4", "T5"],
    ["T3", "T6"],
    ["T5", "T6"],
    ["T5", "T7"],
    ["T6", "T7"],
    ["T7", "T8"],
    ["T4", "T8"],
    ["T8", "T9"],
    ["T9", "T10"]
]

# Preserve the supplied response; don't silently alter the source.
raw_path.write_text(original_plan, encoding="utf-8")

row = {
    "case_id": "REAL_001",
    "source_model": "ChatGPT (user-provided response; exact version unverified)",
    "collection_date": date.today().isoformat(),
    "prompt": (
        "Create a detailed 10-step plan for completing a small software "
        "project from initial requirements to final deployment. List the "
        "steps in their intended execution order. Make dependencies explicit."
    ),
    "original_plan_text": original_plan,
    "required_tasks_json": json.dumps(tasks),
    "precedence_constraints_json": json.dumps(dependencies),
    "task_order_json": json.dumps(tasks),
    "domain": "software_project_planning",
    "independent_validation_notes": (
        "Dependencies extracted from the plan's explicit statements. "
        "Step 3 may run concurrently with Step 2. "
        "Extraction should be independently reviewed before final evaluation."
    ),
    "included_in_evaluation": "pending_review"
}

df = pd.read_csv(csv_path, dtype=str).fillna("")

if "REAL_001" not in df["case_id"].values:
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(csv_path, index=False)

print("EXPERIMENT 65 — FIRST RECORD SAVED")
print("=" * 42)
print("Case ID: REAL_001")
print("Tasks extracted:", len(tasks))
print("Explicit precedence relationships:", len(dependencies))
print("Raw plan preserved:", raw_path.exists())
print("Dataset rows:", len(pd.read_csv(csv_path)))
print("Status: pending independent review")
print("Dataset:", csv_path)


EXPERIMENT 65 — FIRST RECORD SAVED
Case ID: REAL_001
Tasks extracted: 10
Explicit precedence relationships: 13
Raw plan preserved: True
Dataset rows: 1
Status: pending independent review
Dataset: /content/planproof_results/experiment65/real_plan_dataset_template.csv


In [ ]:

from pathlib import Path
import pandas as pd
import json
from collections import defaultdict, deque

base = Path("/content/planproof_results/experiment65")
csv_path = base / "real_plan_dataset_template.csv"

df = pd.read_csv(csv_path, dtype=str).fillna("")
row = df.loc[df["case_id"] == "REAL_001"].iloc[0]

tasks = json.loads(row["required_tasks_json"])
edges = [tuple(e) for e in json.loads(row["precedence_constraints_json"])]
order = json.loads(row["task_order_json"])

task_set = set(tasks)
errors = []

unknown_tasks = sorted(
    {x for edge in edges for x in edge if x not in task_set}
)
self_dependencies = [e for e in edges if e[0] == e[1]]
duplicate_edges = len(edges) - len(set(edges))

if unknown_tasks:
    errors.append(f"Unknown task IDs: {unknown_tasks}")
if self_dependencies:
    errors.append(f"Self-dependencies: {self_dependencies}")
if duplicate_edges:
    errors.append(f"Duplicate dependency pairs: {duplicate_edges}")
if len(order) != len(task_set) or set(order) != task_set:
    errors.append("Task order does not contain every task exactly once.")

# Cycle detection
graph = defaultdict(list)
indegree = {t: 0 for t in tasks}

for before, after in edges:
    if before in task_set and after in task_set and before != after:
        graph[before].append(after)
        indegree[after] += 1

queue = deque(t for t in tasks if indegree[t] == 0)
visited = []

while queue:
    node = queue.popleft()
    visited.append(node)
    for nxt in graph[node]:
        indegree[nxt] -= 1
        if indegree[nxt] == 0:
            queue.append(nxt)

has_cycle = len(visited) != len(task_set)
if has_cycle:
    errors.append("Dependency graph contains a cycle.")

position = {task: i for i, task in enumerate(order)}
order_conflicts = [
    (before, after)
    for before, after in edges
    if before in position and after in position
    and position[before] >= position[after]
]

print("EXPERIMENT 65 — DEPENDENCY REVIEW")
print("=" * 42)
print("Tasks:", len(tasks))
print("Dependency pairs:", len(edges))
print("Unknown task IDs:", len(unknown_tasks))
print("Self-dependencies:", len(self_dependencies))
print("Duplicate dependency pairs:", duplicate_edges)
print("Cycle detected:", has_cycle)
print("Dependencies conflicting with listed order:", len(order_conflicts))

if order_conflicts:
    print("Conflicting pairs:", order_conflicts)

if errors:
    print("\nReview result: ISSUES FOUND")
    for error in errors:
        print("-", error)
else:
    print("\nStructural result: PASS")
    print("Manual review is still required before final evaluation.")


EXPERIMENT 65 — DEPENDENCY REVIEW
Tasks: 10
Dependency pairs: 13
Unknown task IDs: 0
Self-dependencies: 0
Duplicate dependency pairs: 0
Cycle detected: False
Dependencies conflicting with listed order: 0

Structural result: PASS
Manual review is still required before final evaluation.


In [ ]:

from pathlib import Path
import pandas as pd
import json

base = Path("/content/planproof_results/experiment65")
df = pd.read_csv(
    base / "real_plan_dataset_template.csv",
    dtype=str
).fillna("")

row = df.loc[df["case_id"] == "REAL_001"].iloc[0]
edges = json.loads(row["precedence_constraints_json"])

task_names = {
    "T1": "Requirements gathering",
    "T2": "Architecture and tech stack",
    "T3": "UI/UX design",
    "T4": "Environment setup",
    "T5": "Backend and database",
    "T6": "Frontend implementation",
    "T7": "Testing and QA",
    "T8": "Production deployment",
    "T9": "Monitoring and verification",
    "T10": "Documentation and handover",
}

evidence = {
    ("T1", "T2"): "Step 2 explicitly requires Step 1.",
    ("T1", "T3"): "Step 3 explicitly requires Step 1.",
    ("T2", "T4"): "Step 4 explicitly requires Step 2.",
    ("T2", "T5"): "Step 5 explicitly requires the database schema from Step 2.",
    ("T4", "T5"): "Step 5 explicitly requires Step 4.",
    ("T3", "T6"): "Step 6 explicitly requires designs from Step 3.",
    ("T5", "T6"): "Step 6 requires functional APIs or mocked data from Step 5; this is conditional.",
    ("T5", "T7"): "Step 7 explicitly requires Step 5 implementation.",
    ("T6", "T7"): "Step 7 explicitly requires Step 6 implementation.",
    ("T7", "T8"): "Step 8 explicitly requires passing tests from Step 7.",
    ("T4", "T8"): "Step 8 explicitly requires a stable environment from Step 4.",
    ("T8", "T9"): "Step 9 explicitly requires Step 8.",
    ("T9", "T10"): "Step 10 says documentation can begin after Step 9; this may be a soft dependency.",
}

print("REAL_001 — DEPENDENCY EVIDENCE REVIEW")
print("=" * 48)

for before, after in edges:
    print(f"\n{before} -> {after}")
    print(f"  {task_names[before]} -> {task_names[after]}")
    print("  Evidence:", evidence.get((before, after), "Needs manual review"))

print("\nImportant:")
print("- Step 3 may run concurrently with Step 2.")
print("- Step 6 allows mocked data, so T5 -> T6 is conditional.")
print("- Step 10 says documentation can begin after Step 9; "
      "confirm whether this should be a hard dependency.")


REAL_001 — DEPENDENCY EVIDENCE REVIEW

T1 -> T2
  Requirements gathering -> Architecture and tech stack
  Evidence: Step 2 explicitly requires Step 1.

T1 -> T3
  Requirements gathering -> UI/UX design
  Evidence: Step 3 explicitly requires Step 1.

T2 -> T4
  Architecture and tech stack -> Environment setup
  Evidence: Step 4 explicitly requires Step 2.

T2 -> T5
  Architecture and tech stack -> Backend and database
  Evidence: Step 5 explicitly requires the database schema from Step 2.

T4 -> T5
  Environment setup -> Backend and database
  Evidence: Step 5 explicitly requires Step 4.

T3 -> T6
  UI/UX design -> Frontend implementation
  Evidence: Step 6 explicitly requires designs from Step 3.

T5 -> T6
  Backend and database -> Frontend implementation
  Evidence: Step 6 requires functional APIs or mocked data from Step 5; this is conditional.

T5 -> T7
  Backend and database -> Testing and QA
  Evidence: Step 7 explicitly requires Step 5 implementation.

T6 -> T7
  Frontend impleme

In [ ]:

from pathlib import Path
import pandas as pd
import json

base = Path("/content/planproof_results/experiment65")
csv_path = base / "real_plan_dataset_template.csv"

df = pd.read_csv(csv_path, dtype=str).fillna("")
idx = df.index[df["case_id"] == "REAL_001"][0]
row = df.loc[idx]

original_edges = json.loads(row["precedence_constraints_json"])

# Explicitly reviewed mandatory prerequisites.
reviewed_edges = [
    ["T1", "T2"],
    ["T1", "T3"],
    ["T2", "T4"],
    ["T2", "T5"],
    ["T4", "T5"],
    ["T3", "T6"],
    ["T5", "T7"],
    ["T6", "T7"],
    ["T7", "T8"],
    ["T4", "T8"],
    ["T8", "T9"],
]

review_record = {
    "case_id": "REAL_001",
    "review_status": "provisional_manual_review",
    "original_dependency_count": len(original_edges),
    "reviewed_mandatory_dependency_count": len(reviewed_edges),
    "reviewed_mandatory_dependencies": reviewed_edges,
    "conditional_dependency": {
        "edge": ["T5", "T6"],
        "reason": "Frontend may use mocked data instead of functional APIs."
    },
    "soft_dependency": {
        "edge": ["T9", "T10"],
        "reason": "Documentation may begin after Step 9; strict necessity is unclear."
    },
    "limitation": (
        "This is a provisional interpretation of the plan, not independently "
        "verified ground truth. Review decisions must be documented."
    )
}

review_path = base / "case_001_dependency_review.json"
review_path.write_text(json.dumps(review_record, indent=2), encoding="utf-8")

df.loc[idx, "independent_validation_notes"] = (
    "Structural checks passed. Provisional manual review retained 11 "
    "mandatory dependencies; T5->T6 recorded as conditional and T9->T10 "
    "as soft/ambiguous. Independent confirmation still required."
)
df.loc[idx, "included_in_evaluation"] = "pending_final_review"
df.to_csv(csv_path, index=False)

print("EXPERIMENT 65 — REVIEW SAVED")
print("=" * 40)
print("Original dependencies preserved:", len(original_edges))
print("Provisional mandatory dependencies:", len(reviewed_edges))
print("Conditional dependencies documented: 1")
print("Soft/ambiguous dependencies documented: 1")
print("Review file exists:", review_path.exists())
print("Evaluation status: pending_final_review")
print("Review file:", review_path)


EXPERIMENT 65 — REVIEW SAVED
Original dependencies preserved: 13
Provisional mandatory dependencies: 11
Conditional dependencies documented: 1
Soft/ambiguous dependencies documented: 1
Review file exists: True
Evaluation status: pending_final_review
Review file: /content/planproof_results/experiment65/case_001_dependency_review.json


In [ ]:

from pathlib import Path
import pandas as pd
import json

base = Path("/content/planproof_results/experiment65")

csv_path = base / "real_plan_dataset_template.csv"
raw_path = base / "case_001_original_plan.txt"
review_path = base / "case_001_dependency_review.json"

df = pd.read_csv(csv_path, dtype=str).fillna("")
matches = df[df["case_id"] == "REAL_001"]
review = json.loads(review_path.read_text(encoding="utf-8"))

checks = {
    "Exactly one REAL_001 record": len(matches) == 1,
    "Original plan file exists": raw_path.exists(),
    "Review JSON exists": review_path.exists(),
    "Original dependency count matches": (
        len(json.loads(matches.iloc[0]["precedence_constraints_json"]))
        == review["original_dependency_count"]
    ),
    "Reviewed dependency count matches": (
        len(review["reviewed_mandatory_dependencies"])
        == review["reviewed_mandatory_dependency_count"]
    ),
    "Conditional dependency documented": bool(
        review.get("conditional_dependency")
    ),
    "Soft dependency documented": bool(review.get("soft_dependency")),
}

print("EXPERIMENT 65 — INTEGRITY CHECK")
print("=" * 40)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print("-" * 40)
print(f"Dataset rows: {len(df)}")
print(f"Overall: {'PASS' if all(checks.values()) else 'FAIL'}")

if all(checks.values()):
    print("Next: collect REAL_002 from a separate AI-generated plan.")


EXPERIMENT 65 — INTEGRITY CHECK
PASS: Exactly one REAL_001 record
PASS: Original plan file exists
PASS: Review JSON exists
PASS: Original dependency count matches
PASS: Reviewed dependency count matches
PASS: Conditional dependency documented
PASS: Soft dependency documented
----------------------------------------
Dataset rows: 1
Overall: PASS
Next: collect REAL_002 from a separate AI-generated plan.


In [ ]:

from pathlib import Path
import pandas as pd
import json
from datetime import date

base = Path("/content/planproof_results/experiment65")
csv_path = base / "real_plan_dataset_template.csv"

original_plan = """## 14-Day University Examination Study Plan

Here is a comprehensive 10-step study plan structured for a high-intensity, 14-day preparation cycle. Explicit dependencies and parallel tasks are noted to help you manage your time effectively.

### Phase 1: Preparation and Scoping (Days 1–2)

#### 1. Syllabus Breakdown and Topic Prioritization
Execution Order: Step 1 (Day 1)
Dependencies: None.
Parallel Tasks: Can overlap with gathering materials (Step 2).
Details: Review the entire course syllabus, lecture slides, and past papers. Categorize topics into high, medium, and low weightage based on exam patterns. Allocate your limited 14 days proportionally, spending the most time on high-yield, difficult concepts.

#### 2. Gathering Study Materials and Resources
Execution Order: Step 2 (Days 1–2)
Dependencies: None.
Parallel Tasks: Can run in parallel with Step 1.
Details: Collect all textbooks, lecture notes, recommended readings, previous years' question papers, and trusted online references. Organize them digitally or physically so you do not waste time searching for resources later.

### Phase 2: Core Learning and Practice (Days 3–9)

#### 3. Concept Mastery (First Pass)
Execution Order: Step 3 (Days 3–6)
Dependencies: Requires completed Step 1 (Prioritization) and Step 2 (Materials).
Parallel Tasks: Can run alongside Step 4 for topics you finish early.
Details: Go through core concepts systematically, starting with your high-priority topics. Focus on understanding underlying principles rather than rote memorization. Make concise summary notes or flashcards for formulas and definitions.

#### 4. Active Problem Solving and Practice Questions
Execution Order: Step 4 (Days 5–8)
Dependencies: Requires initial familiarity with concepts from Step 3.
Parallel Tasks: Can overlap with the tail-end of Step 3.
Details: Immediately apply what you learn by solving end-of-chapter problems, tutorial sheets, and standard textbook exercises. This bridges the gap between passive reading and active recall.

#### 5. Intermediate Topic-Wise Review
Execution Order: Step 5 (Day 9)
Dependencies: Requires completed core study and practice from Step 3 and Step 4.
Details: Take a day to consolidate what you have learned across the first week. Clear up lingering doubts, revisit tricky derivations or code snippets, and ensure your summary notes are complete.

### Phase 3: Testing and Diagnostics (Days 10–12)

#### 6. Full-Length Mock Test 1 (Diagnostic)
Execution Order: Step 6 (Day 10)
Dependencies: Requires completion of Step 5 (Review).
Details: Sit for your first timed, simulated mock exam under strict exam conditions. Do not look at notes. This exposes remaining blind spots and helps you practice time management under pressure.

#### 7. Mistake Analysis and Remediation
Execution Order: Step 7 (Day 11)
Dependencies: Requires completed mock test from Step 6.
Parallel Tasks: Can happen alongside targeted concept review.
Details: Thoroughly review every incorrect or guessed answer from the mock test. Understand why you got it wrong and restudy those specific sub-topics.

#### 8. Full-Length Mock Test 2 (Refinement)
Execution Order: Step 8 (Day 12)
Dependencies: Requires remediation work from Step 7.
Details: Take a second mock test to measure improvement, refine your exam-taking strategy, and build psychological stamina for the actual exam.

### Phase 4: Final Consolidation and Execution (Days 13–14)

#### 9. High-Yield Revision and Formula Summary Pass
Execution Order: Step 9 (Day 13)
Dependencies: Follows final mock analysis from Step 8.
Details: Review summary notes, key formulas, diagrams, and commonly tested definitions.

#### 10. Exam-Day Preparation and Mental Reset
Execution Order: Step 10 (Day 14 / Exam Eve)
Dependencies: Final step preceding the examination.
Details: Prepare exam logistics, get 7 to 8 hours of sleep, and avoid heavy last-minute cramming.

Would you like help breaking down the schedule for specific subjects or topics you are focusing on?
"""

tasks = [f"T{i}" for i in range(1, 11)]

# Explicit prerequisite relationships extracted from the response.
edges = [
    ["T1", "T3"],
    ["T2", "T3"],
    ["T3", "T4"],
    ["T3", "T5"],
    ["T4", "T5"],
    ["T5", "T6"],
    ["T6", "T7"],
    ["T7", "T8"],
    ["T8", "T9"],
]

row = {
    "case_id": "REAL_002",
    "source_model": "ChatGPT (user-provided response; exact version unverified)",
    "collection_date": date.today().isoformat(),
    "prompt": (
        "Create a detailed 10-step study plan for preparing for a difficult "
        "university examination over 14 days. Include topic selection, "
        "gathering study materials, studying concepts, practice questions, "
        "mock tests, reviewing mistakes, revision, and final exam-day "
        "preparation. State dependencies explicitly and identify parallel tasks."
    ),
    "original_plan_text": original_plan,
    "required_tasks_json": json.dumps(tasks),
    "precedence_constraints_json": json.dumps(edges),
    "task_order_json": json.dumps(tasks),
    "domain": "university_exam_study_planning",
    "independent_validation_notes": (
        "Initial extraction from explicit prerequisites. Steps 1 and 2 "
        "are parallel; Steps 3 and 4 overlap. Dependencies are provisional "
        "and require manual review."
    ),
    "included_in_evaluation": "pending_review"
}

raw_path = base / "case_002_original_plan.txt"
raw_path.write_text(original_plan, encoding="utf-8")

df = pd.read_csv(csv_path, dtype=str).fillna("")
if "REAL_002" not in df["case_id"].values:
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(csv_path, index=False)

print("EXPERIMENT 65 — REAL_002 SAVED")
print("=" * 40)
print("Case ID: REAL_002")
print("Tasks extracted:", len(tasks))
print("Initial dependency pairs:", len(edges))
print("Original plan preserved:", raw_path.exists())
print("Total dataset rows:", len(pd.read_csv(csv_path)))
print("Status: pending dependency review")
print("Dataset:", csv_path)


EXPERIMENT 65 — REAL_002 SAVED
Case ID: REAL_002
Tasks extracted: 10
Initial dependency pairs: 9
Original plan preserved: True
Total dataset rows: 2
Status: pending dependency review
Dataset: /content/planproof_results/experiment65/real_plan_dataset_template.csv


In [ ]:

from pathlib import Path
import pandas as pd
import json
from collections import defaultdict, deque

base = Path("/content/planproof_results/experiment65")
df = pd.read_csv(
    base / "real_plan_dataset_template.csv",
    dtype=str
).fillna("")

matches = df[df["case_id"] == "REAL_002"]
assert len(matches) == 1, f"Expected one REAL_002 record, found {len(matches)}"

row = matches.iloc[0]
tasks = json.loads(row["required_tasks_json"])
edges = [tuple(e) for e in json.loads(row["precedence_constraints_json"])]
order = json.loads(row["task_order_json"])

task_set = set(tasks)
unknown = sorted({t for e in edges for t in e if t not in task_set})
self_edges = [e for e in edges if e[0] == e[1]]
duplicates = len(edges) - len(set(edges))

# Cycle detection
graph = defaultdict(list)
indegree = {t: 0 for t in tasks}

for before, after in edges:
    if before in task_set and after in task_set and before != after:
        graph[before].append(after)
        indegree[after] += 1

queue = deque(t for t in tasks if indegree[t] == 0)
visited = []

while queue:
    node = queue.popleft()
    visited.append(node)
    for nxt in graph[node]:
        indegree[nxt] -= 1
        if indegree[nxt] == 0:
            queue.append(nxt)

has_cycle = len(visited) != len(task_set)

if len(order) == len(task_set) and set(order) == task_set:
    position = {t: i for i, t in enumerate(order)}
    order_conflicts = [
        (a, b) for a, b in edges
        if a in position and b in position and position[a] >= position[b]
    ]
else:
    order_conflicts = ["Task order does not contain every task exactly once"]

checks = {
    "Exactly one REAL_002 record": len(matches) == 1,
    "Ten unique task IDs": len(tasks) == 10 and len(task_set) == 10,
    "No unknown task IDs": len(unknown) == 0,
    "No self-dependencies": len(self_edges) == 0,
    "No duplicate dependencies": duplicates == 0,
    "No dependency cycles": not has_cycle,
    "No listed-order conflicts": len(order_conflicts) == 0,
    "Original plan preserved": (
        base.joinpath("case_002_original_plan.txt").exists()
    ),
}

print("EXPERIMENT 65 — REAL_002 STRUCTURAL CHECK")
print("=" * 46)
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print("Tasks:", len(tasks))
print("Dependency pairs:", len(edges))
if unknown:
    print("Unknown IDs:", unknown)
if self_edges:
    print("Self-dependencies:", self_edges)
if duplicates:
    print("Duplicate count:", duplicates)
if has_cycle:
    print("Cycle detected: True")
if order_conflicts:
    print("Order conflicts:", order_conflicts)

print("-" * 46)
print("Overall:", "PASS" if all(checks.values()) else "FAIL")
print("Note: Structural PASS does not establish semantic correctness.")


EXPERIMENT 65 — REAL_002 STRUCTURAL CHECK
PASS: Exactly one REAL_002 record
PASS: Ten unique task IDs
PASS: No unknown task IDs
PASS: No self-dependencies
PASS: No duplicate dependencies
PASS: No dependency cycles
PASS: No listed-order conflicts
PASS: Original plan preserved
Tasks: 10
Dependency pairs: 9
----------------------------------------------
Overall: PASS
Note: Structural PASS does not establish semantic correctness.


In [ ]:

from pathlib import Path
import pandas as pd
import json

base = Path("/content/planproof_results/experiment65")
df = pd.read_csv(
    base / "real_plan_dataset_template.csv",
    dtype=str
).fillna("")

row = df.loc[df["case_id"] == "REAL_002"].iloc[0]
edges = json.loads(row["precedence_constraints_json"])

task_names = {
    "T1": "Syllabus breakdown and prioritization",
    "T2": "Gathering study materials",
    "T3": "Concept mastery",
    "T4": "Practice questions",
    "T5": "Intermediate topic-wise review",
    "T6": "Mock test 1",
    "T7": "Mistake analysis and remediation",
    "T8": "Mock test 2",
    "T9": "High-yield revision",
    "T10": "Exam-day preparation",
}

evidence = {
    ("T1", "T3"): "Step 3 explicitly requires Step 1.",
    ("T2", "T3"): "Step 3 explicitly requires Step 2.",
    ("T3", "T4"): "Step 4 requires initial familiarity with Step 3.",
    ("T3", "T5"): "Step 5 requires completed Step 3.",
    ("T4", "T5"): "Step 5 requires completed Step 4.",
    ("T5", "T6"): "Step 6 explicitly requires Step 5.",
    ("T6", "T7"): "Step 7 explicitly requires Step 6.",
    ("T7", "T8"): "Step 8 explicitly requires Step 7.",
    ("T8", "T9"): "Step 9 follows final mock analysis from Step 8.",
}

print("REAL_002 — DEPENDENCY EVIDENCE REVIEW")
print("=" * 48)

for before, after in edges:
    print(f"\n{before} -> {after}")
    print(f"  {task_names[before]} -> {task_names[after]}")
    print("  Evidence:", evidence.get((before, after), "Needs review"))

print("\nReview questions:")
print("1. Steps 1 and 2 are independent and can run in parallel.")
print("2. Steps 3 and 4 overlap, so practice need not wait for all concepts.")
print("3. Step 10 is the final preparation step; its explicit dependency "
      "on Step 9 is not represented in the current nine pairs.")
print("4. Step 7 can run alongside targeted concept review.")


REAL_002 — DEPENDENCY EVIDENCE REVIEW

T1 -> T3
  Syllabus breakdown and prioritization -> Concept mastery
  Evidence: Step 3 explicitly requires Step 1.

T2 -> T3
  Gathering study materials -> Concept mastery
  Evidence: Step 3 explicitly requires Step 2.

T3 -> T4
  Concept mastery -> Practice questions
  Evidence: Step 4 requires initial familiarity with Step 3.

T3 -> T5
  Concept mastery -> Intermediate topic-wise review
  Evidence: Step 5 requires completed Step 3.

T4 -> T5
  Practice questions -> Intermediate topic-wise review
  Evidence: Step 5 requires completed Step 4.

T5 -> T6
  Intermediate topic-wise review -> Mock test 1
  Evidence: Step 6 explicitly requires Step 5.

T6 -> T7
  Mock test 1 -> Mistake analysis and remediation
  Evidence: Step 7 explicitly requires Step 6.

T7 -> T8
  Mistake analysis and remediation -> Mock test 2
  Evidence: Step 8 explicitly requires Step 7.

T8 -> T9
  Mock test 2 -> High-yield revision
  Evidence: Step 9 follows final mock analysis

In [ ]:

# ============================================================
# PLANPROOF AI — EXPERIMENT 65 MASTER CELL
# One-run dataset integrity + dependency audit + progress report
# ============================================================

from pathlib import Path
from collections import defaultdict, deque
from datetime import datetime, timezone
import pandas as pd
import json

BASE = Path("/content/planproof_results/experiment65")
DATASET = BASE / "real_plan_dataset_template.csv"
OUT = BASE / "master_audit"
OUT.mkdir(parents=True, exist_ok=True)

TARGET_CASES = 30

# ---------- 1. Load existing dataset ----------
if not DATASET.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET}\n"
        "Run the Experiment 65 template setup first."
    )

df = pd.read_csv(DATASET, dtype=str).fillna("")
required_columns = [
    "case_id", "source_model", "collection_date", "prompt",
    "original_plan_text", "required_tasks_json",
    "precedence_constraints_json", "task_order_json",
    "domain", "independent_validation_notes",
    "included_in_evaluation"
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# ---------- 2. Audit each record ----------
audit_rows = []
dependency_rows = []
seen_ids = set()

for _, row in df.iterrows():
    case_id = row["case_id"].strip()
    errors = []
    warnings = []

    if not case_id:
        errors.append("Missing case_id")
    elif case_id in seen_ids:
        errors.append("Duplicate case_id")
    seen_ids.add(case_id)

    raw_file = BASE / (
        "case_001_original_plan.txt" if case_id == "REAL_001"
        else "case_002_original_plan.txt" if case_id == "REAL_002"
        else f"{case_id.lower()}_original_plan.txt"
    )

    try:
        tasks = json.loads(row["required_tasks_json"])
        edges = json.loads(row["precedence_constraints_json"])
        order = json.loads(row["task_order_json"])

        if not isinstance(tasks, list) or not isinstance(edges, list):
            raise ValueError("Tasks and dependencies must be JSON lists")
        if not isinstance(order, list):
            raise ValueError("Task order must be a JSON list")

        task_set = set(tasks)
        if len(task_set) != len(tasks):
            errors.append("Duplicate task IDs")
        if not tasks:
            errors.append("No tasks recorded")
        if not row["original_plan_text"].strip():
            errors.append("Original plan text is empty")

        unknown = []
        self_edges = []
        normalized_edges = []
        malformed_edges = []

        for edge in edges:
            if not isinstance(edge, list) or len(edge) != 2:
                malformed_edges.append(edge)
                continue

            before, after = edge
            normalized_edges.append((before, after))

            if before not in task_set or after not in task_set:
                unknown.append(edge)
            if before == after:
                self_edges.append(edge)

        if unknown:
            errors.append(f"Unknown task IDs in dependencies: {unknown}")
        if self_edges:
            errors.append(f"Self-dependencies: {self_edges}")
        if malformed_edges:
            errors.append(f"Malformed dependency pairs: {malformed_edges}")

        if len(normalized_edges) != len(set(normalized_edges)):
            errors.append("Duplicate dependency pairs")

        if len(order) != len(task_set) or set(order) != task_set:
            errors.append("Task order must contain every unique task exactly once")

        # Build graph only from usable edges.
        graph = defaultdict(list)
        indegree = {t: 0 for t in tasks}

        for before, after in normalized_edges:
            if (before in task_set and after in task_set
                    and before != after):
                graph[before].append(after)
                indegree[after] += 1

        queue = deque(t for t in tasks if indegree[t] == 0)
        visited = []

        while queue:
            node = queue.popleft()
            visited.append(node)
            for nxt in graph[node]:
                indegree[nxt] -= 1
                if indegree[nxt] == 0:
                    queue.append(nxt)

        cycle = len(visited) != len(task_set)
        if cycle:
            errors.append("Dependency graph contains a cycle")

        conflicts = []
        if len(order) == len(task_set) and set(order) == task_set:
            pos = {task: i for i, task in enumerate(order)}
            conflicts = [
                [a, b] for a, b in normalized_edges
                if a in pos and b in pos and pos[a] >= pos[b]
            ]

        if conflicts:
            warnings.append(
                "Listed order conflicts with dependency pairs; "
                "check whether the order is a strict execution order"
            )

        if not row["independent_validation_notes"].strip():
            warnings.append("No review notes recorded")

        if row["included_in_evaluation"] != "approved":
            warnings.append("Not approved for final evaluation")

        if not raw_file.exists():
            warnings.append("Separate original-plan text file not found")

        review_path = BASE / (
            "case_001_dependency_review.json" if case_id == "REAL_001"
            else "case_002_dependency_review.json" if case_id == "REAL_002"
            else f"{case_id.lower()}_dependency_review.json"
        )

        review_exists = review_path.exists()
        if not review_exists:
            warnings.append("Separate dependency review JSON not found")

        for before, after in normalized_edges:
            dependency_rows.append({
                "case_id": case_id,
                "before_task": before,
                "after_task": after,
                "source_dependency_recorded": True
            })

        audit_rows.append({
            "case_id": case_id,
            "domain": row["domain"],
            "task_count": len(tasks),
            "dependency_count": len(edges),
            "structural_valid": len(errors) == 0,
            "cycle_detected": cycle,
            "order_conflict_count": len(conflicts),
            "raw_file_exists": raw_file.exists(),
            "review_json_exists": review_exists,
            "evaluation_status": row["included_in_evaluation"],
            "error_count": len(errors),
            "warning_count": len(warnings),
            "errors": " | ".join(errors),
            "warnings": " | ".join(warnings)
        })

    except Exception as exc:
        audit_rows.append({
            "case_id": case_id,
            "domain": row.get("domain", ""),
            "task_count": None,
            "dependency_count": None,
            "structural_valid": False,
            "cycle_detected": None,
            "order_conflict_count": None,
            "raw_file_exists": raw_file.exists(),
            "review_json_exists": False,
            "evaluation_status": row["included_in_evaluation"],
            "error_count": 1,
            "warning_count": 0,
            "errors": f"Could not parse or audit record: {exc}",
            "warnings": ""
        })

# ---------- 3. Save detailed audit outputs ----------
audit_df = pd.DataFrame(audit_rows)
deps_df = pd.DataFrame(dependency_rows)

audit_csv = OUT / "all_records_audit.csv"
deps_csv = OUT / "all_dependencies.csv"
summary_json = OUT / "master_summary.json"
report_txt = OUT / "master_report.txt"

audit_df.to_csv(audit_csv, index=False)
deps_df.to_csv(deps_csv, index=False)

total = len(df)
structural_passes = int(audit_df["structural_valid"].fillna(False).sum())
approved = int((df["included_in_evaluation"] == "approved").sum())
review_pending = total - approved
domains = sorted(set(df["domain"].dropna()) - {""})

summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_cases": TARGET_CASES,
    "records_found": total,
    "remaining_to_target": max(0, TARGET_CASES - total),
    "unique_case_ids": int(df["case_id"].nunique()),
    "structurally_valid_records": structural_passes,
    "records_with_structural_errors": total - structural_passes,
    "approved_for_evaluation": approved,
    "not_approved_for_evaluation": review_pending,
    "domains": domains,
    "important_limitations": [
        "Structural validity does not prove dependency semantics are correct.",
        "The cell does not generate genuine AI plans or independently verify their source.",
        "Records require documented human review before approval.",
        "Do not claim real-world effectiveness from this small dataset alone."
    ],
    "audit_csv": str(audit_csv),
    "dependencies_csv": str(deps_csv),
    "dataset_csv": str(DATASET)
}
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

lines = [
    "PLANPROOF AI — EXPERIMENT 65 MASTER REPORT",
    "=" * 48,
    f"Records found: {total}",
    f"Target: {TARGET_CASES}",
    f"Remaining to target: {max(0, TARGET_CASES-total)}",
    f"Structural passes: {structural_passes}/{total}",
    f"Records with structural errors: {total-structural_passes}",
    f"Approved for evaluation: {approved}",
    f"Not approved: {review_pending}",
    f"Domains represented: {', '.join(domains) if domains else 'None'}",
    "",
    "PER-RECORD RESULTS",
    "-" * 48
]

for _, r in audit_df.iterrows():
    lines.append(
        f"{r['case_id']}: structural_valid={r['structural_valid']}; "
        f"tasks={r['task_count']}; dependencies={r['dependency_count']}; "
        f"errors={r['error_count']}; warnings={r['warning_count']}"
    )
    if r["errors"]:
        lines.append(f"  ERRORS: {r['errors']}")
    if r["warnings"]:
        lines.append(f"  WARNINGS: {r['warnings']}")

lines += [
    "",
    "RESEARCH LIMITATION",
    "Structural checks are automated. Semantic dependency review is not.",
    "Only approve records after documented review."
]
report_txt.write_text("\n".join(lines), encoding="utf-8")

# ---------- 4. Display concise result ----------
print("PLANPROOF AI — EXPERIMENT 65 MASTER CELL")
print("=" * 48)
print(f"Records found: {total}/{TARGET_CASES}")
print(f"Remaining to target: {max(0, TARGET_CASES-total)}")
print(f"Structural passes: {structural_passes}/{total}")
print(f"Structurally invalid records: {total-structural_passes}")
print(f"Approved for evaluation: {approved}")
print(f"Not yet approved: {review_pending}")

if not audit_df.empty:
    print("\nPER-RECORD SUMMARY")
    print(audit_df[
        ["case_id", "task_count", "dependency_count",
         "structural_valid", "evaluation_status"]
    ].to_string(index=False))

print("\nFILES SAVED")
print(audit_csv)
print(deps_csv)
print(summary_json)
print(report_txt)
print("\nNote: This audits collected records; it does not invent real plans.")


PLANPROOF AI — EXPERIMENT 65 MASTER CELL
Records found: 2/30
Remaining to target: 28
Structural passes: 2/2
Structurally invalid records: 0
Approved for evaluation: 0
Not yet approved: 2

PER-RECORD SUMMARY
 case_id  task_count  dependency_count  structural_valid    evaluation_status
REAL_001          10                13              True pending_final_review
REAL_002          10                 9              True       pending_review

FILES SAVED
/content/planproof_results/experiment65/master_audit/all_records_audit.csv
/content/planproof_results/experiment65/master_audit/all_dependencies.csv
/content/planproof_results/experiment65/master_audit/master_summary.json
/content/planproof_results/experiment65/master_audit/master_report.txt

Note: This audits collected records; it does not invent real plans.


In [ ]:

# ============================================================
# PLANPROOF AI — EXPERIMENT 65 MASTER CELL V2
# Batch import genuine AI-generated plans + structural audit
# ============================================================

from pathlib import Path
from collections import defaultdict, deque
from datetime import date
import pandas as pd
import json
import re

BASE = Path("/content/planproof_results/experiment65")
DATASET = BASE / "real_plan_dataset_template.csv"
RAW_DIR = BASE / "raw_plans"
AUDIT_DIR = BASE / "batch_audit"

RAW_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET}")

df = pd.read_csv(DATASET, dtype=str).fillna("")

# ------------------------------------------------------------
# ADD NEW PLANS HERE
# Use the complete, genuine response copied from an AI chat.
# Do not invent or paraphrase the original response.
# Start with an empty list; add plans when you have them.
# ------------------------------------------------------------

NEW_PLANS = [
    # Example structure (replace with a real response):
    # {
    #   "prompt": "Your exact prompt",
    #   "response": """Paste the complete original AI response here.""",
    #   "source_model": "Service/model if known",
    #   "domain": "travel_planning"
    # },
]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def next_case_id(existing_ids):
    nums = []
    for cid in existing_ids:
        match = re.fullmatch(r"REAL_(\d+)", str(cid))
        if match:
            nums.append(int(match.group(1)))
    return f"REAL_{max(nums, default=0) + 1:03d}"

def audit_plan(tasks, edges, order):
    errors = []
    task_set = set(tasks)

    if not tasks:
        errors.append("No tasks extracted")
    if len(task_set) != len(tasks):
        errors.append("Duplicate task IDs")

    malformed = [
        e for e in edges
        if not isinstance(e, list) or len(e) != 2
    ]
    if malformed:
        errors.append("Malformed dependency pairs")

    valid_edges = [
        tuple(e) for e in edges
        if isinstance(e, list) and len(e) == 2
    ]

    if any(a not in task_set or b not in task_set
           for a, b in valid_edges):
        errors.append("Unknown task ID in dependency pair")

    if any(a == b for a, b in valid_edges):
        errors.append("Self-dependency")

    if len(valid_edges) != len(set(valid_edges)):
        errors.append("Duplicate dependency pairs")

    if len(order) != len(task_set) or set(order) != task_set:
        errors.append("Order must contain every task exactly once")

    graph = defaultdict(list)
    indegree = {t: 0 for t in tasks}

    for a, b in valid_edges:
        if a in task_set and b in task_set and a != b:
            graph[a].append(b)
            indegree[b] += 1

    queue = deque(t for t in tasks if indegree[t] == 0)
    visited = []

    while queue:
        node = queue.popleft()
        visited.append(node)
        for nxt in graph[node]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                queue.append(nxt)

    cycle = len(visited) != len(task_set)
    if cycle:
        errors.append("Dependency graph contains a cycle")

    conflicts = []
    if len(order) == len(task_set) and set(order) == task_set:
        pos = {t: i for i, t in enumerate(order)}
        conflicts = [
            [a, b] for a, b in valid_edges
            if a in pos and b in pos and pos[a] >= pos[b]
        ]

    return errors, conflicts

# ------------------------------------------------------------
# Import batch
# ------------------------------------------------------------

added = []
skipped = []
existing_ids = df["case_id"].astype(str).tolist()

for item in NEW_PLANS:
    prompt = str(item.get("prompt", "")).strip()
    response = str(item.get("response", "")).strip()
    source = str(item.get("source_model", "Unknown; user-provided")).strip()
    domain = str(item.get("domain", "unspecified")).strip()

    if not prompt or not response:
        skipped.append("Skipped item: prompt or response is empty")
        continue

    # Avoid reimporting the exact same prompt/response pair.
    duplicate = (
        (df["prompt"] == prompt) &
        (df["original_plan_text"] == response)
    ).any()

    if duplicate:
        skipped.append("Skipped exact duplicate prompt/response")
        continue

    case_id = next_case_id(existing_ids)
    existing_ids.append(case_id)

    raw_path = RAW_DIR / f"{case_id.lower()}_original_plan.txt"
    raw_path.write_text(response, encoding="utf-8")

    # This cell preserves the original response but does not pretend
    # to infer ground-truth task dependencies automatically.
    row = {
        "case_id": case_id,
        "source_model": source,
        "collection_date": date.today().isoformat(),
        "prompt": prompt,
        "original_plan_text": response,
        "required_tasks_json": "[]",
        "precedence_constraints_json": "[]",
        "task_order_json": "[]",
        "domain": domain,
        "independent_validation_notes": (
            "Imported in batch. Task/dependency extraction and semantic "
            "review are pending."
        ),
        "included_in_evaluation": "pending_extraction"
    }

    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    added.append(case_id)

df.to_csv(DATASET, index=False)

# ------------------------------------------------------------
# Audit extracted records. Newly imported records remain pending
# until task extraction is completed.
# ------------------------------------------------------------

audit_rows = []

for _, row in df.iterrows():
    case_id = row["case_id"]
    try:
        tasks = json.loads(row["required_tasks_json"])
        edges = json.loads(row["precedence_constraints_json"])
        order = json.loads(row["task_order_json"])

        if not tasks:
            errors = ["Task extraction pending"]
            conflicts = []
        else:
            errors, conflicts = audit_plan(tasks, edges, order)

        audit_rows.append({
            "case_id": case_id,
            "domain": row["domain"],
            "task_count": len(tasks),
            "dependency_count": len(edges),
            "structural_valid": len(errors) == 0,
            "order_conflict_count": len(conflicts),
            "evaluation_status": row["included_in_evaluation"],
            "issues": " | ".join(errors)
        })

    except Exception as exc:
        audit_rows.append({
            "case_id": case_id,
            "domain": row["domain"],
            "task_count": None,
            "dependency_count": None,
            "structural_valid": False,
            "order_conflict_count": None,
            "evaluation_status": row["included_in_evaluation"],
            "issues": str(exc)
        })

audit_df = pd.DataFrame(audit_rows)
audit_path = AUDIT_DIR / "batch_import_audit.csv"
audit_df.to_csv(audit_path, index=False)

print("PLANPROOF AI — BATCH IMPORT COMPLETE")
print("=" * 48)
print("New plans imported:", len(added))
print("New IDs:", added if added else "None")
print("Skipped items:", len(skipped))
for message in skipped:
    print("-", message)
print("Total dataset records:", len(df))
print("Target remaining:", max(0, 30 - len(df)))
print("Structurally valid records:", int(audit_df["structural_valid"].sum()))
print("Records needing extraction/review:",
      int((audit_df["structural_valid"] == False).sum()))
print("\nDataset:", DATASET)
print("Audit:", audit_path)
print("\nReminder: Imported plans need task extraction and review.")


PLANPROOF AI — BATCH IMPORT COMPLETE
New plans imported: 0
New IDs: None
Skipped items: 0
Total dataset records: 2
Target remaining: 28
Structurally valid records: 2
Records needing extraction/review: 0

Dataset: /content/planproof_results/experiment65/real_plan_dataset_template.csv
Audit: /content/planproof_results/experiment65/batch_audit/batch_import_audit.csv

Reminder: Imported plans need task extraction and review.


In [ ]:

from pathlib import Path
import pandas as pd
import json

path = Path("/content/planproof_results/experiment65/real_plan_dataset_template.csv")
df = pd.read_csv(path, dtype=str).fillna("")

def is_pending(value):
    try:
        return len(json.loads(value)) == 0
    except Exception:
        return True

pending_extraction = df["required_tasks_json"].apply(is_pending)
approved = df["included_in_evaluation"].eq("approved")

print("PLANPROOF AI — ACCURATE DATASET STATUS")
print("=" * 44)
print("Total records:", len(df))
print("Records with task extraction pending:", int(pending_extraction.sum()))
print("Records with tasks extracted:", int((~pending_extraction).sum()))
print("Approved for evaluation:", int(approved.sum()))
print("Remaining to target of 30:", max(0, 30 - len(df)))

print("\nRecord status:")
for _, row in df.iterrows():
    pending = is_pending(row["required_tasks_json"])
    print(
        f"{row['case_id']}: "
        f"{'PENDING EXTRACTION' if pending else 'EXTRACTED; REVIEW REQUIRED'}"
        f" | {row['domain']}"
    )


PLANPROOF AI — ACCURATE DATASET STATUS
Total records: 2
Records with task extraction pending: 0
Records with tasks extracted: 2
Approved for evaluation: 0
Remaining to target of 30: 28

Record status:
REAL_001: EXTRACTED; REVIEW REQUIRED | software_project_planning
REAL_002: EXTRACTED; REVIEW REQUIRED | university_exam_study_planning


In [ ]:

# ============================================================
# PLANPROOF AI — MASTER CELL V3
# Consolidated review queue + readiness report
# ============================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timezone

BASE = Path("/content/planproof_results/experiment65")
DATASET = BASE / "real_plan_dataset_template.csv"
OUTPUT = BASE / "master_v3"
OUTPUT.mkdir(parents=True, exist_ok=True)

TARGET = 30

df = pd.read_csv(DATASET, dtype=str).fillna("")

review_rows = []

for _, row in df.iterrows():
    case_id = row["case_id"]

    try:
        tasks = json.loads(row["required_tasks_json"])
        edges = json.loads(row["precedence_constraints_json"])
        order = json.loads(row["task_order_json"])
        parse_ok = True
    except Exception:
        tasks, edges, order = [], [], []
        parse_ok = False

    raw_candidates = [
        BASE / f"{case_id.lower()}_original_plan.txt",
        BASE / "raw_plans" / f"{case_id.lower()}_original_plan.txt"
    ]
    raw_exists = any(p.exists() for p in raw_candidates)

    review_candidates = [
        BASE / f"{case_id.lower()}_dependency_review.json",
        BASE / "reviews" / f"{case_id.lower()}_dependency_review.json"
    ]
    review_path = next((p for p in review_candidates if p.exists()), None)

    if review_path:
        try:
            review_data = json.loads(review_path.read_text(encoding="utf-8"))
            review_status = review_data.get("review_status", "review file exists")
        except Exception:
            review_status = "review file unreadable"
    else:
        review_status = "review file missing"

    task_set = set(tasks)
    issues = []

    if not parse_ok:
        issues.append("JSON parsing failed")
    if not tasks:
        issues.append("Tasks missing")
    if len(task_set) != len(tasks):
        issues.append("Duplicate task IDs")
    if not order or len(order) != len(task_set) or set(order) != task_set:
        issues.append("Task order mismatch")

    valid_edges = [
        e for e in edges
        if isinstance(e, list) and len(e) == 2
    ]
    if len(valid_edges) != len(edges):
        issues.append("Malformed dependency pairs")

    if any(a not in task_set or b not in task_set for a, b in valid_edges):
        issues.append("Unknown task IDs in dependencies")

    if any(a == b for a, b in valid_edges):
        issues.append("Self-dependency")

    if len({tuple(e) for e in valid_edges}) != len(valid_edges):
        issues.append("Duplicate dependencies")

    if not raw_exists:
        issues.append("Separate raw-plan file missing")

    if review_status == "review file missing":
        issues.append("Dependency review missing")

    approved = row["included_in_evaluation"] == "approved"
    if not approved:
        issues.append("Not approved for evaluation")

    review_rows.append({
        "case_id": case_id,
        "domain": row["domain"],
        "task_count": len(tasks),
        "dependency_count": len(edges),
        "raw_plan_preserved": raw_exists,
        "review_status": review_status,
        "approved": approved,
        "ready_for_human_review": (
            parse_ok and len(tasks) > 0 and raw_exists
        ),
        "issues": " | ".join(issues)
    })

queue = pd.DataFrame(review_rows)
queue_path = OUTPUT / "human_review_queue.csv"
queue.to_csv(queue_path, index=False)

total = len(df)
approved_count = int(queue["approved"].sum())
human_review_ready = int(queue["ready_for_human_review"].sum())

summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_records": TARGET,
    "records_collected": total,
    "records_remaining": max(0, TARGET - total),
    "records_ready_for_human_review": human_review_ready,
    "records_approved": approved_count,
    "records_not_approved": total - approved_count,
    "note": (
        "Readiness means the record can be reviewed; it does not "
        "mean semantic correctness or research approval."
    )
}

summary_path = OUTPUT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("PLANPROOF AI — MASTER CELL V3")
print("=" * 44)
print("Records collected:", total, "/", TARGET)
print("Records remaining:", max(0, TARGET - total))
print("Ready for human review:", human_review_ready)
print("Approved for evaluation:", approved_count)

if not queue.empty:
    print("\nREVIEW QUEUE")
    print(queue.to_string(index=False))

print("\nSaved files:")
print(queue_path)
print(summary_path)


PLANPROOF AI — MASTER CELL V3
Records collected: 2 / 30
Records remaining: 28
Ready for human review: 0
Approved for evaluation: 0

REVIEW QUEUE
 case_id                         domain  task_count  dependency_count  raw_plan_preserved       review_status  approved  ready_for_human_review                                                                                   issues
REAL_001      software_project_planning          10                13               False review file missing     False                   False Separate raw-plan file missing | Dependency review missing | Not approved for evaluation
REAL_002 university_exam_study_planning          10                 9               False review file missing     False                   False Separate raw-plan file missing | Dependency review missing | Not approved for evaluation

Saved files:
/content/planproof_results/experiment65/master_v3/human_review_queue.csv
/content/planproof_results/experiment65/master_v3/summary.json


In [ ]:

from pathlib import Path
import pandas as pd
import json

BASE = Path("/content/planproof_results/experiment65")
DATASET = BASE / "real_plan_dataset_template.csv"
RAW_DIR = BASE / "raw_plans"
REVIEW_DIR = BASE / "reviews"
OUT_DIR = BASE / "master_v4"

for folder in (RAW_DIR, REVIEW_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATASET, dtype=str).fillna("")
results = []

for idx, row in df.iterrows():
    case_id = row["case_id"].strip()
    original_text = row["original_plan_text"]

    # Restore separate raw file from the preserved dataset text.
    raw_path = RAW_DIR / f"{case_id.lower()}_original_plan.txt"
    if original_text.strip() and not raw_path.exists():
        raw_path.write_text(original_text, encoding="utf-8")

    # Locate an existing review JSON before creating any new file.
    review_candidates = [
        BASE / f"{case_id.lower()}_dependency_review.json",
        BASE / f"{case_id}_dependency_review.json",
        REVIEW_DIR / f"{case_id.lower()}_dependency_review.json",
        BASE / "case_001_dependency_review.json"
            if case_id == "REAL_001" else BASE / "__not_used__"
    ]
    existing_review = next(
        (p for p in review_candidates if p.exists()), None
    )

    review_status = "missing"
    if existing_review:
        try:
            review_data = json.loads(
                existing_review.read_text(encoding="utf-8")
            )
            review_status = review_data.get(
                "review_status", "existing review found; status unspecified"
            )
        except Exception:
            review_status = "review JSON unreadable"

    # For REAL_002, don't fabricate a completed review.
    tasks = json.loads(row["required_tasks_json"])
    edges = json.loads(row["precedence_constraints_json"])
    order = json.loads(row["task_order_json"])

    structural_issues = []
    if not tasks:
        structural_issues.append("Task extraction missing")
    if len(tasks) != len(set(tasks)):
        structural_issues.append("Duplicate task IDs")
    if len(order) != len(set(tasks)) or set(order) != set(tasks):
        structural_issues.append("Task order mismatch")
    if any(
        not isinstance(e, list) or len(e) != 2
        for e in edges
    ):
        structural_issues.append("Malformed dependency pairs")

    valid_edges = [
        e for e in edges
        if isinstance(e, list) and len(e) == 2
    ]
    if any(a not in set(tasks) or b not in set(tasks)
           for a, b in valid_edges):
        structural_issues.append("Unknown task IDs")

    raw_exists = raw_path.exists()
    review_exists = existing_review is not None

    # Update notes honestly; no automatic semantic approval.
    if not review_exists:
        df.at[idx, "independent_validation_notes"] = (
            str(row["independent_validation_notes"]).strip()
            + " | Separate review JSON still needs to be saved."
        ).strip(" |")

    results.append({
        "case_id": case_id,
        "task_count": len(tasks),
        "dependency_count": len(edges),
        "raw_file_exists": raw_exists,
        "review_json_found": review_exists,
        "review_status": review_status,
        "structural_issues": " | ".join(structural_issues),
        "evaluation_status": row["included_in_evaluation"]
    })

df.to_csv(DATASET, index=False)

queue = pd.DataFrame(results)
queue_path = OUT_DIR / "review_queue.csv"
queue.to_csv(queue_path, index=False)

print("PLANPROOF AI — MASTER CELL V4")
print("=" * 44)
print("Records:", len(df))
print("Raw files restored/found:", int(queue["raw_file_exists"].sum()))
print("Existing review JSONs found:", int(queue["review_json_found"].sum()))
print("Approved for evaluation:",
      int((df["included_in_evaluation"] == "approved").sum()))
print("\nReview queue:")
print(queue.to_string(index=False))
print("\nDataset:", DATASET)
print("Review queue:", queue_path)
print("\nNo records were automatically approved.")


PLANPROOF AI — MASTER CELL V4
Records: 2
Raw files restored/found: 2
Existing review JSONs found: 1
Approved for evaluation: 0

Review queue:
 case_id  task_count  dependency_count  raw_file_exists  review_json_found             review_status structural_issues    evaluation_status
REAL_001          10                13             True               True provisional_manual_review                   pending_final_review
REAL_002          10                 9             True              False                   missing                         pending_review

Dataset: /content/planproof_results/experiment65/real_plan_dataset_template.csv
Review queue: /content/planproof_results/experiment65/master_v4/review_queue.csv

No records were automatically approved.


In [ ]:
import pandas as pd
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
df = pd.read_csv(base / "real_plan_dataset_template.csv")

row = df.loc[df["case_id"] == "REAL_002"].iloc[0]

print("CASE:", row["case_id"])
print("\nORIGINAL AI-GENERATED PLAN:\n")
print(row["original_plan_text"])

print("\nEXTRACTED TASK ORDER:")
print(json.dumps(json.loads(row["task_order_json"]), indent=2))

print("\nEXTRACTED PRECEDENCE CONSTRAINTS:")
print(json.dumps(json.loads(row["precedence_constraints_json"]), indent=2))

CASE: REAL_002

ORIGINAL AI-GENERATED PLAN:

## 14-Day University Examination Study Plan

Here is a comprehensive 10-step study plan structured for a high-intensity, 14-day preparation cycle. Explicit dependencies and parallel tasks are noted to help you manage your time effectively.

### Phase 1: Preparation and Scoping (Days 1–2)

#### 1. Syllabus Breakdown and Topic Prioritization
Execution Order: Step 1 (Day 1)
Dependencies: None.
Parallel Tasks: Can overlap with gathering materials (Step 2).
Details: Review the entire course syllabus, lecture slides, and past papers. Categorize topics into high, medium, and low weightage based on exam patterns. Allocate your limited 14 days proportionally, spending the most time on high-yield, difficult concepts.

#### 2. Gathering Study Materials and Resources
Execution Order: Step 2 (Days 1–2)
Dependencies: None.
Parallel Tasks: Can run in parallel with Step 1.
Details: Collect all textbooks, lecture notes, recommended readings, previous years' 

In [ ]:
import pandas as pd
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
csv_path = base / "real_plan_dataset_template.csv"

df = pd.read_csv(csv_path)

dependencies = [
    ["T1", "T3"],
    ["T2", "T3"],
    ["T3", "T4"],
    ["T3", "T5"],
    ["T4", "T5"],
    ["T5", "T6"],
    ["T6", "T7"],
    ["T7", "T8"],
    ["T8", "T9"],
    ["T9", "T10"]
]

mask = df["case_id"] == "REAL_002"
assert mask.sum() == 1, "Expected exactly one REAL_002 record"

df.loc[mask, "precedence_constraints_json"] = json.dumps(dependencies)
df.loc[mask, "independent_validation_notes"] = (
    "Dependency extraction reviewed against original plan text. "
    "Ten candidate precedence relationships recorded. "
    "Step 3 and Step 4 overlap in schedule, so the dependency "
    "represents initial familiarity rather than full completion. "
    "Pending independent final review; not approved for evaluation."
)
df.loc[mask, "included_in_evaluation"] = False

df.to_csv(csv_path, index=False)

print("REAL_002 dependency update saved.")
print("Candidate dependencies:", len(dependencies))
print("Approved for evaluation: No")
print("Original plan text preserved:", pd.notna(df.loc[mask, "original_plan_text"].iloc[0]))
print("CSV:", csv_path)

REAL_002 dependency update saved.
Candidate dependencies: 10
Approved for evaluation: No
Original plan text preserved: True
CSV: /content/planproof_results/experiment65/real_plan_dataset_template.csv


In [ ]:
import pandas as pd
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
df = pd.read_csv(base / "real_plan_dataset_template.csv")

row = df.loc[df["case_id"] == "REAL_001"].iloc[0]

print("CASE:", row["case_id"])
print("\nORIGINAL AI-GENERATED PLAN:\n")
print(row["original_plan_text"])

print("\nEXTRACTED TASK ORDER:")
print(json.dumps(json.loads(row["task_order_json"]), indent=2))

print("\nEXTRACTED PRECEDENCE CONSTRAINTS:")
print(json.dumps(json.loads(row["precedence_constraints_json"]), indent=2))

CASE: REAL_001

ORIGINAL AI-GENERATED PLAN:

## 10-Step Software Project Plan

Here is a structured, end-to-end plan for executing a small software project. Each step lists its required execution order and dependencies to ensure a smooth workflow from concept to launch.

### Phase 1: Initiation and Definition

#### 1. Requirements Gathering and Scope Definition
Execution Order: Step 1 (First)
Dependencies: None.
Details: Define what the software needs to do, who the target users are, and what the core features (MVP) will be. Establish the project boundaries to prevent scope creep.

#### 2. Architecture and Technology Stack Selection
Execution Order: Step 2
Dependencies: Requires completed Step 1 (Requirements).
Details: Choose the programming languages, frameworks, databases, and hosting infrastructure based on the project requirements. Design the high-level system architecture and data flow.

### Phase 2: Design and Setup

#### 3. UI/UX Design and Prototyping
Execution Order: Step 3
D

In [ ]:
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
review_dir = base / "reviews"
review_dir.mkdir(parents=True, exist_ok=True)

review = {
    "case_id": "REAL_001",
    "review_status": "pending_final_review",
    "original_plan_preserved": True,
    "candidate_dependency_count": 13,
    "conditional_dependencies": [
        {
            "edge": ["T5", "T6"],
            "reason": (
                "Frontend implementation may use mocked data instead "
                "of functional backend API endpoints."
            )
        }
    ],
    "review_notes": [
        "The other extracted dependencies are supported by the plan text.",
        "Conditional dependencies must be handled consistently in evaluation.",
        "This is a preliminary text-based review, not independent validation."
    ],
    "approved_for_evaluation": False
}

path = review_dir / "REAL_001_review.json"
path.write_text(json.dumps(review, indent=2), encoding="utf-8")

print("Review record saved:", path)
print("Status:", review["review_status"])
print("Approved for evaluation:", review["approved_for_evaluation"])

Review record saved: /content/planproof_results/experiment65/reviews/REAL_001_review.json
Status: pending_final_review
Approved for evaluation: False


In [ ]:
import pandas as pd
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
csv_path = base / "real_plan_dataset_template.csv"
df = pd.read_csv(csv_path)

review_paths = {
    "REAL_001": base / "reviews" / "REAL_001_review.json",
    "REAL_002": base / "case_002_dependency_review.json"
}

rows = []

for _, record in df.iterrows():
    case_id = record["case_id"]

    try:
        tasks = json.loads(record["task_order_json"])
        dependencies = json.loads(record["precedence_constraints_json"])
        extraction_ok = (
            isinstance(tasks, list)
            and isinstance(dependencies, list)
            and all(
                isinstance(edge, list)
                and len(edge) == 2
                and edge[0] in tasks
                and edge[1] in tasks
                for edge in dependencies
            )
        )
    except (ValueError, TypeError):
        tasks, dependencies = [], []
        extraction_ok = False

    raw_path = base / "raw_plans" / f"{case_id.lower()}_original_plan.txt"
    review_path = review_paths.get(case_id)

    review_exists = bool(review_path and review_path.exists())
    review_status = "missing"

    if review_exists:
        try:
            review_data = json.loads(review_path.read_text(encoding="utf-8"))
            review_status = review_data.get("review_status", "unknown")
        except (ValueError, OSError):
            review_status = "unreadable"

    original_text = record.get("original_plan_text", "")
    original_preserved = isinstance(original_text, str) and len(original_text.strip()) > 0

    rows.append({
        "case_id": case_id,
        "tasks": len(tasks),
        "dependencies": len(dependencies),
        "extraction_structure_ok": extraction_ok,
        "original_text_preserved_in_csv": original_preserved,
        "raw_file_exists": raw_path.exists(),
        "review_file_exists": review_exists,
        "review_status": review_status,
        "approved_for_evaluation": bool(record.get("included_in_evaluation", False))
    })

audit = pd.DataFrame(rows)
print(audit.to_string(index=False))

print("\nTOTAL RECORDS:", len(audit))
print("APPROVED RECORDS:", int(audit["approved_for_evaluation"].sum()))
print("NOTE: Structural checks do not establish dependency correctness.")

 case_id  tasks  dependencies  extraction_structure_ok  original_text_preserved_in_csv  raw_file_exists  review_file_exists        review_status  approved_for_evaluation
REAL_001     10            13                     True                            True             True                True pending_final_review                     True
REAL_002     10            10                     True                            True             True               False              missing                     True

TOTAL RECORDS: 2
APPROVED RECORDS: 2
NOTE: Structural checks do not establish dependency correctness.


In [ ]:
import pandas as pd
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
csv_path = base / "real_plan_dataset_template.csv"

df = pd.read_csv(csv_path)

mask = df["case_id"].isin(["REAL_001", "REAL_002"])
assert mask.sum() == 2, "Expected REAL_001 and REAL_002"

# Neither record has completed independent final validation.
df.loc[mask, "included_in_evaluation"] = False

if "independent_validation_notes" in df.columns:
    df.loc[df["case_id"] == "REAL_001", "independent_validation_notes"] = (
        "Preliminary dependency review saved; final independent review pending."
    )
    df.loc[df["case_id"] == "REAL_002", "independent_validation_notes"] = (
        "Dependencies extracted and preliminarily reviewed; final independent review pending."
    )

df.to_csv(csv_path, index=False)

check = pd.read_csv(csv_path)
print("Corrected evaluation status:")
print(check[["case_id", "included_in_evaluation"]].to_string(index=False))
print("\nApproved records:", int(check["included_in_evaluation"].fillna(False).astype(bool).sum()))
print("Original plan text preserved:", bool(
    check.loc[mask, "original_plan_text"].notna().all()
))
print("CSV saved:", csv_path)

Corrected evaluation status:
 case_id  included_in_evaluation
REAL_001                   False
REAL_002                   False

Approved records: 0
Original plan text preserved: True
CSV saved: /content/planproof_results/experiment65/real_plan_dataset_template.csv


In [ ]:
from pathlib import Path

base = Path("/content/planproof_results/experiment65")

print("REVIEW JSON FILES FOUND:\n")

for path in sorted(base.rglob("*review*.json")):
    print(path)

print("\nEXPECTED REAL_001 REVIEW:")
print(base / "reviews" / "REAL_001_review.json")

print("\nEXPECTED REAL_002 REVIEW:")
print(base / "case_002_dependency_review.json")

REVIEW JSON FILES FOUND:

/content/planproof_results/experiment65/case_001_dependency_review.json
/content/planproof_results/experiment65/reviews/REAL_001_review.json

EXPECTED REAL_001 REVIEW:
/content/planproof_results/experiment65/reviews/REAL_001_review.json

EXPECTED REAL_002 REVIEW:
/content/planproof_results/experiment65/case_002_dependency_review.json


In [ ]:
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")

review_map = {
    "REAL_001": base / "case_001_dependency_review.json",
    "REAL_002": base / "case_002_dependency_review.json",
}

print("PLANPROOF AI — REVIEW PATH RECONCILIATION")
print("=" * 44)

for case_id, path in review_map.items():
    print(f"\n{case_id}")
    print("Review file:", path)
    print("Exists:", path.exists())

    if path.exists():
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            print("JSON readable: True")
            print("Fields:", list(data.keys()))
        except (json.JSONDecodeError, OSError) as exc:
            print("JSON readable: False")
            print("Error:", str(exc))

print("\nSeparate REAL_001 review record:")
print((base / "reviews" / "REAL_001_review.json").exists())
print("\nNo files were modified.")

PLANPROOF AI — REVIEW PATH RECONCILIATION

REAL_001
Review file: /content/planproof_results/experiment65/case_001_dependency_review.json
Exists: True
JSON readable: True
Fields: ['case_id', 'review_status', 'original_dependency_count', 'reviewed_mandatory_dependency_count', 'reviewed_mandatory_dependencies', 'conditional_dependency', 'soft_dependency', 'limitation']

REAL_002
Review file: /content/planproof_results/experiment65/case_002_dependency_review.json
Exists: False

Separate REAL_001 review record:
True

No files were modified.


In [ ]:
import pandas as pd
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
csv_path = base / "real_plan_dataset_template.csv"
review_path = base / "case_002_dependency_review.json"

assert not review_path.exists(), (
    "A review file already exists. Stop to avoid overwriting it."
)

df = pd.read_csv(csv_path)
row = df.loc[df["case_id"] == "REAL_002"].iloc[0]

dependencies = json.loads(row["precedence_constraints_json"])

review = {
    "case_id": "REAL_002",
    "review_status": "pending_final_review",
    "original_dependency_count": len(dependencies),
    "reviewed_mandatory_dependency_count": None,
    "reviewed_mandatory_dependencies": [],
    "conditional_dependencies": [],
    "soft_dependencies": [],
    "preliminary_candidate_dependencies": dependencies,
    "limitation": (
        "Preliminary text-based extraction only. The ten candidate "
        "relationships have not been independently validated. "
        "Step 3 and Step 4 overlap in the schedule, so the dependency "
        "interpretation requires further review. Not approved for evaluation."
    ),
    "approved_for_evaluation": False
}

review_path.write_text(
    json.dumps(review, indent=2),
    encoding="utf-8"
)

print("REAL_002 review record created.")
print("Status:", review["review_status"])
print("Candidate dependencies:", len(dependencies))
print("Independently validated dependencies: 0")
print("Approved for evaluation:", review["approved_for_evaluation"])
print("Saved:", review_path)

REAL_002 review record created.
Status: pending_final_review
Candidate dependencies: 10
Independently validated dependencies: 0
Approved for evaluation: False
Saved: /content/planproof_results/experiment65/case_002_dependency_review.json


In [ ]:
import pandas as pd
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
csv_path = base / "real_plan_dataset_template.csv"

review_paths = {
    "REAL_001": base / "case_001_dependency_review.json",
    "REAL_002": base / "case_002_dependency_review.json",
}

df = pd.read_csv(csv_path)
results = []

for case_id, review_path in review_paths.items():
    matches = df[df["case_id"] == case_id]
    assert len(matches) == 1, f"Expected exactly one {case_id} row"

    row = matches.iloc[0]
    review = {}

    if review_path.exists():
        review = json.loads(review_path.read_text(encoding="utf-8"))

    tasks = json.loads(row["task_order_json"])
    dependencies = json.loads(row["precedence_constraints_json"])

    results.append({
        "case_id": case_id,
        "tasks": len(tasks),
        "dependencies": len(dependencies),
        "original_text_preserved": (
            isinstance(row["original_plan_text"], str)
            and bool(row["original_plan_text"].strip())
        ),
        "review_file_exists": review_path.exists(),
        "review_status": review.get("review_status", "missing"),
        "independently_validated_count": (
            review.get("independently_validated_dependency_count", 0)
        ),
        "csv_approved": bool(row["included_in_evaluation"]),
        "review_approved": bool(review.get("approved_for_evaluation", False)),
    })

audit = pd.DataFrame(results)
print("PLANPROOF AI — CONSOLIDATED AUDIT")
print("=" * 44)
print(audit.to_string(index=False))
print("\nTotal cases:", len(audit))
print("Cases with review files:", int(audit["review_file_exists"].sum()))
print("CSV-approved cases:", int(audit["csv_approved"].sum()))
print("Review-approved cases:", int(audit["review_approved"].sum()))
print("\nNo files were modified.")

PLANPROOF AI — CONSOLIDATED AUDIT
 case_id  tasks  dependencies  original_text_preserved  review_file_exists             review_status  independently_validated_count  csv_approved  review_approved
REAL_001     10            13                     True                True provisional_manual_review                              0         False            False
REAL_002     10            10                     True                True      pending_final_review                              0         False            False

Total cases: 2
Cases with review files: 2
CSV-approved cases: 0
Review-approved cases: 0

No files were modified.


In [ ]:
import pandas as pd
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
df = pd.read_csv(base / "real_plan_dataset_template.csv")

row = df.loc[df["case_id"] == "REAL_002"].iloc[0]
dependencies = json.loads(row["precedence_constraints_json"])

print("PLANPROOF AI — REAL_002 DEPENDENCY REVIEW")
print("=" * 48)

for i, (before, after) in enumerate(dependencies, start=1):
    note = ""
    if (before, after) == ("T3", "T4"):
        note = " <-- REVIEW: initial familiarity; schedule overlaps"
    print(f"{i:02d}. {before} -> {after}{note}")

print("\nReview status: pending_final_review")
print("Approved for evaluation: No")
print("Files modified: None")

PLANPROOF AI — REAL_002 DEPENDENCY REVIEW
01. T1 -> T3
02. T2 -> T3
03. T3 -> T4 <-- REVIEW: initial familiarity; schedule overlaps
04. T3 -> T5
05. T4 -> T5
06. T5 -> T6
07. T6 -> T7
08. T7 -> T8
09. T8 -> T9
10. T9 -> T10

Review status: pending_final_review
Approved for evaluation: No
Files modified: None


In [ ]:
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
path = base / "case_002_dependency_review.json"

review = json.loads(path.read_text(encoding="utf-8"))

review["review_status"] = "pending_final_review"
review["dependency_interpretation_issues"] = [
    {
        "edge": ["T3", "T4"],
        "issue": (
            "Step 4 requires initial familiarity with Step 3, "
            "but the schedule permits overlap. A simple precedence "
            "edge may incorrectly imply that all of Step 3 must finish first."
        ),
        "decision": "retain_as_candidate_pending_semantic_review"
    }
]
review["approved_for_evaluation"] = False

path.write_text(
    json.dumps(review, indent=2),
    encoding="utf-8"
)

print("REAL_002 review note saved.")
print("Ambiguous candidate edges:", 1)
print("Status:", review["review_status"])
print("Approved for evaluation:", review["approved_for_evaluation"])

REAL_002 review note saved.
Ambiguous candidate edges: 1
Status: pending_final_review
Approved for evaluation: False


In [ ]:
import json
from pathlib import Path

path = Path(
    "/content/planproof_results/experiment65/"
    "case_001_dependency_review.json"
)

review = json.loads(path.read_text(encoding="utf-8"))
print(json.dumps(review, indent=2))

{
  "case_id": "REAL_001",
  "review_status": "provisional_manual_review",
  "original_dependency_count": 13,
  "reviewed_mandatory_dependency_count": 11,
  "reviewed_mandatory_dependencies": [
    [
      "T1",
      "T2"
    ],
    [
      "T1",
      "T3"
    ],
    [
      "T2",
      "T4"
    ],
    [
      "T2",
      "T5"
    ],
    [
      "T4",
      "T5"
    ],
    [
      "T3",
      "T6"
    ],
    [
      "T5",
      "T7"
    ],
    [
      "T6",
      "T7"
    ],
    [
      "T7",
      "T8"
    ],
    [
      "T4",
      "T8"
    ],
    [
      "T8",
      "T9"
    ]
  ],
  "conditional_dependency": {
    "edge": [
      "T5",
      "T6"
    ],
    "reason": "Frontend may use mocked data instead of functional APIs."
  },
  "soft_dependency": {
    "edge": [
      "T9",
      "T10"
    ],
    "reason": "Documentation may begin after Step 9; strict necessity is unclear."
  },
  "limitation": "This is a provisional interpretation of the plan, not independently verified grou

In [ ]:
import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")
path = base / "case_001_dependency_review.json"

review = json.loads(path.read_text(encoding="utf-8"))

review["review_status"] = "pending_final_review"
review["final_review_requirements"] = [
    "Confirm each mandatory dependency against the original plan wording.",
    "Keep T5 -> T6 conditional because mocked data is permitted.",
    "Clarify whether T9 -> T10 means documentation must wait for Step 9.",
    "Record the reviewer and rationale before approving evaluation use."
]
review["approved_for_evaluation"] = False

path.write_text(json.dumps(review, indent=2), encoding="utf-8")

print("REAL_001 final-review requirements saved.")
print("Mandatory dependency candidates:", len(
    review["reviewed_mandatory_dependencies"]
))
print("Conditional dependency:", review["conditional_dependency"]["edge"])
print("Soft dependency:", review["soft_dependency"]["edge"])
print("Status:", review["review_status"])
print("Approved for evaluation:", review["approved_for_evaluation"])

REAL_001 final-review requirements saved.
Mandatory dependency candidates: 11
Conditional dependency: ['T5', 'T6']
Soft dependency: ['T9', 'T10']
Status: pending_final_review
Approved for evaluation: False


In [ ]:

from pathlib import Path
import pandas as pd
import json

# ============================================
# PLANPROOF AI — MASTER COLLECTION WORKFLOW
# ============================================

BASE = Path("/content/planproof_results/experiment65")
CSV_PATH = BASE / "real_plan_dataset_template.csv"
OUT = BASE / "master_collection"
OUT.mkdir(parents=True, exist_ok=True)

TARGET_TOTAL = 30

DOMAINS = [
    "software_development",
    "university_study",
    "research",
    "business",
    "data_science",
    "career_planning",
    "personal_productivity",
    "event_planning",
    "education",
    "health_fitness"
]

# One target slot per domain, repeated three times = 30 slots.
domain_slots = [
    domain
    for _ in range(3)
    for domain in DOMAINS
]

# Load existing dataset; never overwrite original plan text.
df = pd.read_csv(CSV_PATH)

required_columns = [
    "case_id", "source_model", "collection_date", "prompt",
    "original_plan_text", "required_tasks_json",
    "precedence_constraints_json", "task_order_json",
    "domain", "independent_validation_notes",
    "included_in_evaluation"
]

for col in required_columns:
    if col not in df.columns:
        df[col] = None

# Keep all collected records excluded until final review.
df["included_in_evaluation"] = False
df.to_csv(CSV_PATH, index=False)

# Batch-import template: paste only genuine AI responses here later.
batch_template = pd.DataFrame(columns=[
    "case_id",
    "source_model",
    "collection_date",
    "prompt",
    "original_plan_text",
    "domain"
])
batch_path = OUT / "batch_import_template.csv"
batch_template.to_csv(batch_path, index=False)

# Domain coverage tracker.
counts = (
    df["domain"].fillna("unassigned")
    .value_counts()
    .to_dict()
)

coverage = []
for domain in DOMAINS:
    coverage.append({
        "domain": domain,
        "target_slots": 3,
        "currently_collected": int(counts.get(domain, 0)),
        "remaining_to_target": max(
            0, 3 - int(counts.get(domain, 0))
        )
    })

coverage_df = pd.DataFrame(coverage)
coverage_path = OUT / "domain_coverage.csv"
coverage_df.to_csv(coverage_path, index=False)

# Dataset audit.
audit_rows = []
for _, row in df.iterrows():
    def parse_list(value):
        try:
            result = json.loads(value) if pd.notna(value) else []
            return result if isinstance(result, list) else []
        except (ValueError, TypeError):
            return []

    case_id = str(row["case_id"])
    review_candidates = [
        BASE / f"case_{case_id[-3:]}_dependency_review.json",
        BASE / "reviews" / f"{case_id}_review.json"
    ]
    review_exists = any(p.exists() for p in review_candidates)

    original = row["original_plan_text"]
    audit_rows.append({
        "case_id": case_id,
        "domain": row["domain"],
        "original_text_preserved": (
            isinstance(original, str) and bool(original.strip())
        ),
        "task_count": len(parse_list(row["task_order_json"])),
        "dependency_count": len(
            parse_list(row["precedence_constraints_json"])
        ),
        "review_file_found": review_exists,
        "approved_for_evaluation": False
    })

audit_df = pd.DataFrame(audit_rows)
audit_path = OUT / "collection_audit.csv"
audit_df.to_csv(audit_path, index=False)

summary = {
    "target_total": TARGET_TOTAL,
    "currently_collected": int(len(df)),
    "remaining_to_collect": max(0, TARGET_TOTAL - len(df)),
    "approved_for_evaluation": 0,
    "domain_count": len(DOMAINS),
    "domain_slots": len(domain_slots),
    "real_responses_fabricated": False,
    "automatic_approval_enabled": False,
    "batch_template": str(batch_path),
    "domain_coverage": str(coverage_path),
    "collection_audit": str(audit_path)
}

summary_path = OUT / "master_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("PLANPROOF AI — MASTER COLLECTION WORKFLOW")
print("=" * 48)
print("Existing records:", len(df))
print("Target records:", TARGET_TOTAL)
print("Remaining genuine AI plans to collect:",
      summary["remaining_to_collect"])
print("Approved for evaluation: 0")
print("\nDOMAIN COVERAGE:")
print(coverage_df.to_string(index=False))
print("\nFILES CREATED:")
print(batch_path)
print(coverage_path)
print(audit_path)
print(summary_path)
print("\nOriginal plan text was not regenerated.")
print("No plans were fabricated or automatically approved.")


PLANPROOF AI — MASTER COLLECTION WORKFLOW
Existing records: 2
Target records: 30
Remaining genuine AI plans to collect: 28
Approved for evaluation: 0

DOMAIN COVERAGE:
               domain  target_slots  currently_collected  remaining_to_target
 software_development             3                    0                    3
     university_study             3                    0                    3
             research             3                    0                    3
             business             3                    0                    3
         data_science             3                    0                    3
      career_planning             3                    0                    3
personal_productivity             3                    0                    3
       event_planning             3                    0                    3
            education             3                    0                    3
       health_fitness             3                 

In [ ]:
import pandas as pd
from pathlib import Path

path = Path(
    "/content/planproof_results/experiment65/"
    "master_collection/batch_import_template.csv"
)

template = pd.read_csv(path)

print("PLANPROOF AI — BATCH IMPORT TEMPLATE")
print("=" * 42)
print("Columns:")
for column in template.columns:
    print("-", column)

print("\nCurrent rows:", len(template))
print("\nFile:", path)
print("\nNo dataset records were modified.")

PLANPROOF AI — BATCH IMPORT TEMPLATE
Columns:
- case_id
- source_model
- collection_date
- prompt
- original_plan_text
- domain

Current rows: 0

File: /content/planproof_results/experiment65/master_collection/batch_import_template.csv

No dataset records were modified.


In [ ]:

%pip -q install -U google-genai

from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it in Colab Secrets "
        "and enable notebook access."
    )

client = genai.Client(api_key=api_key)

print("Gemini API authentication initialized.")
print("Checking available Gemini models...")

models = list(client.models.list())
available = [
    getattr(model, "name", "")
    for model in models
    if "gemini" in getattr(model, "name", "").lower()
]

for name in available[:20]:
    print("-", name)

print("\nGemini models listed:", len(available))
print("API key value was not displayed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.
Gemini API authentication initialized.
Checking available Gemini models...
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flas

In [ ]:

from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

model_name = "gemini-2.5-flash-lite"

test_prompt = """
Create a tiny test project plan with exactly 3 tasks.
Use task IDs T1, T2, and T3.
State each task's dependencies explicitly.
Return plain text with task descriptions.
Do not add any extra tasks.
"""

try:
    response = client.models.generate_content(
        model=model_name,
        contents=test_prompt
    )

    print("MODEL:", model_name)
    print("TEST STATUS: SUCCESS")
    print("\nGENERATED RESPONSE:\n")
    print(response.text)

except Exception as exc:
    print("TEST STATUS: FAILED")
    print("Error type:", type(exc).__name__)
    print("Error details:", str(exc)[:1200])


TEST STATUS: FAILED
Error type: ClientError
Error details: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash-lite is no longer available to new users. Please update your code to use models/gemini-3.5-flash-lite for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}


In [ ]:

from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

model_name = "gemini-3.5-flash-lite"

test_prompt = """
Create a small project plan with exactly 3 tasks.
Use IDs T1, T2, T3.
State the dependencies for each task.
Return plain text only.
"""

try:
    response = client.interactions.create(
        model=model_name,
        input=test_prompt
    )

    print("TEST STATUS: SUCCESS")
    print("Requested model:", model_name)
    print("\nRESPONSE:\n")
    print(response.output_text)

except Exception as exc:
    print("TEST STATUS: FAILED")
    print("Error type:", type(exc).__name__)
    print("Error details:", str(exc)[:1200])


TEST STATUS: SUCCESS
Requested model: gemini-3.5-flash-lite

RESPONSE:

ID: T1
Name: Requirements Gathering
Dependencies: None

ID: T2
Name: System Design
Dependencies: T1

ID: T3
Name: Implementation
Dependencies: T2


In [ ]:

import json
import time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from google import genai
from google.colab import userdata

# ==========================================
# PLANPROOF AI — RESUMABLE PLAN GENERATOR
# ==========================================

BASE = Path("/content/planproof_results/experiment65")
OUT = BASE / "gemini_generation"
OUT.mkdir(parents=True, exist_ok=True)

MODEL = "gemini-3.5-flash-lite"
TARGET_IDS = [f"REAL_{i:03d}" for i in range(3, 31)]

RAW_JSONL = OUT / "raw_responses.jsonl"
STATUS_CSV = OUT / "generation_status.csv"

DOMAINS = [
    ("software_development",
     "develop and launch a small web application"),
    ("university_study",
     "prepare for five university examinations over 21 days"),
    ("research",
     "conduct a small empirical research study"),
    ("business",
     "launch a small online business"),
    ("data_science",
     "complete a data science analysis project"),
    ("career_planning",
     "prepare and apply for entry-level technical jobs"),
    ("personal_productivity",
     "organize and complete a month-long personal goal"),
    ("event_planning",
     "organize a small academic conference"),
    ("education",
     "design and deliver a short online course"),
    ("health_fitness",
     "organize a general four-week fitness routine")
]

# Load successful prior responses for safe resumption.
completed = {}
if RAW_JSONL.exists():
    with RAW_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                completed[item["case_id"]] = item

# Never overwrite an existing case in the main dataset.
main_csv = BASE / "real_plan_dataset_template.csv"
if main_csv.exists():
    existing_df = pd.read_csv(main_csv)
    existing_ids = set(existing_df["case_id"].astype(str))
else:
    existing_ids = set()

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

status_rows = []
for case_id in TARGET_IDS:
    if case_id in completed:
        status_rows.append({
            "case_id": case_id,
            "status": "already_saved",
            "domain": completed[case_id]["domain"]
        })
    elif case_id in existing_ids:
        status_rows.append({
            "case_id": case_id,
            "status": "already_in_main_dataset_not_regenerated",
            "domain": ""
        })

print("MODEL:", MODEL)
print("Target new plans:", len(TARGET_IDS))
print("Already checkpointed:", len(completed))
print("Existing main-dataset IDs protected:", len(existing_ids))

# Generate one response at a time and checkpoint immediately.
for index, case_id in enumerate(TARGET_IDS):
    if case_id in completed or case_id in existing_ids:
        continue

    domain, scenario = DOMAINS[index % len(DOMAINS)]
    prompt = f"""
Create a realistic, detailed 10-task plan to {scenario}.

Research dataset requirements:
- Use unique task IDs T1 through T10.
- Include each task's name, execution order, and description.
- State direct prerequisites explicitly for every task.
- Identify parallel, conditional, and ambiguous dependencies separately.
- Do not invent a dependency merely because one task has a higher number.
- Explain any conditional relationship in plain language.
- Return the complete plan in readable Markdown.
- Do not claim the plan has been independently validated.
"""

    record = {
        "case_id": case_id,
        "domain": domain,
        "source_provider": "Google Gemini API",
        "source_model_requested": MODEL,
        "collection_timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "prompt": prompt,
        "status": "pending",
        "original_plan_text": None
    }

    try:
        response = client.interactions.create(
            model=MODEL,
            input=prompt
        )

        raw_text = response.output_text
        if not raw_text or not raw_text.strip():
            raise ValueError("API returned an empty text response.")

        record["original_plan_text"] = raw_text
        record["status"] = "generated"
        record["response_characters"] = len(raw_text)

        # Append and flush immediately; never regenerate saved IDs.
        with RAW_JSONL.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()

        completed[case_id] = record
        print(
            f"[SAVED] {case_id} | {domain} | "
            f"{len(raw_text)} characters"
        )

    except Exception as exc:
        print(f"\n[STOPPED] {case_id}")
        print("Error:", type(exc).__name__, str(exc)[:700])
        print("Successful responses remain checkpointed.")
        print("Fix the issue before rerunning; no automatic retries.")
        break

    # Brief pacing between successful calls.
    time.sleep(1)

# Save a status snapshot; raw responses remain in JSONL.
status_rows = []
for case_id in TARGET_IDS:
    if case_id in completed:
        status_rows.append({
            "case_id": case_id,
            "domain": completed[case_id]["domain"],
            "status": "generated",
            "response_characters": len(
                completed[case_id]["original_plan_text"]
            )
        })
    elif case_id in existing_ids:
        status_rows.append({
            "case_id": case_id,
            "domain": "",
            "status": "already_in_main_dataset_not_regenerated",
            "response_characters": 0
        })
    else:
        status_rows.append({
            "case_id": case_id,
            "domain": DOMAINS[
                TARGET_IDS.index(case_id) % len(DOMAINS)
            ][0],
            "status": "not_generated",
            "response_characters": 0
        })

pd.DataFrame(status_rows).to_csv(STATUS_CSV, index=False)

print("\nGENERATION SUMMARY")
print("==================")
print("Checkpointed plans:", len(completed))
print("Target:", len(TARGET_IDS))
print("Raw responses:", RAW_JSONL)
print("Status report:", STATUS_CSV)
print("Main dataset modified: No")
print("Automatically approved: 0")


MODEL: gemini-3.5-flash-lite
Target new plans: 28
Already checkpointed: 0
Existing main-dataset IDs protected: 2
[SAVED] REAL_003 | software_development | 5395 characters
[SAVED] REAL_004 | university_study | 4665 characters
[SAVED] REAL_005 | research | 7108 characters
[SAVED] REAL_006 | business | 6456 characters
[SAVED] REAL_007 | data_science | 5713 characters
[SAVED] REAL_008 | career_planning | 5086 characters
[SAVED] REAL_009 | personal_productivity | 3977 characters
[SAVED] REAL_010 | event_planning | 4705 characters
[SAVED] REAL_011 | education | 5577 characters
[SAVED] REAL_012 | health_fitness | 4272 characters
[SAVED] REAL_013 | software_development | 5843 characters
[SAVED] REAL_014 | university_study | 4756 characters
[SAVED] REAL_015 | research | 5079 characters
[SAVED] REAL_016 | business | 4665 characters
[SAVED] REAL_017 | data_science | 5825 characters
[SAVED] REAL_018 | career_planning | 5244 characters
[SAVED] REAL_019 | personal_productivity | 4667 characters
[SAV

In [ ]:

import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from google import genai
from google.colab import userdata

BASE = Path("/content/planproof_results/experiment65")
RAW_PATH = BASE / "gemini_generation" / "raw_responses.jsonl"
OUT = BASE / "gemini_extraction"
OUT.mkdir(parents=True, exist_ok=True)

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"

# Load the already saved raw responses.
records = {}
with RAW_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            records[item["case_id"]] = item

print("Raw plans available:", len(records))

# Resume safely from previously completed extractions.
checkpoint = OUT / "extractions.jsonl"
extracted = {}

if checkpoint.exists():
    with checkpoint.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                if item.get("status") == "extracted":
                    extracted[item["case_id"]] = item

print("Already extracted:", len(extracted))

for case_id, record in records.items():
    if case_id in extracted:
        continue

    prompt = f"""
Extract a structured representation of this plan.
Do not invent tasks or dependencies.
Use only task IDs that appear in the original plan.
Treat explicit dependencies as candidate edges, not verified ground truth.
Keep conditional, ambiguous and parallel relationships separate.

Return ONLY valid JSON with this schema:
{{
  "tasks": [
    {{"task_id":"T1","name":"...","description":"...",
      "execution_order":1}}
  ],
  "task_order": ["T1"],
  "explicit_dependencies": [
    {{"before":"T1","after":"T2","evidence":"...",
      "type":"mandatory"}}
  ],
  "conditional_dependencies": [],
  "ambiguous_dependencies": [],
  "parallel_relationships": [],
  "warnings": []
}}

For dependency type, use "mandatory" only when the text clearly
states a prerequisite. Otherwise use conditional or ambiguous.
Do not infer a mandatory edge from numbering alone.

ORIGINAL PLAN:
{record["original_plan_text"]}
"""

    try:
        response = client.interactions.create(
            model=MODEL,
            input=prompt
        )
        text = response.output_text.strip()

        # Remove optional Markdown fences before JSON parsing.
        if text.startswith("```"):
            text = text.split("\n", 1)[1]
            text = text.rsplit("```", 1)[0].strip()

        data = json.loads(text)

        task_ids = [
            t.get("task_id") for t in data.get("tasks", [])
        ]
        known_ids = set(task_ids)

        errors = []
        if len(task_ids) != 10:
            errors.append(f"Expected 10 tasks; extracted {len(task_ids)}")
        if len(known_ids) != len(task_ids):
            errors.append("Duplicate task IDs detected")
        if any(not tid for tid in task_ids):
            errors.append("Missing task ID")
        if set(data.get("task_order", [])) != known_ids:
            errors.append("Task order and extracted task IDs differ")

        for edge in data.get("explicit_dependencies", []):
            if (edge.get("before") not in known_ids
                    or edge.get("after") not in known_ids):
                errors.append("Dependency references an unknown task ID")

        result = {
            "case_id": case_id,
            "domain": record["domain"],
            "source_model": record["source_model_requested"],
            "extraction_model_requested": MODEL,
            "extraction_timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "status": "extracted",
            "structural_warnings": errors,
            "extraction": data
        }

        with checkpoint.open("a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            f.flush()

        extracted[case_id] = result
        print(
            f"[EXTRACTED] {case_id} | tasks={len(task_ids)} | "
            f"warnings={len(errors)}"
        )

    except Exception as exc:
        print(f"[STOPPED] {case_id}: {type(exc).__name__}: {str(exc)[:400]}")
        print("Successful extractions are checkpointed.")
        print("Fix the issue before rerunning; no automatic retries.")
        break

print("\nEXTRACTION SUMMARY")
print("==================")
print("Raw plans:", len(records))
print("Successfully extracted:", len(extracted))
print("Remaining:", len(records) - len(extracted))
print("Checkpoint:", checkpoint)
print("Evaluation approval performed: No")


Raw plans available: 28
Already extracted: 0
[EXTRACTED] REAL_003 | tasks=10 | warnings=0
[EXTRACTED] REAL_004 | tasks=10 | warnings=0
[EXTRACTED] REAL_005 | tasks=10 | warnings=0
[EXTRACTED] REAL_006 | tasks=10 | warnings=0
[EXTRACTED] REAL_007 | tasks=10 | warnings=0
[EXTRACTED] REAL_008 | tasks=10 | warnings=0
[EXTRACTED] REAL_009 | tasks=10 | warnings=0
[EXTRACTED] REAL_010 | tasks=10 | warnings=0
[EXTRACTED] REAL_011 | tasks=10 | warnings=0
[EXTRACTED] REAL_012 | tasks=10 | warnings=0
[EXTRACTED] REAL_013 | tasks=10 | warnings=0
[EXTRACTED] REAL_014 | tasks=10 | warnings=0
[EXTRACTED] REAL_015 | tasks=10 | warnings=0
[EXTRACTED] REAL_016 | tasks=10 | warnings=0
[EXTRACTED] REAL_017 | tasks=10 | warnings=0
[EXTRACTED] REAL_018 | tasks=10 | warnings=0
[EXTRACTED] REAL_019 | tasks=10 | warnings=0
[EXTRACTED] REAL_020 | tasks=10 | warnings=0
[EXTRACTED] REAL_021 | tasks=10 | warnings=0
[EXTRACTED] REAL_022 | tasks=10 | warnings=0
[EXTRACTED] REAL_023 | tasks=10 | warnings=0
[EXTRACTED

In [ ]:

import json
from pathlib import Path
import pandas as pd

BASE = Path("/content/planproof_results/experiment65")
SOURCE = BASE / "gemini_extraction" / "extractions.jsonl"
OUT = BASE / "gemini_extraction" / "master_audit"
OUT.mkdir(parents=True, exist_ok=True)

records = {}
with SOURCE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            records[row["case_id"]] = row

audit_rows = []
dependency_rows = []

for case_id, record in sorted(records.items()):
    data = record["extraction"]
    tasks = data.get("tasks", [])
    task_ids = [t.get("task_id") for t in tasks]
    known = set(task_ids)
    edges = data.get("explicit_dependencies", [])
    issues = list(record.get("structural_warnings", []))

    if len(task_ids) != 10:
        issues.append(f"Expected 10 tasks, found {len(task_ids)}")
    if len(known) != len(task_ids):
        issues.append("Duplicate task IDs")
    if any(not t for t in task_ids):
        issues.append("Missing task ID")

    order = data.get("task_order", [])
    if len(order) != len(set(order)):
        issues.append("Duplicate IDs in task order")
    if set(order) != known:
        issues.append("Task order does not match task IDs")

    graph = {t: [] for t in known}
    indegree = {t: 0 for t in known}
    seen_edges = set()

    for edge in edges:
        u, v = edge.get("before"), edge.get("after")
        edge_type = edge.get("type", "unspecified")

        dependency_rows.append({
            "case_id": case_id,
            "before": u,
            "after": v,
            "type": edge_type,
            "evidence": edge.get("evidence", "")
        })

        if u not in known or v not in known:
            issues.append(f"Unknown task reference: {u} -> {v}")
            continue
        if u == v:
            issues.append(f"Self-dependency: {u} -> {v}")
            continue
        if (u, v) in seen_edges:
            issues.append(f"Duplicate dependency: {u} -> {v}")
            continue

        seen_edges.add((u, v))
        graph[u].append(v)
        indegree[v] += 1

    # Kahn's algorithm: detect directed cycles.
    queue = [t for t in known if indegree[t] == 0]
    visited = 0
    while queue:
        u = queue.pop()
        visited += 1
        for v in graph[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)

    has_cycle = visited != len(known)
    if has_cycle:
        issues.append("Cycle detected in explicit dependency graph")

    audit_rows.append({
        "case_id": case_id,
        "domain": record.get("domain", ""),
        "task_count": len(tasks),
        "mandatory_candidate_edges": len(edges),
        "conditional_edges": len(data.get("conditional_dependencies", [])),
        "ambiguous_edges": len(data.get("ambiguous_dependencies", [])),
        "parallel_relationships": len(data.get("parallel_relationships", [])),
        "cycle_detected": has_cycle,
        "structural_audit_pass": len(issues) == 0,
        "issues": " | ".join(issues),
        "approved_for_evaluation": False
    })

audit = pd.DataFrame(audit_rows)
deps = pd.DataFrame(dependency_rows)

audit.to_csv(OUT / "plan_audit.csv", index=False)
deps.to_csv(OUT / "dependency_inventory.csv", index=False)

print("PLANPROOF AI — MASTER STRUCTURAL AUDIT")
print("=" * 46)
print("Plans audited:", len(audit))
print("Plans with structural checks passing:",
      int(audit["structural_audit_pass"].sum()))
print("Plans with issues:",
      int((~audit["structural_audit_pass"]).sum()))
print("Total extracted tasks:", int(audit["task_count"].sum()))
print("Explicit dependency candidates:", len(deps))
print("Cycles detected:", int(audit["cycle_detected"].sum()))
print("Approved for evaluation: 0")
print("\nAUDIT FILE:", OUT / "plan_audit.csv")
print("DEPENDENCY FILE:", OUT / "dependency_inventory.csv")
print("\nISSUES (if any):")
print(audit.loc[
    ~audit["structural_audit_pass"],
    ["case_id", "issues"]
].to_string(index=False))


PLANPROOF AI — MASTER STRUCTURAL AUDIT
Plans audited: 28
Plans with structural checks passing: 28
Plans with issues: 0
Total extracted tasks: 280
Explicit dependency candidates: 337
Cycles detected: 0
Approved for evaluation: 0

AUDIT FILE: /content/planproof_results/experiment65/gemini_extraction/master_audit/plan_audit.csv
DEPENDENCY FILE: /content/planproof_results/experiment65/gemini_extraction/master_audit/dependency_inventory.csv

ISSUES (if any):
Empty DataFrame
Columns: [case_id, issues]
Index: []


In [ ]:

import json
import random
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

BASE = Path("/content/planproof_results/experiment65")
SOURCE = BASE / "gemini_extraction" / "extractions.jsonl"
OUT = BASE / "controlled_corruption"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 20261065
rng = random.Random(SEED)

# Load original extracted plans.
plans = {}
with SOURCE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            plans[row["case_id"]] = row

assert len(plans) == 28, (
    f"Expected 28 extracted plans; found {len(plans)}"
)

# Load existing output IDs if this cell is rerun.
cases_path = OUT / "corrupted_cases.jsonl"
manifest_path = OUT / "corruption_manifest.csv"

existing_ids = set()
if cases_path.exists():
    with cases_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                existing_ids.add(json.loads(line)["test_case_id"])

new_cases = []
manifest = []

def save_case(case_id, domain, defect, original_order,
              corrupted_order, required_tasks, edges,
              operation, seed):
    test_id = f"{case_id}__{defect}"

    if test_id in existing_ids:
        return

    item = {
        "test_case_id": test_id,
        "source_case_id": case_id,
        "domain": domain,
        "defect_type": defect,
        "original_order": list(original_order),
        "corrupted_order": list(corrupted_order),
        "required_tasks": list(required_tasks),
        "precedence_constraints": [list(e) for e in edges],
        "corruption_operation": operation,
        "random_seed": seed,
        "ground_truth_status": "provisional_constraints",
        "approved_for_evaluation": False
    }
    new_cases.append(item)

    manifest.append({
        "test_case_id": test_id,
        "source_case_id": case_id,
        "domain": domain,
        "defect_type": defect,
        "operation": operation,
        "original_task_count": len(original_order),
        "corrupted_task_count": len(corrupted_order),
        "random_seed": seed
    })

for case_index, (case_id, record) in enumerate(sorted(plans.items())):
    data = record["extraction"]
    original = list(data["task_order"])
    required = list(original)

    edges = [
        [e["before"], e["after"]]
        for e in data.get("explicit_dependencies", [])
        if e.get("before") in required
        and e.get("after") in required
    ]

    if len(original) < 3:
        print("Skipping too-small plan:", case_id)
        continue

    # 1. Missing required task.
    missing_rng = random.Random(SEED + case_index * 10 + 1)
    missing_task = missing_rng.choice(required)
    missing_order = list(original)
    missing_order.remove(missing_task)

    save_case(
        case_id, record["domain"], "missing_task",
        original, missing_order, required, edges,
        f"Removed required task {missing_task}",
        SEED + case_index * 10 + 1
    )

    # 2. Duplicate task occurrence.
    duplicate_rng = random.Random(SEED + case_index * 10 + 2)
    duplicate_task = duplicate_rng.choice(original)
    duplicate_order = list(original)
    insert_at = duplicate_rng.randrange(len(duplicate_order) + 1)
    duplicate_order.insert(insert_at, duplicate_task)

    save_case(
        case_id, record["domain"], "duplicate_task",
        original, duplicate_order, required, edges,
        f"Inserted an extra occurrence of {duplicate_task}",
        SEED + case_index * 10 + 2
    )

    # 3. Unknown task ID.
    unknown_rng = random.Random(SEED + case_index * 10 + 3)
    unknown_order = list(original)
    unknown_task = f"UNKNOWN_{case_id[-3:]}"
    insert_at = unknown_rng.randrange(len(unknown_order) + 1)
    unknown_order.insert(insert_at, unknown_task)

    save_case(
        case_id, record["domain"], "unknown_task",
        original, unknown_order, required, edges,
        f"Inserted unknown task {unknown_task}",
        SEED + case_index * 10 + 3
    )

    # 4. Precedence-order violation: reverse a valid edge's endpoints.
    # If no edges exist, do not fabricate a precedence violation.
    valid_edges = [
        (u, v) for u, v in edges
        if u in original and v in original and u != v
    ]

    if valid_edges:
        order_rng = random.Random(SEED + case_index * 10 + 4)
        u, v = order_rng.choice(valid_edges)
        order_order = list(original)

        # Swap endpoints only when this actually reverses their order.
        iu, iv = order_order.index(u), order_order.index(v)
        if iu < iv:
            order_order[iu], order_order[iv] = (
                order_order[iv], order_order[iu]
            )
            operation = f"Swapped {u} and {v} to violate edge {u}->{v}"
        else:
            # Move v before u explicitly.
            order_order.remove(v)
            order_order.insert(order_order.index(u), v)
            operation = f"Moved {v} before {u} to violate edge {u}->{v}"

        save_case(
            case_id, record["domain"], "precedence_violation",
            original, order_order, required, edges,
            operation, SEED + case_index * 10 + 4
        )

# Append new cases only; do not overwrite previous cases.
if new_cases:
    with cases_path.open("a", encoding="utf-8") as f:
        for item in new_cases:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    new_manifest = pd.DataFrame(manifest)
    if manifest_path.exists():
        old_manifest = pd.read_csv(manifest_path)
        new_manifest = pd.concat(
            [old_manifest, new_manifest], ignore_index=True
        )
    new_manifest.to_csv(manifest_path, index=False)

# Read all saved cases for a reproducible summary.
all_cases = []
if cases_path.exists():
    with cases_path.open("r", encoding="utf-8") as f:
        all_cases = [
            json.loads(line) for line in f if line.strip()
        ]

summary = {
    "source_plan_count": len(plans),
    "corrupted_case_count": len(all_cases),
    "new_cases_this_run": len(new_cases),
    "defect_counts": pd.Series(
        [x["defect_type"] for x in all_cases]
    ).value_counts().to_dict(),
    "random_seed": SEED,
    "ground_truth_status": "provisional_constraints",
    "approved_for_evaluation": 0,
    "api_calls_made": 0
}

summary_path = OUT / "corruption_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("PLANPROOF AI — CONTROLLED CORRUPTION")
print("=" * 44)
print("Source plans:", len(plans))
print("New test cases:", len(new_cases))
print("Total saved test cases:", len(all_cases))
print("Defect counts:", summary["defect_counts"])
print("Random seed:", SEED)
print("API calls made: 0")
print("Approved for evaluation: 0")
print("\nCases:", cases_path)
print("Manifest:", manifest_path)
print("Summary:", summary_path)


PLANPROOF AI — CONTROLLED CORRUPTION
Source plans: 28
New test cases: 112
Total saved test cases: 112
Defect counts: {'missing_task': 28, 'duplicate_task': 28, 'unknown_task': 28, 'precedence_violation': 28}
Random seed: 20261065
API calls made: 0
Approved for evaluation: 0

Cases: /content/planproof_results/experiment65/controlled_corruption/corrupted_cases.jsonl
Manifest: /content/planproof_results/experiment65/controlled_corruption/corruption_manifest.csv
Summary: /content/planproof_results/experiment65/controlled_corruption/corruption_summary.json


In [ ]:

import json
from pathlib import Path
from collections import Counter

BASE = Path("/content/planproof_results/experiment65")
CORRUPT = BASE / "controlled_corruption"
SOURCE = BASE / "gemini_extraction" / "extractions.jsonl"

# Load source plans and corrupted test cases
sources = {}
with open(SOURCE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            sources[record["case_id"]] = record

cases = []
with open(CORRUPT / "corrupted_cases.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            cases.append(json.loads(line))

def get_case_id(case):
    return (
        case.get("source_case_id")
        or case.get("original_case_id")
        or case.get("case_id")
    )

def get_order(case):
    for key in ("corrupted_task_order", "task_order", "plan_order"):
        if isinstance(case.get(key), list):
            return case[key]
    return None

def get_required(case, source):
    for key in ("required_tasks", "required_task_ids"):
        if isinstance(case.get(key), list):
            return case[key]
    extraction = source.get("extraction", {})
    return [
        t.get("task_id", t.get("id"))
        for t in extraction.get("tasks", [])
        if isinstance(t, dict) and t.get("task_id", t.get("id")) is not None
    ]

def get_edges(source):
    extraction = source.get("extraction", {})
    edges = extraction.get("explicit_dependencies", [])
    return [
        (e.get("before"), e.get("after"))
        for e in edges
        if e.get("before") is not None and e.get("after") is not None
    ]

def first_present(obj, keys):
    for key in keys:
        if key in obj:
            return obj[key]
    return None

results = []
for case in cases:
    cid = get_case_id(case)
    source = sources.get(cid)
    order = get_order(case)

    # Some schemas store the source ID under a different field.
    if source is None:
        source_id = first_present(
            case, ["source_case_id", "original_case_id", "parent_case_id"]
        )
        source = sources.get(source_id)

    defect = first_present(case, ["defect_type", "corruption_type", "defect"])
    if isinstance(defect, str):
        defect = defect.lower()

    if source is None or order is None:
        results.append({
            "test_case_id": case.get("test_case_id", case.get("case_id")),
            "defect": defect,
            "pass": False,
            "reason": "Could not locate source plan or corrupted task order"
        })
        continue

    required = get_required(case, source)
    edges = get_edges(source)
    counts = Counter(order)
    unknown = [x for x in order if x not in required]
    missing = [x for x in required if counts[x] == 0]
    duplicates = [x for x, n in counts.items() if n > 1]
    positions = {task: i for i, task in enumerate(order)}
    violated = [
        (a, b) for a, b in edges
        if a in positions and b in positions and positions[a] >= positions[b]
    ]

    # Verify the intended defect, not merely the presence of any defect.
    if defect == "missing_task":
        passed = bool(missing)
        reason = f"Missing tasks: {missing}"
    elif defect == "duplicate_task":
        passed = bool(duplicates)
        reason = f"Duplicated tasks: {duplicates}"
    elif defect == "unknown_task":
        passed = bool(unknown)
        reason = f"Unknown tasks: {unknown}"
    elif defect == "precedence_violation":
        passed = bool(violated)
        reason = f"Violated edges: {violated}"
    else:
        passed = False
        reason = "Unrecognized defect label; inspect case schema"

    results.append({
        "test_case_id": case.get("test_case_id", case.get("case_id")),
        "source_case_id": cid,
        "defect": defect,
        "pass": passed,
        "reason": reason
    })

passed = sum(r["pass"] for r in results)
failed = len(results) - passed
by_defect = {}
for r in results:
    key = str(r["defect"])
    by_defect.setdefault(key, {"passed": 0, "failed": 0})
    by_defect[key]["passed" if r["pass"] else "failed"] += 1

audit_dir = CORRUPT / "audit"
audit_dir.mkdir(parents=True, exist_ok=True)
with open(audit_dir / "corruption_audit.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("PLANPROOF AI — CORRUPTION AUDIT")
print("=" * 42)
print(f"Cases audited: {len(results)}")
print(f"Intended defects verified: {passed}")
print(f"Cases needing inspection: {failed}")
print("\nResults by defect:")
for defect, counts in sorted(by_defect.items()):
    print(f"  {defect}: {counts['passed']} passed, {counts['failed']} failed")

if failed:
    print("\nFirst 10 cases needing inspection:")
    for r in [x for x in results if not x["pass"]][:10]:
        print(r)

print(f"\nAudit saved to: {audit_dir / 'corruption_audit.json'}")
print("No API calls made. No source plans or main dataset modified.")


PLANPROOF AI — CORRUPTION AUDIT
Cases audited: 112
Intended defects verified: 0
Cases needing inspection: 112

Results by defect:
  duplicate_task: 0 passed, 28 failed
  missing_task: 0 passed, 28 failed
  precedence_violation: 0 passed, 28 failed
  unknown_task: 0 passed, 28 failed

First 10 cases needing inspection:
{'test_case_id': 'REAL_003__missing_task', 'defect': 'missing_task', 'pass': False, 'reason': 'Could not locate source plan or corrupted task order'}
{'test_case_id': 'REAL_003__duplicate_task', 'defect': 'duplicate_task', 'pass': False, 'reason': 'Could not locate source plan or corrupted task order'}
{'test_case_id': 'REAL_003__unknown_task', 'defect': 'unknown_task', 'pass': False, 'reason': 'Could not locate source plan or corrupted task order'}
{'test_case_id': 'REAL_003__precedence_violation', 'defect': 'precedence_violation', 'pass': False, 'reason': 'Could not locate source plan or corrupted task order'}
{'test_case_id': 'REAL_004__missing_task', 'defect': 'missin

In [ ]:

import json
from pathlib import Path

base = Path("/content/planproof_results/experiment65")

files_to_inspect = {
    "CORRUPTED CASE": (
        base / "controlled_corruption" / "corrupted_cases.jsonl"
    ),
    "SOURCE EXTRACTION": (
        base / "gemini_extraction" / "extractions.jsonl"
    ),
}

for label, path in files_to_inspect.items():
    print("\n" + "=" * 65)
    print(label)
    print("File exists:", path.exists())

    if not path.exists():
        continue

    with open(path, "r", encoding="utf-8") as f:
        first_line = next(
            (line for line in f if line.strip()), None
        )

    if first_line:
        record = json.loads(first_line)
        print("Top-level keys:", list(record.keys()))
        print("First record:")
        print(json.dumps(record, indent=2, ensure_ascii=False)[:7000])

print("\nInspection only — no files modified and no API calls made.")



CORRUPTED CASE
File exists: True
Top-level keys: ['test_case_id', 'source_case_id', 'domain', 'defect_type', 'original_order', 'corrupted_order', 'required_tasks', 'precedence_constraints', 'corruption_operation', 'random_seed', 'ground_truth_status', 'approved_for_evaluation']
First record:
{
  "test_case_id": "REAL_003__missing_task",
  "source_case_id": "REAL_003",
  "domain": "software_development",
  "defect_type": "missing_task",
  "original_order": [
    "T1",
    "T2",
    "T3",
    "T4",
    "T5",
    "T6",
    "T7",
    "T8",
    "T9",
    "T10"
  ],
  "corrupted_order": [
    "T1",
    "T2",
    "T4",
    "T5",
    "T6",
    "T7",
    "T8",
    "T9",
    "T10"
  ],
  "required_tasks": [
    "T1",
    "T2",
    "T3",
    "T4",
    "T5",
    "T6",
    "T7",
    "T8",
    "T9",
    "T10"
  ],
  "precedence_constraints": [
    [
      "T1",
      "T2"
    ],
    [
      "T1",
      "T3"
    ],
    [
      "T2",
      "T4"
    ],
    [
      "T3",
      "T5"
    ],
    [
      "

In [ ]:

import json
import csv
from pathlib import Path
from collections import Counter, defaultdict

BASE = Path("/content/planproof_results/experiment65")
CORRUPT = BASE / "controlled_corruption"
SOURCE = BASE / "gemini_extraction" / "extractions.jsonl"

with open(SOURCE, encoding="utf-8") as f:
    sources = {
        r["case_id"]: r
        for line in f if line.strip()
        for r in [json.loads(line)]
    }

with open(CORRUPT / "corrupted_cases.jsonl", encoding="utf-8") as f:
    cases = [json.loads(line) for line in f if line.strip()]

results = []

for c in cases:
    source = sources.get(c["source_case_id"])
    order = c.get("corrupted_order", [])
    original = c.get("original_order", [])
    required = c.get("required_tasks", [])
    edges = [tuple(e) for e in c.get("precedence_constraints", [])]
    counts = Counter(order)
    positions = {task: i for i, task in enumerate(order)}

    missing = [t for t in required if counts[t] == 0]
    duplicates = [t for t, n in counts.items() if n > 1]
    unknown = [t for t in order if t not in required]
    violated = [
        [a, b] for a, b in edges
        if a in positions and b in positions and positions[a] >= positions[b]
    ]

    defect = c.get("defect_type")
    checks = {
        "missing_task": bool(missing),
        "duplicate_task": bool(duplicates),
        "unknown_task": bool(unknown),
        "precedence_violation": bool(violated),
    }

    source_edges = set()
    if source:
        source_edges = {
            (e.get("before"), e.get("after"))
            for e in source.get("extraction", {}).get("explicit_dependencies", [])
            if e.get("before") is not None and e.get("after") is not None
        }

    results.append({
        "test_case_id": c.get("test_case_id"),
        "source_case_id": c.get("source_case_id"),
        "defect_type": defect,
        "intended_defect_verified": checks.get(defect, False),
        "original_order_preserved_in_record": isinstance(original, list) and len(original) > 0,
        "source_plan_found": source is not None,
        "constraint_count": len(edges),
        "constraints_match_source": set(edges) == source_edges,
        "missing_tasks": missing,
        "duplicate_tasks": duplicates,
        "unknown_tasks": unknown,
        "violated_edges": violated,
        "approved_for_evaluation": c.get("approved_for_evaluation", False),
    })

audit_dir = CORRUPT / "audit"
audit_dir.mkdir(parents=True, exist_ok=True)
audit_csv = audit_dir / "corruption_audit_corrected.csv"

with open(audit_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print("PLANPROOF AI — CORRECTED CORRUPTION AUDIT")
print("=" * 48)
print("Cases audited:", len(results))
print("Source plans found:", sum(r["source_plan_found"] for r in results), "/", len(results))
print("Intended defects verified:",
      sum(r["intended_defect_verified"] for r in results), "/", len(results))
print("Original order recorded:",
      sum(r["original_order_preserved_in_record"] for r in results), "/", len(results))
print("Constraints match source extraction:",
      sum(r["constraints_match_source"] for r in results), "/", len(results))
print("Approved cases:",
      sum(bool(r["approved_for_evaluation"]) for r in results))

print("\nResults by defect:")
for defect in sorted({r["defect_type"] for r in results}):
    group = [r for r in results if r["defect_type"] == defect]
    good = sum(r["intended_defect_verified"] for r in group)
    print(f"  {defect}: {good}/{len(group)} verified")

print("\nCases with unverified intended defects:")
for r in results:
    if not r["intended_defect_verified"]:
        print(r["test_case_id"], "->", r["defect_type"],
              "| missing:", r["missing_tasks"],
              "| duplicates:", r["duplicate_tasks"],
              "| unknown:", r["unknown_tasks"],
              "| violated edges:", r["violated_edges"])

print("\nAudit saved:", audit_csv)
print("No API calls made. No source plans or main dataset modified.")


PLANPROOF AI — CORRECTED CORRUPTION AUDIT
Cases audited: 112
Source plans found: 112 / 112
Intended defects verified: 112 / 112
Original order recorded: 112 / 112
Constraints match source extraction: 112 / 112
Approved cases: 0

Results by defect:
  duplicate_task: 28/28 verified
  missing_task: 28/28 verified
  precedence_violation: 28/28 verified
  unknown_task: 28/28 verified

Cases with unverified intended defects:

Audit saved: /content/planproof_results/experiment65/controlled_corruption/audit/corruption_audit_corrected.csv
No API calls made. No source plans or main dataset modified.


In [ ]:

import inspect

FUNCTION_NAMES = [
    "validate_plan",
    "repair_plan",
    "shared_task_order_change_rate",
    "full_plan_repair_cost",
    "exact_optimize_order_fair",
    "adaptive_repair_plan",
]

print("PLANPROOF AI — REPAIR PIPELINE READINESS")
print("=" * 48)

available = {}

for name in FUNCTION_NAMES:
    obj = globals().get(name)

    if callable(obj):
        available[name] = obj
        try:
            signature = inspect.signature(obj)
        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        print(f"\n[FOUND] {name}{signature}")

        try:
            source = inspect.getsource(obj)
            print(source[:1800])
            if len(source) > 1800:
                print("... [source truncated]")
        except (OSError, TypeError):
            print("Source unavailable in this runtime.")
    else:
        print(f"\n[MISSING] {name}")

print("\n" + "=" * 48)
print(f"Functions found: {len(available)}/{len(FUNCTION_NAMES)}")
print("No plans modified. No API calls made.")


PLANPROOF AI — REPAIR PIPELINE READINESS

[FOUND] validate_plan(plan)
def validate_plan(plan):
    order = plan["task_order"]
    required = plan["required_tasks"]
    constraints = plan["precedence_constraints"]

    violations = []

    if len(order) != len(set(order)):
        violations.append("Duplicate task IDs")

    order_set = set(order)
    required_set = set(required)

    missing = required_set - order_set
    unknown = order_set - required_set

    if missing:
        violations.append(f"Missing tasks: {sorted(missing)}")

    if unknown:
        violations.append(f"Unknown tasks: {sorted(unknown)}")

    positions = {}
    for i, task_id in enumerate(order):
        positions.setdefault(task_id, i)

    for before, after in constraints:
        if before not in positions or after not in positions:
            violations.append(
                f"Cannot check constraint {before} -> {after}: task missing"
            )
        elif positions[before] >= positions[after]:
   

In [ ]:

import json
import csv
import time
from pathlib import Path
from collections import Counter, defaultdict

BASE = Path("/content/planproof_results/experiment65")
INPUT = BASE / "controlled_corruption" / "corrupted_cases.jsonl"
OUT = BASE / "experiment66_repair_evaluation"
OUT.mkdir(parents=True, exist_ok=True)

with open(INPUT, "r", encoding="utf-8") as f:
    cases = [json.loads(line) for line in f if line.strip()]

results = []

for case in cases:
    plan = {
        "task_order": list(case["corrupted_order"]),
        "required_tasks": list(case["required_tasks"]),
        "precedence_constraints": list(case["precedence_constraints"]),
    }

    # Validate the corrupted input.
    before = validate_plan(plan)

    # Measure structural edits and original-order changes.
    edit_cost = full_plan_repair_cost(
        plan["task_order"], plan["required_tasks"]
    )

    start = time.perf_counter()
    try:
        repaired = repair_plan(plan)
        elapsed = time.perf_counter() - start

        success = (
            isinstance(repaired, dict)
            and repaired.get("success", False)
            and isinstance(repaired.get("repaired_plan"), dict)
        )

        if success:
            repaired_plan = repaired["repaired_plan"]
            repaired_order = repaired_plan.get("task_order", [])
            after = validate_plan({
                "task_order": repaired_order,
                "required_tasks": plan["required_tasks"],
                "precedence_constraints": plan["precedence_constraints"],
            })
            order_change = shared_task_order_change_rate(
                plan["task_order"], repaired_order
            )
        else:
            repaired_order = None
            after = {"valid": False, "violations": [repaired.get("message", "Repair failed")]}
            order_change = None

    except Exception as exc:
        elapsed = time.perf_counter() - start
        success = False
        repaired_order = None
        after = {"valid": False, "violations": [f"{type(exc).__name__}: {exc}"]}
        order_change = None

    results.append({
        "test_case_id": case["test_case_id"],
        "source_case_id": case["source_case_id"],
        "defect_type": case["defect_type"],
        "input_detected_invalid": not before["valid"],
        "input_violation_count": len(before["violations"]),
        "repair_success": success,
        "repaired_plan_valid": after["valid"],
        "remaining_violation_count": len(after["violations"]),
        "repair_method": "stable_repair",
        "order_change_rate": order_change,
        "missing_tasks_to_restore": edit_cost["missing_tasks_to_restore"],
        "duplicate_occurrences_to_remove": edit_cost["duplicate_occurrences_to_remove"],
        "unknown_occurrences_to_remove": edit_cost["unknown_occurrences_to_remove"],
        "structural_edit_count": edit_cost["structural_edit_count"],
        "elapsed_seconds": round(elapsed, 6),
        "repaired_order_json": json.dumps(repaired_order) if repaired_order is not None else "",
        "input_violations_json": json.dumps(before["violations"]),
        "remaining_violations_json": json.dumps(after["violations"]),
        "approved_for_evaluation": False,
    })

csv_path = OUT / "experiment66_repair_results.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

summary = {
    "experiment": "Experiment 66 — Stable Repair on Controlled Corruptions",
    "cases": len(results),
    "api_calls": 0,
    "approved_for_evaluation": 0,
    "input_detected_invalid": sum(r["input_detected_invalid"] for r in results),
    "repair_success": sum(r["repair_success"] for r in results),
    "valid_repairs": sum(r["repaired_plan_valid"] for r in results),
    "defect_summary": {},
}

for defect in sorted({r["defect_type"] for r in results}):
    group = [r for r in results if r["defect_type"] == defect]
    summary["defect_summary"][defect] = {
        "cases": len(group),
        "input_detected_invalid": sum(r["input_detected_invalid"] for r in group),
        "repair_success": sum(r["repair_success"] for r in group),
        "valid_repairs": sum(r["repaired_plan_valid"] for r in group),
        "mean_order_change_rate": (
            round(sum(r["order_change_rate"] for r in group if r["order_change_rate"] is not None)
                  / sum(r["order_change_rate"] is not None for r in group), 6)
            if any(r["order_change_rate"] is not None for r in group) else None
        ),
    }

with open(OUT / "experiment66_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("PLANPROOF AI — EXPERIMENT 66")
print("=" * 48)
print("Cases evaluated:", summary["cases"])
print("Invalid inputs detected:", summary["input_detected_invalid"], "/", len(results))
print("Repair calls successful:", summary["repair_success"], "/", len(results))
print("Repaired plans valid:", summary["valid_repairs"], "/", len(results))
print("\nResults by defect:")
for defect, s in summary["defect_summary"].items():
    print(
        f"  {defect}: detected {s['input_detected_invalid']}/{s['cases']}, "
        f"repair success {s['repair_success']}/{s['cases']}, "
        f"valid repaired {s['valid_repairs']}/{s['cases']}, "
        f"mean order-change rate={s['mean_order_change_rate']}"
    )

print("\nResults CSV:", csv_path)
print("Summary JSON:", OUT / "experiment66_summary.json")
print("API calls: 0 | Original dataset unchanged | Evaluation approvals: 0")


PLANPROOF AI — EXPERIMENT 66
Cases evaluated: 112
Invalid inputs detected: 112 / 112
Repair calls successful: 112 / 112
Repaired plans valid: 112 / 112

Results by defect:
  duplicate_task: detected 28/28, repair success 28/28, valid repaired 28/28, mean order-change rate=0.016667
  missing_task: detected 28/28, repair success 28/28, valid repaired 28/28, mean order-change rate=0.032738
  precedence_violation: detected 28/28, repair success 28/28, valid repaired 28/28, mean order-change rate=0.049206
  unknown_task: detected 28/28, repair success 28/28, valid repaired 28/28, mean order-change rate=0.0

Results CSV: /content/planproof_results/experiment65/experiment66_repair_evaluation/experiment66_repair_results.csv
Summary JSON: /content/planproof_results/experiment65/experiment66_repair_evaluation/experiment66_summary.json
API calls: 0 | Original dataset unchanged | Evaluation approvals: 0


In [ ]:

import json
import csv
import hashlib
from pathlib import Path
from collections import Counter

BASE = Path("/content/planproof_results/experiment65")
CASES_FILE = BASE / "controlled_corruption" / "corrupted_cases.jsonl"
RESULTS_FILE = (
    BASE / "experiment66_repair_evaluation"
    / "experiment66_repair_results.csv"
)
OUT = BASE / "experiment67_regression"
OUT.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Record a fingerprint before reading the original test set.
cases_hash_before = sha256_file(CASES_FILE)

with open(CASES_FILE, encoding="utf-8") as f:
    cases = [json.loads(line) for line in f if line.strip()]

with open(RESULTS_FILE, newline="", encoding="utf-8") as f:
    prior_results = {
        row["test_case_id"]: row
        for row in csv.DictReader(f)
    }

audit = []

for case in cases:
    cid = case["test_case_id"]
    result = prior_results.get(cid)

    required = case["required_tasks"]
    edges = [tuple(e) for e in case["precedence_constraints"]]
    original = case["corrupted_order"]

    if result is None or not result.get("repaired_order_json"):
        audit.append({
            "test_case_id": cid,
            "all_invariants_pass": False,
            "reason": "Missing repair result or repaired order"
        })
        continue

    repaired_order = json.loads(result["repaired_order_json"])
    counts = Counter(repaired_order)
    required_set = set(required)

    missing = sorted(required_set - set(repaired_order))
    unknown = sorted(set(repaired_order) - required_set)
    duplicates = sorted(t for t, n in counts.items() if n > 1)

    positions = {t: i for i, t in enumerate(repaired_order)}
    violated = [
        [a, b] for a, b in edges
        if a not in positions or b not in positions
        or positions[a] >= positions[b]
    ]

    exact_task_set = (
        len(repaired_order) == len(required)
        and set(repaired_order) == required_set
        and len(set(repaired_order)) == len(repaired_order)
    )

    validator_result = validate_plan({
        "task_order": repaired_order,
        "required_tasks": required,
        "precedence_constraints": edges
    })

    checks = {
        "all_required_tasks_present_once": exact_task_set,
        "no_missing_tasks": not missing,
        "no_unknown_tasks": not unknown,
        "no_duplicate_tasks": not duplicates,
        "all_precedence_constraints_satisfied": not violated,
        "validator_confirms_valid": validator_result["valid"],
        "previous_evaluation_says_valid": (
            result.get("repaired_plan_valid", "").lower() == "true"
        ),
        "original_order_retained_in_case": (
            isinstance(original, list) and len(original) > 0
        ),
    }

    audit.append({
        "test_case_id": cid,
        "source_case_id": case["source_case_id"],
        "defect_type": case["defect_type"],
        **checks,
        "all_invariants_pass": all(checks.values()),
        "missing_tasks": json.dumps(missing),
        "unknown_tasks": json.dumps(unknown),
        "duplicate_tasks": json.dumps(duplicates),
        "violated_edges": json.dumps(violated),
        "approved_for_evaluation": False,
    })

cases_hash_after = sha256_file(CASES_FILE)
original_cases_unchanged = cases_hash_before == cases_hash_after

audit_path = OUT / "experiment67_invariant_audit.csv"
with open(audit_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(audit[0].keys()))
    writer.writeheader()
    writer.writerows(audit)

summary = {
    "experiment": "Experiment 67 — Repair Invariants and Regression",
    "cases_checked": len(audit),
    "all_invariants_pass": sum(
        bool(r["all_invariants_pass"]) for r in audit
    ),
    "failed_cases": [
        r["test_case_id"] for r in audit if not r["all_invariants_pass"]
    ],
    "original_corruption_file_unchanged": original_cases_unchanged,
    "api_calls": 0,
    "approved_for_evaluation": 0,
    "audit_file": str(audit_path),
}

with open(OUT / "experiment67_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("PLANPROOF AI — EXPERIMENT 67")
print("=" * 48)
print("Cases checked:", summary["cases_checked"])
print("All invariants passed:",
      summary["all_invariants_pass"], "/", summary["cases_checked"])
print("Failed cases:", len(summary["failed_cases"]))
print("Original corruption file unchanged:", original_cases_unchanged)
print("API calls: 0")
print("Evaluation approvals: 0")

if summary["failed_cases"]:
    print("\nFirst failures:")
    for r in audit:
        if not r["all_invariants_pass"]:
            print(r["test_case_id"], {
                k: v for k, v in r.items()
                if k.endswith("_pass") and not v
            })

print("\nAudit CSV:", audit_path)
print("Summary JSON:", OUT / "experiment67_summary.json")


PLANPROOF AI — EXPERIMENT 67
Cases checked: 112
All invariants passed: 112 / 112
Failed cases: 0
Original corruption file unchanged: True
API calls: 0
Evaluation approvals: 0

Audit CSV: /content/planproof_results/experiment65/experiment67_regression/experiment67_invariant_audit.csv
Summary JSON: /content/planproof_results/experiment65/experiment67_regression/experiment67_summary.json


In [ ]:

import json
import csv
import time
from pathlib import Path
from collections import defaultdict

BASE = Path("/content/planproof_results/experiment65")
INPUT = BASE / "controlled_corruption" / "corrupted_cases.jsonl"
OUT = BASE / "experiment68_method_comparison"
OUT.mkdir(parents=True, exist_ok=True)

with open(INPUT, encoding="utf-8") as f:
    cases = [json.loads(line) for line in f if line.strip()]

results = []

def make_case(record):
    return {
        "task_order": list(record["corrupted_order"]),
        "required_tasks": list(record["required_tasks"]),
        "precedence_constraints": list(record["precedence_constraints"]),
    }

def evaluate_order(order, case):
    if not isinstance(order, list):
        return False, None
    verdict = validate_plan({
        "task_order": order,
        "required_tasks": case["required_tasks"],
        "precedence_constraints": case["precedence_constraints"],
    })
    rate = shared_task_order_change_rate(case["task_order"], order)
    return verdict["valid"], rate

for index, record in enumerate(cases, start=1):
    case = make_case(record)
    cid = record["test_case_id"]

    # Stable repair
    start = time.perf_counter()
    try:
        stable_result = repair_plan(case)
        stable_seconds = time.perf_counter() - start
        stable_ok = (
            isinstance(stable_result, dict)
            and stable_result.get("success", False)
            and isinstance(stable_result.get("repaired_plan"), dict)
        )
        stable_order = (
            stable_result["repaired_plan"].get("task_order")
            if stable_ok else None
        )
    except Exception as exc:
        stable_seconds = time.perf_counter() - start
        stable_ok, stable_order = False, None
        stable_error = f"{type(exc).__name__}: {exc}"
    else:
        stable_error = ""

    stable_valid, stable_change = evaluate_order(stable_order, case)

    # Adaptive repair
    start = time.perf_counter()
    try:
        adaptive_result = adaptive_repair_plan(
            case, time_budget_seconds=5
        )
        adaptive_seconds = time.perf_counter() - start
        adaptive_ok = (
            isinstance(adaptive_result, dict)
            and adaptive_result.get("success", False)
            and isinstance(adaptive_result.get("order"), list)
        )
        adaptive_order = (
            adaptive_result.get("order") if adaptive_ok else None
        )
        adaptive_method = adaptive_result.get("method", "unknown")
        adaptive_optimal = adaptive_result.get("optimality_proven", False)
        adaptive_status = adaptive_result.get("solver_status")
    except Exception as exc:
        adaptive_seconds = time.perf_counter() - start
        adaptive_ok, adaptive_order = False, None
        adaptive_method, adaptive_optimal, adaptive_status = (
            "exception", False, None
        )
        adaptive_error = f"{type(exc).__name__}: {exc}"
    else:
        adaptive_error = ""

    adaptive_valid, adaptive_change = evaluate_order(adaptive_order, case)

    # Exact optimizer
    start = time.perf_counter()
    try:
        exact_result = exact_optimize_order_fair(
            case, time_limit_seconds=5
        )
        exact_seconds = time.perf_counter() - start
        exact_order = (
            exact_result.get("order")
            if isinstance(exact_result, dict) else None
        )
        exact_status = (
            exact_result.get("status", "unknown")
            if isinstance(exact_result, dict) else "unknown"
        )
        exact_optimal = (
            exact_result.get("optimal", False)
            if isinstance(exact_result, dict) else False
        )
    except Exception as exc:
        exact_seconds = time.perf_counter() - start
        exact_order = None
        exact_status, exact_optimal = "exception", False
        exact_error = f"{type(exc).__name__}: {exc}"
    else:
        exact_error = ""

    exact_valid, exact_change = evaluate_order(exact_order, case)

    results.append({
        "test_case_id": cid,
        "source_case_id": record["source_case_id"],
        "defect_type": record["defect_type"],

        "stable_success": stable_ok,
        "stable_valid": stable_valid,
        "stable_order_change_rate": stable_change,
        "stable_seconds": round(stable_seconds, 6),
        "stable_error": stable_error,

        "adaptive_success": adaptive_ok,
        "adaptive_valid": adaptive_valid,
        "adaptive_order_change_rate": adaptive_change,
        "adaptive_method": adaptive_method,
        "adaptive_optimality_proven": adaptive_optimal,
        "adaptive_solver_status": adaptive_status,
        "adaptive_seconds": round(adaptive_seconds, 6),
        "adaptive_error": adaptive_error,

        "exact_valid": exact_valid,
        "exact_order_change_rate": exact_change,
        "exact_solver_status": exact_status,
        "exact_optimality_proven": exact_optimal,
        "exact_seconds": round(exact_seconds, 6),
        "exact_error": exact_error,

        "approved_for_evaluation": False,
    })

    if index % 10 == 0:
        print(f"Processed {index}/{len(cases)} cases")

csv_path = OUT / "experiment68_comparison.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

def mean_valid(rows, key):
    values = [
        r[key] for r in rows
        if isinstance(r[key], (int, float))
    ]
    return sum(values) / len(values) if values else None

summary = {
    "experiment": "Experiment 68 — Repair Method Comparison",
    "cases": len(results),
    "api_calls": 0,
    "approved_for_evaluation": 0,
    "methods": {},
    "paired_comparisons": {},
}

for method, valid_key, rate_key, time_key in [
    ("stable", "stable_valid", "stable_order_change_rate", "stable_seconds"),
    ("adaptive", "adaptive_valid", "adaptive_order_change_rate", "adaptive_seconds"),
    ("exact", "exact_valid", "exact_order_change_rate", "exact_seconds"),
]:
    summary["methods"][method] = {
        "valid_repairs": sum(bool(r[valid_key]) for r in results),
        "mean_order_change_rate": mean_valid(results, rate_key),
        "mean_runtime_seconds": mean_valid(results, time_key),
    }

for method in ("adaptive", "exact"):
    key = f"{method}_order_change_rate"
    paired = [
        (r["stable_order_change_rate"], r[key])
        for r in results
        if isinstance(r["stable_order_change_rate"], (int, float))
        and isinstance(r[key], (int, float))
        and r["stable_valid"] and r[f"{method}_valid"]
    ]
    improvements = [s - m for s, m in paired]
    summary["paired_comparisons"][f"stable_vs_{method}"] = {
        "paired_valid_cases": len(paired),
        "mean_improvement_in_order_change_rate": (
            sum(improvements) / len(improvements) if improvements else None
        ),
        "wins": sum(x > 1e-12 for x in improvements),
        "ties": sum(abs(x) <= 1e-12 for x in improvements),
        "losses": sum(x < -1e-12 for x in improvements),
    }

with open(OUT / "experiment68_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nPLANPROOF AI — EXPERIMENT 68")
print("=" * 52)
print("Cases:", len(results))

for method, stats in summary["methods"].items():
    print(f"\n{method.upper()}")
    print("  Valid repairs:", stats["valid_repairs"], "/", len(results))
    print("  Mean order-change rate:", stats["mean_order_change_rate"])
    print("  Mean runtime (seconds):", stats["mean_runtime_seconds"])

print("\nPAIRED COMPARISONS")
for name, stats in summary["paired_comparisons"].items():
    print(f"  {name}: {stats}")

print("\nExact optimizer status counts:")
for status in sorted({str(r["exact_solver_status"]) for r in results}):
    print(f"  {status}: {sum(str(r['exact_solver_status']) == status for r in results)}")

print("\nCSV:", csv_path)
print("Summary:", OUT / "experiment68_summary.json")
print("API calls: 0 | Evaluation approvals: 0")


Processed 10/112 cases
Processed 20/112 cases
Processed 30/112 cases
Processed 40/112 cases
Processed 50/112 cases
Processed 60/112 cases
Processed 70/112 cases
Processed 80/112 cases
Processed 90/112 cases
Processed 100/112 cases
Processed 110/112 cases

PLANPROOF AI — EXPERIMENT 68
Cases: 112

STABLE
  Valid repairs: 112 / 112
  Mean order-change rate: 0.024652777777777777
  Mean runtime (seconds): 7.824999999999999e-05

ADAPTIVE
  Valid repairs: 112 / 112
  Mean order-change rate: 0.01626984126984127
  Mean runtime (seconds): 0.031235892857142854

EXACT
  Valid repairs: 112 / 112
  Mean order-change rate: 0.01626984126984127
  Mean runtime (seconds): 0.023471812499999998

PAIRED COMPARISONS
  stable_vs_adaptive: {'paired_valid_cases': 112, 'mean_improvement_in_order_change_rate': 0.008382936507936508, 'wins': 9, 'ties': 103, 'losses': 0}
  stable_vs_exact: {'paired_valid_cases': 112, 'mean_improvement_in_order_change_rate': 0.008382936507936508, 'wins': 9, 'ties': 103, 'losses': 0}


In [ ]:

import csv
import json
import math
import statistics
from pathlib import Path

from scipy.stats import wilcoxon, binomtest, ttest_rel

BASE = Path("/content/planproof_results/experiment65")
INPUT = BASE / "experiment68_method_comparison" / "experiment68_comparison.csv"
OUT = BASE / "experiment69_statistics"
OUT.mkdir(parents=True, exist_ok=True)

with open(INPUT, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

def number(row, key):
    value = row.get(key, "")
    if value is None or value == "":
        return None
    return float(value)

# Compare only cases where both methods produced valid plans and
# both order-change metrics are available.
pairs = []
for row in rows:
    stable = number(row, "stable_order_change_rate")
    adaptive = number(row, "adaptive_order_change_rate")
    exact = number(row, "exact_order_change_rate")

    if (
        row.get("stable_valid", "").lower() == "true"
        and row.get("adaptive_valid", "").lower() == "true"
        and stable is not None and adaptive is not None
    ):
        pairs.append({
            "case_id": row["test_case_id"],
            "stable": stable,
            "adaptive": adaptive,
            "exact": exact,
            "stable_minus_adaptive": stable - adaptive,
            "stable_minus_exact": (
                stable - exact if exact is not None else None
            ),
        })

def analyze(pairs, diff_key, method_label, seed=20261069):
    diffs = [p[diff_key] for p in pairs if p[diff_key] is not None]
    n = len(diffs)

    if n == 0:
        return {"method": method_label, "paired_cases": 0}

    wins = sum(d > 1e-12 for d in diffs)
    ties = sum(abs(d) <= 1e-12 for d in diffs)
    losses = sum(d < -1e-12 for d in diffs)
    nonzero = [d for d in diffs if abs(d) > 1e-12]

    # Exact two-sided sign test among non-tied pairs.
    sign_p = (
        binomtest(wins, wins + losses, p=0.5, alternative="two-sided").pvalue
        if wins + losses else 1.0
    )

    # Wilcoxon signed-rank test, omitting zero differences.
    try:
        wilcoxon_p = (
            float(wilcoxon(nonzero, alternative="two-sided",
                           method="auto").pvalue)
            if nonzero else 1.0
        )
    except Exception:
        wilcoxon_p = None

    # Paired t-test; interpret cautiously because differences are discrete
    # and most cases may be tied.
    try:
        ttest_p = (
            float(ttest_rel(
                [p["stable"] for p in pairs],
                [p["adaptive"] if diff_key == "stable_minus_adaptive"
                 else p["exact"] for p in pairs]
            ).pvalue)
            if n > 1 else None
        )
    except Exception:
        ttest_p = None

    # Deterministic nonparametric percentile bootstrap CI for the mean.
    import random
    rng = random.Random(seed)
    bootstrap_means = []
    B = 10000

    for _ in range(B):
        sample = [diffs[rng.randrange(n)] for _ in range(n)]
        bootstrap_means.append(sum(sample) / n)

    bootstrap_means.sort()
    lower = bootstrap_means[int(0.025 * B)]
    upper = bootstrap_means[min(B - 1, int(0.975 * B))]

    return {
        "method": method_label,
        "paired_cases": n,
        "mean_improvement": statistics.mean(diffs),
        "median_improvement": statistics.median(diffs),
        "sample_sd_of_paired_differences": (
            statistics.stdev(diffs) if n > 1 else 0.0
        ),
        "wins": wins,
        "ties": ties,
        "losses": losses,
        "relative_mean_reduction_percent": (
            100 * statistics.mean(diffs) /
            statistics.mean([p["stable"] for p in pairs])
            if statistics.mean([p["stable"] for p in pairs]) else None
        ),
        "bootstrap_95_percentile_ci_mean": [lower, upper],
        "sign_test_two_sided_p": sign_p,
        "wilcoxon_signed_rank_two_sided_p": wilcoxon_p,
        "paired_t_test_two_sided_p": ttest_p,
        "bootstrap_replicates": B,
        "random_seed": seed,
    }

adaptive_stats = analyze(
    pairs, "stable_minus_adaptive", "Stable vs Adaptive"
)
exact_pairs = [p for p in pairs if p["stable_minus_exact"] is not None]
exact_stats = analyze(
    exact_pairs, "stable_minus_exact", "Stable vs Exact", seed=20261070
)

# Compare adaptive directly with exact.
adaptive_exact_diffs = [
    p["adaptive"] - p["exact"]
    for p in pairs if p["exact"] is not None
]
adaptive_exact = {
    "paired_cases": len(adaptive_exact_diffs),
    "adaptive_lower_change_rate_wins": sum(
        d < -1e-12 for d in adaptive_exact_diffs
    ),
    "ties": sum(abs(d) <= 1e-12 for d in adaptive_exact_diffs),
    "exact_lower_change_rate_wins": sum(
        d > 1e-12 for d in adaptive_exact_diffs
    ),
}

summary = {
    "experiment": "Experiment 69 — Statistical Significance",
    "source": str(INPUT),
    "api_calls": 0,
    "approved_for_evaluation": 0,
    "stable_vs_adaptive": adaptive_stats,
    "stable_vs_exact": exact_stats,
    "adaptive_vs_exact": adaptive_exact,
    "interpretation_note": (
        "Positive improvement means lower order-change rate than stable repair. "
        "Confidence intervals describe paired benchmark differences, not "
        "generalization to independently reviewed real-world plans."
    ),
}

with open(OUT / "experiment69_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("PLANPROOF AI — EXPERIMENT 69")
print("=" * 52)

for label, stats in [
    ("STABLE VS ADAPTIVE", adaptive_stats),
    ("STABLE VS EXACT", exact_stats),
]:
    print("\n" + label)
    for key in [
        "paired_cases", "mean_improvement", "median_improvement",
        "wins", "ties", "losses", "relative_mean_reduction_percent",
        "bootstrap_95_percentile_ci_mean", "sign_test_two_sided_p",
        "wilcoxon_signed_rank_two_sided_p", "paired_t_test_two_sided_p"
    ]:
        print(f"  {key}: {stats.get(key)}")

print("\nADAPTIVE VS EXACT")
print(json.dumps(adaptive_exact, indent=2))
print("\nSaved:", OUT / "experiment69_summary.json")
print("API calls: 0 | Evaluation approvals: 0")


PLANPROOF AI — EXPERIMENT 69

STABLE VS ADAPTIVE
  paired_cases: 112
  mean_improvement: 0.008382936507936508
  median_improvement: 0.0
  wins: 9
  ties: 103
  losses: 0
  relative_mean_reduction_percent: 34.00402414486922
  bootstrap_95_percentile_ci_mean: [0.0021825396825396826, 0.01755952380952381]
  sign_test_two_sided_p: 0.00390625
  wilcoxon_signed_rank_two_sided_p: 0.00390625
  paired_t_test_two_sided_p: 0.03751604165940024

STABLE VS EXACT
  paired_cases: 112
  mean_improvement: 0.008382936507936508
  median_improvement: 0.0
  wins: 9
  ties: 103
  losses: 0
  relative_mean_reduction_percent: 34.00402414486922
  bootstrap_95_percentile_ci_mean: [0.002132936507936508, 0.01755952380952381]
  sign_test_two_sided_p: 0.00390625
  wilcoxon_signed_rank_two_sided_p: 0.00390625
  paired_t_test_two_sided_p: 0.03751604165940024

ADAPTIVE VS EXACT
{
  "paired_cases": 112,
  "adaptive_lower_change_rate_wins": 0,
  "ties": 112,
  "exact_lower_change_rate_wins": 0
}

Saved: /content/planproof

In [ ]:

import json
import csv
from pathlib import Path
from collections import Counter

BASE = Path("/content/planproof_results/experiment65")
MAIN_CSV = BASE / "real_plan_dataset_template.csv"
EXTRACTIONS = BASE / "gemini_extraction" / "extractions.jsonl"
REVIEWS_DIR = BASE / "reviews"
if not REVIEWS_DIR.exists():
    REVIEWS_DIR = BASE / "master_collection"

OUT = BASE / "experiment70_review_readiness"
OUT.mkdir(parents=True, exist_ok=True)

# Load the main dataset without modifying it.
with open(MAIN_CSV, newline="", encoding="utf-8") as f:
    main_rows = list(csv.DictReader(f))

# Load genuine Gemini-generated plan extractions.
with open(EXTRACTIONS, encoding="utf-8") as f:
    extraction_rows = [
        json.loads(line) for line in f if line.strip()
    ]

# Search existing review JSON files recursively.
review_files = list(BASE.rglob("*review*.json"))
review_files += list(BASE.rglob("*dependency*.json"))
review_files = list(dict.fromkeys(review_files))

reviews_by_case = {}
review_file_records = []

for path in review_files:
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, dict):
            continue

        cid = (
            data.get("case_id")
            or data.get("source_case_id")
            or data.get("plan_id")
        )

        review_file_records.append({
            "path": str(path),
            "case_id": cid,
            "top_level_keys": list(data.keys()),
        })

        if cid:
            reviews_by_case.setdefault(cid, []).append({
                "path": str(path),
                "data": data,
            })
    except Exception as exc:
        review_file_records.append({
            "path": str(path),
            "case_id": None,
            "top_level_keys": [],
            "read_error": str(exc),
        })

extraction_ids = {r["case_id"] for r in extraction_rows}
main_ids = {r.get("case_id", "") for r in main_rows}
audit_rows = []

for row in main_rows:
    cid = row.get("case_id", "")
    review_matches = reviews_by_case.get(cid, [])

    approved_reviews = []
    independent_reviews = []

    for item in review_matches:
        data = item["data"]
        approved = data.get("approved_for_evaluation") is True
        reviewer = (
            data.get("reviewer")
            or data.get("reviewer_id")
            or data.get("independent_reviewer")
        )
        independence = data.get("independent_review", False) is True

        if approved:
            approved_reviews.append(item["path"])
        if reviewer or independence:
            independent_reviews.append(item["path"])

    included = str(
        row.get("included_in_evaluation", "")
    ).strip().lower() in ("true", "1", "yes")

    audit_rows.append({
        "case_id": cid,
        "domain": row.get("domain", ""),
        "has_original_plan_text": bool(
            row.get("original_plan_text", "").strip()
        ),
        "has_extraction": cid in extraction_ids,
        "review_file_count": len(review_matches),
        "review_files": "; ".join(x["path"] for x in review_matches),
        "independence_evidence_file_count": len(independent_reviews),
        "approved_review_file_count": len(approved_reviews),
        "included_in_evaluation": included,
        "eligible_under_conservative_rule": (
            bool(row.get("original_plan_text", "").strip())
            and cid in extraction_ids
            and bool(independent_reviews)
            and bool(approved_reviews)
        ),
    })

# Save separate audit artifacts.
audit_csv = OUT / "experiment70_review_readiness.csv"
with open(audit_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(audit_rows[0].keys()))
    writer.writeheader()
    writer.writerows(audit_rows)

with open(OUT / "review_file_inventory.json", "w", encoding="utf-8") as f:
    json.dump(review_file_records, f, indent=2)

summary = {
    "experiment": "Experiment 70 — Independent Review Readiness",
    "main_dataset_records": len(main_rows),
    "extraction_records": len(extraction_rows),
    "review_files_discovered": len(review_file_records),
    "records_with_original_text": sum(
        r["has_original_plan_text"] for r in audit_rows
    ),
    "records_with_extraction": sum(
        r["has_extraction"] for r in audit_rows
    ),
    "records_with_review_files": sum(
        r["review_file_count"] > 0 for r in audit_rows
    ),
    "records_with_independence_evidence": sum(
        r["independence_evidence_file_count"] > 0 for r in audit_rows
    ),
    "records_with_approved_review_files": sum(
        r["approved_review_file_count"] > 0 for r in audit_rows
    ),
    "records_eligible_under_conservative_rule": sum(
        r["eligible_under_conservative_rule"] for r in audit_rows
    ),
    "records_marked_included": sum(
        r["included_in_evaluation"] for r in audit_rows
    ),
    "api_calls": 0,
    "main_dataset_modified": False,
}

with open(OUT / "experiment70_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("PLANPROOF AI — EXPERIMENT 70")
print("=" * 52)
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nPer-record readiness:")
for r in audit_rows:
    print(
        r["case_id"],
        "| reviews:", r["review_file_count"],
        "| independent evidence:", r["independence_evidence_file_count"],
        "| approved:", r["approved_review_file_count"],
        "| eligible:", r["eligible_under_conservative_rule"],
    )

print("\nAudit CSV:", audit_csv)
print("Review inventory:", OUT / "review_file_inventory.json")
print("Summary JSON:", OUT / "experiment70_summary.json")
print("No API calls. Main dataset unchanged.")


PLANPROOF AI — EXPERIMENT 70
experiment: Experiment 70 — Independent Review Readiness
main_dataset_records: 2
extraction_records: 28
review_files_discovered: 3
records_with_original_text: 2
records_with_extraction: 0
records_with_review_files: 2
records_with_independence_evidence: 0
records_with_approved_review_files: 0
records_eligible_under_conservative_rule: 0
records_marked_included: 0
api_calls: 0
main_dataset_modified: False

Per-record readiness:
REAL_001 | reviews: 2 | independent evidence: 0 | approved: 0 | eligible: False
REAL_002 | reviews: 1 | independent evidence: 0 | approved: 0 | eligible: False

Audit CSV: /content/planproof_results/experiment65/experiment70_review_readiness/experiment70_review_readiness.csv
Review inventory: /content/planproof_results/experiment65/experiment70_review_readiness/review_file_inventory.json
Summary JSON: /content/planproof_results/experiment65/experiment70_review_readiness/experiment70_summary.json
No API calls. Main dataset unchanged.


In [ ]:

import json
from pathlib import Path

BASE = Path("/content/planproof_results/experiment65")

review_files = sorted(set(
    list(BASE.rglob("*review*.json"))
    + list(BASE.rglob("*dependency*.json"))
))

print("PLANPROOF AI — EXPERIMENT 71")
print("=" * 52)
print("Review files discovered:", len(review_files))

for path in review_files:
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"\n[READ ERROR] {path}\n{exc}")
        continue

    print("\n" + "-" * 52)
    print("FILE:", path)
    print("TOP-LEVEL KEYS:",
          list(data.keys()) if isinstance(data, dict) else type(data).__name__)

    if isinstance(data, dict):
        for key in [
            "case_id", "source_case_id", "plan_id",
            "status", "review_status", "reviewer",
            "reviewer_id", "independent_reviewer",
            "independent_review", "approved_for_evaluation",
            "review_date", "review_notes", "decision",
            "mandatory_dependencies", "conditional_dependencies",
            "ambiguous_dependencies", "candidate_edges",
        ]:
            if key in data:
                print(f"{key}:")
                print(json.dumps(data[key], indent=2, ensure_ascii=False)[:3500])

        # Show any additional fields that may contain review evidence.
        shown = {
            "case_id", "source_case_id", "plan_id",
            "status", "review_status", "reviewer",
            "reviewer_id", "independent_reviewer",
            "independent_review", "approved_for_evaluation",
            "review_date", "review_notes", "decision",
            "mandatory_dependencies", "conditional_dependencies",
            "ambiguous_dependencies", "candidate_edges",
        }
        extra = {k: v for k, v in data.items() if k not in shown}
        if extra:
            print("OTHER FIELDS:")
            print(json.dumps(extra, indent=2, ensure_ascii=False)[:2500])

print("\nInspection only. No files modified. No API calls made.")


PLANPROOF AI — EXPERIMENT 71
Review files discovered: 4

----------------------------------------------------
FILE: /content/planproof_results/experiment65/case_001_dependency_review.json
TOP-LEVEL KEYS: ['case_id', 'review_status', 'original_dependency_count', 'reviewed_mandatory_dependency_count', 'reviewed_mandatory_dependencies', 'conditional_dependency', 'soft_dependency', 'limitation', 'final_review_requirements', 'approved_for_evaluation']
case_id:
"REAL_001"
review_status:
"pending_final_review"
approved_for_evaluation:
false
OTHER FIELDS:
{
  "original_dependency_count": 13,
  "reviewed_mandatory_dependency_count": 11,
  "reviewed_mandatory_dependencies": [
    [
      "T1",
      "T2"
    ],
    [
      "T1",
      "T3"
    ],
    [
      "T2",
      "T4"
    ],
    [
      "T2",
      "T5"
    ],
    [
      "T4",
      "T5"
    ],
    [
      "T3",
      "T6"
    ],
    [
      "T5",
      "T7"
    ],
    [
      "T6",
      "T7"
    ],
    [
      "T7",
      "T8"
    ],
 

In [ ]:

import json
import csv
from pathlib import Path

BASE = Path("/content/planproof_results/experiment65")
SOURCE = BASE / "gemini_extraction" / "extractions.jsonl"
OUT = BASE / "experiment72_human_review"
OUT.mkdir(parents=True, exist_ok=True)

with open(SOURCE, encoding="utf-8") as f:
    plans = [json.loads(line) for line in f if line.strip()]

edge_rows = []
plan_rows = []

for plan in plans:
    cid = plan["case_id"]
    extraction = plan.get("extraction", {})
    tasks = extraction.get("tasks", [])
    task_names = {
        t.get("task_id", t.get("id")): t.get("name", "")
        for t in tasks
    }

    edges = extraction.get("explicit_dependencies", [])
    conditional = extraction.get("conditional_dependencies", [])
    ambiguous = extraction.get("ambiguous_dependencies", [])

    plan_rows.append({
        "case_id": cid,
        "domain": plan.get("domain", ""),
        "source_model": plan.get("source_model", ""),
        "original_plan_text": "",
        "reviewer_id": "",
        "review_date": "",
        "review_status": "pending_human_review",
        "approved_for_evaluation": False,
    })

    combined = []
    for e in edges:
        combined.append(("explicit_candidate", e))
    for e in conditional:
        combined.append(("conditional_candidate", e))
    for e in ambiguous:
        combined.append(("ambiguous_candidate", e))

    seen = set()
    for category, edge in combined:
        before = edge.get("before")
        after = edge.get("after")
        key = (before, after, category)
        if key in seen:
            continue
        seen.add(key)

        edge_rows.append({
            "case_id": cid,
            "domain": plan.get("domain", ""),
            "before_task_id": before,
            "before_task_name": task_names.get(before, ""),
            "after_task_id": after,
            "after_task_name": task_names.get(after, ""),
            "model_candidate_category": category,
            "model_evidence": edge.get("evidence", ""),
            "human_decision": "",
            "human_rationale": "",
            "reviewer_id": "",
            "review_date": "",
            "approved_for_evaluation": False,
        })

plan_path = OUT / "plan_review_register.csv"
edge_path = OUT / "dependency_review_worksheet.csv"

with open(plan_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(plan_rows[0].keys()))
    writer.writeheader()
    writer.writerows(plan_rows)

with open(edge_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(edge_rows[0].keys()))
    writer.writeheader()
    writer.writerows(edge_rows)

summary = {
    "experiment": "Experiment 72 — Human Review Worksheet",
    "source_plans": len(plans),
    "candidate_edge_rows": len(edge_rows),
    "plans_auto_approved": 0,
    "edges_auto_approved": 0,
    "api_calls": 0,
    "files": [str(plan_path), str(edge_path)],
}

with open(OUT / "experiment72_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("PLANPROOF AI — EXPERIMENT 72")
print("=" * 48)
print("Plans in review register:", len(plan_rows))
print("Dependency candidates:", len(edge_rows))
print("Automatically approved plans:", 0)
print("Automatically approved dependencies:", 0)
print("API calls:", 0)
print("\nPlan register:", plan_path)
print("Dependency worksheet:", edge_path)
print("Summary:", OUT / "experiment72_summary.json")


PLANPROOF AI — EXPERIMENT 72
Plans in review register: 28
Dependency candidates: 409
Automatically approved plans: 0
Automatically approved dependencies: 0
API calls: 0

Plan register: /content/planproof_results/experiment65/experiment72_human_review/plan_review_register.csv
Dependency worksheet: /content/planproof_results/experiment65/experiment72_human_review/dependency_review_worksheet.csv
Summary: /content/planproof_results/experiment65/experiment72_human_review/experiment72_summary.json


In [ ]:

import json
import csv
from pathlib import Path
from collections import Counter, defaultdict

BASE = Path("/content/planproof_results/experiment65")
INPUT = BASE / "experiment72_human_review" / "dependency_review_worksheet.csv"
OUT = BASE / "experiment73_review_priorities"
OUT.mkdir(parents=True, exist_ok=True)

with open(INPUT, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

per_plan = defaultdict(list)
edge_categories = defaultdict(set)
edge_evidence = defaultdict(list)

for r in rows:
    cid = r["case_id"]
    before = r.get("before_task_id", "").strip()
    after = r.get("after_task_id", "").strip()
    category = r.get("model_candidate_category", "").strip()
    evidence = r.get("model_evidence", "").strip()

    per_plan[cid].append(r)

    if before and after:
        key = (cid, before, after)
        edge_categories[key].add(category)
        if evidence:
            edge_evidence[key].append(evidence)

# Repeated category means the same directed edge appears under
# multiple extraction categories; this is a review flag, not an error.
repeated_edges = {
    key: categories
    for key, categories in edge_categories.items()
    if len(categories) > 1
}

plan_report = []
for cid, items in sorted(per_plan.items()):
    unique_edges = {
        (r.get("before_task_id", "").strip(),
         r.get("after_task_id", "").strip())
        for r in items
        if r.get("before_task_id", "").strip()
        and r.get("after_task_id", "").strip()
    }

    ambiguous_count = sum(
        "ambiguous" in r.get("model_candidate_category", "").lower()
        for r in items
    )
    conditional_count = sum(
        "conditional" in r.get("model_candidate_category", "").lower()
        for r in items
    )
    missing_evidence_count = sum(
        not r.get("model_evidence", "").strip()
        for r in items
    )
    repeated_count = sum(
        key[0] == cid for key in repeated_edges
    )

    # Transparent triage score: prioritize ambiguity, then missing
    # evidence, then repeated classifications. Not a correctness score.
    priority_score = (
        3 * ambiguous_count
        + 2 * missing_evidence_count
        + repeated_count
        + conditional_count
    )

    plan_report.append({
        "case_id": cid,
        "candidate_rows": len(items),
        "unique_directed_edges": len(unique_edges),
        "ambiguous_candidate_rows": ambiguous_count,
        "conditional_candidate_rows": conditional_count,
        "missing_evidence_rows": missing_evidence_count,
        "edges_in_multiple_categories": repeated_count,
        "review_priority_score": priority_score,
        "review_status": "pending_human_review",
        "approved_for_evaluation": False,
    })

plan_report.sort(
    key=lambda r: (-r["review_priority_score"], r["case_id"])
)

edge_report = []
for (cid, before, after), categories in sorted(edge_categories.items()):
    edge_report.append({
        "case_id": cid,
        "before_task_id": before,
        "after_task_id": after,
        "candidate_categories": "; ".join(sorted(categories)),
        "appears_in_multiple_categories": len(categories) > 1,
        "evidence_count": len(edge_evidence[(cid, before, after)]),
        "review_status": "pending_human_review",
        "human_decision": "",
        "human_rationale": "",
        "approved_for_evaluation": False,
    })

def save_csv(path, data):
    if not data:
        path.write_text("", encoding="utf-8")
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(data[0].keys()))
        writer.writeheader()
        writer.writerows(data)

plan_path = OUT / "plan_review_priority.csv"
edge_path = OUT / "unique_edge_review_register.csv"
save_csv(plan_path, plan_report)
save_csv(edge_path, edge_report)

summary = {
    "experiment": "Experiment 73 — Review Priority Triage",
    "input_candidate_rows": len(rows),
    "plans_in_report": len(plan_report),
    "unique_plan_edge_pairs": len(edge_report),
    "edges_appearing_in_multiple_categories": len(repeated_edges),
    "candidate_rows_missing_evidence": sum(
        not r.get("model_evidence", "").strip() for r in rows
    ),
    "plans_approved": 0,
    "edges_approved": 0,
    "api_calls": 0,
    "priority_score_is_correctness_measure": False,
    "files": [str(plan_path), str(edge_path)],
}

with open(OUT / "experiment73_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("PLANPROOF AI — EXPERIMENT 73")
print("=" * 52)
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nTOP 10 PLANS TO REVIEW FIRST")
for r in plan_report[:10]:
    print(
        r["case_id"],
        "| priority:", r["review_priority_score"],
        "| ambiguous:", r["ambiguous_candidate_rows"],
        "| conditional:", r["conditional_candidate_rows"],
        "| missing evidence:", r["missing_evidence_rows"],
        "| repeated-category edges:", r["edges_in_multiple_categories"],
    )

print("\nPlan priority report:", plan_path)
print("Unique edge register:", edge_path)
print("Summary:", OUT / "experiment73_summary.json")
print("No API calls. No approvals. Source files unchanged.")


PLANPROOF AI — EXPERIMENT 73
experiment: Experiment 73 — Review Priority Triage
input_candidate_rows: 409
plans_in_report: 28
unique_plan_edge_pairs: 359
edges_appearing_in_multiple_categories: 49
candidate_rows_missing_evidence: 0
plans_approved: 0
edges_approved: 0
api_calls: 0
priority_score_is_correctness_measure: False
files: ['/content/planproof_results/experiment65/experiment73_review_priorities/plan_review_priority.csv', '/content/planproof_results/experiment65/experiment73_review_priorities/unique_edge_review_register.csv']

TOP 10 PLANS TO REVIEW FIRST
REAL_003 | priority: 18 | ambiguous: 4 | conditional: 2 | missing evidence: 0 | repeated-category edges: 4
REAL_029 | priority: 17 | ambiguous: 4 | conditional: 1 | missing evidence: 0 | repeated-category edges: 4
REAL_030 | priority: 12 | ambiguous: 2 | conditional: 2 | missing evidence: 0 | repeated-category edges: 4
REAL_011 | priority: 10 | ambiguous: 3 | conditional: 1 | missing evidence: 0 | repeated-category edges: 0
REA

In [ ]:

from pathlib import Path
import json
import pandas as pd
from datetime import datetime, timezone

# ============================================================
# EXPERIMENT 74: HUMAN REVIEW PACKET — REAL_003
# Read-only inputs; new output directory; zero API calls.
# ============================================================

BASE = Path("/content/planproof_results/experiment65")

RAW_FILE = BASE / "gemini_generation/raw_responses.jsonl"
EXTRACTION_FILE = BASE / "gemini_extraction/extractions.jsonl"
WORKSHEET_FILE = BASE / "experiment72_human_review/dependency_review_worksheet.csv"

OUT = BASE / "experiment74_review_packet_REAL_003"
OUT.mkdir(parents=True, exist_ok=True)

CASE_ID = "REAL_003"

# ---------- Check inputs ----------
for path in [RAW_FILE, EXTRACTION_FILE, WORKSHEET_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required input missing: {path}")

def read_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(
                        f"Invalid JSON in {path}, line {line_number}: {exc}"
                    )
    return records

raw_records = read_jsonl(RAW_FILE)
extractions = read_jsonl(EXTRACTION_FILE)
worksheet = pd.read_csv(WORKSHEET_FILE, dtype=str).fillna("")

raw_matches = [r for r in raw_records if r.get("case_id") == CASE_ID]
extraction_matches = [r for r in extractions if r.get("case_id") == CASE_ID]
candidate_rows = worksheet[worksheet["case_id"] == CASE_ID].copy()

if len(raw_matches) != 1:
    raise ValueError(f"Expected one raw response for {CASE_ID}; found {len(raw_matches)}")
if len(extraction_matches) != 1:
    raise ValueError(f"Expected one extraction for {CASE_ID}; found {len(extraction_matches)}")
if candidate_rows.empty:
    raise ValueError(f"No dependency review rows found for {CASE_ID}")

raw_record = raw_matches[0]
extraction_record = extraction_matches[0]
extraction = extraction_record.get("extraction", {})

original_text = raw_record.get("original_plan_text", "")
if not original_text:
    raise ValueError("Original Gemini response text is empty; refusing to create an incomplete packet.")

tasks = extraction.get("tasks", [])
task_order = extraction.get("task_order", [])
if not tasks:
    raise ValueError("No extracted task records found.")

# ---------- Normalize task information ----------
task_by_id = {}
for task in tasks:
    if not isinstance(task, dict):
        continue
    tid = str(task.get("task_id", task.get("id", ""))).strip()
    if tid:
        task_by_id[tid] = task

def task_name(tid):
    task = task_by_id.get(str(tid), {})
    return str(task.get("task_name", task.get("name", "")))

def task_description(tid):
    task = task_by_id.get(str(tid), {})
    return str(task.get("description", task.get("task_description", "")))

# ---------- Combine repeated candidate edges ----------
# Preserve all categories/evidence from the existing worksheet.
required_columns = [
    "before_task_id", "after_task_id",
    "model_candidate_category", "model_evidence"
]
missing_columns = [c for c in required_columns if c not in candidate_rows.columns]
if missing_columns:
    raise ValueError(f"Worksheet is missing columns: {missing_columns}")

combined = {}

for _, row in candidate_rows.iterrows():
    before = str(row["before_task_id"]).strip()
    after = str(row["after_task_id"]).strip()
    if not before or not after:
        continue

    key = (before, after)
    if key not in combined:
        combined[key] = {
            "case_id": CASE_ID,
            "domain": str(row.get("domain", raw_record.get("domain", ""))),
            "before_task_id": before,
            "before_task_name": task_name(before),
            "before_task_description": task_description(before),
            "after_task_id": after,
            "after_task_name": task_name(after),
            "after_task_description": task_description(after),
            "model_candidate_categories": [],
            "model_evidence_all": [],
            "human_decision": "",
            "human_rationale": "",
            "reviewer_id": "",
            "review_date": "",
            "approved_for_evaluation": False,
        }

    category = str(row["model_candidate_category"]).strip()
    evidence = str(row["model_evidence"]).strip()

    if category and category not in combined[key]["model_candidate_categories"]:
        combined[key]["model_candidate_categories"].append(category)
    if evidence and evidence not in combined[key]["model_evidence_all"]:
        combined[key]["model_evidence_all"].append(evidence)

edge_rows = []
for item in combined.values():
    item["model_candidate_categories"] = " | ".join(item["model_candidate_categories"])
    item["model_evidence_all"] = " || ".join(item["model_evidence_all"])
    item["allowed_human_decisions"] = (
        "mandatory | conditional | unsupported | ambiguous | needs_more_context"
    )
    edge_rows.append(item)

edges_df = pd.DataFrame(edge_rows)
if not edges_df.empty:
    edges_df = edges_df.sort_values(
        ["before_task_id", "after_task_id"], kind="stable"
    ).reset_index(drop=True)

# ---------- Human-readable packet ----------
packet_file = OUT / f"{CASE_ID}_human_review_packet.md"
with packet_file.open("w", encoding="utf-8") as f:
    f.write(f"# Human Dependency Review Packet — {CASE_ID}\n\n")
    f.write("## Review status\n\n")
    f.write("- Status: **PENDING HUMAN REVIEW**\n")
    f.write("- Approved for evaluation: **No**\n")
    f.write("- API calls made by this cell: **0**\n")
    f.write("- This packet does not assert that the model's dependency list is complete or correct.\n")
    f.write("- Review each proposed edge against the original plan and its context.\n\n")

    f.write("## Provenance\n\n")
    f.write(f"- Case ID: `{CASE_ID}`\n")
    f.write(f"- Source provider: `{raw_record.get('source_provider', 'not recorded')}`\n")
    f.write(f"- Source model requested: `{raw_record.get('source_model_requested', 'not recorded')}`\n")
    f.write(f"- Collection timestamp: `{raw_record.get('collection_timestamp_utc', 'not recorded')}`\n")
    f.write(f"- Domain: `{raw_record.get('domain', 'not recorded')}`\n")
    f.write(f"- Extraction status: `{extraction_record.get('status', 'not recorded')}`\n")
    f.write("- Dependency extraction and generation used the same model family; this is not independent ground truth.\n\n")

    f.write("## Original Gemini-generated plan\n\n")
    f.write("The text below is copied from the saved raw response. It has not been rewritten for this review packet.\n\n")
    f.write("```text\n")
    f.write(original_text.rstrip() + "\n")
    f.write("```\n\n")

    f.write("## Extracted task list\n\n")
    f.write("Task descriptions are extracted model fields and may need comparison with the original text.\n\n")
    f.write("| Task ID | Task name | Description |\n|---|---|---|\n")
    for tid in task_order:
        tid = str(tid)
        name = task_name(tid).replace("|", "\\|").replace("\n", " ")
        desc = task_description(tid).replace("|", "\\|").replace("\n", " ")
        f.write(f"| {tid} | {name} | {desc} |\n")
    for tid, task in task_by_id.items():
        if tid not in [str(x) for x in task_order]:
            name = task_name(tid).replace("|", "\\|").replace("\n", " ")
            desc = task_description(tid).replace("|", "\\|").replace("\n", " ")
            f.write(f"| {tid} | {name} | {desc} |\n")

    f.write("\n## Proposed dependency edges — human decision required\n\n")
    f.write(f"Unique directed candidate edges in this packet: **{len(edges_df)}**\n\n")
    f.write("The categories and evidence below are model-produced suggestions, not verified facts.\n\n")
    for i, row in edges_df.iterrows():
        f.write(f"### Edge {i+1}: {row['before_task_id']} → {row['after_task_id']}\n\n")
        f.write(f"- Before task: {row['before_task_name']}\n")
        f.write(f"- Before description: {row['before_task_description']}\n")
        f.write(f"- After task: {row['after_task_name']}\n")
        f.write(f"- After description: {row['after_task_description']}\n")
        f.write(f"- Model candidate categories: {row['model_candidate_categories']}\n")
        f.write(f"- Model evidence: {row['model_evidence_all'] or '(none supplied)'}\n")
        f.write("- Human decision: **[leave blank until reviewed]**\n")
        f.write("- Human rationale: **[leave blank until reviewed]**\n")
        f.write("- Reviewer ID / review date: **[leave blank]**\n\n")

    f.write("## Allowed human decisions\n\n")
    f.write("- `mandatory`: the plan supports a strict prerequisite relationship.\n")
    f.write("- `conditional`: the relationship holds only under stated assumptions.\n")
    f.write("- `unsupported`: the plan does not support the proposed strict dependency.\n")
    f.write("- `ambiguous`: available wording does not permit a confident decision.\n")
    f.write("- `needs_more_context`: additional source information is required.\n\n")
    f.write("Do not mark an edge mandatory merely because it appears in the model extraction. ")
    f.write("Record the specific textual or contextual reason for each decision. ")
    f.write("Do not approve the case for evaluation until the review protocol's required checks are completed.\n")

# ---------- Save editable CSV decision worksheet ----------
csv_file = OUT / f"{CASE_ID}_dependency_decisions.csv"
edges_df.to_csv(csv_file, index=False, encoding="utf-8")

# ---------- Save task register ----------
task_rows = []
for tid in task_order:
    tid = str(tid)
    task_rows.append({
        "case_id": CASE_ID,
        "task_id": tid,
        "task_name": task_name(tid),
        "task_description": task_description(tid),
        "review_notes": "",
        "reviewer_id": "",
        "review_date": "",
        "approved_for_evaluation": False,
    })

tasks_file = OUT / f"{CASE_ID}_task_register.csv"
pd.DataFrame(task_rows).to_csv(tasks_file, index=False, encoding="utf-8")

summary = {
    "experiment": 74,
    "case_id": CASE_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "raw_source_found": True,
    "extraction_found": True,
    "task_count": len(task_rows),
    "unique_candidate_edges": len(edges_df),
    "human_review_status": "pending",
    "approved_for_evaluation": False,
    "api_calls_made": 0,
    "existing_main_dataset_modified": False,
    "files": [str(packet_file), str(csv_file), str(tasks_file)],
}
summary_file = OUT / "experiment74_summary.json"
summary_file.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("EXPERIMENT 74 COMPLETE")
print("Case:", CASE_ID)
print("Tasks:", len(task_rows))
print("Unique candidate edges:", len(edges_df))
print("Human review: PENDING")
print("Approved for evaluation: False")
print("API calls: 0")
print("Existing main dataset modified: No")
print("\nCreated files:")
for path in [packet_file, csv_file, tasks_file, summary_file]:
    print("-", path)


EXPERIMENT 74 COMPLETE
Case: REAL_003
Tasks: 10
Unique candidate edges: 12
Human review: PENDING
Approved for evaluation: False
API calls: 0
Existing main dataset modified: No

Created files:
- /content/planproof_results/experiment65/experiment74_review_packet_REAL_003/REAL_003_human_review_packet.md
- /content/planproof_results/experiment65/experiment74_review_packet_REAL_003/REAL_003_dependency_decisions.csv
- /content/planproof_results/experiment65/experiment74_review_packet_REAL_003/REAL_003_task_register.csv
- /content/planproof_results/experiment65/experiment74_review_packet_REAL_003/experiment74_summary.json


In [ ]:

from pathlib import Path
import pandas as pd
import json

OUT = Path(
    "/content/planproof_results/experiment65/"
    "experiment74_review_packet_REAL_003"
)

decisions_file = OUT / "REAL_003_dependency_decisions.csv"
tasks_file = OUT / "REAL_003_task_register.csv"
summary_file = OUT / "experiment74_summary.json"
packet_file = OUT / "REAL_003_human_review_packet.md"

# Check that every expected file exists.
expected = [decisions_file, tasks_file, summary_file, packet_file]
missing = [str(p) for p in expected if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

edges = pd.read_csv(decisions_file).fillna("")
tasks = pd.read_csv(tasks_file).fillna("")
summary = json.loads(summary_file.read_text(encoding="utf-8"))
packet = packet_file.read_text(encoding="utf-8")

# Integrity checks.
duplicate_edges = edges.duplicated(
    subset=["before_task_id", "after_task_id"]
).sum()

blank_decisions = (edges["human_decision"].astype(str).str.strip() == "").sum()
approved_edges = (
    edges["approved_for_evaluation"].astype(str).str.lower() == "true"
).sum()

approved_tasks = (
    tasks["approved_for_evaluation"].astype(str).str.lower() == "true"
).sum()

checks = {
    "expected_task_count_10": len(tasks) == 10,
    "expected_unique_edge_count_12": len(edges) == 12,
    "no_duplicate_directed_edges": duplicate_edges == 0,
    "all_human_decisions_blank": blank_decisions == len(edges),
    "no_edges_approved": approved_edges == 0,
    "no_tasks_approved": approved_tasks == 0,
    "original_plan_in_packet": "Original Gemini-generated plan" in packet,
    "task_list_in_packet": "Extracted task list" in packet,
    "dependency_review_in_packet": "Proposed dependency edges" in packet,
    "summary_says_pending": summary.get("human_review_status") == "pending",
    "summary_says_not_approved": summary.get("approved_for_evaluation") is False,
}

print("EXPERIMENT 75 — REVIEW PACKET INTEGRITY AUDIT")
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print(f"\nTasks: {len(tasks)}")
print(f"Unique candidate edges: {len(edges)}")
print(f"Blank human decisions: {blank_decisions}")
print(f"Approved edges: {approved_edges}")
print(f"Approved tasks: {approved_tasks}")

if all(checks.values()):
    print("\nRESULT: ALL CHECKS PASSED")
    print("Next: human review of REAL_003 dependencies.")
else:
    print("\nRESULT: AUDIT FAILED — inspect the failed checks before continuing.")


EXPERIMENT 75 — REVIEW PACKET INTEGRITY AUDIT
PASS: expected_task_count_10
PASS: expected_unique_edge_count_12
PASS: no_duplicate_directed_edges
PASS: all_human_decisions_blank
PASS: no_edges_approved
PASS: no_tasks_approved
PASS: original_plan_in_packet
PASS: task_list_in_packet
PASS: dependency_review_in_packet
PASS: summary_says_pending
PASS: summary_says_not_approved

Tasks: 10
Unique candidate edges: 12
Blank human decisions: 12
Approved edges: 0
Approved tasks: 0

RESULT: ALL CHECKS PASSED
Next: human review of REAL_003 dependencies.


In [ ]:

from pathlib import Path
import pandas as pd

BASE = Path("/content/planproof_results/experiment65")
OUT = BASE / "experiment74_review_packet_REAL_003"

packet_file = OUT / "REAL_003_human_review_packet.md"
edges_file = OUT / "REAL_003_dependency_decisions.csv"

# Display the original plan.
text = packet_file.read_text(encoding="utf-8")
start_marker = "## Original Gemini-generated plan"
end_marker = "## Extracted task list"

start = text.find(start_marker)
end = text.find(end_marker)

if start == -1 or end == -1 or end <= start:
    raise ValueError("Could not isolate the original plan section.")

print("=" * 75)
print("REAL_003 — ORIGINAL GEMINI-GENERATED PLAN")
print("=" * 75)
print(text[start:end])

# Display all proposed dependency edges.
edges = pd.read_csv(edges_file).fillna("")

print("\n" + "=" * 75)
print("PROPOSED DEPENDENCIES — REVIEW EVIDENCE")
print("=" * 75)

for _, row in edges.iterrows():
    print(f"\n{row['before_task_id']} -> {row['after_task_id']}")
    print("Before:", row["before_task_name"])
    print("After: ", row["after_task_name"])
    print("Model category:", row["model_candidate_categories"])
    print("Evidence:", row["model_evidence_all"] or "(none supplied)")

print("\nTotal proposed edges:", len(edges))
print("No decisions or approvals were changed.")


REAL_003 — ORIGINAL GEMINI-GENERATED PLAN
## Original Gemini-generated plan

The text below is copied from the saved raw response. It has not been rewritten for this review packet.

```text
## Small Web Application Development and Launch Plan

This plan outlines a realistic, detailed 10-task roadmap to design, build, test, and launch a small web application from scratch.

### Task Inventory

*   **T1: Product Scope and Requirements Definition**
    *   *Execution Order:* 1
    *   *Description:* Define the core feature set, target user personas, technical stack, and acceptance criteria for the MVP (Minimum Viable Product).
    *   *Direct Prerequisites:* None.

*   **T2: UI/UX Wireframing and Design**
    *   *Execution Order:* 2
    *   *Description:* Create low-fidelity wireframes and high-fidelity user interface mockups for all core application screens.
    *   *Direct Prerequisites:* T1.

*   **T3: Database Schema and Architecture Design**
    *   *Execution Order:* 3
    *   *Desc

In [ ]:

from pathlib import Path
import pandas as pd
import json

OUT = Path(
    "/content/planproof_results/experiment65/"
    "experiment74_review_packet_REAL_003"
)

source = OUT / "REAL_003_dependency_decisions.csv"
destination = OUT / "REAL_003_provisional_review_v1.csv"

df = pd.read_csv(source).fillna("")

decisions = {
    ("T1", "T2"): ("mandatory", "Original plan explicitly lists T1 as T2's direct prerequisite."),
    ("T1", "T3"): ("mandatory", "Original plan explicitly lists T1 as T3's direct prerequisite."),
    ("T2", "T4"): ("mandatory", "Original plan explicitly lists T2 as T4's direct prerequisite."),
    ("T3", "T5"): ("mandatory", "Original plan explicitly lists T3 as T5's direct prerequisite."),
    ("T3", "T6"): ("ambiguous", "API-spec changes are discussed, but the plan does not directly list T3 as a prerequisite of T6."),
    ("T3", "T8"): ("mandatory", "Original plan explicitly lists T3 as T8's direct prerequisite."),
    ("T4", "T6"): ("mandatory", "Original plan explicitly lists T4 as a direct prerequisite of T6."),
    ("T5", "T6"): ("mandatory", "Original plan explicitly lists T5 as a direct prerequisite of T6."),
    ("T6", "T7"): ("mandatory", "Original plan explicitly lists T6 as T7's direct prerequisite."),
    ("T7", "T9"): ("mandatory", "Original plan explicitly lists T7 as a direct prerequisite of T9."),
    ("T8", "T9"): ("mandatory", "Original plan explicitly lists T8 as a direct prerequisite of T9."),
    ("T9", "T10"): ("conditional", "The plan states that T10 requires T9 to finish, while critical findings may delay launch."),
}

if len(df) != 12:
    raise ValueError(f"Expected 12 candidate edges, found {len(df)}")

for index, row in df.iterrows():
    key = (
        str(row["before_task_id"]).strip(),
        str(row["after_task_id"]).strip()
    )
    if key not in decisions:
        raise ValueError(f"Unexpected edge: {key}")

    decision, rationale = decisions[key]
    df.loc[index, "proposed_human_decision"] = decision
    df.loc[index, "proposed_human_rationale"] = rationale
    df.loc[index, "review_stage"] = "provisional_model_assisted_review"
    df.loc[index, "reviewer_confirmation"] = "pending"
    df.loc[index, "approved_for_evaluation"] = False

df.to_csv(destination, index=False, encoding="utf-8")

print("EXPERIMENT 76 — PROVISIONAL REVIEW SAVED")
print("Edges reviewed:", len(df))
print("\nProvisional decision counts:")
print(df["proposed_human_decision"].value_counts().to_string())
print("\nReviewer confirmation: pending for every edge")
print("Approved for evaluation:", int(
    (df["approved_for_evaluation"].astype(str).str.lower() == "true").sum()
))
print("Original worksheet modified: No")
print("Saved:", destination)


EXPERIMENT 76 — PROVISIONAL REVIEW SAVED
Edges reviewed: 12

Provisional decision counts:
proposed_human_decision
mandatory      10
ambiguous       1
conditional     1

Reviewer confirmation: pending for every edge
Approved for evaluation: 0
Original worksheet modified: No
Saved: /content/planproof_results/experiment65/experiment74_review_packet_REAL_003/REAL_003_provisional_review_v1.csv


In [ ]:

from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/planproof_results/experiment65")
OUT = BASE / "experiment74_review_packet_REAL_003"

packet_path = OUT / "REAL_003_human_review_packet.md"
edges_path = OUT / "REAL_003_provisional_review_v1.csv"

text = packet_path.read_text(encoding="utf-8")
edges = pd.read_csv(edges_path).fillna("")

# Extract direct-prerequisite statements from the original plan.
# This deliberately checks explicit wording only; it cannot establish
# that the plan itself contains every real-world dependency.
pattern = re.compile(
    r"\*\s+\*\*T(\d+):.*?"
    r"\*Direct Prerequisites:\*\s*([^.\n]*?)\.",
    re.DOTALL
)

explicit_edges = set()
for match in pattern.finditer(text):
    after_id = f"T{match.group(1)}"
    prerequisites = match.group(2).strip()

    if prerequisites.lower() == "none" or not prerequisites:
        continue

    for before_num in re.findall(r"T(\d+)", prerequisites):
        explicit_edges.add((f"T{before_num}", after_id))

candidate_edges = set(
    zip(
        edges["before_task_id"].astype(str),
        edges["after_task_id"].astype(str)
    )
)

missing_from_candidates = sorted(explicit_edges - candidate_edges)
not_explicitly_listed = sorted(candidate_edges - explicit_edges)

print("EXPERIMENT 77 — EXPLICIT DEPENDENCY COVERAGE")
print("Explicit prerequisite edges found in original plan:", len(explicit_edges))
print("Candidate edges in review worksheet:", len(candidate_edges))
print("Explicit edges missing from candidates:", len(missing_from_candidates))

if missing_from_candidates:
    for edge in missing_from_candidates:
        print("MISSING:", edge[0], "->", edge[1])
else:
    print("PASS: Every explicitly stated prerequisite appears in the candidate list.")

print("\nCandidate edges not directly listed in the prerequisite fields:",
      len(not_explicitly_listed))
for edge in not_explicitly_listed:
    print("CHECK:", edge[0], "->", edge[1])

print("\nNo files modified. No API calls made.")


EXPERIMENT 77 — EXPLICIT DEPENDENCY COVERAGE
Explicit prerequisite edges found in original plan: 11
Candidate edges in review worksheet: 12
Explicit edges missing from candidates: 0
PASS: Every explicitly stated prerequisite appears in the candidate list.

Candidate edges not directly listed in the prerequisite fields: 1
CHECK: T3 -> T6

No files modified. No API calls made.


In [1]:

from pathlib import Path

ROOT = Path("/content/planproof_results")

targets = [
    "experiment64_summary.json",
    "experiment70_review_readiness_summary.json",
    "experiment64_research_report.md",
    "experiment70_review_readiness_inventory.json",
]

print("FINALIZATION — EXISTING FILE SEARCH")
print("=" * 55)

for target in targets:
    matches = list(ROOT.rglob(target))
    print(f"\n{target}: {len(matches)} match(es)")
    for path in matches[:10]:
        print(" ", path)

print("\nSearch complete. No files modified and no API calls made.")


FINALIZATION — EXISTING FILE SEARCH

experiment64_summary.json: 0 match(es)

experiment70_review_readiness_summary.json: 0 match(es)

experiment64_research_report.md: 0 match(es)

experiment70_review_readiness_inventory.json: 0 match(es)

Search complete. No files modified and no API calls made.


In [2]:

from pathlib import Path
import json
import pandas as pd
from datetime import datetime, timezone

BASE = Path("/content/planproof_results/experiment65")
OUT = BASE / "finalization"
OUT.mkdir(parents=True, exist_ok=True)

sources = {
    "Experiment 66 — Repair evaluation":
        BASE / "experiment66_repair_evaluation/experiment66_summary.json",
    "Experiment 67 — Regression audit":
        BASE / "experiment67_regression/experiment67_summary.json",
    "Experiment 68 — Method comparison":
        BASE / "experiment68_method_comparison/experiment68_summary.json",
    "Experiment 69 — Statistical analysis":
        BASE / "experiment69_statistics/experiment69_summary.json",
    "Experiment 72 — Human review worksheet":
        BASE / "experiment72_human_review/experiment72_summary.json",
    "Experiment 73 — Review priorities":
        BASE / "experiment73_review_priorities/experiment73_summary.json",
    "Experiment 74 — Review packet":
        BASE / "experiment74_review_packet_REAL_003/experiment74_summary.json",
}

report = []
report.append("# PlanProof AI — Consolidated Research Results\n")
report.append(
    f"Generated: {datetime.now(timezone.utc).isoformat()}\n"
)
report.append("## Scope and research-integrity notes\n")
report.append(
    "- Synthetic benchmark results must not be described as real-world validation.\n"
    "- Gemini-generated plans and same-model dependency extraction do not provide independent ground truth.\n"
    "- The controlled-corruption evaluation measures performance on intentionally corrupted benchmark cases.\n"
    "- REAL_003 has 12 provisional dependency decisions; all remain pending confirmation.\n"
    "- No real-plan case is approved for evaluation based on the review status currently recorded.\n"
    "- Missing source files or metrics are reported as unavailable, not reconstructed by guesswork.\n"
)

report.append("\n## Existing experiment summaries\n")

inventory_rows = []
for title, path in sources.items():
    if path.exists():
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            report.append(f"\n### {title}\n")
            report.append("```json\n")
            report.append(json.dumps(data, indent=2, ensure_ascii=False))
            report.append("\n```\n")
            inventory_rows.append({
                "source": title,
                "path": str(path),
                "status": "FOUND"
            })
        except Exception as exc:
            report.append(f"\n### {title}\n\nSummary could not be parsed: {exc}\n")
            inventory_rows.append({
                "source": title,
                "path": str(path),
                "status": "PARSE_ERROR"
            })
    else:
        report.append(f"\n### {title}\n\nSource summary unavailable at expected path: `{path}`.\n")
        inventory_rows.append({
            "source": title,
            "path": str(path),
            "status": "NOT_FOUND"
        })

# Include existing comparison/statistics CSVs if available.
report.append("\n## Existing detailed result tables\n")
table_paths = [
    BASE / "experiment66_repair_evaluation/experiment66_repair_results.csv",
    BASE / "experiment68_method_comparison/experiment68_comparison.csv",
    BASE / "experiment69_statistics/experiment69_statistics.csv",
]

for path in table_paths:
    if not path.exists():
        report.append(f"\n- Not found: `{path}`\n")
        continue

    try:
        df = pd.read_csv(path)
        report.append(f"\n### `{path.name}`\n")
        report.append(f"- Rows: {len(df)}\n")
        report.append(f"- Columns: {', '.join(map(str, df.columns))}\n")
        report.append("- Preview:\n")
        report.append("```text\n")
        report.append(df.head(10).to_string(index=False))
        report.append("\n```\n")
    except Exception as exc:
        report.append(f"\n- Could not read `{path}`: {exc}\n")

report.append("\n## Next decisions\n")
report.append(
    "1. Verify the consolidated metrics against the saved result tables.\n"
    "2. Complete or explicitly limit the human-review dataset; do not treat pending decisions as labels.\n"
    "3. Write the methodology, results, limitations, and reproducibility sections.\n"
    "4. Avoid additional experiments unless a specific missing requirement blocks the paper.\n"
)

report_path = OUT / "planproof_consolidated_research_report_draft.md"
inventory_path = OUT / "consolidation_inventory.csv"

report_path.write_text("\n".join(report), encoding="utf-8")
pd.DataFrame(inventory_rows).to_csv(inventory_path, index=False)

print("CONSOLIDATION COMPLETE")
print("Summary sources found:", sum(r["status"] == "FOUND" for r in inventory_rows))
print("Summary sources missing:", sum(r["status"] == "NOT_FOUND" for r in inventory_rows))
print("\nCreated:")
print(report_path)
print(inventory_path)
print("\nNo API calls made.")
print("No experiments rerun.")
print("Original research files were not modified.")


CONSOLIDATION COMPLETE
Summary sources found: 0
Summary sources missing: 7

Created:
/content/planproof_results/experiment65/finalization/planproof_consolidated_research_report_draft.md
/content/planproof_results/experiment65/finalization/consolidation_inventory.csv

No API calls made.
No experiments rerun.
Original research files were not modified.


In [3]:

from pathlib import Path

ROOT = Path("/content/planproof_results")

print("PLANPROOF FILESYSTEM DIAGNOSTIC")
print("=" * 60)
print("Root exists:", ROOT.exists())

if ROOT.exists():
    print("\nTop-level entries:")
    for p in sorted(ROOT.iterdir()):
        print(("DIR " if p.is_dir() else "FILE"), p.name)

    print("\nExperiment 65 entries:")
    exp65 = ROOT / "experiment65"
    if exp65.exists():
        for p in sorted(exp65.iterdir()):
            print(("DIR " if p.is_dir() else "FILE"), p.name)
    else:
        print("experiment65 directory not found")

    print("\nSummary JSON files found:")
    summaries = list(ROOT.rglob("*summary*.json"))
    for p in summaries[:100]:
        print(p)
    print("Total summary JSON files:", len(summaries))

    print("\nResearch result CSV files:")
    csvs = list(ROOT.rglob("*.csv"))
    for p in csvs[:100]:
        print(p)
    print("Total CSV files:", len(csvs))
else:
    print("The research root directory is not accessible in this runtime.")

print("\nDiagnostic only: no files modified, no API calls.")


PLANPROOF FILESYSTEM DIAGNOSTIC
Root exists: True

Top-level entries:
DIR  experiment65

Experiment 65 entries:
DIR  finalization

Summary JSON files found:
Total summary JSON files: 0

Research result CSV files:
/content/planproof_results/experiment65/finalization/consolidation_inventory.csv
Total CSV files: 1

Diagnostic only: no files modified, no API calls.


In [4]:

from pathlib import Path

locations = [
    Path("/content/drive/MyDrive/planproof_results"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
]

print("PLANPROOF RECOVERY CHECK")
print("=" * 55)

for root in locations:
    print(f"\nChecking: {root}")
    if not root.exists():
        print("  Not mounted / not found")
        continue

    if root.name == "MyDrive":
        matches = [
            p for p in root.iterdir()
            if "planproof" in p.name.lower()
        ]
        for p in matches[:30]:
            print(" ", p)
        if not matches:
            print("  No top-level PlanProof-named entries")
    else:
        for name in [
            "experiment64",
            "experiment65",
            "experiment66_repair_evaluation",
            "experiment68_method_comparison",
            "experiment69_statistics",
        ]:
            matches = list(root.glob(f"**/{name}"))
            for p in matches[:10]:
                print(" ", p)

print("\nRecovery check only. No files changed.")


PLANPROOF RECOVERY CHECK

Checking: /content/drive/MyDrive/planproof_results
  Not mounted / not found

Checking: /content/drive/MyDrive
  Not mounted / not found

Checking: /content
  /content/planproof_results/experiment65

Recovery check only. No files changed.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

print("PLANPROOF — GOOGLE DRIVE RECOVERY")
print("=" * 55)

# Find folders that may contain the saved research.
matches = []
for p in DRIVE.iterdir():
    if "planproof" in p.name.lower():
        matches.append(p)

if matches:
    print("Potential project folders:")
    for p in matches:
        print(f"\n{p}")
        if p.is_dir():
            for child in sorted(p.iterdir())[:30]:
                print("  ", child.name)
else:
    print("No top-level PlanProof-named folder found.")

# Check the expected location directly.
expected = DRIVE / "planproof_results"
print("\nExpected results directory:", expected)
print("Exists:", expected.exists())

if expected.exists():
    for p in sorted(expected.rglob("*")):
        if p.is_file() and (
            "summary" in p.name.lower()
            or "experiment68" in str(p).lower()
            or "experiment69" in str(p).lower()
        ):
            print("Found:", p)

print("\nRead-only search complete. No experiments rerun.")


PLANPROOF — GOOGLE DRIVE RECOVERY
No top-level PlanProof-named folder found.

Expected results directory: /content/drive/MyDrive/planproof_results
Exists: False

Read-only search complete. No experiments rerun.


In [7]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

print("SEARCHING GOOGLE DRIVE FOR PLANPROOF ARTIFACTS")
print("=" * 55)

found = []
extensions = {".json", ".csv", ".md", ".ipynb", ".zip"}

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()
    relevant_name = (
        "planproof" in name
        or "experiment64" in name
        or "experiment68" in name
        or "experiment69" in name
        or "experiment70" in name
    )

    if relevant_name and path.suffix.lower() in extensions:
        found.append(path)

for path in found[:100]:
    print(path)

print(f"\nRelevant files found: {len(found)}")

if not found:
    print(
        "\nNo matching artifacts found in MyDrive. "
        "This does not rule out files in another Drive, "
        "a local download, or another Colab notebook."
    )

print("\nRead-only search complete.")


SEARCHING GOOGLE DRIVE FOR PLANPROOF ARTIFACTS
/content/drive/MyDrive/Colab Notebooks/PlanProof_AI_Research.ipynb

Relevant files found: 1

Read-only search complete.


In [8]:

from pathlib import Path
import json
import re

NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/PlanProof_AI_Research.ipynb"
)

if not NOTEBOOK.exists():
    raise FileNotFoundError(f"Notebook not found: {NOTEBOOK}")

notebook = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
cells = notebook.get("cells", [])

print("PLANPROOF NOTEBOOK RECOVERY AUDIT")
print("=" * 65)
print("Notebook:", NOTEBOOK.name)
print("Cells:", len(cells))
print("Code cells:", sum(c.get("cell_type") == "code" for c in cells))

terms = re.compile(
    r"experiment\s*\d+|planproof_results|mean improvement|"
    r"wilcoxon|bootstrap|optimal|repair|summary\.json|"
    r"real_\d+|api calls|approved",
    re.IGNORECASE
)

matches = []
saved_outputs = 0

for i, cell in enumerate(cells):
    source = "".join(cell.get("source", []))

    if cell.get("cell_type") == "markdown" and terms.search(source):
        matches.append(("MARKDOWN", i, source[:1800]))

    if cell.get("cell_type") == "code":
        outputs = cell.get("outputs", [])
        execution_count = cell.get("execution_count")
        for output in outputs:
            text_parts = []

            if "text" in output:
                text_parts.append("".join(output["text"]))

            data = output.get("data", {})
            if "text/plain" in data:
                text_parts.append("".join(data["text/plain"]))

            output_text = "\n".join(text_parts)
            if output_text:
                saved_outputs += 1
                if terms.search(source) or terms.search(output_text):
                    matches.append((
                        f"CODE OUTPUT (execution={execution_count})",
                        i,
                        output_text[:2200]
                    ))

print("Saved text outputs:", saved_outputs)
print("Relevant sections found:", len(matches))

for kind, index, content in matches[:35]:
    print(f"\n--- {kind} | CELL {index} ---")
    print(content)

print("\nNotebook inspection complete.")
print("Notebook was not executed or modified.")


PLANPROOF NOTEBOOK RECOVERY AUDIT
Notebook: PlanProof_AI_Research.ipynb
Cells: 236
Code cells: 236
Saved text outputs: 317
Relevant sections found: 290

--- CODE OUTPUT (execution=None) | CELL 3 ---
PLAN REPAIR TEST
Success: True
Message: Repaired plan passed verification.
Original order: ['literature_review', 'write_results', 'run_experiments', 'submit_report']
Repaired order: ['literature_review', 'run_experiments', 'write_results', 'submit_report']
Final verification: {'valid': True, 'violations': []}


--- CODE OUTPUT (execution=None) | CELL 4 ---
[PASS] Valid plan is accepted
[PASS] Missing required task is restored
[PASS] Duplicate tasks are removed
[PASS] Impossible dependency cycle is rejected

All four robustness tests passed.


--- CODE OUTPUT (execution=None) | CELL 5 ---
PLANPROOF AI — BENCHMARK RESULTS
Random seed: 42
Cases: 500
Average tasks per case: 17.18

INITIAL PLANS
Valid: 25
Invalid: 475
Initial validity rate: 5.00%

AFTER REPAIR
Successfully repaired and verified:

In [9]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

targets = {
    "research_report.md",
    "PlanProof_AI_research_artifacts.zip",
    "benchmark_summary.csv",
    "per_case_benchmark_results.csv",
    "paired_comparisons.csv",
    "manifest.json",
}

print("PLANPROOF — ARTIFACT RECOVERY")
print("=" * 55)

found = []

for path in ROOT.rglob("*"):
    if path.is_file() and path.name in targets:
        found.append(path)

if found:
    for path in found:
        print(f"FOUND: {path} ({path.stat().st_size:,} bytes)")
else:
    print("No matching artifacts found in Google Drive.")

print(f"\nTotal matching files: {len(found)}")
print("Read-only search: no experiments rerun, no files modified.")


PLANPROOF — ARTIFACT RECOVERY
No matching artifacts found in Google Drive.

Total matching files: 0
Read-only search: no experiments rerun, no files modified.


In [10]:

from pathlib import Path
import shutil
import hashlib

source = Path(
    "/content/drive/MyDrive/Colab Notebooks/PlanProof_AI_Research.ipynb"
)

backup_dir = Path(
    "/content/drive/MyDrive/PlanProof_Recovery"
)
backup_dir.mkdir(parents=True, exist_ok=True)

backup = backup_dir / "PlanProof_AI_Research_recovery.ipynb"

if not source.exists():
    raise FileNotFoundError(f"Original notebook not found: {source}")

source_hash = hashlib.sha256(source.read_bytes()).hexdigest()

# Preserve any previous recovery backup instead of overwriting it.
if backup.exists():
    existing_hash = hashlib.sha256(backup.read_bytes()).hexdigest()
    if existing_hash != source_hash:
        backup = backup_dir / "PlanProof_AI_Research_recovery_2.ipynb"

shutil.copy2(source, backup)

backup_hash = hashlib.sha256(backup.read_bytes()).hexdigest()

print("NOTEBOOK BACKUP")
print("================")
print("Original:", source)
print("Backup:", backup)
print("Original SHA-256:", source_hash)
print("Backup SHA-256:", backup_hash)
print("Identical:", source_hash == backup_hash)
print("Backup verified:", source_hash == backup_hash)


NOTEBOOK BACKUP
Original: /content/drive/MyDrive/Colab Notebooks/PlanProof_AI_Research.ipynb
Backup: /content/drive/MyDrive/PlanProof_Recovery/PlanProof_AI_Research_recovery.ipynb
Original SHA-256: 69cdea18701e4825a41d8a447ad0ca5c39b0376c282e1dd59f0c4f59dabfdcc8
Backup SHA-256: 69cdea18701e4825a41d8a447ad0ca5c39b0376c282e1dd59f0c4f59dabfdcc8
Identical: True
Backup verified: True


In [11]:

from pathlib import Path
import shutil
import hashlib

source = Path(
    "/content/drive/MyDrive/Colab Notebooks/PlanProof_AI_Research.ipynb"
)

backup_dir = Path(
    "/content/drive/MyDrive/PlanProof_Recovery"
)
backup_dir.mkdir(parents=True, exist_ok=True)

backup = backup_dir / "PlanProof_AI_Research_recovery.ipynb"

if not source.exists():
    raise FileNotFoundError(f"Original notebook not found: {source}")

source_hash = hashlib.sha256(source.read_bytes()).hexdigest()

# Preserve any previous recovery backup instead of overwriting it.
if backup.exists():
    existing_hash = hashlib.sha256(backup.read_bytes()).hexdigest()
    if existing_hash != source_hash:
        backup = backup_dir / "PlanProof_AI_Research_recovery_2.ipynb"

shutil.copy2(source, backup)

backup_hash = hashlib.sha256(backup.read_bytes()).hexdigest()

print("NOTEBOOK BACKUP")
print("================")
print("Original:", source)
print("Backup:", backup)
print("Original SHA-256:", source_hash)
print("Backup SHA-256:", backup_hash)
print("Identical:", source_hash == backup_hash)
print("Backup verified:", source_hash == backup_hash)


NOTEBOOK BACKUP
Original: /content/drive/MyDrive/Colab Notebooks/PlanProof_AI_Research.ipynb
Backup: /content/drive/MyDrive/PlanProof_Recovery/PlanProof_AI_Research_recovery_2.ipynb
Original SHA-256: 53ad7ffdf8f92f75234cb32ed442337d743f08f8126af4679c7f110d36fece91
Backup SHA-256: 53ad7ffdf8f92f75234cb32ed442337d743f08f8126af4679c7f110d36fece91
Identical: True
Backup verified: True


In [12]:

import json
from pathlib import Path

# Locate the newest recovery copy without modifying the original notebook.
recovery_dir = Path("/content/drive/MyDrive/PlanProof_Recovery")
backups = sorted(
    recovery_dir.glob("PlanProof_AI_Research_recovery*.ipynb"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not backups:
    raise FileNotFoundError(
        f"No recovery notebook found in {recovery_dir}"
    )

notebook_path = backups[0]
with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = (
    "experiment", "benchmark", "optimal", "scalability",
    "paired", "confidence interval", "validity", "manifest",
    "research_report", "mean order", "p-value", "p value",
    "order change", "repair success"
)

inventory = []
for cell_index, cell in enumerate(notebook.get("cells", [])):
    for output_index, output in enumerate(cell.get("outputs", [])):
        parts = []

        if "text" in output:
            value = output["text"]
            parts.append(
                "".join(value) if isinstance(value, list) else str(value)
            )

        for mime, value in output.get("data", {}).items():
            if mime in ("text/plain", "text/markdown"):
                parts.append(
                    "".join(value) if isinstance(value, list) else str(value)
                )

        text = "\n".join(parts).strip()
        if text and any(word in text.lower() for word in keywords):
            inventory.append({
                "cell_index": cell_index,
                "output_index": output_index,
                "characters": len(text),
                "saved_output": text
            })

output_path = recovery_dir / "recovered_evidence_inventory.json"
output_path.write_text(
    json.dumps(inventory, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("PLANPROOF — EVIDENCE RECOVERY")
print("=" * 38)
print("Notebook:", notebook_path.name)
print("Total notebook cells:", len(notebook.get("cells", [])))
print("Relevant saved outputs:", len(inventory))
print("Inventory saved:", output_path)

print("\nLARGEST RELEVANT OUTPUTS")
print("=" * 38)
for item in sorted(
    inventory, key=lambda x: x["characters"], reverse=True
)[:12]:
    preview = " ".join(item["saved_output"].split())[:500]
    print(
        f"\nCell {item['cell_index']} | "
        f"{item['characters']} characters"
    )
    print(preview)

print("\nNotebook code was not executed.")
print("No research experiments were rerun.")


PLANPROOF — EVIDENCE RECOVERY
Notebook: PlanProof_AI_Research_recovery_2.ipynb
Total notebook cells: 238
Relevant saved outputs: 203
Inventory saved: /content/drive/MyDrive/PlanProof_Recovery/recovered_evidence_inventory.json

LARGEST RELEVANT OUTPUTS

Cell 236 | 18995 characters
PLANPROOF NOTEBOOK RECOVERY AUDIT ================================================================= Notebook: PlanProof_AI_Research.ipynb Cells: 236 Code cells: 236 Saved text outputs: 317 Relevant sections found: 290 --- CODE OUTPUT (execution=None) | CELL 3 --- PLAN REPAIR TEST Success: True Message: Repaired plan passed verification. Original order: ['literature_review', 'write_results', 'run_experiments', 'submit_report'] Repaired order: ['literature_review', 'run_experiments', 'write_result

Cell 154 | 10647 characters
EXPERIMENT 50 — GENERATOR SEARCH ============================================= Matching notebook cells: 6 --- Cell 71 | Match score: 5 --- # EXPERIMENT 50 — STEP 2: FIND THE BENCHMARK GENER

In [13]:

from pathlib import Path
from datetime import datetime

out_dir = Path("/content/drive/MyDrive/PlanProof_Recovery")
out_dir.mkdir(parents=True, exist_ok=True)

report_path = out_dir / "PlanProof_AI_Consolidated_Research_Report.md"

report = r"""# PlanProof AI: Constraint-Aware Task-Plan Repair

## Consolidated Research Report

**Evidence basis:** Saved outputs from the recovered research notebook and previously recorded experiment summaries.
**Purpose:** Consolidate existing evidence without rerunning experiments.
**Status:** Research report draft; independent semantic validation remains incomplete.

---

## 1. Abstract

PlanProof AI investigates constraint-aware repair of task plans. Given a required set of tasks, an initial task ordering, and precedence constraints, the system seeks a valid ordering while minimizing disruption to the original order.

The research compares stable topological repair with optimization-based approaches, including local search, faster search variants, and an adaptive repair strategy. The saved experiments report improvements in order preservation for optimization-based methods, alongside a runtime trade-off as task sizes increase.

A separate workflow explores generating realistic task plans with a language model and reviewing candidate dependencies. Those results must be distinguished from the synthetic benchmark results: generated plans and model-proposed dependencies do not constitute independently verified ground truth.

## 2. Research problem

A task plan can be invalid because it omits required tasks, contains duplicate or unknown task identifiers, or violates precedence constraints.

A repair method should:
1. Restore structural validity.
2. Respect all supplied precedence constraints.
3. Preserve as much of the original relative task order as practical.
4. Avoid unnecessary changes to already-valid plans.
5. Make the trade-off between repair quality and runtime measurable.

## 3. Methods

### 3.1 Stable repair

Stable topological sorting provides a deterministic baseline that attempts to preserve the relative order of tasks while satisfying precedence constraints.

### 3.2 Optimization-based repair

Local-search and faster-search variants attempt to reduce the order-change objective beyond the baseline. Exhaustive or exact optimization was also used on bounded cases to establish optimality where the solver reported an optimal solution.

### 3.3 Adaptive repair

The adaptive method starts with a valid stable-repair fallback and attempts optimization within a time budget. The fallback is retained if optimization does not produce a better valid candidate.

### 3.4 Evaluation measures

The saved research outputs include:
- Structural validity and repair success.
- Order-change rate.
- Runtime, including median and high-percentile runtime in scalability experiments.
- Paired comparisons between repair methods.
- Optimality checks on bounded benchmark cases.

Order-change rates should be interpreted according to the implementation's defined metric; they should not be treated as a direct measure of semantic correctness.

## 4. Recovered benchmark evidence

The following results are recorded in the saved research outputs or previously recorded experiment summaries. They are not claims of independent real-world validation.

### 4.1 Multi-seed benchmark

A recorded experiment used 10 random seeds with 500 synthetic cases per seed.

- Stable repair: mean order-change rate of 27.37%.
- FIFO comparison method: mean order-change rate of 39.61%.
- Difference: 12.24 percentage points in favor of stable repair.
- Stable repair had a lower recorded order-change rate in all 10 seeds.
- No repair failures were recorded in that benchmark.

### 4.2 Bounded optimality evaluation

An early evaluation over 30 cases reported:
- Stable repair mean order-change rate: 16.13%.
- Optimal mean order-change rate: 15.69%.
- Stable repair matched the optimal cost in 29 of 30 cases.
- A counterexample showed stable repair at 60% versus 46.67% for the optimal solution.
- Local search matched the optimal cost in all 30 cases in that recorded evaluation.

A later 100-case evaluation recorded optimal-cost matches in all 100 cases. These results apply to the evaluated cases and objective, not to arbitrary plans.

### 4.3 Scalability

A recorded benchmark used 100 cases at each of 10, 15, 20, and 30 tasks.

| Tasks | Stable order change | Local-search order change | Stable runtime (ms) | Local-search runtime (ms) |
|---:|---:|---:|---:|---:|
| 10 | 19.67% | 17.91% | 0.030 | 1.654 |
| 15 | 26.23% | 23.77% | 0.044 | 9.697 |
| 20 | 32.04% | 28.62% | 0.066 | 39.821 |
| 30 | 38.59% | 34.86% | 0.146 | 315.229 |

These are recorded measurements from one saved scalability run. Runtime depends on the implementation, environment, and search budget.

### 4.4 Larger benchmark comparison

A recorded 400-plan benchmark compared stable repair, best-search, and fast-search methods, with 100 plans at each task size.

| Tasks | Stable order change | Best-search order change | Fast-search order change |
|---:|---:|---:|---:|
| 10 | 23.00% | 21.18% | 21.18% |
| 15 | 28.00% | 24.51% | 24.51% |
| 20 | 31.98% | 28.39% | 28.39% |
| 30 | 37.99% | 34.07% | 34.08% |

All three methods were recorded as valid on all 400 plans. In this benchmark, search methods reduced order change relative to stable repair. The saved results also show that search methods cost more runtime than stable repair, particularly at larger sizes.

### 4.5 Independent synthetic-seed summaries

Previously recorded summaries describe two additional 100-case synthetic evaluations:

- Seed 20261050: stable mean order change 0.376916; adaptive 0.344620; exact 0.344480. Adaptive repair was valid on 100/100 cases and was never worse than stable repair in the recorded paired comparison.
- Seed 20261056: stable mean order change 0.365861; adaptive 0.336883; exact 0.336665. All methods were recorded as valid, and adaptive repair matched exact repair on 67 cases.

These figures are historical summaries. The original per-case artifacts were not found in the current recovery directory, so they should be independently reproduced before being used as final publication-grade evidence.

## 5. Language-model-generated plans and dependency review

The recorded Gemini workflow generated 28 plans, with 10 tasks per plan and 337 explicit candidate dependency edges. The generation responses were reportedly preserved in a prior runtime, but those raw files were not recovered in the current runtime.

The workflow also created controlled corruptions involving missing tasks, duplicate tasks, unknown tasks, and precedence violations. A later saved corruption-audit output reported 112 cases needing inspection, with 28 cases failing the audit in each defect category. Therefore, those cases must not be described as fully verified from that audit.

Another recorded repair evaluation reported 112/112 repaired cases as valid. That structural result does not by itself establish that every dependency is semantically correct.

Dependency extraction used the same model family as plan generation. Consequently, agreement between generated plans and extracted dependencies is not independent ground truth.

The review workflow recorded:
- 28 plans and 409 candidate review rows in the triage output.
- Zero approved plans and zero approved edges at that stage.
- A review packet for plan REAL_003 containing 10 tasks and 12 candidate edges.
- Provisional decisions for REAL_003, with final reviewer confirmation still pending.

Human review and reviewer provenance are required before any dependency labels are described as approved ground truth.

## 6. Limitations

1. **Synthetic benchmark scope:** Synthetic cases enable controlled comparisons but may not reflect the complexity of real project plans.
2. **Objective limitations:** A lower order-change score does not prove that the resulting plan is useful, realistic, or semantically correct.
3. **Ground-truth limitations:** Model-generated plans and same-family dependency extraction are not independent validation.
4. **Human review:** The recovered review evidence does not establish completed independent approval of the candidate dependency labels.
5. **Artifact recovery:** The current Drive recovery folder did not contain the old experiment bundle or report. Some numerical summaries survive in saved notebook outputs or prior recorded summaries, but not all underlying per-case files were recovered.
6. **Runtime comparability:** Timing measurements should be reported with hardware, software, solver, and budget details before making broad performance claims.
7. **Statistical claims:** Inferential statistics should be accompanied by recoverable per-case data and a documented analysis script before final publication.

## 7. Conclusions supported by the current evidence

The recorded synthetic benchmarks support the conclusion that optimization-based repair can reduce order change compared with stable repair on the evaluated cases, while requiring additional runtime. Stable topological repair remains a useful fast baseline. Adaptive repair is a promising way to combine a reliable fallback with a bounded optimization attempt.

The available evidence does **not** yet justify claiming that PlanProof AI has independently validated semantic dependencies for real-world plans or demonstrated general superiority on real project data.

## 8. Recommended next actions

1. Preserve the recovered notebook and evidence inventory.
2. Recover the original per-case datasets, paired-comparison files, and experiment configurations if available.
3. Rebuild tables from the original CSV/JSON results where possible.
4. Complete independent human review of a documented sample of real-plan dependencies.
5. Record reviewer identity or anonymized reviewer ID, decision, rationale, and timestamp.
6. Run a final reproducible analysis only after the necessary source artifacts are available.
7. Update all numerical claims to match the recovered source files and explicitly report any unrecoverable evidence.

---

**Report status:** Consolidated draft based on recovered evidence. Not yet a claim of completed independent real-world validation.
"""

report_path.write_text(report, encoding="utf-8")

print("REPORT CREATED")
print("==============================")
print("Path:", report_path)
print("Size:", f"{report_path.stat().st_size:,}", "bytes")
print("Sections:", sum(
    1 for line in report.splitlines() if line.startswith("## ")
))
print("Independent real-plan validation claimed: No")
print("Research experiments rerun: No")


REPORT CREATED
Path: /content/drive/MyDrive/PlanProof_Recovery/PlanProof_AI_Consolidated_Research_Report.md
Size: 10,002 bytes
Sections: 9
Independent real-plan validation claimed: No
Research experiments rerun: No


In [14]:

import json
from pathlib import Path

base = Path("/content/drive/MyDrive/PlanProof_Recovery")
inventory_path = base / "recovered_evidence_inventory.json"
report_path = base / "PlanProof_AI_Consolidated_Research_Report.md"

if not inventory_path.exists() or not report_path.exists():
    raise FileNotFoundError("Report or evidence inventory is missing.")

inventory = json.loads(inventory_path.read_text(encoding="utf-8"))
report = report_path.read_text(encoding="utf-8")
evidence_text = "\n".join(
    item.get("saved_output", "") for item in inventory
).lower()

claims = {
    "10 seeds and 500 cases per seed": ["10 random seeds", "500"],
    "Stable 27.37% vs FIFO 39.61%": ["27.37%", "39.61%"],
    "30-case optimality comparison": ["16.13%", "15.69%"],
    "Stable repair matched 29/30": ["29 of 30"],
    "100-case optimality result": ["100 cases", "100"],
    "Scalability results for 10–30 tasks": [
        "19.67%", "26.23%", "32.04%", "38.59%"
    ],
    "400-plan benchmark results": [
        "23.00%", "28.00%", "31.98%", "37.99%"
    ],
    "Real-plan review approvals remain zero": [
        "plans approved: 0", "edges approved: 0"
    ]
}

print("PLANPROOF — CLAIM AUDIT")
print("=" * 45)

for claim, phrases in claims.items():
    found = [phrase for phrase in phrases if phrase.lower() in evidence_text]
    status = "EVIDENCE FOUND" if len(found) == len(phrases) else "CHECK SOURCE"
    print(f"\n{claim}: {status}")
    if found:
        print("  Matched:", ", ".join(found))
    missing = [phrase for phrase in phrases if phrase.lower() not in evidence_text]
    if missing:
        print("  Not found in inventory:", ", ".join(missing))

print("\nReport exists:", report_path.exists())
print("Report size:", f"{report_path.stat().st_size:,}", "bytes")
print("\nNote: String matches are not proof of correctness.")
print("Claims marked CHECK SOURCE need manual verification.")
print("No experiments rerun; no files modified.")


PLANPROOF — CLAIM AUDIT

10 seeds and 500 cases per seed: CHECK SOURCE
  Matched: 500
  Not found in inventory: 10 random seeds

Stable 27.37% vs FIFO 39.61%: EVIDENCE FOUND
  Matched: 27.37%, 39.61%

30-case optimality comparison: EVIDENCE FOUND
  Matched: 16.13%, 15.69%

Stable repair matched 29/30: CHECK SOURCE
  Not found in inventory: 29 of 30

100-case optimality result: EVIDENCE FOUND
  Matched: 100 cases, 100

Scalability results for 10–30 tasks: EVIDENCE FOUND
  Matched: 19.67%, 26.23%, 32.04%, 38.59%

400-plan benchmark results: CHECK SOURCE
  Not found in inventory: 23.00%, 28.00%, 31.98%, 37.99%

Real-plan review approvals remain zero: CHECK SOURCE
  Not found in inventory: plans approved: 0, edges approved: 0

Report exists: True
Report size: 10,002 bytes

Note: String matches are not proof of correctness.
Claims marked CHECK SOURCE need manual verification.
No experiments rerun; no files modified.


In [15]:

import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/PlanProof_Recovery/"
    "recovered_evidence_inventory.json"
)
items = json.loads(path.read_text(encoding="utf-8"))

targets = [
    "29/30", "29 of 30", "29/30 cases",
    "23.00%", "28.00%", "31.98%", "37.99%",
    "plans approved", "edges approved",
    "10 random seeds", "seed"
]

print("TARGETED SOURCE CHECK")
print("=" * 50)

for target in targets:
    matches = []
    for item in items:
        text = item.get("saved_output", "")
        pos = text.lower().find(target.lower())
        if pos >= 0:
            start = max(0, pos - 250)
            end = min(len(text), pos + len(target) + 350)
            matches.append((item.get("cell_index"), text[start:end]))

    print(f"\nTARGET: {target} | Matches: {len(matches)}")
    for cell_index, excerpt in matches[:3]:
        print(f"Cell {cell_index}:")
        print(" ".join(excerpt.split()))

print("\nRead-only check complete. No experiments rerun.")


TARGETED SOURCE CHECK

TARGET: 29/30 | Matches: 3
Cell 9:
EXHAUSTIVE OPTIMALITY EXPERIMENT Cases evaluated: 30 Stable mean change: 16.13% Optimal mean change: 15.69% Mean optimality gap: 0.44 percentage points Stable matched optimal cost: 29/30
Cell 12:
THREE-METHOD COMPARISON Cases evaluated: 30 Stable repair mean change: 16.13% Local-search mean change: 15.69% Exhaustive optimum mean change: 15.69% Stable optimal-cost matches: 29/30 Local-search optimal-cost matches: 30/30 Mean local-search optimality gap: 0.00 percentage points Mean runtime per case (milliseconds) Stable: 0.021 Local search: 0.293 Exhaustive: 21.233
Cell 236:
tal FIFO failures: 0 --- CODE OUTPUT (execution=None) | CELL 9 --- EXHAUSTIVE OPTIMALITY EXPERIMENT Cases evaluated: 30 Stable mean change: 16.13% Optimal mean change: 15.69% Mean optimality gap: 0.44 percentage points Stable matched optimal cost: 29/30 --- CODE OUTPUT (execution=None) | CELL 10 --- COUNTEREXAMPLE FOUND Case: 6 Original order: ['T5', 'T2', 'T1'

In [16]:

from pathlib import Path

base = Path("/content/drive/MyDrive/PlanProof_Recovery")
output = base / "PlanProof_AI_Final_Corrected_Report.md"
output.parent.mkdir(parents=True, exist_ok=True)

report = r"""# PlanProof AI: Constraint-Aware Task-Plan Repair

**Document status:** Corrected research report based on recovered notebook evidence
**Evidence policy:** Saved outputs are used as evidence of recorded results, not as proof of independent reproducibility.

## 1. Abstract

PlanProof AI investigates repair of task plans under required-task and precedence constraints. Given an initial task ordering, the system aims to restore validity while minimizing changes to the original relative ordering.

The recovered notebook contains experiments comparing stable topological repair with local-search and other optimization-based methods. The recorded synthetic benchmarks show that optimization can reduce order change relative to stable repair, at the cost of additional runtime.

A separate workflow explores language-model-generated plans and dependency review. Independent semantic validation of real-plan dependencies remains incomplete.

## 2. Problem definition

A task plan can be invalid because it omits required tasks, contains duplicate or unknown task identifiers, or violates precedence constraints.

PlanProof evaluates whether a repair method can:
- restore structural validity;
- satisfy supplied precedence constraints;
- reduce unnecessary changes to task order;
- leave already-valid plans unchanged when appropriate; and
- balance repair quality against runtime.

Order preservation is a structural objective. It does not independently establish that the plan or its dependencies are semantically correct.

## 3. Methods

### 3.1 Stable repair

Stable topological sorting provides a deterministic baseline that attempts to preserve the original relative order while satisfying precedence constraints.

### 3.2 Optimization-based repair

Local-search and faster-search variants attempt to reduce order change further. Bounded exhaustive optimization provides a reference optimum for cases where the search space can be evaluated and optimality is established.

### 3.3 Adaptive repair

The recorded adaptive method begins with a stable-repair fallback and attempts optimization within a time budget. A valid fallback protects against optimization failure or lack of improvement.

### 3.4 Metrics

The saved experiments report structural validity, repair success, order-change rate, runtime, and optimal-cost matches. Results should be interpreted using the exact definitions and settings of each experiment.

## 4. Verified-in-saved-output benchmark findings

“Verified” in this section means the numerical result was located in the recovered notebook's saved output. It does not mean the experiment was independently rerun.

### 4.1 Exhaustive optimality experiment: 30 cases

The saved output reports:
- Cases evaluated: 30.
- Stable repair mean change: 16.13%.
- Optimal mean change: 15.69%.
- Mean optimality gap: 0.44 percentage points.
- Stable repair matched optimal cost in 29/30 cases.

A separate saved three-method comparison reports local search matched the optimum in 30/30 cases. These findings support the conclusion that stable repair is not always optimal for the recorded order-change objective.

### 4.2 Expanded optimality experiment: 100 cases

The saved output reports:
- Cases evaluated: 100.
- Optimal-cost matches: 100/100.
- Optimal-cost match rate: 100%.
- Mean per-seed optimality gap: 0.0000 percentage points.

This is evidence for the evaluated cases and implementation, not a guarantee of optimality for arbitrary inputs.

### 4.3 Scalability experiment

The saved notebook output reports 100 cases per task size, with seed 2027.

| Tasks | Stable order change | Local-search order change | Stable runtime (ms) | Local-search runtime (ms) |
|---:|---:|---:|---:|---:|
| 10 | 19.67% | 17.91% | 0.030 | 1.654 |
| 15 | 26.23% | 23.77% | 0.044 | 9.697 |
| 20 | 32.04% | 28.62% | 0.066 | 39.821 |
| 30 | 38.59% | 34.86% | 0.146 | 315.229 |

For these recorded cases, local search reduced mean order change compared with stable repair, while requiring more runtime. These timing values depend on the original execution environment and should not be generalized without implementation and hardware details.

### 4.4 Other benchmark claims requiring source verification

The consolidated draft previously included the following claims:
- A 10-seed comparison reporting stable repair at 27.37% versus FIFO at 39.61%.
- A 400-plan comparison reporting order-change percentages for stable, best-search, and fast-search methods.

The targeted inventory search did not locate all source details for these claims. They are **not included as verified numerical findings in this corrected report**. They may be restored if the original benchmark outputs or per-case files are recovered and checked.

## 5. Language-model-generated plans and dependency review

The historical workflow records generation of 28 plans, with 10 tasks per plan and 337 explicit candidate dependency edges. The prior runtime reportedly stored raw generation responses, but those files were not recovered in the current recovery directory.

The workflow also included controlled corruptions for missing tasks, duplicate tasks, unknown tasks, and precedence violations. One recovered audit output reports 112 cases needing inspection, with 28 audit failures in each defect category. Therefore, the corrupted cases must not be described as fully verified based on that audit output alone.

A separate recorded repair evaluation reported 112/112 repaired cases as structurally valid. Structural validity does not prove semantic dependency correctness.

Dependency extraction used the same model family as plan generation. Agreement between the generator and extractor is therefore not independent ground truth.

The review workflow recorded a packet for plan REAL_003 with 10 tasks and 12 candidate edges. Its decisions were provisional and human confirmation remained pending. The targeted search did not independently recover the final approval-status records, so this report does not assert a verified final count of approved plans or edges.

## 6. Limitations

1. **Synthetic data:** Synthetic benchmarks support controlled algorithm comparisons but may not represent the complexity of real project plans.
2. **Objective scope:** Lower order change does not prove semantic correctness or practical usefulness.
3. **Ground truth:** Language-model-generated plans and same-family dependency extraction are not independent validation.
4. **Human review:** The recovered evidence does not establish completed independent human approval of real-plan dependency labels.
5. **Artifact recovery:** Some historical summary claims could not be traced to complete original tables or per-case files during recovery.
6. **Runtime:** Timing comparisons require environment, hardware, and solver details for reproducibility.
7. **Reproducibility:** Saved outputs are not a substitute for the original datasets, configuration files, and analysis scripts.

## 7. Conclusions

The recovered notebook evidence supports the following limited conclusions:

- Stable topological repair is a useful baseline for restoring structural validity under supplied constraints.
- On the recorded scalability cases, local search reduced order change relative to stable repair, but required more runtime.
- Stable repair did not match the recorded optimum in every case of the 30-case exhaustive comparison.
- A separate 100-case saved evaluation reported optimal-cost matches in all evaluated cases.

The current evidence does not establish independently validated semantic dependencies for real-world plans or general superiority on real project data.

## 8. Recommended completion criteria

Before treating the work as publication-ready:

1. Recover original per-case benchmark datasets and experiment configurations where possible.
2. Reconcile every reported numerical claim against its source output.
3. Conduct independent human review of a documented sample of real-plan dependencies.
4. Preserve reviewer decisions, rationale, and timestamps.
5. Make the benchmark analysis reproducible from preserved source data.
6. Clearly separate synthetic results, model-generated examples, structural tests, and human-validated real-plan evidence.

## 9. Final status

**Status:** Consolidated corrected report based on recovered notebook outputs.

**Independent real-plan semantic validation:** Not established by the recovered evidence.

**Unverified historical numerical claims:** Excluded from the verified findings pending source recovery.

**Experiments rerun to create this report:** None.
"""

output.write_text(report, encoding="utf-8")

print("CORRECTED REPORT CREATED")
print("==============================")
print("File:", output)
print("Size:", f"{output.stat().st_size:,}", "bytes")
print("Original report overwritten: No")
print("Experiments rerun: No")
print("Independent real-plan validation claimed: No")


CORRECTED REPORT CREATED
File: /content/drive/MyDrive/PlanProof_Recovery/PlanProof_AI_Final_Corrected_Report.md
Size: 8,578 bytes
Original report overwritten: No
Experiments rerun: No
Independent real-plan validation claimed: No
